# Repeat Purchase Prediction
## 01. Data Understanding and Exploratory Data Analysis

### Notebook purpose

This notebook explores the datasets provided for the Repeat Purchase Prediction
project. It documents their structure, assesses data quality and investigates
patterns relevant to future modelling.

The analysis aims to:
- Establish what each dataset and observation represents.
- Understand the variables, identifiers and relationships between datasets.
- Identify missing values, duplicates, inconsistent records and unusual values.
- Explore variable distributions and relationships with the target.
- Compare training and test feature distributions.
- Identify potential data leakage and implications for validation.
- Document findings and recommendations for data preparation.

### Analysis principles

Original datasets will remain unchanged. Any exploratory transformations will
be documented and performed on separate objects.

### Planned structure

1. Environment setup and dataset size assessment
2. Project context and dataset documentation
3. Data loading and structural overview
4. Identifier checks and relationships between datasets
5. Data quality assessment
6. Target distribution
7. Univariate analysis
8. Relationships between features and the target
9. Relationships between features
10. Temporal and customer-level analysis, where supported
11. Training and test distribution comparison
12. Leakage risks and validation considerations
13. Findings and recommendations for data preparation

Data cleaning and feature preparation will be implemented in
“02_preprocessing.ipynb”. Model training and evaluation will be covered in
“03_modelling.ipynb”.

## 1. Environment Setup and Dataset Size Assessment

We begin by importing the required libraries and locating the data folder.
Before loading the datasets, we check their file sizes to assess the resources
needed for exploration.

In [ ]:
# Handle file paths without relying on computer-specific locations.
from pathlib import Path

# Locate the project from the current folder or one of its parent folders.
PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "data").is_dir()
        and (folder / "notebooks").is_dir()
    ),
    None,
)

# Give a clear error if the expected project structure cannot be found.
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate the project. Open the notebook within the repository."
    )

DATA_DIR = PROJECT_ROOT / "data"

# Check the expected datasets without loading their contents.
DATA_FILES = [
    "train.csv",
    "test.csv",
    "clientes.csv",
    "sample_submission.csv",
]

total_size_mib = 0

for filename in DATA_FILES:
    file_path = DATA_DIR / filename

    if not file_path.is_file():
        raise FileNotFoundError(f"Missing dataset: {file_path}")

    # Convert file size from bytes to mebibytes (MiB).
    size_mib = file_path.stat().st_size / (1024 ** 2)
    total_size_mib += size_mib

    print(f"{filename}: {size_mib:,.2f} MiB")

print(f"\nTotal size: {total_size_mib:,.2f} MiB")

### 1.1. Imports and Display Settings

Import the libraries used throughout the analysis and configure display
settings to keep tables readable and plots consistent.

In [ ]:
# Data manipulation and numerical calculations.
import pandas as pd
import numpy as np

# Charts for exploring distributions and relationships.
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical functions for later analyses.
from scipy import stats

# Display multiple tables explicitly within a notebook cell.
from IPython.display import display

# Show more columns and longer text when inspecting DataFrames.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

# Format displayed numbers without changing the underlying values.
pd.set_option("display.float_format", "{:,.3f}".format)

# Use consistent styling and colours across plots.
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

## 2. Project Context and Dataset Documentation

### 2.1. Prediction Objective

The goal of this project is to predict `repeat_purchase_90d`, a binary target
indicating whether a repeat purchase occurs within 90 days.

We will explore the data, prepare the features and compare classification
models before generating predictions for every observation in `test.csv`.

The `ID` column will be preserved to match predictions to the observations
in the submission.

### 2.2. Available Datasets

| File | Role |
|------|------|
| `train.csv` | Labelled data used for exploration, model development and validation. |
| `test.csv` | Observations for which final predictions will be generated. |
| `clientes.csv` | Optional customer information that can be linked through `customer_id`. |
| `sample_submission.csv` | Template for the prediction submission. |

### 2.3. Organisation of the Work

| Notebook | Scope |
|----------|-------|
| `01_data_understanding.ipynb` | Data exploration, descriptive statistics, quality checks and analysis of distributions and relationships. |
| `02_preprocessing.ipynb` | Data cleaning, customer-data integration, categorical encoding, feature engineering, scaling and feature selection. |
| `03_modelling.ipynb` | Validation strategy, model training, performance evaluation and submission generation. |


## 3. Data Loading and Structural Overview

Load the datasets and inspect their dimensions, column names and first rows
to understand their structure.

In [ ]:
# Load the datasets into separate DataFrames.
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
clientes = pd.read_csv(DATA_DIR / "clientes.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Group the datasets to inspect them consistently.
datasets = {
    "train": train,
    "test": test,
    "clientes": clientes,
    "sample_submission": sample_submission,
}

# Show the dimensions, column names and first five rows of each dataset.
for name, df in datasets.items():
    print(f"\nDataset: {name}")
    print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]:,}")
    print(f"Columns: {df.columns.tolist()}")

    display(df.head())

### 3.1. Initial Observations

| Dataset | Rows | Columns |
|---------|-----:|--------:|
| Training | 6,534 | 39 |
| Test | 986 | 38 |
| Customers | 1,143 | 13 |
| Submission template | 986 | 2 |

The training and test column lists contain the same variables, except for
`repeat_purchase_90d`, which is present only in training.

Each training or test row appears to represent a purchase occasion, combining
current purchase information with customer history. We will check identifiers
and repeated customers to clarify the observation structure.

The variables cover:
- Current purchase amounts, discounts, costs and invoice details.
- Previous purchase frequency, spending and time intervals.
- Purchase counts and spending over 30, 90, 180 and 365 days.
- Customer identifiers, operational categories and purchase dates.

The customer table provides additional geographic and customer information
that may be linked through `customer_id`.

Missing values are visible in both purchase and customer data. Some purchase
counts are displayed as decimals in training and integers in test, which
requires checking the underlying data types and values.

The preview also shows later purchase dates in test than in training.
The complete date ranges will be examined before choosing a validation strategy.

The submission template has the same number of rows as test. Its target
values will be treated as examples of the submission format, not as test labels.

### 3.2. Data Types and Column Overview

Inspect the stored data types, non-missing counts and number of distinct
values in each column. This helps identify categorical variables, possible
identifiers and columns requiring further checks.

In [ ]:
# Focus detailed column exploration on training and customer data.
for name, df in {"train": train, "clientes": clientes}.items():
    column_overview = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_missing": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean() * 100,
        "unique_non_missing": df.nunique(dropna=True),
    })

    column_overview.index.name = "variable"

    print(f"\nColumn overview: {name}")
    display(column_overview)

### 3.3. Variable Dictionary

The tables below describe all variables in the purchase and customer datasets.
Descriptions are based on the assignment, column names and observed values.
Exact accounting definitions, monetary units and historical window boundaries
require confirmation from the dataset documentation.

Analytical types describe how variables should be interpreted, independently
of their stored data types.

#### 3.3.1. Current Purchase Information

| Variable | Analytical type | Description |
|----------|-----------------|-------------|
| `Total a Pagar` | Numerical — monetary | Total amount payable for the current purchase. |
| `Total Bruto` | Numerical — monetary | Gross purchase amount, apparently before discounts. |
| `Total Liquido` | Numerical — monetary | Net purchase amount, apparently after discounts and before tax. |
| `Total Imp_` | Numerical — monetary | Tax amount associated with the purchase, inferred from the column name and preview. |
| `Total Portes` | Numerical — monetary | Shipping or delivery charge. |
| `Total Desc_ Global` | Numerical — monetary | Discount applied to the purchase as a whole. |
| `Total Desc_ Linha` | Numerical — monetary | Total discounts applied at line-item level. |
| `Custo` | Numerical — monetary | Recorded cost associated with the purchase; the accounting basis is not specified. |
| `N_ Detalhes` | Numerical — count | Number of purchase detail lines, inferred from the column name. |
| `invoice_count_current_purchase` | Numerical — count | Number of invoices associated with the current purchase occasion. |

#### 3.3.2. Customer Purchase History

| Variable | Analytical type | Description |
|----------|-----------------|-------------|
| `days_since_previous_purchase` | Numerical — duration | Days between the current purchase and the customer's previous purchase. |
| `customer_tenure_days` | Numerical — duration | Customer tenure in days at the purchase date; the starting reference requires confirmation. |
| `prior_purchase_count` | Numerical — count | Number of purchases recorded before the current purchase. |
| `prior_total_spend` | Numerical — monetary | Total spending across previous purchases. |
| `prior_mean_purchase_value` | Numerical — monetary | Mean value of previous purchases. |
| `prior_std_purchase_value` | Numerical — monetary | Standard deviation of previous purchase values. |
| `prior_min_purchase_value` | Numerical — monetary | Minimum value of previous purchases. |
| `prior_max_purchase_value` | Numerical — monetary | Maximum value of previous purchases. |
| `prior_mean_gap_days` | Numerical — duration | Mean interval in days between previous purchases. |
| `prior_std_gap_days` | Numerical — duration | Standard deviation of intervals between previous purchases. |

#### 3.3.3. Purchase Activity in Historical Windows

These variables appear to summarise activity in windows preceding the current
purchase. Whether the current purchase is excluded and how window boundaries
are handled require confirmation.

| Variable | Analytical type | Description |
|----------|-----------------|-------------|
| `purchase_count_30d` | Numerical — count | Number of purchases in the 30-day window. |
| `spend_30d` | Numerical — monetary | Total spending in the 30-day window. |
| `purchase_count_90d` | Numerical — count | Number of purchases in the 90-day window. |
| `spend_90d` | Numerical — monetary | Total spending in the 90-day window. |
| `purchase_count_180d` | Numerical — count | Number of purchases in the 180-day window. |
| `spend_180d` | Numerical — monetary | Total spending in the 180-day window. |
| `purchase_count_365d` | Numerical — count | Number of purchases in the 365-day window. |
| `spend_365d` | Numerical — monetary | Total spending in the 365-day window. |

#### 3.3.4. Identifiers, Operational Categories, Date and Target

| Variable | Analytical type | Description |
|----------|-----------------|-------------|
| `customer_id` | Identifier | Customer identifier shared with the customer table. Multiple purchases can belong to the same customer. |
| `payment_type` | Categorical — nominal | Recorded payment category. |
| `salesperson` | Categorical — nominal | Salesperson category associated with the purchase. |
| `commercial_zone` | Categorical — nominal | Commercial zone associated with the purchase. |
| `transport_method` | Categorical — nominal | Recorded transport or delivery category. |
| `warehouse` | Categorical — nominal | Warehouse associated with the purchase. |
| `document_series` | Categorical — nominal | Series or category of the purchase document. |
| `purchase_date` | Temporal | Date of the purchase, currently stored as text. |
| `ID` | Identifier | Observation identifier used to align predictions with the submission. |
| `random_noise` | Numerical — continuous | Variable whose name suggests artificial random noise; its usefulness will be assessed during feature selection. |
| `repeat_purchase_90d` | Binary target | Indicator of whether a repeat purchase occurs within 90 days. Present only in training. |

The operational category labels appear to be artificial aliases. Their wording
will not be interpreted as a literal description of business operations.

#### 3.3.5. Customer Information

| Variable | Analytical type | Description |
|----------|-----------------|-------------|
| `customer_id` | Identifier | Customer-table key used to link customer information to purchases. |
| `country_or_region` | Categorical — nominal | Recorded country or region category. |
| `commercial_region` | Categorical — nominal | Commercial region assigned to the customer. |
| `island` | Categorical — nominal | Recorded island category, where available. |
| `municipality` | Categorical — nominal | Recorded municipality category. |
| `city` | Categorical — nominal | Recorded city category. |
| `postal_area` | Categorical — nominal | Recorded postal area category. |
| `postal_code` | Categorical — nominal | Postal code category; its label should not be treated as a numerical quantity. |
| `street` | Categorical — nominal | Recorded street category. |
| `address` | Categorical — nominal | Recorded address label, potentially acting as a customer-level identifier. |
| `customer_type` | Categorical — intended | Customer classification field; all values are missing in the supplied table. |
| `company_name` | Categorical — nominal | Company label, potentially acting as a customer-level identifier. |
| `customer_since_year` | Temporal — year | Recorded year when the customer relationship began. |

### 3.4. Findings from the Column Overview

- Training contains 6,534 unique observation IDs and 610 distinct customers.
  Customers therefore have repeated observations, which must be considered
  when choosing a validation strategy.
- The customer table contains 1,143 unique customer IDs, with no missing keys.
  Purchase-to-customer matching still needs to be checked.
- The target has two distinct values and no missing observations.
- Many training variables have exactly 65 missing values. We will check
  whether these belong to the same rows.
- Historical variables have higher missingness, reaching 22.62% for
  `prior_std_gap_days`. Some missing values may reflect insufficient purchase
  history rather than recording errors.
- Count variables are stored as floats in training. Missing values can explain
  this storage type, but we still need to check for fractional or invalid counts.
- `purchase_date` is stored as text and requires date-parsing checks.
- `customer_type` is entirely missing and provides no observed information
  in the supplied customer table.
- `island`, `city` and `street` have substantial missingness. Missing geographic
  values may have different meanings depending on the field.
- Address and company labels have many distinct values. Their potential role
  as identifiers requires review before including them as model features.

### 3.5. Descriptive Statistics — Training Data

Summarise numerical features using counts, central tendency, dispersion and
percentiles. These statistics help identify differences in scale, asymmetry
and values requiring closer inspection.

Identifiers and the target are excluded from the numerical summary because
their averages and standard deviations have no useful interpretation here.
They will be examined separately.

In [ ]:
# Separate identifiers and the target from numerical features.
ID_COLUMNS = ["ID", "customer_id"]
TARGET = "repeat_purchase_90d"

numeric_features = [
    column
    for column in train.select_dtypes(include="number").columns
    if column not in ID_COLUMNS + [TARGET]
]

# Include tail percentiles to help inspect unusually small or large values.
numeric_summary = train[numeric_features].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).T

# Add the interquartile range as a measure of dispersion.
numeric_summary["IQR"] = (
    numeric_summary["75%"] - numeric_summary["25%"]
)

# Positive skewness indicates a longer right tail; negative a longer left tail.
numeric_summary["skewness"] = train[numeric_features].skew()

display(numeric_summary)

#### Interpretation of Numerical Statistics

**Current purchase values**

Purchase amounts are right-skewed. For example, `Total a Pagar` has a mean
of 925.56 and a median of 476.74, with a maximum of 22,297.21.
Large purchases pull the mean upwards, so the median and interquartile range
provide useful descriptions of typical purchases.

Costs, taxes and line discounts also show positive skewness. Their largest
values require inspection, but size alone does not establish that they are errors.

**Shipping, discounts and invoice counts**

`Total Portes` has a median and third quartile of zero, while
`Total Desc_ Global` remains zero at the 95th percentile. Both variables
are concentrated at zero, with relatively few positive values.

`invoice_count_current_purchase` has a median of one and a maximum of five.
Its zero interquartile range reflects the concentration at one invoice.
An IQR-based outlier rule would flag every value above one, even though
multiple invoices may be valid.

**Purchase history and customer activity**

Customers vary substantially in their recorded purchase history.
`prior_purchase_count` has a median of 11, a mean of 23.46 and a maximum
of 276. Historical spending is also right-skewed: `prior_total_spend`
has a median of 6,754.31 and a mean of 18,908.35.

The median `days_since_previous_purchase` is 57 days, compared with a mean
of 101.53 days. Some customers therefore have much longer gaps between purchases.

The 30-day purchase count and spending variables both have medians of zero,
indicating that at least half of their non-missing observations record
no activity in that window.

**Values requiring consistency checks**

`days_since_previous_purchase` and `prior_mean_gap_days` both reach −4.
Negative elapsed times conflict with their intended interpretation and
require investigation.

`prior_mean_purchase_value` and `spend_365d` both reach 1,000,000,000.
Their means are consequently far above their medians:
- Previous mean purchase value: mean 2,821,825.32 versus median 724.90.
- Annual spending: mean 2,396,534.88 versus median 2,426.61.

These extreme values may be placeholders or corrupted entries. We will
inspect the affected rows and compare them with related history variables.

The minimum recorded `prior_mean_purchase_value` is −2,478.66, while
all non-missing values of `prior_min_purchase_value` are positive.
We need to inspect the affected rows to check whether the historical
mean is consistent with the corresponding minimum and maximum.

**Random noise**

`random_noise` has a mean close to zero, a standard deviation close to one
and little skewness. These summaries resemble those of a standard normal
distribution, but do not establish normality or independence from the target.

Distributions will be examined visually in Section 7 using histograms,
boxplots and empirical cumulative distribution plots.

### 3.6. Categorical Summary — Training Data

Summarise the operational categories using their number of observed values,
most frequent category and its share of non-missing observations.

Purchase dates are excluded because they will be analysed as temporal data.

In [ ]:
# Define operational categories explicitly according to their meaning.
categorical_features = [
    "payment_type",
    "salesperson",
    "commercial_zone",
    "transport_method",
    "warehouse",
    "document_series",
]

# Summarise category counts and the most frequent value in each feature.
categorical_summary = train[categorical_features].describe().T

# Use descriptive labels for the output columns.
categorical_summary = categorical_summary.rename(columns={
    "count": "non_missing",
    "unique": "unique_categories",
    "top": "most_frequent_category",
    "freq": "most_frequent_count",
})

# Calculate the leading category's share among non-missing observations.
categorical_summary["most_frequent_pct"] = (
    categorical_summary["most_frequent_count"]
    / categorical_summary["non_missing"]
    * 100
)

display(categorical_summary)

#### Interpretation of Categorical Statistics

The percentages below refer to non-missing observations.

**Category concentration**

`warehouse` is almost constant: its most frequent category accounts for
99.67% of observed values, leaving only 21 observations in the other category.

`commercial_zone` is also highly concentrated, with 96.77% of observed
values belonging to Zone 4. The remaining categories have limited representation.

This imbalance may make comparisons involving rare categories unstable.
It does not, by itself, justify removing either feature.

**Payment and salesperson categories**

`payment_type` and `salesperson` each contain nine observed categories.
Their leading categories account for 72.76% and 64.61% of non-missing
observations, respectively.

Full frequency tables will show how the remaining observations are
distributed and whether some categories are particularly rare.

**Transport and document categories**

`transport_method` has 80 categories, substantially more than the other
operational variables. Its most frequent category represents 17.84%
of observed values. Category frequencies will help assess whether rare
levels need special treatment during preprocessing.

`document_series` contains three categories, with the largest representing
42.33% of observed values. It is less concentrated than warehouse or
commercial zone.

These summaries describe category frequencies. Relationships with repeat
purchase will be examined separately.

## 4. Identifiers and Observation Structure

Check observation IDs and repeated records before examining customer activity.
A customer can have several legitimate purchases, so repeated customer IDs
are not automatically duplicates.

### 4.1. Observation IDs and Duplicate Records

Check whether `ID` is complete and unique. Then compare records with the ID
excluded, since an observation copied under a new ID would otherwise go unnoticed.

A further check excludes `random_noise`, because different noise values could
also hide otherwise identical records. These checks flag records for inspection;
they do not remove observations.

In [ ]:
# Count missing IDs and repeated IDs beyond their first occurrence.
id_checks = pd.Series({
    "Total rows": len(train),
    "Missing IDs": train["ID"].isna().sum(),
    "Unique non-missing IDs": train["ID"].nunique(),
    "Repeated non-missing IDs beyond first": (
        train.loc[train["ID"].notna(), "ID"].duplicated().sum()
    ),
})

display(id_checks.to_frame("count"))

# Define comparison columns without modifying the original DataFrame.
columns_without_id = [
    column for column in train.columns
    if column != "ID"
]

columns_without_id_noise = [
    column for column in train.columns
    if column not in ["ID", "random_noise"]
]

# Compare duplicates under different definitions.
duplicate_definitions = {
    "All columns": train.columns.tolist(),
    "Excluding ID": columns_without_id,
    "Excluding ID and random_noise": columns_without_id_noise,
}

duplicate_results = []

for check_name, columns in duplicate_definitions.items():
    duplicate_results.append({
        "check": check_name,
        # Counts additional copies after the first record in each group.
        "duplicates_beyond_first": train.duplicated(
            subset=columns, keep="first"
        ).sum(),
        # Counts every record belonging to a duplicated group.
        "rows_in_duplicate_groups": train.duplicated(
            subset=columns, keep=False
        ).sum(),
    })

duplicate_summary = pd.DataFrame(duplicate_results).set_index("check")

display(duplicate_summary)

# Inspect potential duplicates using the broadest comparison above.
potential_duplicates = train.duplicated(
    subset=columns_without_id_noise,
    keep=False,
)

if potential_duplicates.any():
    print("Potential duplicate records — first 20 rows:")
    display(
        train.loc[potential_duplicates]
        .sort_values(["customer_id", "purchase_date", "ID"])
        .head(20)
    )
else:
    print("No duplicate records found after excluding ID and random_noise.")

#### Interpretation of Identifier and Duplicate Checks

All 6,534 observation IDs are present and unique.

Including every column produces no exact duplicates. Excluding `ID`
identifies 10 additional matching records across 16 rows, forming six groups.
Excluding `random_noise` produces the same result.

Inspection shows that all 16 flagged rows contain only an observation ID,
customer ID and target value. Their purchase dates, amounts, history and
operational categories are missing.

Pandas treats matching missing values as equal when detecting duplicates.
Consequently, these rows match on customer and target, but there is insufficient
information to establish whether they represent the same purchase.

No rows are removed at this stage. These records will be investigated further
as part of the missing-data analysis.

### 4.2. Repeated Customers and Purchase Granularity

Measure how observations are distributed across customers and check whether
the same customer appears more than once on the same recorded purchase date.

Repeated customers are expected in purchase data. Repeated customer-date pairs
require inspection because they may represent separate purchases, duplicated
records or a different aggregation level.

In [ ]:
# Count observations per customer, including records with missing purchase data.
customer_record_counts = train.groupby("customer_id").size()

# Identify customers represented by more than one observation.
repeated_customers = customer_record_counts > 1

customer_summary = pd.Series({
    "Unique customers": len(customer_record_counts),
    "Customers with one observation": (customer_record_counts == 1).sum(),
    "Customers with multiple observations": repeated_customers.sum(),
    "Rows belonging to repeated customers": (
        customer_record_counts.loc[repeated_customers].sum()
    ),
    "Median observations per customer": customer_record_counts.median(),
    "Maximum observations per customer": customer_record_counts.max(),
})

display(customer_summary.to_frame("value"))

# Describe the full distribution to assess customer representation.
display(
    customer_record_counts.describe(
        percentiles=[0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    ).to_frame("observations_per_customer")
)

# Exclude missing keys: unknown dates cannot establish a shared purchase date.
known_purchase_keys = train.dropna(
    subset=["customer_id", "purchase_date"]
)

# Count records for each customer and recorded date.
customer_date_counts = (
    known_purchase_keys
    .groupby(["customer_id", "purchase_date"])
    .size()
    .rename("record_count")
)

repeated_customer_dates = customer_date_counts[
    customer_date_counts > 1
]

print(
    "\nRows excluded from customer-date check due to missing keys:",
    len(train) - len(known_purchase_keys),
)
print("Repeated customer-date pairs:", len(repeated_customer_dates))
print(
    "Rows in repeated customer-date pairs:",
    int(repeated_customer_dates.sum()),
)

# Show the most repeated pairs for subsequent inspection.
display(
    repeated_customer_dates
    .sort_values(ascending=False)
    .head(20)
    .to_frame()
)

#### Interpretation of Customer Repetition and Purchase Granularity

Training contains 610 customers, of whom 453 have multiple observations
and 157 appear once. Repeated customers account for 6,377 rows, or 97.60%
of the training data.

Customer representation is uneven. The median is four observations per
customer, compared with a mean of 10.71, and the most represented customer
has 277 observations. These counts describe representation in the supplied
training sample, not necessarily each customer's complete purchase history.

Among the 6,469 rows with non-missing customer and date keys, no
customer-date pair is repeated. This supports interpreting an observation
as a purchase occasion at customer-date level, potentially covering
multiple invoices.

The remaining 65 rows have missing purchase dates, so their customer-date
uniqueness cannot be assessed.

Repeated customer IDs are expected and are not grounds for removing rows.
For validation, a random row split could place the same customers in both
training and validation. A chronological split assesses future purchases,
whereas a customer-grouped split assesses generalisation to unseen customers.
The final choice will depend on the prediction setting and test structure.

### 4.3. Matching Inputs and Different Target Values

Check whether records with identical observed inputs have different target
values. Observation IDs and random noise are excluded from the comparison,
while customer IDs remain included.

Matching incomplete records do not necessarily describe the same purchase.
The results must therefore be interpreted alongside the availability of
purchase information.

In [ ]:
# Compare input records without the target, observation ID or random noise.
comparison_features = [
    column for column in train.columns
    if column not in ["ID", "random_noise", TARGET]
]

# Keep missing values in the grouping so incomplete records are also examined.
input_groups = (
    train.groupby(comparison_features, dropna=False)
    .agg(
        record_count=("ID", "size"),
        target_values=(TARGET, "nunique"),
    )
    .reset_index()
)

# Find matching input groups containing both target classes.
conflicting_groups = input_groups.loc[
    (input_groups["record_count"] > 1)
    & (input_groups["target_values"] > 1)
]

print("Matching input groups with different targets:", len(conflicting_groups))
print(
    "Rows in these groups:",
    int(conflicting_groups["record_count"].sum()),
)

# Recover the corresponding records for inspection.
if not conflicting_groups.empty:
    conflicting_records = train.merge(
        conflicting_groups[comparison_features],
        on=comparison_features,
        how="inner",
        validate="many_to_one",
    )

    # Count available purchase fields, excluding identifiers and the target.
    purchase_fields = [
        column for column in comparison_features
        if column != "customer_id"
    ]

    conflicting_records["observed_purchase_fields"] = (
        conflicting_records[purchase_fields].notna().sum(axis=1)
    )

    display(
        conflicting_records[
            [
                "ID", "customer_id", "purchase_date", TARGET,
                "observed_purchase_fields",
            ]
        ]
        .sort_values(["customer_id", "ID"])
    )
else:
    print("No matching input groups contain different target values.")

#### Interpretation of Matching Inputs and Different Targets

Two matching-input groups contain different target values, involving four
rows and two customers: `427118` and `475032`.

All four rows have zero observed purchase fields. Only their observation
IDs, customer IDs and target values are available.

These results do not establish contradictory labels for the same purchase.
The records may represent different purchase occasions whose details are
missing. No labels are changed and no records are removed at this stage.

## 5. Data Quality Assessment

Investigate missing information, invalid values and inconsistencies.
Checks are performed on the original training data; cleaning decisions
will be implemented in the preprocessing notebook.

### 5.1. Missing Values by Column and Row

Measure missingness across columns, rows and individual cells. Identify
records with no purchase information and check whether the variables
with exactly 65 missing values share the same affected rows.

In [ ]:
# Exclude identifiers and the target when measuring purchase-data availability.
purchase_columns = [
    column for column in train.columns
    if column not in ["ID", "customer_id", TARGET, "random_noise"]
]

# Measure missingness across purchase fields.
purchase_missing = train[purchase_columns].isna()
missing_per_row = purchase_missing.sum(axis=1)
no_purchase_information = purchase_missing.all(axis=1)

# Summarise rows, columns and cells separately.
missing_overview = pd.Series({
    "Purchase fields assessed": len(purchase_columns),
    "Rows with any missing purchase field": purchase_missing.any(axis=1).sum(),
    "Rows with any missing purchase field (%)": (
        purchase_missing.any(axis=1).mean() * 100
    ),
    "Rows with all purchase fields missing": no_purchase_information.sum(),
    "Rows with all purchase fields missing (%)": (
        no_purchase_information.mean() * 100
    ),
    "Purchase columns containing missing values": (
        purchase_missing.any(axis=0).sum()
    ),
    "Missing purchase cells": purchase_missing.sum().sum(),
    "Missing purchase cells (%)": (
        purchase_missing.to_numpy().mean() * 100
    ),
})

display(missing_overview.to_frame("value"))

# Show missingness for every training column, retaining complete columns.
missing_by_column = pd.DataFrame({
    "missing_count": train.isna().sum(),
    "missing_pct": train.isna().mean() * 100,
}).sort_values("missing_count", ascending=False)

display(missing_by_column)

# Show how many rows have each number of missing purchase fields.
row_missing_distribution = (
    missing_per_row.value_counts()
    .sort_index()
    .rename_axis("missing_purchase_fields")
    .to_frame("rows")
)

row_missing_distribution["row_pct"] = (
    row_missing_distribution["rows"] / len(train) * 100
)

display(row_missing_distribution)

# Check whether all columns with 65 missing values share the same null mask.
columns_missing_65 = train.columns[train.isna().sum().eq(65)].tolist()

if columns_missing_65:
    reference_mask = train[columns_missing_65[0]].isna()

    same_missing_rows = all(
        train[column].isna().equals(reference_mask)
        for column in columns_missing_65
    )

    print("\nColumns with exactly 65 missing values:", columns_missing_65)
    print("All share the same missing rows:", same_missing_rows)

    # Test whether that shared pattern is exactly the fully missing purchase rows.
    if same_missing_rows:
        print(
            "Same rows as those with all purchase fields missing:",
            reference_mask.equals(no_purchase_information),
        )

#### Interpretation of Missing Values

Of the 6,534 training observations, 2,605 (39.87%) have at least one missing
purchase field, while 3,929 (60.13%) are complete across the 35 assessed fields.
Missing cells represent 4.27% of these fields overall.

Missingness therefore affects a substantial number of observations, even
though its share across individual cells is relatively small. Dropping every
incomplete row would remove almost 40% of training data.

**Rows without purchase information**

There are 65 observations (0.99%) with all 35 purchase fields missing.
Their `random_noise` values are also missing; only `ID`, `customer_id`
and the target remain available.

Every column with exactly 65 missing values shares this same set of rows.
This confirms a common missingness pattern across these records, although
the cause is unknown.

This pattern also explains the incomplete records flagged by the earlier
duplicate checks. Their treatment will be considered separately from
partially missing observations.

**Partially missing observations**

The most common incomplete pattern is one missing purchase field,
affecting 1,287 observations (19.70%). Another 511 observations (7.82%)
have seven missing fields. Counts alone do not establish whether these
rows share the same missing columns.

Historical summaries have the highest missingness:
- `prior_std_gap_days`: 22.62%.
- `prior_mean_gap_days`: 20.02%.
- `prior_std_purchase_value`: 17.03%.

Some historical statistics may be undefined when a customer has insufficient
previous purchases. We will compare their missingness with
`prior_purchase_count` before treating all missing values as recording errors.

No imputation or row removal is applied during this assessment.

### 5.2. Missingness Patterns and Purchase History

Visualise missingness by purchase field and examine whether missing historical
statistics coincide with limited previous purchase history.

The 65 rows with no purchase information are shown in the overall assessment
but excluded from the history comparison because their prior purchase counts
are unknown.

In [ ]:
# Calculate missingness across the purchase fields defined in Section 5.1.
missing_pct = (
    train[purchase_columns]
    .isna()
    .mean()
    .mul(100)
    .sort_values()
)

# Plot every purchase field so the assessment covers all 35 variables.
fig, ax = plt.subplots(figsize=(10, 11))

ax.barh(
    missing_pct.index,
    missing_pct.values,
    color=sns.color_palette("colorblind")[0],
)

# Label each bar with its percentage.
for position, percentage in enumerate(missing_pct.values):
    ax.text(
        percentage + 0.25,
        position,
        f"{percentage:.2f}%",
        va="center",
        fontsize=9,
    )

ax.set_title("Missing Purchase Information by Variable")
ax.set_xlabel("Missing observations (%)")
ax.set_ylabel("")
ax.set_xlim(0, missing_pct.max() + 5)
ax.grid(axis="y", visible=False)

plt.tight_layout()
plt.show()

#### Interpretation

Historical purchase statistics have the highest missingness:
`prior_std_gap_days` is missing in 22.62% of observations,
`prior_mean_gap_days` in 20.02% and `prior_std_purchase_value` in 17.03%.

All 35 purchase fields share a baseline of 65 missing observations (0.99%),
corresponding to rows with no purchase information. Several fields have
additional missing values beyond these rows.

Some historical statistics require multiple previous purchases to be defined.
We will compare their missingness with `prior_purchase_count` to investigate
whether unavailable history explains these patterns.

#### 5.2.1. Historical Missingness by Prior Purchase Count

Compare missing historical information across observations with zero, one,
two and three or more prior purchases.

Some statistics require sufficient history: a previous purchase value needs
at least one prior purchase, while intervals between prior purchases require
at least two. Standard deviations may require additional observations,
depending on how they were calculated.

Rows with missing, negative, fractional or non-finite prior purchase counts
are excluded from this comparison and reported separately.

In [ ]:
# Select historical features whose availability may depend on purchase history.
history_features = [
    "days_since_previous_purchase",
    "prior_total_spend",
    "prior_mean_purchase_value",
    "prior_std_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
    "prior_mean_gap_days",
    "prior_std_gap_days",
]

prior_count = train["prior_purchase_count"]

# Separate unknown counts from observed counts that are invalid for a count.
missing_count = prior_count.isna()

valid_count = (
    prior_count.notna()
    & np.isfinite(prior_count)
    & prior_count.ge(0)
    & prior_count.mod(1).eq(0)
)

invalid_count = prior_count.notna() & ~valid_count

print("Rows excluded — missing prior purchase count:", missing_count.sum())
print("Rows excluded — invalid prior purchase count:", invalid_count.sum())

# Work on a separate copy to preserve the original training data.
history_data = train.loc[
    valid_count,
    ["prior_purchase_count"] + history_features,
].copy()

# Group counts into stages where historical statistics become available.
history_groups = ["0", "1", "2", "3+"]

history_data["prior_purchase_group"] = pd.cut(
    history_data["prior_purchase_count"],
    bins=[-0.5, 0.5, 1.5, 2.5, np.inf],
    labels=history_groups,
)

# Report group sizes so percentages can be interpreted in context.
group_sizes = (
    history_data.groupby("prior_purchase_group", observed=False)
    .size()
    .rename("rows")
    .to_frame()
)

group_sizes["share_of_assessed_rows_pct"] = (
    group_sizes["rows"] / len(history_data) * 100
)

display(group_sizes)

# Calculate the percentage missing within each purchase-history group.
history_missing_pct = (
    history_data[history_features]
    .isna()
    .groupby(history_data["prior_purchase_group"], observed=False)
    .mean()
    .mul(100)
    .reindex(history_groups)
)

display(history_missing_pct)

# Include group sizes directly in the heatmap labels.
plot_data = history_missing_pct.T
plot_data.columns = [
    f"{group} prior\n(n={group_sizes.loc[group, 'rows']:,})"
    for group in history_groups
]

fig, ax = plt.subplots(figsize=(10, 6))

sns.heatmap(
    plot_data,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    cbar_kws={"label": "Missing observations (%)"},
    ax=ax,
)

ax.set_title("Historical Missingness by Prior Purchase Count")
ax.set_xlabel("Prior purchases")
ax.set_ylabel("")
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)

plt.tight_layout()
plt.show()

#### Interpretation

The comparison includes 6,469 observations. The 65 rows with missing prior
purchase counts were excluded, and no observed counts were negative,
fractional or non-finite.

Of the assessed observations, 603 have no prior purchases, 445 have one,
365 have two and 5,056 have three or more. These groups describe history
at each purchase occasion, rather than distinct customer groups.

**Missingness consistent with insufficient history**

For observations with no prior purchases, all assessed historical features
are missing except `prior_total_spend`, which is populated. We will check
whether this spending value is consistently zero.

`prior_std_purchase_value` is missing whenever there are fewer than two
prior purchases and is available for every observation with at least two.

`prior_mean_gap_days` is always missing with fewer than two prior purchases.
`prior_std_gap_days` is always missing with fewer than three and is available
for every observation with three or more.

These patterns are consistent with statistics calculated from prior purchases
only, using sample standard deviations: two purchase values are needed to
calculate their standard deviation, while three purchases provide two
intervals for calculating the standard deviation of purchase gaps.

**Missingness despite sufficient history**

Some historical features remain missing when their purchase-count
requirements are met:

- `days_since_previous_purchase`: 4.27% with one prior purchase,
  5.48% with two and 3.05% with three or more.
- `prior_mean_purchase_value`: 4.49%, 3.01% and 3.22% in these same groups.
- `prior_mean_gap_days`: 5.75% with two prior purchases and 3.44%
  with three or more.

Insufficient purchase history does not explain these additional gaps.
Their cause requires further investigation.

**Implications for preprocessing**

Missing values caused by insufficient history should be distinguished from
unexplained missing values. A single imputation rule may conceal this difference.
Purchase-history counts and suitable missingness indicators will therefore
be considered when designing preprocessing.

#### 5.2.2. Consistency of Limited Purchase History

Check whether observations with no prior purchases have zero historical
spending.

With exactly one prior purchase, total spending, mean, minimum and maximum
purchase value should agree if they summarise the same purchase amount.
These checks allow a small numerical tolerance and flag discrepancies
for investigation rather than automatically correcting them.

In [ ]:
# Store each check's result and affected row IDs for later inspection.
history_check_results = []
history_check_ids = {}


def check_history_equality(label, row_mask, left_column, right_column=None):
    """Compare available, finite values within a selected purchase-history group."""
    columns = [left_column]

    if right_column is not None:
        columns.append(right_column)

    values = train.loc[row_mask, columns]

    # Missing or non-finite values cannot be assessed by this equality check.
    assessable = (
        values.notna().all(axis=1)
        & np.isfinite(values).all(axis=1)
    )

    left = values.loc[assessable, left_column]

    # Compare either two columns or one column against zero.
    right = (
        values.loc[assessable, right_column]
        if right_column is not None
        else 0.0
    )

    # Allow minor floating-point differences and rounding to monetary units.
    matches = np.isclose(left, right, rtol=1e-9, atol=0.01)
    flagged_indices = left.index[~matches]

    history_check_results.append({
        "check": label,
        "eligible_rows": len(values),
        "assessed_rows": int(assessable.sum()),
        "unassessable_rows": int((~assessable).sum()),
        "flagged_rows": len(flagged_indices),
    })

    history_check_ids[label] = train.loc[flagged_indices, "ID"].tolist()


# No prior purchases should imply zero total historical spending.
check_history_equality(
    "Zero prior purchases: total spending equals zero",
    train["prior_purchase_count"].eq(0),
    "prior_total_spend",
)

# One purchase should give the same total, mean, minimum and maximum.
for column in [
    "prior_mean_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
]:
    check_history_equality(
        f"One prior purchase: total spending equals {column}",
        train["prior_purchase_count"].eq(1),
        "prior_total_spend",
        column,
    )

display(pd.DataFrame(history_check_results).set_index("check"))

# Inspect the union of flagged records without changing the original data.
flagged_ids = {
    record_id
    for record_ids in history_check_ids.values()
    for record_id in record_ids
}

if flagged_ids:
    print(f"\nDistinct records flagged: {len(flagged_ids)}")

    display(
        train.loc[
            train["ID"].isin(flagged_ids),
            [
                "ID",
                "customer_id",
                "purchase_date",
                "prior_purchase_count",
                "prior_total_spend",
                "prior_mean_purchase_value",
                "prior_min_purchase_value",
                "prior_max_purchase_value",
            ],
        ].sort_values("ID")
    )
else:
    print("All assessed records satisfy these limited-history checks.")

#### Interpretation of Limited-History Consistency Checks

All 603 observations with no prior purchases have zero historical spending,
consistent with the absence of previous purchase activity.

Among the 445 observations with exactly one prior purchase:
- Total spending equals the minimum purchase value in every observation.
- The mean differs from total spending in six of the 425 assessable observations.
  The remaining 20 have missing mean values.
- The maximum differs from total spending in three observations.

The checks flag nine distinct records. The mean discrepancies include two
negative values, three inflated positive values and one value of
1,000,000,000. The three inconsistent maximum values are all greater
than the corresponding total and minimum.

If these features summarise the same historical purchases, one prior purchase
should produce identical total, mean, minimum and maximum values.
The observed discrepancies therefore indicate internal inconsistencies,
rather than simply unusually large purchases.

Agreement between total spending and minimum value provides a useful
cross-check, but does not prove that either field is correct.
Potential corrections will be considered in the preprocessing notebook;
the original values remain unchanged here.

### 5.3. Numerical Validity Checks

Check numerical features for infinite values, negative values and fractional
counts. Count variables must contain non-negative whole numbers, while
durations and standard deviations must be non-negative.

Negative monetary values are flagged for investigation because their validity
depends on accounting definitions, including how returns or credits are recorded.
Negative values in `random_noise` are expected and are not flagged as errors.

Missing values are counted separately from observed values that fail a check.
No values are changed or removed.

In [ ]:
# Define count features according to their meaning, rather than stored dtype.
count_features = [
    "N_ Detalhes",
    "invoice_count_current_purchase",
    "prior_purchase_count",
    "purchase_count_30d",
    "purchase_count_90d",
    "purchase_count_180d",
    "purchase_count_365d",
]

# Durations and standard deviations should not be negative.
non_negative_features = [
    "days_since_previous_purchase",
    "customer_tenure_days",
    "prior_std_purchase_value",
    "prior_mean_gap_days",
    "prior_std_gap_days",
]

# Monetary negatives require interpretation before being classified as errors.
monetary_features = [
    "Total a Pagar",
    "Total Bruto",
    "Total Liquido",
    "Total Imp_",
    "Total Portes",
    "Total Desc_ Global",
    "Total Desc_ Linha",
    "Custo",
    "prior_total_spend",
    "prior_mean_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
    "spend_30d",
    "spend_90d",
    "spend_180d",
    "spend_365d",
]

# Retain per-variable flags for subsequent row-level inspection.
numerical_flags = pd.DataFrame(
    False,
    index=train.index,
    columns=numeric_features,
)

validity_results = []

for column in numeric_features:
    values = train[column]

    # Distinguish missing entries, finite observations and infinities.
    missing = values.isna()
    finite = values.notna() & np.isfinite(values)
    infinite = values.notna() & ~np.isfinite(values)

    negative = finite & values.lt(0)

    # Only count features are checked for fractional values.
    fractional = pd.Series(False, index=train.index)

    if column in count_features:
        fractional = finite & ~np.isclose(
            values,
            values.round(),
            rtol=0,
            atol=1e-9,
        )

    # Apply negative-value screening only where it is relevant.
    screen_negative = column in (
        count_features + non_negative_features + monetary_features
    )

    numerical_flags[column] = (
        infinite
        | fractional
        | (negative if screen_negative else False)
    )

    validity_results.append({
        "variable": column,
        "missing": int(missing.sum()),
        "finite_observations": int(finite.sum()),
        "infinite_values": int(infinite.sum()),
        "negative_values": int(negative.sum()),
        "fractional_counts": (
            int(fractional.sum()) if column in count_features else pd.NA
        ),
        "flagged_rows": int(numerical_flags[column].sum()),
    })

validity_summary = pd.DataFrame(validity_results).set_index("variable")

display(validity_summary)

# A record may fail multiple checks, so count distinct affected rows separately.
any_numerical_flag = numerical_flags.any(axis=1)

print(
    "\nDistinct rows flagged for numerical investigation:",
    int(any_numerical_flag.sum()),
)

# Display flagged cells in a compact table rather than printing every column.
flagged_cells = []

for column in numeric_features:
    flagged_index = numerical_flags.index[numerical_flags[column]]

    for row_index in flagged_index:
        flagged_cells.append({
            "ID": train.at[row_index, "ID"],
            "customer_id": train.at[row_index, "customer_id"],
            "variable": column,
            "value": train.at[row_index, column],
        })

if flagged_cells:
    display(
        pd.DataFrame(flagged_cells)
        .sort_values(["variable", "ID"])
        .head(30)
    )
else:
    print("No values were flagged by these numerical checks.")

#### Interpretation of Numerical Validity Checks

No numerical feature contains infinite values. All observed count values
are non-negative whole numbers, despite being stored as floats.

The checks flag 63 distinct observations, representing 0.96% of training:
- 16 negative values in `days_since_previous_purchase`.
- 16 negative values in `prior_mean_gap_days`.
- 16 negative values in `prior_mean_purchase_value`.
- 15 negative values in `spend_365d`.

The flagged-cell counts sum to 63, matching the distinct-row count.
Each affected row therefore fails one of these checks.

Negative elapsed times conflict with the meaning of purchase intervals.
All displayed negative time values equal −4, suggesting a repeated
placeholder or corruption pattern. Its origin has not been established.

Negative historical monetary values require further investigation.

The 3,357 negative values in `random_noise` are not flagged because negative
values are valid for this variable.

These checks assess basic numerical validity. They do not establish that
unflagged values are correct: extreme positive values and inconsistencies
between columns still require examination.

No values are changed or removed.

In [ ]:
# Inspect negative-value frequencies in the four flagged features.
negative_features = [
    "days_since_previous_purchase",
    "prior_mean_gap_days",
    "prior_mean_purchase_value",
    "spend_365d",
]

for column in negative_features:
    negative_values = train.loc[train[column].lt(0), column]

    print(f"\nNegative values: {column}")
    display(
        negative_values.value_counts()
        .sort_index()
        .rename_axis("value")
        .to_frame("frequency")
    )

# Inspect related history fields for every negative monetary observation.
negative_money_rows = (
    train["prior_mean_purchase_value"].lt(0)
    | train["spend_365d"].lt(0)
)

display(
    train.loc[
        negative_money_rows,
        [
            "ID",
            "customer_id",
            "prior_purchase_count",
            "prior_total_spend",
            "prior_mean_purchase_value",
            "prior_min_purchase_value",
            "prior_max_purchase_value",
            "spend_180d",
            "spend_365d",
        ],
    ].sort_values("ID")
)

#### Interpretation of Negative Values

**Negative time intervals**

Both `days_since_previous_purchase` and `prior_mean_gap_days` contain
16 negative observations, all recorded as −4.

Negative intervals are inconsistent with elapsed time between purchases.
The repeated value suggests a possible placeholder or systematic recording
issue, although its origin cannot be established from these results alone.

**Negative historical mean purchase values**

The 16 negative values of `prior_mean_purchase_value` occur alongside
positive historical spending, purchase counts and minimum purchase values.

A negative mean cannot lie between the positive minimum and maximum
of the same purchase history. These records therefore contain inconsistent
historical summaries.

Their magnitudes also appear to match `prior_total_spend` divided by
`prior_purchase_count`, with the sign reversed. We will check this
relationship numerically before drawing a conclusion.

**Negative annual spending**

`spend_365d` contains 15 negative observations, ranging from −159,115.18
to −167.18. The corresponding 180-day spending values are non-negative.

If these variables represent accumulated spending from non-negative
purchase amounts, this pattern is inconsistent with their intended meaning.
Returns or accounting adjustments could affect this interpretation, so
the spending definitions and relationships require further checks.

For observation `10004848`, annual spending is −159,115.18 while total
prior spending is 49,328.65, making this record particularly important
to investigate.

**Implications for preprocessing**

These observations will remain unchanged during exploration. Any later
correction or replacement will need a documented rule supported by the
consistency checks.

In [ ]:
# Inspect whether negative historical means match the expected mean
# in magnitude, assuming spending and counts describe the same purchases.
negative_mean_rows = train["prior_mean_purchase_value"].lt(0)

negative_mean_check = train.loc[
    negative_mean_rows,
    [
        "ID",
        "prior_purchase_count",
        "prior_total_spend",
        "prior_mean_purchase_value",
    ],
].copy()

# Calculate the mean implied by total spending and purchase count.
negative_mean_check["expected_mean"] = (
    negative_mean_check["prior_total_spend"]
    / negative_mean_check["prior_purchase_count"]
)

# Allow a small numerical tolerance when comparing monetary values.
negative_mean_check["matches_sign_reversal"] = np.isclose(
    negative_mean_check["prior_mean_purchase_value"],
    -negative_mean_check["expected_mean"],
    rtol=1e-9,
    atol=0.01,
)

display(negative_mean_check)

print(
    "Negative means matching a sign reversal:",
    f"{negative_mean_check['matches_sign_reversal'].sum()} "
    f"of {len(negative_mean_check)}",
)

All 16 negative historical means match the negative of
`prior_total_spend / prior_purchase_count`, within the numerical tolerance
used in the check.

Their corresponding historical minimum and maximum purchase values are
positive, so the negative means also fall outside the recorded range.

These findings support a sign error in `prior_mean_purchase_value`.
A correction based on the historical spending and purchase count will
be considered in the preprocessing notebook, after checking these
relationships across the remaining observations.

### 5.4. Consistency Between Historical Purchase Statistics

Historical purchase statistics should agree when they summarise the same
previous purchases.

For observations with at least one prior purchase, we will check whether:

- Mean purchase value equals total spending divided by purchase count.
- Minimum purchase value does not exceed maximum purchase value.
- Mean purchase value lies between the minimum and maximum.

Each check uses observations with valid purchase counts and finite values
for the fields involved. Missing or non-finite values are reported as
unassessable rather than treated as inconsistencies.

Small numerical differences are allowed to account for rounding.
Flagged records will be inspected without changing the original data.

In [ ]:
# Identify observations with a finite, positive, whole-number purchase count.
prior_count = train["prior_purchase_count"]

valid_prior_count = (
    prior_count.notna()
    & np.isfinite(prior_count)
    & prior_count.gt(0)
    & np.isclose(
        prior_count,
        prior_count.round(),
        rtol=0,
        atol=1e-9,
    )
)

# Calculate the mean implied by spending and purchase count.
# This separate object leaves the original training data unchanged.
expected_prior_mean = (
    train["prior_total_spend"]
    / prior_count.where(valid_prior_count)
)

# Store summaries and row-level flags for later inspection.
historical_consistency_results = []
historical_consistency_flags = pd.DataFrame(index=train.index)


def record_history_check(name, columns, condition):
    """Summarise a consistency check using finite, eligible observations."""

    # Require all fields involved in this check to be present and finite.
    available = (
        train[columns].notna().all(axis=1)
        & np.isfinite(train[columns]).all(axis=1)
    )

    assessed = valid_prior_count & available

    # A row is flagged only when it can be assessed and fails the check.
    flagged = assessed & ~condition.fillna(False)
    historical_consistency_flags[name] = flagged

    historical_consistency_results.append({
        "check": name,
        "eligible_rows": int(valid_prior_count.sum()),
        "assessed_rows": int(assessed.sum()),
        "unassessable_rows": int(
            (valid_prior_count & ~available).sum()
        ),
        "flagged_rows": int(flagged.sum()),
        "flagged_pct_of_assessed": (
            flagged.sum() / assessed.sum() * 100
            if assessed.any()
            else np.nan
        ),
    })


# Allow one cent of absolute difference and a small relative tolerance.
record_history_check(
    "Mean equals total spending / purchase count",
    ["prior_total_spend", "prior_mean_purchase_value"],
    pd.Series(
        np.isclose(
            train["prior_mean_purchase_value"],
            expected_prior_mean,
            rtol=1e-9,
            atol=0.01,
        ),
        index=train.index,
    ),
)

# Minimum should not exceed maximum, allowing for small rounding differences.
record_history_check(
    "Minimum does not exceed maximum",
    ["prior_min_purchase_value", "prior_max_purchase_value"],
    train["prior_min_purchase_value"].le(
        train["prior_max_purchase_value"] + 0.01
    ),
)

# Check the lower and upper bounds separately to locate inconsistencies.
record_history_check(
    "Mean is not below minimum",
    ["prior_mean_purchase_value", "prior_min_purchase_value"],
    train["prior_mean_purchase_value"].ge(
        train["prior_min_purchase_value"] - 0.01
    ),
)

record_history_check(
    "Mean is not above maximum",
    ["prior_mean_purchase_value", "prior_max_purchase_value"],
    train["prior_mean_purchase_value"].le(
        train["prior_max_purchase_value"] + 0.01
    ),
)

# Display the results of each check.
historical_consistency_summary = pd.DataFrame(
    historical_consistency_results
).set_index("check")

display(historical_consistency_summary)

# Count distinct flagged records, since one row may fail several checks.
any_history_inconsistency = historical_consistency_flags.any(axis=1)

print(
    "Distinct records flagged:",
    int(any_history_inconsistency.sum()),
)

In [ ]:
# Show the recorded statistics alongside the independently calculated mean.
historical_inconsistencies = train.loc[
    any_history_inconsistency,
    [
        "ID",
        "customer_id",
        "prior_purchase_count",
        "prior_total_spend",
        "prior_mean_purchase_value",
        "prior_min_purchase_value",
        "prior_max_purchase_value",
    ],
].copy()

historical_inconsistencies["expected_mean"] = (
    expected_prior_mean.loc[any_history_inconsistency]
)

# Include each check's flag so we can see why a record was selected.
historical_inconsistencies = historical_inconsistencies.join(
    historical_consistency_flags.loc[any_history_inconsistency]
)

print("Historical inconsistencies — first 30 records:")
display(historical_inconsistencies.sort_values("ID").head(30))

#### Interpretation of Historical Consistency Checks

Of the 5,866 observations with a valid positive prior purchase count,
5,672 could be assessed for consistency of the historical mean.
The remaining 194 observations have a missing mean purchase value.

The recorded mean differs from total spending divided by purchase count
in 78 observations, representing 1.38% of assessed rows.

Of these inconsistent means:
- 16 fall below the recorded minimum. These correspond to the negative
  means previously identified as sign reversals.
- 61 exceed the recorded maximum.
- One remains within the minimum–maximum range despite disagreeing with
  the mean calculated from spending and purchase count.

The inspected records include means of 1,000,000,000 and other positive
values substantially above their corresponding historical maximum.

No recorded minimum exceeds its corresponding maximum. However, this
check alone does not establish that either value is correct: an inflated
maximum could still satisfy that relationship.

These results show why numerical range checks and relationships between
variables are both necessary. A value may be positive, or even lie within
the recorded range, while remaining inconsistent with related statistics.

The original values remain unchanged. Possible corrections will be
considered during preprocessing, after checking the reliability of
the related historical fields.

In [ ]:
# Find means that disagree with total spending / purchase count
# but remain within the recorded minimum and maximum.
within_range_inconsistency = (
    historical_consistency_flags[
        "Mean equals total spending / purchase count"
    ]
    & ~historical_consistency_flags["Mean is not below minimum"]
    & ~historical_consistency_flags["Mean is not above maximum"]
)

print(
    "Inconsistent means within the recorded range:",
    int(within_range_inconsistency.sum()),
)

display(
    historical_inconsistencies.loc[within_range_inconsistency]
)

Observation `10007142` illustrates an inconsistency that range checks
alone would miss. Its recorded historical mean of 1,953.075 lies between
the minimum of 4.54 and maximum of 2,306.25, but total spending divided
by purchase count gives approximately 218.962.

Assuming these fields summarise the same previous purchases, the recorded
mean is inconsistent with the spending and count. This supports checking
relationships between variables alongside their individual ranges.

### 5.5. Consistency Between Purchase Windows

#### 5.5.1. Purchase Counts

If the purchase windows end at the same reference date and count the same
type of purchase, shorter windows should not contain more purchases
than longer windows:

`purchase_count_30d ≤ purchase_count_90d ≤ purchase_count_180d ≤ purchase_count_365d`

The 365-day count should also not exceed `prior_purchase_count`, assuming
both exclude the current purchase and use the same purchase definition.

Each comparison uses rows where both counts are present, finite,
non-negative and whole numbers. Other rows are reported as unassessable.

These checks identify inconsistencies without changing the original data.

In [ ]:
# Define comparisons between nested windows and total purchase history.
purchase_count_pairs = [
    ("purchase_count_30d", "purchase_count_90d"),
    ("purchase_count_90d", "purchase_count_180d"),
    ("purchase_count_180d", "purchase_count_365d"),
    ("purchase_count_365d", "prior_purchase_count"),
]

window_count_results = []
window_count_flags = pd.DataFrame(index=train.index)

for shorter_window, longer_window in purchase_count_pairs:
    shorter = train[shorter_window]
    longer = train[longer_window]

    # Assess only rows with two valid counts.
    assessed = (
        shorter.notna()
        & longer.notna()
        & np.isfinite(shorter)
        & np.isfinite(longer)
        & shorter.ge(0)
        & longer.ge(0)
        & np.isclose(shorter, shorter.round(), rtol=0, atol=1e-9)
        & np.isclose(longer, longer.round(), rtol=0, atol=1e-9)
    )

    # A shorter window containing more purchases fails the comparison.
    flagged = assessed & shorter.gt(longer)
    check_name = f"{shorter_window} > {longer_window}"

    # True explicitly means that this inconsistency was found.
    window_count_flags[check_name] = flagged

    window_count_results.append({
        "inconsistency": check_name,
        "assessed_rows": int(assessed.sum()),
        "unassessable_rows": int((~assessed).sum()),
        "flagged_rows": int(flagged.sum()),
        "flagged_pct_of_assessed": (
            flagged.sum() / assessed.sum() * 100
            if assessed.any()
            else np.nan
        ),
    })

# Summarise each comparison.
window_count_summary = pd.DataFrame(
    window_count_results
).set_index("inconsistency")

display(window_count_summary)

# Count distinct records because a row could fail multiple comparisons.
any_window_count_inconsistency = window_count_flags.any(axis=1)

print(
    "Distinct records with inconsistent purchase counts:",
    int(any_window_count_inconsistency.sum()),
)

In [ ]:
# Inspect flagged records alongside the specific comparisons they fail.
window_count_inconsistencies = train.loc[
    any_window_count_inconsistency,
    [
        "ID",
        "customer_id",
        "purchase_date",
        "prior_purchase_count",
        "purchase_count_30d",
        "purchase_count_90d",
        "purchase_count_180d",
        "purchase_count_365d",
    ],
].copy()

window_count_inconsistencies = window_count_inconsistencies.join(
    window_count_flags.loc[any_window_count_inconsistency]
)

if window_count_inconsistencies.empty:
    print("No purchase-count inconsistencies found among assessed rows.")
else:
    print("Purchase-count inconsistencies — first 30 records:")
    display(
        window_count_inconsistencies.sort_values("ID").head(30)
    )

#### Interpretation of Purchase-Count Consistency

All 6,469 assessed observations satisfy the expected ordering of purchase
counts across the 30-, 90-, 180- and 365-day windows.

The 365-day count also never exceeds the total prior purchase count.
The remaining 65 observations could not be assessed because their
purchase information is missing.

These results support consistency between the supplied purchase counts.
They do not independently establish that the counts are accurate or
that the windows exclude the current purchase.

#### 5.5.2. Historical Spending

If spending summarises non-negative purchase amounts over windows ending
at the same reference date, accumulated spending should not decrease
as the window expands.

We will compare spending across adjacent windows and check whether
365-day spending exceeds total prior spending. A tolerance of one cent
will allow for small rounding differences.

These comparisons are diagnostic. Returns, accounting adjustments or
differences in how spending is defined could affect the expected ordering.
Negative spending values will remain included in the comparisons and
have already been flagged separately.

In [ ]:
# Compare spending across nested windows and total purchase history.
spending_pairs = [
    ("spend_30d", "spend_90d"),
    ("spend_90d", "spend_180d"),
    ("spend_180d", "spend_365d"),
    ("spend_365d", "prior_total_spend"),
]

window_spending_results = []
window_spending_flags = pd.DataFrame(index=train.index)

for shorter_window, longer_window in spending_pairs:
    shorter = train[shorter_window]
    longer = train[longer_window]

    # Assess rows where both monetary values are present and finite.
    assessed = (
        shorter.notna()
        & longer.notna()
        & np.isfinite(shorter)
        & np.isfinite(longer)
    )

    # Flag decreases larger than one cent as the window expands.
    flagged = assessed & shorter.gt(longer + 0.01)
    check_name = f"{shorter_window} > {longer_window}"

    window_spending_flags[check_name] = flagged

    window_spending_results.append({
        "comparison": check_name,
        "assessed_rows": int(assessed.sum()),
        "unassessable_rows": int((~assessed).sum()),
        "flagged_rows": int(flagged.sum()),
        "flagged_pct_of_assessed": (
            flagged.sum() / assessed.sum() * 100
            if assessed.any()
            else np.nan
        ),
    })

# Display comparison results.
window_spending_summary = pd.DataFrame(
    window_spending_results
).set_index("comparison")

display(window_spending_summary)

# Count distinct records across all spending comparisons.
any_window_spending_flag = window_spending_flags.any(axis=1)

print(
    "Distinct records flagged for spending investigation:",
    int(any_window_spending_flag.sum()),
)

# Inspect the relevant spending and purchase-count fields.
window_spending_investigation = train.loc[
    any_window_spending_flag,
    [
        "ID",
        "customer_id",
        "prior_purchase_count",
        "prior_total_spend",
        "purchase_count_30d",
        "spend_30d",
        "purchase_count_90d",
        "spend_90d",
        "purchase_count_180d",
        "spend_180d",
        "purchase_count_365d",
        "spend_365d",
    ],
].copy()

# True identifies the comparison that flagged each record.
window_spending_investigation = window_spending_investigation.join(
    window_spending_flags.loc[any_window_spending_flag]
)

if window_spending_investigation.empty:
    print("No spending-order violations found among assessed rows.")
else:
    print("Spending records for investigation — first 30:")
    display(
        window_spending_investigation.sort_values("ID").head(30)
    )

#### Interpretation of Historical Spending Checks

No spending-order violations were found between the 30- and 90-day
windows or between the 90- and 180-day windows across 6,469 assessed rows.

Comparisons involving annual spending flagged 69 distinct observations:
- In 15 rows, 365-day spending is below 180-day spending.
- In 54 rows, 365-day spending exceeds total prior spending.

The 15 decreases correspond to the 15 negative annual spending values
identified earlier: all have non-negative 180-day spending.

The inspected records also contain unusually large positive annual
spending values, including repeated entries of 1,000,000,000.

Observation `10001907` is particularly clear: it records no prior
purchases, zero spending in the shorter windows and zero total prior
spending, but annual spending of 1,000,000,000.

These findings provide further evidence of inconsistencies involving
`spend_365d`, under the assumed common definition of historical spending.
They do not establish that every annual value passing these checks is
correct, or that the other spending fields are free from errors.

For comparisons involving annual spending, 260 rows could not be assessed
because `spend_365d` is missing. This includes the 65 rows missing all
purchase information.

The flagged values remain unchanged during exploration. Their treatment
will be documented in the preprocessing notebook.

#### 5.5.3. Zero Purchase Counts and Recorded Spending

If purchase counts and spending describe the same transactions within
each window, an observation with zero purchases should have zero spending.

We will check this relationship for the 30-, 90-, 180- and 365-day windows.
A tolerance of one cent will accommodate small rounding differences.

Rows with a zero purchase count but missing or non-finite spending cannot
be assessed. Non-zero spending will be flagged for investigation.

A positive purchase count with zero spending is not automatically treated
as an error, since zero-value purchases or accounting adjustments may
be possible.

In [ ]:
# Match each purchase count with spending from the same time window.
count_spending_pairs = [
    ("purchase_count_30d", "spend_30d"),
    ("purchase_count_90d", "spend_90d"),
    ("purchase_count_180d", "spend_180d"),
    ("purchase_count_365d", "spend_365d"),
]

zero_count_results = []
zero_count_flags = pd.DataFrame(index=train.index)

for count_column, spending_column in count_spending_pairs:
    purchase_count = train[count_column]
    spending = train[spending_column]

    # This check applies only to observations recording zero purchases.
    eligible = purchase_count.eq(0)

    # Missing or infinite spending cannot be assessed.
    assessed = (
        eligible
        & spending.notna()
        & np.isfinite(spending)
    )

    # Flag either positive or negative spending beyond the tolerance.
    spending_is_zero = pd.Series(
        np.isclose(spending, 0, rtol=0, atol=0.01),
        index=train.index,
    )

    flagged = assessed & ~spending_is_zero
    check_name = f"{count_column} = 0 but {spending_column} != 0"

    zero_count_flags[check_name] = flagged

    zero_count_results.append({
        "inconsistency": check_name,
        "eligible_rows": int(eligible.sum()),
        "assessed_rows": int(assessed.sum()),
        "unassessable_rows": int((eligible & ~assessed).sum()),
        "flagged_rows": int(flagged.sum()),
        "flagged_pct_of_assessed": (
            flagged.sum() / assessed.sum() * 100
            if assessed.any()
            else np.nan
        ),
    })

# Display results separately for each time window.
zero_count_summary = pd.DataFrame(
    zero_count_results
).set_index("inconsistency")

display(zero_count_summary)

# Count distinct records, allowing for overlapping flags across windows.
any_zero_count_flag = zero_count_flags.any(axis=1)

print(
    "Distinct records with zero purchases and non-zero spending:",
    int(any_zero_count_flag.sum()),
)

In [ ]:
# Inspect spending and counts together for the flagged observations.
zero_count_investigation = train.loc[
    any_zero_count_flag,
    [
        "ID",
        "customer_id",
        "prior_purchase_count",
        "prior_total_spend",
        "purchase_count_30d",
        "spend_30d",
        "purchase_count_90d",
        "spend_90d",
        "purchase_count_180d",
        "spend_180d",
        "purchase_count_365d",
        "spend_365d",
    ],
].copy()

# True indicates which window contains the mismatch.
zero_count_investigation = zero_count_investigation.join(
    zero_count_flags.loc[any_zero_count_flag]
)

if zero_count_investigation.empty:
    print("No zero-count spending mismatches found among assessed rows.")
else:
    print("Zero-count spending mismatches — first 30 records:")
    display(
        zero_count_investigation.sort_values("ID").head(30)
    )

# Identify whether this check finds records missed by spending-order checks.
new_zero_count_flags = (
    any_zero_count_flag & ~any_window_spending_flag
)

print(
    "Additional records beyond the spending-order checks:",
    int(new_zero_count_flags.sum()),
)

#### Interpretation of Zero-Count Spending Checks

All assessed observations with zero purchases in the 30-, 90- and
180-day windows also record zero spending within the one-cent tolerance.
These checks cover 4,342, 2,889 and 1,410 observations respectively.

For the 365-day window, 843 observations record zero purchases.
Of these, 811 could be assessed and 32 have missing annual spending.

One assessed observation, `10001907`, records annual spending of
1,000,000,000 despite having zero purchases in every window and zero
total prior spending.

This observation was already flagged by the spending-order checks.
The zero-count checks therefore identify no additional records, but
provide further evidence of inconsistency in its annual spending.

The 32 missing annual spending values are unassessable, rather than
confirmed matches or mismatches.

### 5.6. Consistency Between Current Purchase Amounts

We will examine whether the recorded amounts support the following
relationships:

- Net value equals gross value minus line and global discounts.
- Total payable equals net value plus tax.
- Total payable equals net value plus tax and shipping.

The two payable formulas help assess whether shipping needs to be added
separately. Their agreement rates must be interpreted alongside the
observations with non-zero shipping.

These checks assume that discounts are monetary amounts and that the
fields describe the same purchase. Differences may reflect rounding,
accounting conventions or inconsistent records.

Only observations with all required values present and finite will be
assessed. A one-cent absolute tolerance and a small relative tolerance
will accommodate numerical differences.

In [ ]:
# Define candidate relationships and the fields required for each one.
amount_relationships = [
    {
        "name": "Net = gross - line discounts - global discounts",
        "recorded": "Total Liquido",
        "required": [
            "Total Liquido",
            "Total Bruto",
            "Total Desc_ Linha",
            "Total Desc_ Global",
        ],
        "expected": (
            train["Total Bruto"]
            - train["Total Desc_ Linha"]
            - train["Total Desc_ Global"]
        ),
    },
    {
        "name": "Payable = net + tax",
        "recorded": "Total a Pagar",
        "required": ["Total a Pagar", "Total Liquido", "Total Imp_"],
        "expected": train["Total Liquido"] + train["Total Imp_"],
    },
    {
        "name": "Payable = net + tax + shipping",
        "recorded": "Total a Pagar",
        "required": [
            "Total a Pagar",
            "Total Liquido",
            "Total Imp_",
            "Total Portes",
        ],
        "expected": (
            train["Total Liquido"]
            + train["Total Imp_"]
            + train["Total Portes"]
        ),
    },
]

amount_check_results = []
amount_difference_flags = pd.DataFrame(index=train.index)
amount_residuals = pd.DataFrame(index=train.index)

for relationship in amount_relationships:
    name = relationship["name"]
    required = relationship["required"]
    recorded = train[relationship["recorded"]]
    expected = relationship["expected"]

    # Require every field used by the candidate formula to be finite.
    assessed = (
        train[required].notna().all(axis=1)
        & np.isfinite(train[required]).all(axis=1)
    )

    matches = pd.Series(
        np.isclose(recorded, expected, rtol=1e-9, atol=0.01),
        index=train.index,
    )

    differs = assessed & ~matches

    # Residual: positive means the recorded amount exceeds the formula.
    residual = (recorded - expected).where(assessed)

    amount_difference_flags[name] = differs
    amount_residuals[name] = residual

    amount_check_results.append({
        "candidate_relationship": name,
        "assessed_rows": int(assessed.sum()),
        "unassessable_rows": int((~assessed).sum()),
        "matching_rows": int((assessed & matches).sum()),
        "differing_rows": int(differs.sum()),
        "matching_pct": (
            (assessed & matches).sum() / assessed.sum() * 100
            if assessed.any()
            else np.nan
        ),
        "median_absolute_difference": residual.abs().median(),
        "maximum_absolute_difference": residual.abs().max(),
    })

display(
    pd.DataFrame(amount_check_results).set_index(
        "candidate_relationship"
    )
)

In [ ]:
# Compare both payable formulas on the same rows with non-zero shipping.
payable_fields = [
    "Total a Pagar",
    "Total Liquido",
    "Total Imp_",
    "Total Portes",
]

shipping_assessed = (
    train[payable_fields].notna().all(axis=1)
    & np.isfinite(train[payable_fields]).all(axis=1)
    & train["Total Portes"].abs().gt(0.01)
)

shipping_formula_results = []

for relationship in amount_relationships[1:]:
    name = relationship["name"]

    matching = shipping_assessed & ~amount_difference_flags[name]

    shipping_formula_results.append({
        "candidate_relationship": name,
        "assessed_non_zero_shipping_rows": int(shipping_assessed.sum()),
        "matching_rows": int(matching.sum()),
        "matching_pct": (
            matching.sum() / shipping_assessed.sum() * 100
            if shipping_assessed.any()
            else np.nan
        ),
    })

display(
    pd.DataFrame(shipping_formula_results).set_index(
        "candidate_relationship"
    )
)

# Inspect examples with non-zero shipping and each formula's residual.
shipping_examples = train.loc[
    shipping_assessed,
    ["ID"] + payable_fields,
].copy()

shipping_examples = shipping_examples.join(
    amount_residuals.iloc[:, 1:].add_prefix("Residual: ")
)

display(shipping_examples.sort_values("ID").head(20))

#### Interpretation of Current Purchase Amounts

All 6,469 assessed observations satisfy both relationships within the
specified numerical tolerance:

- `Total Liquido = Total Bruto − Total Desc_ Linha − Total Desc_ Global`
- `Total a Pagar = Total Liquido + Total Imp_`

The remaining 65 observations could not be assessed because their
purchase information is missing.

Adding `Total Portes` separately produces differences in 1,053 observations.
These are exactly the assessed observations with shipping amounts above
the one-cent threshold. In this subset, the formula without additional
shipping matches every row, while the formula adding shipping matches none.

The supplied totals therefore support calculating payable value as net
value plus tax, without adding shipping separately. These results alone
do not establish whether shipping is already included in another amount
or recorded separately for another purpose.

The accounting relationships also show that several monetary variables
contain mathematically overlapping information. This will be considered
during the analysis of feature relationships and feature selection.

Passing these checks establishes internal consistency, but does not
independently verify the accuracy of the recorded amounts.

### 5.7. Date Quality and Temporal Consistency

#### 5.7.1. Purchase-Date Parsing and Coverage

`purchase_date` was loaded as text. We will parse it using the expected
year-month-day format and distinguish missing dates from values that
cannot be interpreted as valid dates.

We will also examine the earliest and latest valid dates and the number
of observations recorded each year.

The original column will remain unchanged. Parsed dates will be stored
separately for exploratory checks and plots.

In [ ]:
# Parse dates separately, preserving the original purchase_date column.
purchase_dates = pd.to_datetime(
    train["purchase_date"],
    format="%Y-%m-%d",
    errors="coerce",
)

# Separate originally missing dates from non-missing values that failed parsing.
missing_purchase_dates = train["purchase_date"].isna()

invalid_purchase_dates = (
    train["purchase_date"].notna()
    & purchase_dates.isna()
)

date_quality_summary = pd.DataFrame({
    "count": {
        "Total observations": len(train),
        "Originally missing dates": int(missing_purchase_dates.sum()),
        "Non-missing dates that failed parsing": int(
            invalid_purchase_dates.sum()
        ),
        "Successfully parsed dates": int(purchase_dates.notna().sum()),
        "Distinct valid dates": int(purchase_dates.nunique()),
    }
})

display(date_quality_summary)

# Report the date range using successfully parsed values.
display(pd.DataFrame({
    "date": {
        "Earliest valid purchase date": purchase_dates.min(),
        "Latest valid purchase date": purchase_dates.max(),
    }
}))

# Inspect any non-missing values that could not be parsed.
if invalid_purchase_dates.any():
    print("Purchase dates that failed parsing:")
    display(
        train.loc[
            invalid_purchase_dates,
            ["ID", "customer_id", "purchase_date"],
        ]
    )
else:
    print("All non-missing purchase dates were parsed successfully.")

In [ ]:
# Count observations by calendar year, excluding unavailable dates.
purchases_by_year = (
    purchase_dates.dropna()
    .dt.year
    .value_counts()
    .sort_index()
    .rename_axis("year")
    .to_frame("observations")
)

# Percentages refer only to observations with successfully parsed dates.
purchases_by_year["share_of_valid_dates_pct"] = (
    purchases_by_year["observations"]
    / purchase_dates.notna().sum()
    * 100
)

display(purchases_by_year)

# Include intervening years in the chart, even if they have no observations.
if purchase_dates.notna().any():
    first_year = int(purchase_dates.min().year)
    last_year = int(purchase_dates.max().year)

    yearly_plot_counts = purchases_by_year["observations"].reindex(
        range(first_year, last_year + 1),
        fill_value=0,
    )

    ax = yearly_plot_counts.plot.bar(
        figsize=(11, 5),
        color=sns.color_palette("colorblind")[0],
        rot=45,
    )

    ax.set_title("Training observations by purchase year")
    ax.set_xlabel("Purchase year")
    ax.set_ylabel("Number of observations")

    # Label each bar with its observation count.
    ax.bar_label(ax.containers[0], padding=3, fontsize=9)
    ax.margins(y=0.15)

    plt.tight_layout()
    plt.show()
else:
    print("No valid purchase dates available for the yearly plot.")

#### Interpretation of Purchase-Date Coverage

All 6,469 non-missing purchase dates were parsed successfully.
The 65 missing dates belong to the observations previously identified
as lacking all purchase information.

The valid dates span 11 July 2015 to 15 February 2025, covering
1,492 distinct dates.

Yearly observation counts vary, with the highest count in 2021
(782 observations), followed by 2023 (767) and 2024 (757).

The first and last years have partial coverage: the recorded dates
begin in July 2015 and end in February 2025. Their counts therefore
should not be compared directly with those of complete calendar years.
In particular, the 78 observations in 2025 do not establish a decline
in purchasing activity.

The graph describes observations included in the supplied training
dataset. Without knowing how these observations were selected, changes
in their frequency cannot be interpreted directly as business growth
or decline.

The broad date range makes temporal validation worth considering.
We will compare training and test date coverage later to assess how
closely validation should reflect prediction on later purchases.

#### 5.7.2. Previous-Purchase Interval and Customer Tenure

If `customer_tenure_days` measures the time since the customer's first
purchase, the interval since their most recent previous purchase should
not exceed that tenure.

We will assess observations with at least one prior purchase and finite,
non-negative values for both time variables.

Missing or invalid values will be reported separately. The interpretation
of any flagged records depends on the supplied definition of customer
tenure.

In [ ]:
# Read the relevant fields without changing the original data.
prior_count = train["prior_purchase_count"]
previous_gap = train["days_since_previous_purchase"]
customer_tenure = train["customer_tenure_days"]

# Require a finite, positive, whole-number prior purchase count.
eligible_tenure_check = (
    prior_count.notna()
    & np.isfinite(prior_count)
    & prior_count.gt(0)
    & np.isclose(prior_count, prior_count.round(), rtol=0, atol=1e-9)
)

# Separate unavailable time values from observed invalid values.
time_fields = train[
    ["days_since_previous_purchase", "customer_tenure_days"]
]

missing_time_values = time_fields.isna().any(axis=1)

invalid_time_values = (
    (~np.isfinite(time_fields) & time_fields.notna()).any(axis=1)
    | time_fields.lt(0).any(axis=1)
)

# Assess only eligible observations with usable time values.
assessed_tenure_check = (
    eligible_tenure_check
    & ~missing_time_values
    & ~invalid_time_values
)

# Allow a tiny tolerance for floating-point representation.
gap_exceeds_tenure = (
    assessed_tenure_check
    & previous_gap.gt(customer_tenure + 1e-9)
)

tenure_check_summary = pd.DataFrame({
    "count": {
        "Total training observations": len(train),
        "Eligible observations with prior purchases": int(
            eligible_tenure_check.sum()
        ),
        "Eligible rows with missing time values": int(
            (eligible_tenure_check & missing_time_values).sum()
        ),
        "Eligible rows with invalid observed time values": int(
            (eligible_tenure_check & invalid_time_values).sum()
        ),
        "Assessed observations": int(assessed_tenure_check.sum()),
        "Unassessable eligible observations": int(
            (eligible_tenure_check & ~assessed_tenure_check).sum()
        ),
        "Previous-purchase interval exceeds tenure": int(
            gap_exceeds_tenure.sum()
        ),
    }
})

display(tenure_check_summary)

# Missing and invalid categories can overlap; their counts are not additive.
if assessed_tenure_check.any():
    print(
        "Flagged share of assessed observations:",
        f"{gap_exceeds_tenure.sum() / assessed_tenure_check.sum() * 100:.3f}%",
    )

# Inspect any violations and quantify how far the interval exceeds tenure.
tenure_investigation = train.loc[
    gap_exceeds_tenure,
    [
        "ID",
        "customer_id",
        "purchase_date",
        "prior_purchase_count",
        "days_since_previous_purchase",
        "customer_tenure_days",
    ],
].copy()

tenure_investigation["excess_days"] = (
    tenure_investigation["days_since_previous_purchase"]
    - tenure_investigation["customer_tenure_days"]
)

if tenure_investigation.empty:
    print("No previous-purchase intervals exceed tenure among assessed rows.")
else:
    display(
        tenure_investigation.sort_values(
            "excess_days", ascending=False
        ).head(30)
    )

#### Interpretation of Previous-Purchase Intervals and Tenure

Of the 5,866 observations with a valid positive prior purchase count,
5,657 could be assessed. None has an interval since the previous
purchase greater than customer tenure.

The remaining 209 eligible observations could not be assessed:
193 have missing time values and 16 contain the negative intervals
identified earlier. These two groups do not overlap in this result.

The assessed observations therefore satisfy the expected relationship,
assuming customer tenure measures time since the first purchase.
This does not independently establish that either time variable is
accurate.

#### 5.7.3. Temporal Consistency Within Customers

We will compare each dated observation with the preceding dated
observation for the same customer in training.

Under a consistent definition of purchase history:
- Prior purchase count should increase by at least one between distinct
  purchase occasions.
- Customer tenure should increase by the elapsed number of days,
  if measured from a fixed starting date.
- The interval since the actual previous purchase should not exceed
  the interval since the previous purchase observed in training.

Training may omit some purchases. Therefore, purchase counts can increase
by more than one, and the actual previous-purchase interval can be shorter
than the interval between observed records.

Comparisons require valid values for the fields involved. Each flag
identifies a relationship requiring investigation, rather than an
automatic correction.

In [ ]:
# Keep dated observations with known customer identifiers.
# Retain missing history values so each pair uses consecutive observed dates.
customer_timeline = train.loc[
    purchase_dates.notna() & train["customer_id"].notna(),
    [
        "ID",
        "customer_id",
        "prior_purchase_count",
        "customer_tenure_days",
        "days_since_previous_purchase",
    ],
].copy()

customer_timeline["purchase_date_parsed"] = purchase_dates.loc[
    customer_timeline.index
]

customer_timeline = customer_timeline.sort_values(
    ["customer_id", "purchase_date_parsed", "ID"]
)

# Retrieve the preceding dated observation within each customer.
previous_fields = [
    "ID",
    "purchase_date_parsed",
    "prior_purchase_count",
    "customer_tenure_days",
]

previous_observations = (
    customer_timeline.groupby("customer_id")[previous_fields]
    .shift(1)
    .add_prefix("previous_observed_")
)

customer_timeline = customer_timeline.join(previous_observations)

# Calculate elapsed days between consecutive observed purchases.
customer_timeline["observed_gap_days"] = (
    customer_timeline["purchase_date_parsed"]
    - customer_timeline["previous_observed_purchase_date_parsed"]
).dt.total_seconds() / 86400

# First observations have no preceding record to compare.
# Restrict these checks to strictly later purchase dates.
eligible_temporal_pairs = customer_timeline["observed_gap_days"].gt(0)

print("Dated observations retained:", len(customer_timeline))
print(
    "Consecutive observation pairs assessed for eligibility:",
    int(eligible_temporal_pairs.sum()),
)

In [ ]:
temporal_check_results = []
temporal_flags = pd.DataFrame(index=customer_timeline.index)


def record_temporal_check(name, fields, condition, counts=False):
    """Assess a temporal relationship using valid values for its fields."""

    values = customer_timeline[fields]

    usable = (
        values.notna().all(axis=1)
        & np.isfinite(values).all(axis=1)
        & values.ge(0).all(axis=1)
    )

    # Purchase counts must also be whole numbers.
    if counts:
        whole_numbers = pd.DataFrame(
            np.isclose(values, values.round(), rtol=0, atol=1e-9),
            index=values.index,
            columns=values.columns,
        )
        usable = usable & whole_numbers.all(axis=1)

    assessed = eligible_temporal_pairs & usable
    flagged = assessed & ~condition.fillna(False)

    temporal_flags[name] = flagged

    temporal_check_results.append({
        "inconsistency": name,
        "eligible_pairs": int(eligible_temporal_pairs.sum()),
        "assessed_pairs": int(assessed.sum()),
        "unassessable_pairs": int(
            (eligible_temporal_pairs & ~usable).sum()
        ),
        "flagged_pairs": int(flagged.sum()),
        "flagged_pct_of_assessed": (
            flagged.sum() / assessed.sum() * 100
            if assessed.any()
            else np.nan
        ),
    })


# Each preceding observed purchase should contribute to later history.
record_temporal_check(
    "Prior purchase count fails to increase",
    [
        "prior_purchase_count",
        "previous_observed_prior_purchase_count",
    ],
    customer_timeline["prior_purchase_count"].ge(
        customer_timeline["previous_observed_prior_purchase_count"] + 1
    ),
    counts=True,
)

# A fixed tenure starting date implies tenure increases with elapsed time.
tenure_change = (
    customer_timeline["customer_tenure_days"]
    - customer_timeline["previous_observed_customer_tenure_days"]
)

record_temporal_check(
    "Tenure change differs from elapsed days",
    [
        "customer_tenure_days",
        "previous_observed_customer_tenure_days",
    ],
    pd.Series(
        np.isclose(
            tenure_change,
            customer_timeline["observed_gap_days"],
            rtol=0,
            atol=1e-6,
        ),
        index=customer_timeline.index,
    ),
)

# An actual previous purchase cannot precede a known intervening purchase.
record_temporal_check(
    "Previous-purchase interval exceeds observed gap",
    ["days_since_previous_purchase"],
    customer_timeline["days_since_previous_purchase"].le(
        customer_timeline["observed_gap_days"] + 1e-9
    ),
)

display(
    pd.DataFrame(temporal_check_results).set_index("inconsistency")
)

any_temporal_flag = temporal_flags.any(axis=1)

print(
    "Distinct later observations flagged:",
    int(any_temporal_flag.sum()),
)

In [ ]:
# Display the earlier and later records together with their temporal flags.
temporal_investigation = customer_timeline.loc[
    any_temporal_flag,
    [
        "customer_id",
        "previous_observed_ID",
        "ID",
        "previous_observed_purchase_date_parsed",
        "purchase_date_parsed",
        "observed_gap_days",
        "previous_observed_prior_purchase_count",
        "prior_purchase_count",
        "previous_observed_customer_tenure_days",
        "customer_tenure_days",
        "days_since_previous_purchase",
    ],
].copy()

temporal_investigation = temporal_investigation.join(
    temporal_flags.loc[any_temporal_flag]
)

if temporal_investigation.empty:
    print("No temporal inconsistencies found among assessed pairs.")
else:
    print("Temporal comparisons requiring investigation — first 30:")
    display(temporal_investigation.head(30))

#### Interpretation of Temporal Consistency Within Customers

The 6,469 dated observations produce 5,862 consecutive purchase pairs
within customers.

All 5,862 pairs satisfy both historical progression checks:
- Prior purchase count increases by at least one.
- Customer tenure increases by the number of days between observations.

The tenure results are consistent with measuring each customer's tenure
from a fixed starting date. They do not establish what that date represents.

For the previous-purchase interval, 5,654 pairs could be assessed.
None has a recorded interval greater than the gap between the two
observed purchases. The remaining 208 pairs have missing or invalid
interval values and could not be assessed.

No additional temporal inconsistencies were identified by these checks.
However, passing them does not establish that the training dataset
contains every purchase or that all historical values are accurate.

### 5.8. Categorical Label Quality

We will inspect the categorical purchase variables for:
- Empty strings or labels containing only whitespace.
- Leading or trailing whitespace.
- Distinct labels that become identical after trimming whitespace
  and ignoring capitalisation.

Normalised labels will be used only for comparison. Potential variants
will be inspected before deciding whether they represent the same category.

Missing values remain separate from observed text labels.

In [ ]:
# Inspect the purchase categories already identified in Section 3.
category_quality_results = []
category_label_variants = {}

for column in categorical_features:
    # Use nullable strings so missing values remain missing.
    labels = train[column].astype("string")
    trimmed_labels = labels.str.strip()

    # Detect blank labels and surrounding whitespace.
    blank_labels = labels.notna() & trimmed_labels.eq("").fillna(False)

    surrounding_whitespace = (
        labels.notna()
        & labels.ne(trimmed_labels).fillna(False)
    )

    # Normalise temporarily for comparison, without changing train.
    normalised_labels = trimmed_labels.str.casefold()

    # Inspect distinct non-blank labels and their normalised equivalents.
    label_mapping = pd.DataFrame({
        "original_label": labels,
        "normalised_label": normalised_labels,
    }).loc[
        labels.notna() & ~blank_labels
    ].drop_duplicates()

    # Find normalised labels associated with multiple original spellings.
    variants_per_label = (
        label_mapping.groupby("normalised_label")["original_label"]
        .nunique()
    )

    variant_groups = variants_per_label[
        variants_per_label.gt(1)
    ].index

    category_label_variants[column] = label_mapping.loc[
        label_mapping["normalised_label"].isin(variant_groups)
    ].sort_values(["normalised_label", "original_label"])

    category_quality_results.append({
        "variable": column,
        "missing_values": int(labels.isna().sum()),
        "blank_labels": int(blank_labels.sum()),
        "rows_with_surrounding_whitespace": int(
            surrounding_whitespace.sum()
        ),
        "distinct_non_blank_labels": int(
            labels.loc[labels.notna() & ~blank_labels].nunique()
        ),
        "distinct_normalised_non_blank_labels": int(
            normalised_labels.loc[
                labels.notna() & ~blank_labels
            ].nunique()
        ),
        "potential_variant_groups": len(variant_groups),
    })

# Summarise formatting issues for each categorical variable.
category_quality_summary = pd.DataFrame(
    category_quality_results
).set_index("variable")

display(category_quality_summary)

# Display potential variants only where they were found.
for column, variants in category_label_variants.items():
    if not variants.empty:
        print(f"\nPotential label variants: {column}")
        display(variants)

if all(
    variants.empty for variants in category_label_variants.values()
):
    print("No label variants found after trimming and ignoring case.")

#### Interpretation of Categorical Label Quality

No empty or whitespace-only labels, leading or trailing spaces,
or label variants differing only in capitalisation were found.

The number of distinct categories remains unchanged after temporary
normalisation for all six variables. These checks therefore provide
no evidence that formatting creates additional categories.

Missing values remain present, ranging from 65 observations in
`commercial_zone` and `document_series` to 261 in `transport_method`.

These results establish consistency in the inspected label formatting.
They do not verify the meaning of the categories or whether individual
observations have been assigned correctly.

### 5.9. Summary of Training Data Quality Checks

The main findings are:

- 65 observations lack all purchase information while retaining their
  identifiers and target.
- Historical missingness often follows insufficient purchase history,
  but additional missing values occur where statistics could be defined.
- No infinite numerical values or negative or fractional purchase counts
  were found.
- Two interval variables contain 16 negative values each, all equal to −4.
- Historical means disagree with spending divided by purchase count
  in 78 assessed observations, including 16 confirmed sign reversals.
- Annual spending fails the ordering checks in 69 observations.
  One also records non-zero spending despite zero annual purchases.
- Current purchase amounts satisfy the supported gross-to-net and
  net-to-payable relationships across all assessed rows.
- All non-missing purchase dates parse successfully. The assessed
  within-customer temporal relationships also pass their checks.
- No blank categorical labels, surrounding whitespace or variants
  differing only in capitalisation were found.

Original values remain unchanged. These findings will guide documented
preprocessing decisions.

Distribution plots and outlier exploration will follow. Passing the
checks above does not establish that every remaining value is plausible
or accurate.

## 6. Target Distribution

The target, `repeat_purchase_90d`, indicates whether a customer makes
a repeat purchase within the following 90 days:

- `0`: no repeat purchase within the prediction window.
- `1`: repeat purchase within the prediction window.

We will check the observed labels and measure the frequency of each class.
Class balance will inform the choice of validation strategy and
classification metrics.

The proportions describe purchase observations in training, rather than
distinct customers, since customers can appear multiple times.

In [ ]:
# Check that observed target values belong to the expected binary labels.
target_values = train[TARGET]

unexpected_target = (
    target_values.notna()
    & ~target_values.isin([0, 1])
)

print("Missing target values:", int(target_values.isna().sum()))
print("Unexpected target values:", int(unexpected_target.sum()))

if unexpected_target.any():
    display(train.loc[unexpected_target, ["ID", TARGET]])

# Summarise the two expected classes.
target_distribution = (
    target_values.loc[target_values.isin([0, 1])]
    .value_counts()
    .reindex([0, 1], fill_value=0)
    .rename_axis(TARGET)
    .to_frame("observations")
)

valid_target_count = target_distribution["observations"].sum()

target_distribution["percentage"] = (
    target_distribution["observations"]
    / valid_target_count
    * 100
    if valid_target_count > 0
    else np.nan
)

target_distribution["meaning"] = [
    "No repeat purchase within 90 days",
    "Repeat purchase within 90 days",
]

display(target_distribution)

# Calculate the accuracy obtained by always predicting the majority class.
if valid_target_count > 0:
    majority_class = target_distribution["observations"].idxmax()
    majority_accuracy = (
        target_distribution["observations"].max()
        / valid_target_count
    )

    print("Majority class:", majority_class)
    print(f"Majority-class baseline accuracy: {majority_accuracy:.2%}")

In [ ]:
# Plot class counts and label them with their shares of valid targets.
fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(
    ["0: No repeat purchase", "1: Repeat purchase"],
    target_distribution["observations"],
    color=sns.color_palette("colorblind")[:2],
)

for bar, count, percentage in zip(
    bars,
    target_distribution["observations"],
    target_distribution["percentage"],
):
    ax.annotate(
        f"{count:,}\n({percentage:.2f}%)",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
    )

ax.set_title("Repeat purchase within 90 days — training class balance")
ax.set_xlabel("Target class")
ax.set_ylabel("Number of observations")
ax.margins(y=0.20)

plt.tight_layout()
plt.show()

## 7. Univariate Analysis

We will examine individual feature distributions to understand their
shape, spread, concentration at zero and extreme values.

Histograms show how observations are distributed, while boxplots
summarise the median, interquartile range and values beyond the whiskers.
Points beyond the whiskers are statistical outliers, not automatically
recording errors.

### 7.1. Current Purchase Amounts and Costs

We will first examine gross value, net value, total payable, tax and cost.

Each variable will be displayed using:
- A histogram covering its full observed finite range.
- A boxplot retaining the extreme observations.
- A histogram of `log1p(value)` to make differences among smaller values
  easier to see when the distribution is strongly right-skewed.

The logarithmic view uses non-negative values only and includes zero.
It is an exploratory representation, not a preprocessing decision.
Missing and non-finite values are excluded from plots and reported below.

In [ ]:
# Group current purchase amounts separately from shipping and discounts.
current_amount_features = [
    "Total Bruto",
    "Total Liquido",
    "Total a Pagar",
    "Total Imp_",
    "Custo",
]

# Report the observations included in each graphical view.
current_amount_coverage = []

for column in current_amount_features:
    values = train[column]
    finite_mask = values.notna() & np.isfinite(values)

    current_amount_coverage.append({
        "variable": column,
        "missing": int(values.isna().sum()),
        "non_finite_non_missing": int(
            (values.notna() & ~np.isfinite(values)).sum()
        ),
        "finite_observations": int(finite_mask.sum()),
        "negative_finite_values": int(
            (finite_mask & values.lt(0)).sum()
        ),
        "observations_in_log_view": int(
            (finite_mask & values.ge(0)).sum()
        ),
    })

display(
    pd.DataFrame(current_amount_coverage).set_index("variable")
)

In [ ]:
# Define the plotting function once, then call it in separate cells.
def plot_amount_distribution(column):
    """Show the full distribution, boxplot and log1p view of one variable."""

    # Select finite observations without changing the original data.
    values = train.loc[
        train[column].notna() & np.isfinite(train[column]),
        column,
    ]

    if values.empty:
        print(f"No finite observations available for {column}.")
        return

    colours = sns.color_palette("colorblind")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Full-range histogram with mean and median reference lines.
    sns.histplot(
        x=values, bins=40, color=colours[0], ax=axes[0]
    )

    axes[0].axvline(
        values.mean(),
        color=colours[1],
        linestyle="--",
        label=f"Mean: {values.mean():,.2f}",
    )
    axes[0].axvline(
        values.median(),
        color=colours[2],
        linestyle=":",
        label=f"Median: {values.median():,.2f}",
    )

    axes[0].set_title("Full-range distribution")
    axes[0].set_xlabel(column)
    axes[0].set_ylabel("Observations")
    axes[0].legend(fontsize=9)

    # Retain extreme observations in the boxplot.
    sns.boxplot(
        x=values,
        whis=1.5,
        showfliers=True,
        color=colours[0],
        ax=axes[1],
    )

    axes[1].set_title("Boxplot — 1.5 × IQR whiskers")
    axes[1].set_xlabel(column)

    # Display non-negative observations on a log1p scale, including zeros.
    non_negative_values = values.loc[values.ge(0)]

    if not non_negative_values.empty:
        sns.histplot(
            x=np.log1p(non_negative_values),
            bins=40,
            color=colours[0],
            ax=axes[2],
        )
    else:
        axes[2].text(
            0.5, 0.5, "No non-negative observations",
            ha="center", va="center", transform=axes[2].transAxes,
        )

    axes[2].set_title("Log1p distribution")
    axes[2].set_xlabel(f"log(1 + {column})")
    axes[2].set_ylabel("Observations")

    fig.suptitle(f"{column} — training distribution", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# Examine purchase value before discounts.
plot_amount_distribution("Total Bruto")

#### Interpretation of Gross Purchase Value

`Total Bruto` is strongly right-skewed: most observations are concentrated
at lower purchase values, with a long tail extending above 20,000.

The mean of 886.15 is almost twice the median of 445.74, reflecting
the influence of large purchases. The median therefore provides a
more representative description of the centre of the distribution.

The boxplot shows many observations above the upper 1.5 × IQR whisker.
These are statistical outliers, but their size alone does not establish
that they are errors.

The log1p view reveals a broadly unimodal, more symmetric distribution,
making the variation among smaller purchases easier to see.
Its appearance does not establish normality or justify a transformation
for modelling on its own.

In [ ]:
# Examine purchase value after the recorded discounts.
plot_amount_distribution("Total Liquido")

#### Interpretation of Net Purchase Value

`Total Liquido` remains strongly right-skewed after discounts, with most
observations concentrated at lower values and a long upper tail.

Its mean of 773.34 exceeds the median of 389.91. Both are lower than
the corresponding gross-value statistics, consistent with the discount
relationship verified earlier.

The boxplot shows many observations beyond the upper whisker, including
values above 15,000. These require contextual inspection rather than
automatic removal.

The log1p view reveals a broadly unimodal and more symmetric distribution.
Discounts reduce purchase amounts, but the plots show that substantial
variation in purchase size remains.

In [ ]:
# Examine the total amount payable.
plot_amount_distribution("Total a Pagar")

#### Interpretation of Total Payable

`Total a Pagar` is strongly right-skewed, with most observations at
lower amounts and a long tail reaching approximately 22,300.

The mean of 925.56 is substantially higher than the median of 476.74,
showing the influence of large purchases. Both exceed the corresponding
net-value statistics, consistent with adding the recorded tax.

The boxplot highlights many observations above the upper whisker.
The accounting checks found that payable equals net value plus tax
across all assessed rows, including these large amounts. This supports
internal consistency but does not independently verify their accuracy.

The log1p view makes the central distribution easier to distinguish
and shows a broadly unimodal shape, similar to gross and net purchase
values. Their relationships will be examined directly in the
multivariate analysis.

In [ ]:
# Examine the recorded tax amount.
plot_amount_distribution("Total Imp_")

#### Interpretation of Tax Amounts

`Total Imp_` is strongly right-skewed. Its mean of 152.22 exceeds
the median of 79.74, with a long upper tail extending above 4,000.

The boxplot identifies many observations above the upper whisker.
Large tax amounts should be examined alongside their corresponding
purchase values rather than judged in isolation.

The log1p histogram shows a main concentration of positive tax amounts
and a separate concentration at zero or very small values. The first
bin includes both, so its height does not give the exact number of
zero-tax observations.

Zero tax may reflect legitimate purchase circumstances, but the plot
alone cannot establish their meaning. We will quantify exact zeros
and examine tax relative to net purchase value in the relationship
analysis.

In [ ]:
# Examine the recorded purchase cost.
plot_amount_distribution("Custo")

#### Interpretation of Purchase Costs

`Custo` is strongly right-skewed, with a mean of 328.17 and a median
of 152.54. Most observations have relatively low costs, while the
upper tail extends above 7,000.

The boxplot identifies many values beyond the upper whisker.
These should be assessed alongside purchase size and net value:
a high cost alone does not establish an error.

The log1p view separates the main distribution of positive costs
from a pronounced concentration at zero or very small values.
The exact number of zero-cost observations requires a numerical count,
since the first histogram bin also includes small positive values.

Zero recorded costs may have a different meaning from missing costs.
Their interpretation should be investigated before deciding whether
any treatment is appropriate.

### 7.2. Zero Values and IQR Outlier Flags

We will count exact zero values separately from missing observations
and quantify the statistical outliers shown in the boxplots.

The IQR rule flags observations below `Q1 − 1.5 × IQR` or above
`Q3 + 1.5 × IQR`, where `IQR = Q3 − Q1`.

These flags describe position within a distribution. They do not
establish that an observation is incorrect, particularly for strongly
right-skewed purchase amounts.

Percentages use finite observations as their denominator. No values
will be removed or capped during this analysis.

In [ ]:
# Quantify zeros and IQR flags for the five plotted monetary variables.
current_amount_iqr_results = []
current_amount_iqr_flags = pd.DataFrame(index=train.index)

for column in current_amount_features:
    values = train[column]
    finite_mask = values.notna() & np.isfinite(values)
    finite_values = values.loc[finite_mask]
    assessed_count = len(finite_values)

    # Calculate quartiles and the standard boxplot fences.
    q1 = finite_values.quantile(0.25)
    q3 = finite_values.quantile(0.75)
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    below_fence = finite_mask & values.lt(lower_fence)
    above_fence = finite_mask & values.gt(upper_fence)
    flagged = below_fence | above_fence

    # Use exact equality to distinguish zeros from small positive amounts.
    exact_zeros = finite_mask & values.eq(0)

    current_amount_iqr_flags[column] = flagged

    current_amount_iqr_results.append({
        "variable": column,
        "finite_observations": assessed_count,
        "exact_zeros": int(exact_zeros.sum()),
        "zero_pct": (
            exact_zeros.sum() / assessed_count * 100
            if assessed_count > 0 else np.nan
        ),
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_fence": lower_fence,
        "upper_fence": upper_fence,
        "below_lower_fence": int(below_fence.sum()),
        "above_upper_fence": int(above_fence.sum()),
        "flagged_pct": (
            flagged.sum() / assessed_count * 100
            if assessed_count > 0 else np.nan
        ),
        "zero_IQR": bool(iqr == 0) if assessed_count > 0 else pd.NA,
    })

current_amount_iqr_summary = pd.DataFrame(
    current_amount_iqr_results
).set_index("variable")

display(current_amount_iqr_summary)

# A purchase may be flagged in several mathematically related variables.
any_current_amount_iqr_flag = current_amount_iqr_flags.any(axis=1)

print(
    "Distinct observations beyond at least one IQR fence:",
    int(any_current_amount_iqr_flag.sum()),
)

In [ ]:
# Inspect the ten largest payable amounts together with related fields.
# Rank only finite payable values.
finite_payable = (
    train["Total a Pagar"].notna()
    & np.isfinite(train["Total a Pagar"])
)

largest_payable_purchases = (
    train.loc[
        finite_payable,
        [
            "ID",
            "customer_id",
            "purchase_date",
            "Total Bruto",
            "Total Desc_ Linha",
            "Total Desc_ Global",
            "Total Liquido",
            "Total Imp_",
            "Total a Pagar",
            "Custo",
            "N_ Detalhes",
            "invoice_count_current_purchase",
        ],
    ]
    .nlargest(10, "Total a Pagar")
    .copy()
)

print("Ten largest purchases by total payable:")
display(largest_payable_purchases)

#### Interpretation of Zero Values and IQR Flags

Gross value, net value and total payable contain no exact zeros among
their 6,469 finite observations.

Tax is exactly zero in 213 observations (3.29%), while cost is exactly
zero in 486 observations (7.51%). These are observed values, distinct
from the 65 missing values in each variable.

The IQR rule flags between 8.24% and 9.15% of observations per variable.
All flags occur above the upper fence, consistent with the right-skewed
distributions. Each variable has a positive IQR.

Across the five variables, 816 distinct observations exceed at least
one fence. Flags overlap because a large purchase can also have high
gross value, net value, tax and cost.

The largest payable purchase is 22,297.21 and contains 268 recorded
details. Several other large purchases also have many details, which
provides context for their size without confirming their accuracy.

Customer `227020` accounts for four of the ten largest payable purchases.
Large amounts therefore recur for some customers rather than appearing
only as isolated observations.

Some records require further contextual review. Observation `10004393`
has net value of 10,538.20 but cost of 179.92, while observation
`10007062` has net value of 9,762.60 and zero recorded cost.
These differences warrant examining cost relative to purchase value.

The inspected purchases satisfy the accounting relationships checked
earlier. Their IQR flags identify distributional extremes rather than
demonstrating recording errors. No observations are removed or capped
at this stage.

### 7.3. Shipping and Discounts

Shipping and discount amounts can be concentrated at zero.
We will distinguish their frequency of use from the size of positive
recorded amounts.

For each variable, we will show:
- Counts of zero and positive values among finite observations.
- A histogram of positive amounts.
- A boxplot of positive amounts, retaining extreme observations.

Missing values are reported separately. The positive-only plots exclude
zeros explicitly and do not represent the full distribution.

IQR flags calculated from the full distribution will also be summarised.
When the IQR is zero, the fences collapse to the same value, making this
rule unsuitable as an automatic criterion for identifying errors.

In [ ]:
# Examine shipping and the two recorded discount amounts.
shipping_discount_features = [
    "Total Portes",
    "Total Desc_ Global",
    "Total Desc_ Linha",
]

shipping_discount_results = []

for column in shipping_discount_features:
    values = train[column]
    finite_values = values.loc[
        values.notna() & np.isfinite(values)
    ]

    positive_values = finite_values.loc[finite_values.gt(0)]
    finite_count = len(finite_values)

    # Calculate IQR fences using the full finite distribution, including zeros.
    q1 = finite_values.quantile(0.25)
    q3 = finite_values.quantile(0.75)
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    outside_fences = (
        finite_values.lt(lower_fence)
        | finite_values.gt(upper_fence)
    )

    shipping_discount_results.append({
        "variable": column,
        "missing": int(values.isna().sum()),
        "finite_observations": finite_count,
        "exact_zeros": int(finite_values.eq(0).sum()),
        "zero_pct": (
            finite_values.eq(0).mean() * 100
            if finite_count > 0 else np.nan
        ),
        "positive_observations": len(positive_values),
        "negative_observations": int(finite_values.lt(0).sum()),
        "median_positive_amount": positive_values.median(),
        "maximum_positive_amount": positive_values.max(),
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "upper_fence": upper_fence,
        "outside_IQR_fences": int(outside_fences.sum()),
    })

display(
    pd.DataFrame(shipping_discount_results).set_index("variable")
)

In [ ]:
def plot_shipping_discount(column):
    """Separate zero-value frequency from the size of positive amounts."""

    values = train[column]
    finite_values = values.loc[
        values.notna() & np.isfinite(values)
    ]

    if finite_values.empty:
        print(f"No finite observations available for {column}.")
        return

    positive_values = finite_values.loc[finite_values.gt(0)]
    colours = sns.color_palette("colorblind")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Include negative values if present so the frequency chart is complete.
    labels = ["Zero", "Positive"]
    counts = [
        int(finite_values.eq(0).sum()),
        len(positive_values),
    ]

    if finite_values.lt(0).any():
        labels.append("Negative")
        counts.append(int(finite_values.lt(0).sum()))

    bars = axes[0].bar(
        labels,
        counts,
        color=colours[:len(labels)],
    )

    for bar, count in zip(bars, counts):
        axes[0].annotate(
            f"{count:,}\n({count / len(finite_values) * 100:.2f}%)",
            xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
        )

    axes[0].set_title("Recorded values — finite observations")
    axes[0].set_ylabel("Observations")
    axes[0].margins(y=0.20)

    # Show the complete positive range without the zero-value concentration.
    if not positive_values.empty:
        sns.histplot(
            x=positive_values,
            bins=30,
            color=colours[0],
            ax=axes[1],
        )

        axes[1].axvline(
            positive_values.median(),
            color=colours[2],
            linestyle=":",
            label=f"Median: {positive_values.median():,.2f}",
        )

        axes[1].legend(fontsize=9)

        # These boxplot quartiles refer to positive values only.
        sns.boxplot(
            x=positive_values,
            whis=1.5,
            showfliers=True,
            color=colours[0],
            ax=axes[2],
        )
    else:
        for ax in axes[1:]:
            ax.text(
                0.5, 0.5, "No positive observations",
                ha="center", va="center", transform=ax.transAxes,
            )

    axes[1].set_title("Positive amounts — full range")
    axes[1].set_xlabel(column)
    axes[1].set_ylabel("Observations")

    axes[2].set_title("Positive amounts — boxplot")
    axes[2].set_xlabel(column)

    fig.suptitle(f"{column} — training distribution", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# Shipping: frequency of zero values and distribution of positive amounts.
plot_shipping_discount("Total Portes")

#### Interpretation of Shipping Amounts

`Total Portes` is zero in 5,416 finite observations (83.72%) and positive
in 1,053 (16.28%). Zero recorded shipping should not automatically be
interpreted as free delivery, since its meaning depends on how charges
are recorded.

Among positive amounts, the median is 4.89. The distribution is strongly
right-skewed, with most charges at relatively low values and a sparse
upper tail reaching approximately 181.

The positive-only boxplot highlights unusually large charges relative
to other positive shipping amounts. These require contextual inspection
rather than automatic removal.

Because more than 75% of the full distribution is zero, both quartiles
and the IQR are zero. Applying the full-distribution IQR rule therefore
flags every positive amount, making it unhelpful for distinguishing
ordinary positive charges from unusually large ones.

Separating zero frequency from positive amounts reveals both how often
shipping is recorded and how much is recorded when positive.

In [ ]:
# Global discounts: frequency and size of positive recorded discounts.
plot_shipping_discount("Total Desc_ Global")

#### Interpretation of Global Discounts

`Total Desc_ Global` is zero in 6,359 finite observations (98.30%).
Only 110 observations (1.70%) contain a positive global discount.

Among positive discounts, the median is 9.21. Their distribution is
right-skewed, with most amounts concentrated at lower values and
a few extending above 200.

The full distribution has a zero IQR, so its standard IQR rule flags
every positive discount. The positive-only boxplot provides a more
informative comparison of discount sizes.

Positive global discounts are uncommon in training. Their amount
should also be examined relative to gross purchase value, since a large
discount may accompany a large purchase. Their usefulness for prediction
will be assessed through relationships with the target and validation.

In [ ]:
# Line discounts: frequency and size of positive recorded discounts.
plot_shipping_discount("Total Desc_ Linha")

#### Interpretation of Line Discounts

`Total Desc_ Linha` is positive in 4,028 finite observations (62.27%)
and zero in 2,441 (37.73%). Positive line discounts are therefore much
more common than positive global discounts, which occur in only 1.70%
of finite observations.

Among positive line discounts, the median is 61.52. The distribution
is strongly right-skewed, with most amounts concentrated at lower
values and a long tail extending above 4,000.

The positive-only boxplot identifies many large discounts relative
to other positive amounts. Their interpretation requires comparison
with gross purchase value: a large absolute discount may accompany
a large purchase without representing an unusually high discount rate.

Unlike shipping and global discounts, the full line-discount distribution
does not have both quartiles equal to zero. Its IQR flags still require
careful interpretation because of the strong right skew.

The frequency and relative size of discounts will be examined alongside
purchase value and the target in later relationship analyses.

### 7.4. Current Purchase Counts

`N_ Detalhes` records the number of purchase details or lines, while
`invoice_count_current_purchase` records the number of invoices associated
with the current purchase.

These variables describe different aspects of purchase size. A detail
count should not be interpreted as a quantity of individual units without
supporting documentation.

We will examine their distributions and calculate IQR flags. Counts
beyond the fences will be treated as observations for investigation,
rather than automatically classified as errors.

In [ ]:
# Summarise the two current-purchase count variables.
current_count_features = [
    "N_ Detalhes",
    "invoice_count_current_purchase",
]

current_count_results = []

for column in current_count_features:
    values = train[column]
    finite_values = values.loc[
        values.notna() & np.isfinite(values)
    ]

    q1 = finite_values.quantile(0.25)
    q3 = finite_values.quantile(0.75)
    iqr = q3 - q1

    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr

    outside_fences = (
        finite_values.lt(lower_fence)
        | finite_values.gt(upper_fence)
    )

    current_count_results.append({
        "variable": column,
        "missing": int(values.isna().sum()),
        "finite_observations": len(finite_values),
        "minimum": finite_values.min(),
        "median": finite_values.median(),
        "mean": finite_values.mean(),
        "maximum": finite_values.max(),
        "exact_zeros": int(finite_values.eq(0).sum()),
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "upper_fence": upper_fence,
        "outside_IQR_fences": int(outside_fences.sum()),
        "flagged_pct": (
            outside_fences.mean() * 100
            if not finite_values.empty else np.nan
        ),
    })

display(
    pd.DataFrame(current_count_results).set_index("variable")
)

In [ ]:
# Select finite detail counts without changing the original column.
detail_counts = train.loc[
    train["N_ Detalhes"].notna() & np.isfinite(train["N_ Detalhes"]),
    "N_ Detalhes",
]

colours = sns.color_palette("colorblind")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Integer-centred bins retain the discrete nature of the counts.
detail_bins = np.arange(
    detail_counts.min() - 0.5,
    detail_counts.max() + 1.5,
    1,
)

sns.histplot(
    x=detail_counts,
    bins=detail_bins,
    color=colours[0],
    ax=axes[0],
)

axes[0].axvline(
    detail_counts.mean(),
    color=colours[1],
    linestyle="--",
    label=f"Mean: {detail_counts.mean():.2f}",
)
axes[0].axvline(
    detail_counts.median(),
    color=colours[2],
    linestyle=":",
    label=f"Median: {detail_counts.median():.0f}",
)

axes[0].set_title("Purchase details — full-range distribution")
axes[0].set_xlabel("Number of purchase details")
axes[0].set_ylabel("Observations")
axes[0].legend()

# Keep large counts visible in the boxplot.
sns.boxplot(
    x=detail_counts,
    whis=1.5,
    showfliers=True,
    color=colours[0],
    ax=axes[1],
)

axes[1].set_title("Purchase details — boxplot")
axes[1].set_xlabel("Number of purchase details")

plt.tight_layout()
plt.show()

#### Interpretation of Purchase Detail Counts

`N_ Detalhes` is strongly right-skewed. The median is 11 details,
while the mean is 21.52, reflecting purchases with substantially
larger detail counts.

The middle 50% of observations lie between 4 and 29 details.
Counts range from 1 to 423, with no recorded zeros.

The upper IQR fence is 66.5, so counts of 67 or more are flagged.
This affects 436 observations (6.74% of finite values).
These purchases require contextual inspection, but a large number
of details alone does not establish an error.

The long tail indicates substantial variation in purchase size.
We will examine detail counts alongside monetary amounts to understand
whether larger counts tend to accompany larger purchase values.

In [ ]:
# Build an exact frequency table for each observed invoice count.
invoice_values = train.loc[
    train["invoice_count_current_purchase"].notna()
    & np.isfinite(train["invoice_count_current_purchase"]),
    "invoice_count_current_purchase",
]

invoice_frequency = (
    invoice_values.value_counts()
    .sort_index()
    .rename_axis("invoice_count")
    .to_frame("observations")
)

invoice_frequency["percentage"] = (
    invoice_frequency["observations"] / len(invoice_values) * 100
)

display(invoice_frequency)

# Plot each observed count separately, with exact frequencies and shares.
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    [f"{value:g}" for value in invoice_frequency.index],
    invoice_frequency["observations"],
    color=colours[0],
)

for bar, count, percentage in zip(
    bars,
    invoice_frequency["observations"],
    invoice_frequency["percentage"],
):
    ax.annotate(
        f"{count:,}\n({percentage:.2f}%)",
        xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
        xytext=(0, 5),
        textcoords="offset points",
        ha="center",
        va="bottom",
    )

ax.set_title("Invoices associated with the current purchase")
ax.set_xlabel("Number of invoices")
ax.set_ylabel("Observations")
ax.margins(y=0.20)

plt.tight_layout()
plt.show()

#### Interpretation of Invoice Counts

Most purchases have one associated invoice: 6,116 observations,
representing 94.54% of finite values.

Two invoices occur in 312 observations (4.82%). Counts of three,
four and five invoices occur in 14, 21 and 6 observations respectively.

Both quartiles equal one, giving an IQR of zero. The IQR rule therefore
flags all 353 observations with multiple invoices (5.46%).
This reflects concentration at one invoice rather than evidence
that multiple invoices are invalid.

The variable has limited overall variation, although multiple-invoice
purchases may differ from single-invoice purchases. Their relationship
with purchase amounts and repeat purchasing will be examined later.

Both current-purchase count variables have 65 missing observations,
corresponding to the rows lacking all purchase information.

### 7.5. Purchase History and Customer Tenure

`prior_purchase_count` describes the number of purchases recorded before
the current purchase. `customer_tenure_days` describes customer tenure
at that observation, expressed in days.

We will examine their distributions, typical values and upper tails.
Zero values remain included and are counted separately from missing values.

These distributions describe purchase observations rather than distinct
customers. Customers with more observations contribute more frequently,
and their history and tenure can change between purchases.

In [ ]:
# Summarise purchase history and tenure using finite observations.
history_activity_features = [
    "prior_purchase_count",
    "customer_tenure_days",
]

history_activity_results = []

for column in history_activity_features:
    values = train[column]
    finite_values = values.loc[
        values.notna() & np.isfinite(values)
    ]

    history_activity_results.append({
        "variable": column,
        "missing": int(values.isna().sum()),
        "finite_observations": len(finite_values),
        "minimum": finite_values.min(),
        "Q1": finite_values.quantile(0.25),
        "median": finite_values.median(),
        "mean": finite_values.mean(),
        "Q3": finite_values.quantile(0.75),
        "95th_percentile": finite_values.quantile(0.95),
        "maximum": finite_values.max(),
        "exact_zeros": int(finite_values.eq(0).sum()),
        "zero_pct": (
            finite_values.eq(0).mean() * 100
            if not finite_values.empty else np.nan
        ),
        "skewness": finite_values.skew(),
    })

display(
    pd.DataFrame(history_activity_results).set_index("variable")
)

In [ ]:
def plot_history_activity(column, integer_bins=False):
    """Show a full-range histogram and boxplot for one history variable."""

    # Preserve original values and select finite observations for plotting.
    values = train.loc[
        train[column].notna() & np.isfinite(train[column]),
        column,
    ]

    if values.empty:
        print(f"No finite observations available for {column}.")
        return

    colours = sns.color_palette("colorblind")
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Use integer-centred bins for purchase counts.
    # Wider bins keep the day-based tenure distribution readable.
    bins = (
        np.arange(values.min() - 0.5, values.max() + 1.5, 1)
        if integer_bins
        else 40
    )

    sns.histplot(
        x=values,
        bins=bins,
        color=colours[0],
        ax=axes[0],
    )

    axes[0].axvline(
        values.mean(),
        color=colours[1],
        linestyle="--",
        label=f"Mean: {values.mean():,.2f}",
    )
    axes[0].axvline(
        values.median(),
        color=colours[2],
        linestyle=":",
        label=f"Median: {values.median():,.2f}",
    )

    axes[0].set_title("Full-range distribution")
    axes[0].set_xlabel(column)
    axes[0].set_ylabel("Observations")
    axes[0].legend()

    # Retain extreme observations without interpreting them as errors.
    sns.boxplot(
        x=values,
        whis=1.5,
        showfliers=True,
        color=colours[0],
        ax=axes[1],
    )

    axes[1].set_title("Boxplot — 1.5 × IQR whiskers")
    axes[1].set_xlabel(column)

    fig.suptitle(f"{column} — training distribution", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# Examine the number of purchases preceding each current purchase.
plot_history_activity("prior_purchase_count", integer_bins=True)

#### Interpretation of Prior Purchase Counts

`prior_purchase_count` is strongly right-skewed, with skewness of 3.54.
The median is 11 previous purchases, compared with a mean of 23.46.

The middle 50% of observations have between 3 and 27 prior purchases.
The 95th percentile is 88, while the maximum reaches 276, showing
substantial variation in recorded purchase history.

There are 603 observations with zero prior purchases (9.32% of finite
values). These indicate no recorded purchase history before the current
purchase, which does not necessarily mean no earlier activity outside
the available records.

The upper IQR fence is 63 purchases. Values above this threshold appear
as boxplot outliers, but extensive purchase history is not inherently
invalid. The temporal checks found that prior counts increase
consistently across successive assessed observations.

These statistics describe purchase occasions rather than unique
customers. Multiple observations from active customers can contribute
to the long upper tail.

In [ ]:
# Examine customer tenure in days at each purchase observation.
plot_history_activity("customer_tenure_days")

#### Interpretation — Customer Tenure

`customer_tenure_days` has a median of 918 days and a mean of
1,111.38 days. The distribution is moderately right-skewed, with
a skewness of 0.63.

The middle 50% of observations have tenure between 308 and 1,759 days.
The maximum is 3,506 days, showing substantial variation in recorded
customer history.

There are 603 observations with zero tenure, representing 9.32% of
non-missing values. These contribute to the concentration near zero
in the histogram. The comparison below examines their relationship
with the absence of prior purchases.

The boxplot shows no observations beyond the 1.5 × IQR fences.
This does not establish that every value is correct, but the earlier
temporal checks also found no inconsistencies among assessed observations.

These statistics describe purchase observations. Customers with more
observations contribute more often to the distribution.

In [ ]:
# Compare zero tenure with zero prior purchases on rows where both
# variables are observed, finite and non-negative.
count = train["prior_purchase_count"]
tenure = train["customer_tenure_days"]

assessed = (
    count.notna()
    & tenure.notna()
    & np.isfinite(count)
    & np.isfinite(tenure)
    & count.ge(0)
    & count.eq(np.floor(count))
    & tenure.ge(0)
)

# Counts in each combination show whether the two zero patterns coincide.
zero_history_comparison = pd.crosstab(
    count.loc[assessed].eq(0).rename("No prior purchases"),
    tenure.loc[assessed].eq(0).rename("Zero tenure"),
).reindex(index=[False, True], columns=[False, True], fill_value=0)

print(f"Rows assessed: {assessed.sum():,}")
print(f"Rows excluded: {(~assessed).sum():,}")
display(zero_history_comparison)

#### Interpretation — Zero Tenure and Purchase History

All 603 observations with no recorded prior purchases also have zero
customer tenure. The remaining 5,866 assessed observations have both
positive tenure and at least one prior purchase. No mismatches were found.

This supports interpreting zero tenure as the start of the recorded
purchase history. It does not establish when the customer first registered
or whether purchases occurred outside the available records.

The 65 observations with missing purchase information were excluded
from this comparison.

### 7.6. Purchase Counts Across Historical Windows

These variables describe recorded purchase activity over the previous
30, 90, 180 and 365 days.

We examine their distributions, the proportion of observations with no
purchases and the concentration of activity among frequent purchasers.

The earlier consistency checks confirmed that purchase counts do not
decrease as the window expands and do not exceed the total prior purchase
count among assessed observations.

The windows overlap, so their counts should not be added together.

In [ ]:
# Keep the windows in chronological order for comparisons.
window_count_features = [
    "purchase_count_30d",
    "purchase_count_90d",
    "purchase_count_180d",
    "purchase_count_365d",
]

window_count_results = []

for column in window_count_features:
    # Assess observed, finite values without changing the original data.
    values = train.loc[
        train[column].notna() & np.isfinite(train[column]), column
    ]

    window_count_results.append({
        "variable": column,
        "missing": train[column].isna().sum(),
        "finite_observations": len(values),
        "minimum": values.min(),
        "Q1": values.quantile(0.25),
        "median": values.median(),
        "mean": values.mean(),
        "Q3": values.quantile(0.75),
        "95th_percentile": values.quantile(0.95),
        "maximum": values.max(),
        "exact_zeros": values.eq(0).sum(),
        "zero_pct": values.eq(0).mean() * 100,
        "skewness": values.skew(),
    })

window_count_summary = (
    pd.DataFrame(window_count_results)
    .set_index("variable")
)

display(window_count_summary)

In [ ]:
def plot_window_purchase_count(column):
    """Show the distribution and zero share for one purchase-count window."""

    # Use finite observations; retain zeros in the histogram and boxplot.
    values = train.loc[
        train[column].notna() & np.isfinite(train[column]), column
    ]

    if values.empty:
        print(f"No finite observations available for {column}.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Centre histogram bins on integers because purchases are counted.
    bins = np.arange(values.min() - 0.5, values.max() + 1.5, 1)

    sns.histplot(x=values, bins=bins, ax=axes[0], color="steelblue")
    axes[0].axvline(
        values.mean(),
        color="darkorange",
        linestyle="--",
        label=f"Mean: {values.mean():.2f}",
    )
    axes[0].axvline(
        values.median(),
        color="seagreen",
        linestyle=":",
        label=f"Median: {values.median():.2f}",
    )
    axes[0].set_title("Full-range distribution")
    axes[0].set_xlabel("Number of purchases")
    axes[0].set_ylabel("Observations")
    axes[0].legend()

    # Display all values, including points beyond the standard IQR whiskers.
    sns.boxplot(
        x=values,
        ax=axes[1],
        color="steelblue",
        whis=1.5,
        showfliers=True,
    )
    axes[1].set_title("Boxplot — 1.5 × IQR whiskers")
    axes[1].set_xlabel("Number of purchases")

    # Separate absence of recorded activity from positive purchase counts.
    activity_counts = pd.Series({
        "Zero purchases": values.eq(0).sum(),
        "At least one purchase": values.gt(0).sum(),
    })

    axes[2].bar(
        activity_counts.index,
        activity_counts.values,
        color=["lightgrey", "steelblue"],
    )

    for position, count in enumerate(activity_counts):
        axes[2].text(
            position,
            count,
            f"{count:,}\n({count / len(values) * 100:.2f}%)",
            ha="center",
            va="bottom",
        )

    axes[2].set_ylim(0, max(activity_counts.max() * 1.2, 1))
    axes[2].set_title("Recorded activity")
    axes[2].set_ylabel("Observations")

    fig.suptitle(f"{column} — training distribution", fontsize=15)
    plt.tight_layout()
    plt.show()

In [ ]:
# Explore the shortest window first.
plot_window_purchase_count("purchase_count_30d")

#### Interpretation — Purchases in the Previous 30 Days

Most observations have no recorded purchases in the previous 30 days:
4,342 observations (67.12%). The remaining 2,127 (32.88%) have at
least one purchase.

The median is zero and the mean is 0.60 purchases. The distribution
is strongly right-skewed, with a skewness of 3.31. The third quartile
is one purchase and the 95th percentile is three, while the maximum
reaches 11.

With Q1 = 0 and Q3 = 1, the upper IQR fence is 2.5. The boxplot therefore
flags counts of three or more. These represent relatively frequent
purchasing and are not necessarily errors; the earlier count checks
found no invalid counts or inconsistencies between historical windows.

Zero purchases in this window indicates no recorded recent activity.
It does not necessarily mean that the customer has no purchase history.
The 65 observations with missing counts are excluded from these statistics.

In [ ]:
# Examine activity over the previous 90 days.
plot_window_purchase_count("purchase_count_90d")

#### Interpretation — Purchases in the Previous 90 Days

There are 3,580 observations (55.34%) with at least one recorded purchase
in the previous 90 days, while 2,889 (44.66%) have none. The share with
recorded activity is higher than in the 30-day window (32.88%), as expected
for a longer, overlapping period.

The median is one purchase and the mean is 1.63. The distribution remains
strongly right-skewed, with a skewness of 3.34. The middle 50% of counts
lie between zero and two purchases, while the 95th percentile is eight
and the maximum is 22.

The upper IQR fence is five purchases, so counts of six or more appear
beyond the boxplot whisker. These indicate unusually frequent purchasing
relative to the distribution and do not, by themselves, establish errors.

This variable describes historical activity in the previous 90 days.
The target describes repeat purchasing in the following 90 days; sharing
the same window length does not make them equivalent.

In [ ]:
# Examine activity over the previous 180 days.
plot_window_purchase_count("purchase_count_180d")

#### Interpretation — Purchases in the Previous 180 Days

Most observations have recorded activity in the previous 180 days:
5,059 (78.20%) have at least one purchase, while 1,410 (21.80%) have none.

The median is two purchases and the mean is 3.37. The middle 50% of
counts lie between one and three purchases. The distribution remains
strongly right-skewed, with a skewness of 3.22, a 95th percentile of
16 purchases and a maximum of 37.

The upper IQR fence is six purchases, so counts of seven or more
appear beyond the boxplot whisker. The long tail reflects a smaller
number of observations with much higher purchase frequency; these
counts should not be treated as errors solely because they are unusual.

The share with recorded activity increases from 55.34% in the 90-day
window to 78.20% in the 180-day window. This reflects the longer period
covered and does not, by itself, indicate increasing activity over time.

In [ ]:
# Examine activity over the previous 365 days.
plot_window_purchase_count("purchase_count_365d")

#### Interpretation — Purchases in the Previous 365 Days

There are 5,626 observations (86.97%) with at least one recorded purchase
in the previous 365 days, while 843 (13.03%) have none.

The median is three purchases and the mean is 6.56. The middle 50% of
counts lie between one and seven purchases. The distribution is strongly
right-skewed, with a skewness of 3.20, a 95th percentile of 31 purchases
and a maximum of 60.

The upper IQR fence is 16 purchases, so counts of 17 or more appear
beyond the boxplot whisker. These observations represent comparatively
frequent purchasing and are not necessarily invalid.

Small concentrations are visible in the upper tail. Repeated observations
from frequent customers may contribute to these patterns, but this needs
customer-level investigation before drawing conclusions.

#### Comparison Across Historical Windows

The proportion of observations with no recorded purchases decreases
from 67.12% over 30 days to 44.66% over 90 days, 21.80% over 180 days
and 13.03% over 365 days.

Median purchase counts increase from zero to one, two and three,
respectively. All four distributions have long right tails, showing
substantial differences in recorded purchase frequency.

These are overlapping windows measured at each purchase observation.
Higher counts in longer windows do not establish a trend over time,
and customers with short recorded histories may not have been observed
for the full window.

All four variables have 6,469 observed values and the same 65 missing
observations. Their relationships with the target will be explored
in the later bivariate analysis.

### 7.7. Historical Purchase Values and Spending

We examine total prior spending, the mean, variability and range of
previous purchase values, and spending over the historical windows.

The earlier checks identified inconsistencies in some historical values,
including negative amounts and extremely large entries. These remain
in the initial descriptive statistics so their influence is visible.

Any additional plots that exclude flagged values will state the exclusion
rule and the number of observations excluded. The original data will
remain unchanged.

In [ ]:
# Include purchase-value statistics and spending across all four windows.
historical_spending_features = [
    "prior_total_spend",
    "prior_mean_purchase_value",
    "prior_std_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
    "spend_30d",
    "spend_90d",
    "spend_180d",
    "spend_365d",
]

historical_spending_results = []

for column in historical_spending_features:
    # Retain all finite values, including negative and suspicious amounts.
    # This is a descriptive overview, not a cleaning step.
    values = train.loc[
        train[column].notna() & np.isfinite(train[column]), column
    ]

    historical_spending_results.append({
        "variable": column,
        "missing": train[column].isna().sum(),
        "finite_observations": len(values),
        "minimum": values.min(),
        "Q1": values.quantile(0.25),
        "median": values.median(),
        "mean": values.mean(),
        "Q3": values.quantile(0.75),
        "95th_percentile": values.quantile(0.95),
        "99th_percentile": values.quantile(0.99),
        "maximum": values.max(),
        "exact_zeros": values.eq(0).sum(),
        "zero_pct": values.eq(0).mean() * 100,
        "negative_values": values.lt(0).sum(),
        "skewness": values.skew(),
    })

historical_spending_summary = (
    pd.DataFrame(historical_spending_results)
    .set_index("variable")
)

display(historical_spending_summary)

#### Total Prior Spending

`prior_total_spend` describes accumulated spending before the current
purchase. Its distribution reflects differences in both purchase
frequency and purchase value.

We display the full distribution, a boxplot and a logarithmic view.
The logarithmic view uses `log1p`, which retains zero spending and
compresses large amounts to make the distribution easier to inspect.

In [ ]:
# Display total prior spending using the existing three-panel function.
plot_amount_distribution("prior_total_spend")

#### Interpretation — Total Prior Spending

`prior_total_spend` is strongly right-skewed. The mean is 18,908.35,
compared with a median of 6,754.31. A relatively small number of high
spending observations pull the mean upwards.

The boxplot shows many values beyond the upper IQR whisker. Large
accumulated spending may reflect frequent purchases, high purchase
values or a longer recorded history, so these values require context
rather than automatic removal.

The logarithmic view makes the positive spending distribution easier
to inspect. It also shows a separate concentration at zero, corresponding
to the 603 observations with no recorded prior purchases identified
in the earlier consistency checks.

The positive values appear more balanced on the logarithmic scale,
but this plot alone does not establish a normal distribution.

#### Mean Previous Purchase Value

`prior_mean_purchase_value` describes the average value of previous
purchases. Earlier checks found 78 observed means that differed from
total prior spending divided by the prior purchase count.

We first examine the supplied values, including negative amounts and
extreme entries. These may dominate the plot and obscure the main
distribution.

The full-range histogram and boxplot retain all finite values.
The `log1p` panel includes only non-negative finite values; the number
of negative values excluded from that panel is reported below.

In [ ]:
# Report coverage explicitly because negative values cannot be included
# in the non-negative log1p view used by this plotting function.
column = "prior_mean_purchase_value"

finite_values = train.loc[
    train[column].notna() & np.isfinite(train[column]), column
]

print(f"Missing values: {train[column].isna().sum():,}")
print(f"Finite values in the full-range plots: {len(finite_values):,}")
print(
    "Negative values excluded from the log1p panel: "
    f"{finite_values.lt(0).sum():,}"
)

# Preserve the supplied values; no corrections or clipping are applied.
plot_amount_distribution(column)

#### Interpretation — Supplied Mean Previous Purchase Values

The supplied values have a mean of 2,821,825.32 and a median of 724.90.
Entries reaching 1,000,000,000 dominate the arithmetic mean and make
the full-range histogram and boxplot difficult to interpret.

The logarithmic panel reveals the main concentration of positive values,
while the extreme entries remain separated on the right. The 16 negative
values are excluded from this panel but retained in the full-range plots.

Earlier consistency checks identified 78 observed means that differed
from total prior spending divided by the prior purchase count. All
16 negative means matched a sign reversal of that calculation.

We therefore examine an additional diagnostic view using observed means
that pass the arithmetic consistency check. This view does not correct
the data or establish that the underlying spending and count are accurate.

#### Diagnostic View — Arithmetically Consistent Means

The following plots retain finite, non-negative observed means that
pass the earlier comparison with total prior spending divided by
prior purchase count.

Missing means are not reconstructed. Values excluded from this view
remain in the original dataset and in the preceding analysis.

In [ ]:
# Reuse the earlier consistency check: True indicates a failed check.
mean_check_failed = historical_consistency_flags[
    "Mean equals total spending / purchase count"
]

observed_mean = train["prior_mean_purchase_value"]

# Require an assessable expected mean as well as an observed finite value.
finite_observed_mean = observed_mean.notna() & np.isfinite(observed_mean)

consistent_mean_mask = (
    finite_observed_mean
    & expected_prior_mean.notna()
    & np.isfinite(expected_prior_mean)
    & observed_mean.ge(0)
    & ~mean_check_failed
)

consistent_mean_values = observed_mean.loc[consistent_mean_mask]

print(f"Finite observed means: {finite_observed_mean.sum():,}")
print(f"Means retained in diagnostic view: {len(consistent_mean_values):,}")
print(
    "Finite observed means excluded from diagnostic view: "
    f"{(finite_observed_mean & ~consistent_mean_mask).sum():,}"
)

# Summarise the retained values so the interpretation uses this subset.
display(
    consistent_mean_values.describe(
        percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
    ).to_frame("arithmetically_consistent_mean")
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Show the complete range of retained values without clipping.
sns.histplot(x=consistent_mean_values, bins=40, ax=axes[0])
axes[0].axvline(
    consistent_mean_values.mean(),
    color="darkorange",
    linestyle="--",
    label=f"Mean: {consistent_mean_values.mean():,.2f}",
)
axes[0].axvline(
    consistent_mean_values.median(),
    color="seagreen",
    linestyle=":",
    label=f"Median: {consistent_mean_values.median():,.2f}",
)
axes[0].set_title("Diagnostic distribution")
axes[0].set_xlabel("Mean previous purchase value")
axes[0].legend()

# Keep statistical outliers visible within the diagnostic subset.
sns.boxplot(x=consistent_mean_values, whis=1.5, ax=axes[1])
axes[1].set_title("Boxplot — 1.5 × IQR whiskers")
axes[1].set_xlabel("Mean previous purchase value")

# Compress the scale without changing or replacing the original values.
sns.histplot(x=np.log1p(consistent_mean_values), bins=40, ax=axes[2])
axes[2].set_title("Log1p diagnostic distribution")
axes[2].set_xlabel("log(1 + mean previous purchase value)")

fig.suptitle(
    "prior_mean_purchase_value — arithmetic check passed",
    fontsize=15,
)
plt.tight_layout()
plt.show()

#### Interpretation — Arithmetically Consistent Means

Among observations retained in this diagnostic view, the mean previous
purchase value is 870.02 and the median is 718.96. The distribution
remains right-skewed, with a smaller number of high purchase averages
extending the upper tail.

The mean is substantially lower than the supplied-data mean of
2,821,825.32, while the median changes more modestly from 724.90
to 718.96. This illustrates how strongly the inconsistent extreme
entries affect the arithmetic mean.

Several values still appear beyond the upper boxplot whisker.
Passing the arithmetic check does not make these values ordinary,
but their size alone does not establish an error.

The logarithmic view reveals concentrations within the main distribution.
It does not establish normality or distinct customer segments.

These findings apply to the retained subset. Passing this check confirms
agreement between the recorded mean, spending and purchase count within
the chosen tolerance; it does not independently verify those inputs.

#### Variability of Previous Purchase Values

`prior_std_purchase_value` measures how much previous purchase values
vary around their historical mean, in the same units as purchase value.

A small standard deviation indicates similar previous purchase amounts,
while a large value indicates greater variation. Its interpretation also
depends on the average purchase value and the number of prior purchases.

The earlier missingness analysis found that this variable is missing
for observations with zero or one prior purchase. These missing values
should not be interpreted as zero variability.

In [ ]:
# Inspect the supplied historical standard deviations without modification.
plot_amount_distribution("prior_std_purchase_value")

#### Interpretation — Variability of Previous Purchase Values

`prior_std_purchase_value` has 5,421 observed values and 1,113 missing
values (17.03%). The earlier analysis linked this missingness to
insufficient purchase history.

The median standard deviation is 569.08 and the mean is 737.47.
The middle 50% of observed values lie between 258.56 and 981.09.
The distribution is right-skewed, with a skewness of 2.07.

The 95th percentile is 1,979.62 and the maximum reaches 6,770.22.
These larger values indicate substantial variation in previous purchase
amounts. They require interpretation alongside the historical mean,
purchase count and range rather than automatic removal.

Two observations have a standard deviation of zero, indicating no
recorded variation in their previous purchase values, subject to the
precision of the supplied statistic. This differs from a missing value,
where variability is unavailable.

The logarithmic view makes the main distribution easier to inspect.
It does not establish normality or verify the recorded statistics.

#### Minimum Previous Purchase Value

`prior_min_purchase_value` records the smallest purchase value in
the available history before the current purchase. It is a purchase
amount, not a count or a measure of time.

We examine its distribution to understand the lower end of customers'
recorded purchase values. Its interpretation depends on the amount of
available history: customers with more prior purchases have more
opportunities to record a small purchase.

In [ ]:
# Explore the smallest recorded previous purchase value.
plot_amount_distribution("prior_min_purchase_value")

#### Interpretation — Minimum Previous Purchase Value

`prior_min_purchase_value` has a median of 64.34 and a mean of 185.46.
The distribution is strongly right-skewed, with a skewness of 5.91.
The middle 50% of observed minima lie between 15.45 and 162.61.

The smallest recorded minimum is 0.93, while the largest is 6,298.98.
The 95th percentile is 820.70, indicating that some observations have
relatively high values even for their smallest previous purchase.

The upper IQR fence is approximately 383.35. Values above this appear
beyond the boxplot whisker, but a high historical minimum can be plausible,
particularly for customers with few previous purchases or consistently
large purchase amounts.

The logarithmic view reveals several concentrations that are difficult
to see on the original scale. Repeated observations from the same
customers may contribute to these patterns; they do not establish
distinct customer groups.

There are 668 missing values: the 603 observations with no prior
purchases and the 65 observations missing all purchase information.
None of the observed minima are zero or negative.

#### Maximum Previous Purchase Value

`prior_max_purchase_value` records the largest purchase value in
the available history before the current purchase.

We examine the upper end of recorded purchase values. A large maximum
may represent a genuine large purchase, but it should also be assessed
against total prior spending and the other historical statistics.

Earlier checks found no minimum exceeding its corresponding maximum.
This alone does not establish that the maxima are accurate.

In [ ]:
# Explore the largest recorded previous purchase value.
plot_amount_distribution("prior_max_purchase_value")

#### Interpretation — Maximum Previous Purchase Value

`prior_max_purchase_value` has a median of 1,783.24 and a mean of
2,946.38. The middle 50% of observed maxima lie between 834.53
and 3,504.42.

The distribution has an extreme right tail, with a skewness of 21.34.
The 99th percentile is 14,433.41, while the largest recorded value
reaches 267,526.19. This extreme value makes the main distribution
difficult to inspect on the original scale.

The logarithmic view reveals the main concentration more clearly,
while retaining the unusually large positive values.

There are 668 missing values and no observed zero or negative maximum
purchase values. However, these checks alone do not establish accuracy.
For three observations with exactly one prior purchase, the recorded
maximum purchase value differed from total prior spending, although
the two should describe the same purchase amount.

#### Additional Check — Historical Maximum and Total Spending

If total prior spending is the sum of the same non-negative purchase
amounts used to calculate the historical maximum, the maximum should
not exceed that total.

We check this relationship with a numerical tolerance. Differences
are flagged for investigation because the rule depends on the variables
sharing the same purchase history and amount definition.

In [ ]:
# Identify observations with a valid positive prior purchase count.
prior_count = train["prior_purchase_count"]
prior_maximum = train["prior_max_purchase_value"]
prior_total = train["prior_total_spend"]

eligible_max_total = (
    prior_count.notna()
    & np.isfinite(prior_count)
    & prior_count.gt(0)
    & prior_count.eq(np.floor(prior_count))
)

# Assess only finite, non-negative maximum and total spending values.
assessed_max_total = (
    eligible_max_total
    & prior_maximum.notna()
    & prior_total.notna()
    & np.isfinite(prior_maximum)
    & np.isfinite(prior_total)
    & prior_maximum.ge(0)
    & prior_total.ge(0)
)

# Allow small numerical differences when comparing monetary values.
max_total_close = pd.Series(
    np.isclose(prior_maximum, prior_total, rtol=1e-5, atol=0.01),
    index=train.index,
)

max_exceeds_total = (
    assessed_max_total
    & prior_maximum.gt(prior_total)
    & ~max_total_close
)

max_total_summary = pd.Series({
    "Eligible observations": eligible_max_total.sum(),
    "Assessed observations": assessed_max_total.sum(),
    "Unassessable eligible observations": (
        eligible_max_total & ~assessed_max_total
    ).sum(),
    "Maximum exceeds total spending": max_exceeds_total.sum(),
}).to_frame("count")

display(max_total_summary)

# Inspect the largest differences without modifying any supplied values.
max_total_records = train.loc[
    max_exceeds_total,
    [
        "ID",
        "customer_id",
        "prior_purchase_count",
        "prior_total_spend",
        "prior_min_purchase_value",
        "prior_mean_purchase_value",
        "prior_max_purchase_value",
    ],
].copy()

max_total_records["maximum_minus_total"] = (
    max_total_records["prior_max_purchase_value"]
    - max_total_records["prior_total_spend"]
)

display(
    max_total_records.sort_values(
        "maximum_minus_total", ascending=False
    ).head(20)
)

#### Interpretation — Historical Maximum and Total Spending

For 45 observations (0.77% of the 5,866 assessed), the recorded maximum
previous purchase value exceeds total prior spending beyond the chosen
numerical tolerance.

The largest discrepancy occurs for ID 10003019: the historical maximum
is 267,526.19, while total prior spending is 131,548.00. The difference
is 135,978.19. This is also the largest maximum observed in the
distribution analysis.

Under the assumption that both variables summarise the same non-negative
purchase amounts, these records are inconsistent. The differences are
too large to be explained by rounding alone.

Some displayed records have a historical mean consistent with spending
divided by purchase count, despite their inconsistent maximum. This
shows why checking several relationships is useful.

The remaining observations pass this particular check, but passing does
not independently establish accuracy. No values have been changed.

#### Spending in the Previous 30 Days

`spend_30d` describes recorded spending over the previous 30 days.

We examine the concentration at zero and the distribution of positive
amounts. Earlier checks found no mismatches between zero purchase counts
and non-zero spending in this window.

In [ ]:
# Retain zero spending and all finite positive amounts in the plots.
plot_amount_distribution("spend_30d")

#### Interpretation — Spending in the Previous 30 Days

There are 4,342 observations with zero recorded spending, representing
67.12% of the 6,469 observed values. Both the first quartile and median
are zero, while the third quartile is 338.24.

The mean is 457.14. The distribution is strongly right-skewed, with
a skewness of 5.50, a 95th percentile of 2,528.62 and a maximum
of 22,297.21.

The upper IQR fence is 845.60. Values above this appear beyond the
boxplot whisker, but large spending may result from frequent purchases,
large individual purchases or both. These flags alone do not establish
errors.

The logarithmic view separates the concentration at zero from positive
spending. Zero spending describes the absence of recorded spending
in this window; it is different from the 65 missing observations.

Because zero values dominate the distribution, the overall median
does not describe typical spending among observations with recent
purchase activity.

#### Spending in the Previous 90 Days

`spend_90d` describes recorded spending over the previous 90 days.

This window includes the 30-day period. Earlier checks found no
observations where 30-day spending exceeded 90-day spending among
assessed rows.

In [ ]:
# Examine the full distribution of spending over the previous 90 days.
plot_amount_distribution("spend_90d")

#### Interpretation — Spending in the Previous 90 Days

`spend_90d` has a median of 198.50 and a mean of 1,253.53.
There are 2,889 observations with zero spending, representing 44.66%
of the 6,469 observed values.

The distribution is strongly right-skewed, with a skewness of 3.72.
The first quartile is zero and the third quartile is 1,514.56.
The 95th percentile reaches 5,828.66, while the maximum is 30,234.88.

The upper IQR fence is 3,786.40. Spending above this appears beyond
the boxplot whisker, but its plausibility depends on purchase frequency
and individual purchase amounts.

The logarithmic view shows a concentration at zero and a separate
distribution of positive spending. Compared with the 30-day window,
fewer observations have zero spending, as expected when a longer
historical period is included.

These differences between overlapping windows do not establish growth
over time. The 65 missing observations are excluded from the plots
and descriptive statistics.

In [ ]:
# Examine spending over the previous 180 days, retaining zero values.
plot_amount_distribution("spend_180d")

#### Interpretation — Spending in the Previous 180 Days

`spend_180d` has a median of 1,117.68 and a mean of 2,650.60.
The middle 50% of observed values lie between 149.16 and 3,404.44.

There are 1,410 observations with zero spending, representing 21.80%
of the 6,469 observed values. The concentration at zero is smaller
than in the shorter historical windows.

The distribution remains strongly right-skewed, with a skewness of 3.15.
The 95th percentile is 10,479.21 and the maximum reaches 49,746.97.

The upper IQR fence is 8,287.36. Values above this appear beyond the
boxplot whisker and require context from purchase frequency and amounts,
rather than automatic removal.

The logarithmic view makes the positive spending distribution easier
to inspect while retaining the separate concentration at zero.
The 65 missing observations are excluded from these statistics.

#### Spending in the Previous 365 Days

`spend_365d` describes recorded spending over the previous 365 days.

Earlier checks identified 15 negative values and 54 observations where
annual spending exceeded total prior spending. Some entries reach
1,000,000,000, so the supplied values may obscure the main distribution.

We first plot the supplied data. The full-range histogram and boxplot
retain all finite values. The logarithmic panel excludes negative values,
with the exclusion count reported explicitly.

In [ ]:
# Report coverage before plotting the supplied annual spending values.
column = "spend_365d"

finite_annual_spending = train.loc[
    train[column].notna() & np.isfinite(train[column]), column
]

print(f"Missing values: {train[column].isna().sum():,}")
print(f"Finite values in the full-range plots: {len(finite_annual_spending):,}")
print(
    "Negative values excluded from the log1p panel: "
    f"{finite_annual_spending.lt(0).sum():,}"
)

# Keep suspicious values visible; no correction or clipping is applied.
plot_amount_distribution(column)

#### Interpretation — Supplied Annual Spending

`spend_365d` has 6,274 observed values and 260 missing values.
Its mean is 2,396,534.88, compared with a median of 2,426.61.

The middle 50% of observed values lie between 637.78 and 7,045.42.
However, entries reaching 1,000,000,000 dominate the arithmetic mean
and compress the main distribution in the histogram and boxplot.

There are 15 negative values, with a minimum of −159,115.18.
These remain in the original-scale plots but are excluded from
the logarithmic panel. There are also 810 exact zeros.

The logarithmic view reveals the main concentration of positive spending,
a separate concentration at zero and the extreme entries on the right.

Earlier spending-order checks flagged 69 distinct observations:
15 where annual spending was below 180-day spending and 54 where
it exceeded total prior spending. These inconsistencies require
attention before using this variable for modelling.

#### Diagnostic View — Annual Spending Passing Consistency Checks

To inspect the main distribution, we retain observed, non-negative
annual spending for which the related values are assessable and
the earlier spending-order and zero-count checks pass.

No values are corrected or reconstructed. Passing these checks
does not establish accuracy, and this subset may differ from the
full dataset.

In [ ]:
# Require finite related values so unassessable records cannot be
# mistaken for records that passed the consistency checks.
annual_check_columns = [
    "spend_180d",
    "spend_365d",
    "prior_total_spend",
    "purchase_count_365d",
]

annual_check_values = train[annual_check_columns]

annual_checks_assessable = (
    annual_check_values.notna().all(axis=1)
    & np.isfinite(annual_check_values).all(axis=1)
    & annual_check_values.ge(0).all(axis=1)
    & train["purchase_count_365d"].eq(
        np.floor(train["purchase_count_365d"])
    )
)

# Reuse the flags from the earlier checks; True means an inconsistency.
annual_diagnostic_mask = (
    annual_checks_assessable
    & ~any_window_spending_flag
    & ~any_zero_count_flag
)

annual_diagnostic_values = train.loc[
    annual_diagnostic_mask, "spend_365d"
]

annual_finite_mask = (
    train["spend_365d"].notna()
    & np.isfinite(train["spend_365d"])
)

print(f"Finite supplied annual values: {annual_finite_mask.sum():,}")
print(f"Values retained: {len(annual_diagnostic_values):,}")
print(
    "Finite values excluded from diagnostic view: "
    f"{(annual_finite_mask & ~annual_diagnostic_mask).sum():,}"
)

# Report statistics for this subset separately from the supplied data.
display(
    annual_diagnostic_values.describe(
        percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
    ).to_frame("annual_spending_diagnostic")
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Retain the full range of values passing the selected checks.
sns.histplot(x=annual_diagnostic_values, bins=40, ax=axes[0])
axes[0].axvline(
    annual_diagnostic_values.mean(),
    color="darkorange",
    linestyle="--",
    label=f"Mean: {annual_diagnostic_values.mean():,.2f}",
)
axes[0].axvline(
    annual_diagnostic_values.median(),
    color="seagreen",
    linestyle=":",
    label=f"Median: {annual_diagnostic_values.median():,.2f}",
)
axes[0].set_title("Diagnostic distribution")
axes[0].set_xlabel("Annual spending")
axes[0].legend()

# Keep observations beyond the IQR whiskers visible.
sns.boxplot(x=annual_diagnostic_values, whis=1.5, ax=axes[1])
axes[1].set_title("Boxplot — 1.5 × IQR whiskers")
axes[1].set_xlabel("Annual spending")

# Include zero spending in the logarithmic view.
sns.histplot(x=np.log1p(annual_diagnostic_values), bins=40, ax=axes[2])
axes[2].set_title("Log1p diagnostic distribution")
axes[2].set_xlabel("log(1 + annual spending)")

fig.suptitle("spend_365d — selected consistency checks passed", fontsize=15)
plt.tight_layout()
plt.show()

#### Interpretation — Annual Spending Passing Consistency Checks

The diagnostic view retains 6,205 observations and excludes 69 of
the 6,274 finite supplied values (1.10%) because they fail the selected
consistency checks.

Among retained observations, mean annual spending is 5,214.50 and
the median is 2,386.22. The middle 50% lie between 637.65 and 6,871.41.
The 95th percentile is 20,066.85 and the maximum is 75,013.40.

Compared with the supplied-data mean of 2,396,534.88, the diagnostic
mean is substantially lower. The median changes more modestly,
from 2,426.61 to 2,386.22. This illustrates the strong influence
of the inconsistent extreme values on the arithmetic mean.

The retained distribution remains right-skewed. Large values still
appear beyond the boxplot whisker, showing that statistical extremes
remain even after excluding records that fail these checks.

The logarithmic view reveals a concentration at zero and the main
distribution of positive spending. Passing the selected checks does
not independently establish accuracy, and no original values have
been modified.

#### Findings from Historical Spending Analysis

Historical spending and purchase-value statistics generally have
right-skewed distributions. Medians and quartiles provide useful
descriptions alongside means and extreme values.

Zero spending is common in shorter historical windows and should
be distinguished from missing information.

Consistency checks identified problems that distribution plots alone
could not explain: incorrect historical means, maximum purchase values
exceeding total spending and inconsistent annual spending.

Diagnostic views made the main distributions easier to inspect without
changing the supplied data. Decisions about correcting or excluding
values will be documented in the preprocessing notebook.

### 7.8. Time Between Purchases

We examine three variables measured in days:

- `days_since_previous_purchase`: time between the previous purchase
  and the current purchase.
- `prior_mean_gap_days`: average interval between previous purchases.
- `prior_std_gap_days`: variability of the intervals between previous
  purchases.

These variables describe different aspects of purchasing behaviour.
The latest interval may differ from the customer's historical average,
while the standard deviation describes how regular those earlier
intervals were.

Earlier checks identified 16 negative values in each of the first two
variables, all equal to −4. We retain these in the initial summary.
Missing values are considered separately because some arise from
insufficient purchase history.

In [ ]:
# Summarise the supplied timing variables, including negative values.
purchase_gap_features = [
    "days_since_previous_purchase",
    "prior_mean_gap_days",
    "prior_std_gap_days",
]

purchase_gap_results = []

for column in purchase_gap_features:
    values = train.loc[
        train[column].notna() & np.isfinite(train[column]), column
    ]

    purchase_gap_results.append({
        "variable": column,
        "missing": train[column].isna().sum(),
        "missing_pct": train[column].isna().mean() * 100,
        "finite_observations": len(values),
        "minimum": values.min(),
        "Q1": values.quantile(0.25),
        "median": values.median(),
        "mean": values.mean(),
        "Q3": values.quantile(0.75),
        "95th_percentile": values.quantile(0.95),
        "99th_percentile": values.quantile(0.99),
        "maximum": values.max(),
        "exact_zeros": values.eq(0).sum(),
        "negative_values": values.lt(0).sum(),
        "skewness": values.skew(),
    })

purchase_gap_summary = (
    pd.DataFrame(purchase_gap_results)
    .set_index("variable")
)

display(purchase_gap_summary)

#### Time Since the Previous Purchase

The following diagnostic plots exclude missing, infinite and negative
values. Negative elapsed times conflict with the intended interpretation
of this variable and were already inspected in the data quality section.

All remaining values are displayed without clipping. Long intervals
are retained because they may represent genuine gaps between purchases.
The original dataset remains unchanged.

In [ ]:
def plot_purchase_gap(column):
    """Plot non-negative finite timing values and report exclusions."""

    supplied = train[column]
    finite_mask = supplied.notna() & np.isfinite(supplied)
    retained_mask = finite_mask & supplied.ge(0)
    values = supplied.loc[retained_mask]

    # Distinguish unavailable values from observed invalid elapsed times.
    print(f"Variable: {column}")
    print(f"Missing values: {supplied.isna().sum():,}")
    print(
        "Infinite values excluded: "
        f"{(supplied.notna() & ~np.isfinite(supplied)).sum():,}"
    )
    print(
        "Negative values excluded: "
        f"{(finite_mask & supplied.lt(0)).sum():,}"
    )
    print(f"Values plotted: {len(values):,}")

    if values.empty:
        print("No non-negative finite values available.")
        return

    # Report statistics for precisely the observations shown in the plots.
    display(
        values.describe(
            percentiles=[0.25, 0.5, 0.75, 0.95, 0.99]
        ).to_frame("diagnostic_values")
    )

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Retain the full range of non-negative elapsed times.
    sns.histplot(x=values, bins=40, ax=axes[0], color="steelblue")
    axes[0].axvline(
        values.mean(),
        color="darkorange",
        linestyle="--",
        label=f"Mean: {values.mean():.2f}",
    )
    axes[0].axvline(
        values.median(),
        color="seagreen",
        linestyle=":",
        label=f"Median: {values.median():.2f}",
    )
    axes[0].set_title("Non-negative values — full range")
    axes[0].set_xlabel("Days")
    axes[0].set_ylabel("Observations")
    axes[0].legend()

    # Show long intervals beyond the standard IQR whiskers.
    sns.boxplot(x=values, whis=1.5, ax=axes[1], color="steelblue")
    axes[1].set_title("Boxplot — 1.5 × IQR whiskers")
    axes[1].set_xlabel("Days")

    # Compress long intervals while retaining any zero values.
    sns.histplot(
        x=np.log1p(values), bins=40, ax=axes[2], color="steelblue"
    )
    axes[2].set_title("Log1p distribution")
    axes[2].set_xlabel("log(1 + days)")
    axes[2].set_ylabel("Observations")

    fig.suptitle(f"{column} — diagnostic distribution", fontsize=15)
    plt.tight_layout()
    plt.show()

In [ ]:
# Start with the interval immediately preceding the current purchase.
plot_purchase_gap("days_since_previous_purchase")

#### Interpretation — Days Since Previous Purchase

After excluding the 16 negative values, the median interval since the
previous purchase is 57 days and the mean is 101.83 days. The middle
50% of assessed observations fall between 17 and 119 days.

The larger mean reflects a long right tail. The 95th percentile is
337.20 days, while the maximum reaches 3,199 days. Some observations
therefore represent purchases following substantially longer periods
without recorded activity.

The upper 1.5 × IQR fence is 272 days. Values above this threshold
are unusual relative to the central distribution, but a long interval
does not by itself establish an error. The earlier temporal checks
found no intervals exceeding customer tenure or the gap between
consecutive observed purchases among assessed records.

There are 861 missing values. The earlier missingness analysis showed
that these include observations without prior purchases, observations
without purchase information and additional missing historical values.

The diagnostic plots retain 5,657 observations. The original data remain
unchanged, and the 16 negative intervals require a separate
preprocessing decision.

#### Mean Interval Between Previous Purchases

`prior_mean_gap_days` describes the average interval between purchases
in the customer's recorded history before the current purchase.

Unlike `days_since_previous_purchase`, which describes the most recent
interval, this variable summarises historical purchase frequency.

We inspect its distribution using finite, non-negative values.
Missing values and negative intervals are reported separately.

In [ ]:
# Examine historical average purchase intervals and report excluded values.
plot_purchase_gap("prior_mean_gap_days")

#### Interpretation — Mean Interval Between Previous Purchases

After excluding the 16 negative values, the mean historical purchase
interval is 96.18 days and the median is 68.61 days. The middle 50%
of assessed observations fall between 42.87 and 114 days.

The distribution is right-skewed. Most observations have relatively
short average intervals, while a smaller number have much longer
historical gaps. The 95th percentile is 263.46 days and the maximum
reaches 3,199 days.

The full-range histogram compresses the central distribution because
of this long tail. The log1p panel makes its structure easier to see,
without changing the original values.

The boxplot identifies unusually large intervals, but these are not
automatically errors. Their interpretation depends on the customer's
purchase history and the number of intervals used to calculate
the average.

There are 1,308 missing values. The earlier missingness checks showed
that this statistic is unavailable for observations with fewer than
two prior purchases, with additional missing values among observations
with sufficient recorded history.

These results describe purchase observations rather than one value
per customer. Customers with more observations contribute more often.

#### Variation in Historical Purchase Intervals

`prior_std_gap_days` measures how much the intervals between previous
purchases vary.

A smaller standard deviation indicates more consistent historical
intervals. A larger value indicates greater variation, although it
does not describe how frequently the customer purchases.

We inspect finite, non-negative values and report any exclusions.

In [ ]:
# Examine variation in historical purchase intervals.
plot_purchase_gap("prior_std_gap_days")

#### Interpretation — Variation in Historical Purchase Intervals

`prior_std_gap_days` has a median of 50.64 days and a mean of
74.14 days. The middle 50% of observed values fall between
32.74 and 78.63 days.

The distribution is right-skewed. The 95th percentile reaches
220.58 days and the maximum is 2,212.54 days, indicating substantial
variation in historical purchase intervals for some observations.

These values measure variation rather than the average time between
purchases. A customer can purchase infrequently but at consistent
intervals, resulting in a large average gap and a small standard
deviation.

The boxplot flags values above approximately 147.46 days using the
1.5 × IQR rule. These deserve inspection, but large variation can
reflect genuine irregular purchasing behaviour.

One observation has a standard deviation of zero. This indicates no
variation in its recorded historical intervals at the supplied
precision; it is different from a missing value.

There are 1,478 missing values, representing 22.62% of training
observations. The earlier checks showed that missingness occurs
when there are fewer than three prior purchases, consistent with
insufficient intervals to calculate a sample standard deviation.

All 5,056 observed values are finite and non-negative. No observed
values were excluded from these plots.

#### Findings from Purchase-Interval Analysis

The most recent purchase interval, historical average interval and
historical interval variability describe different aspects of
customer activity.

All three distributions have long right tails. Medians and quartiles
therefore complement the means when describing typical observations.

Missing historical statistics must be interpreted alongside the
number of prior purchases. An unavailable statistic does not mean
that its value is zero.

The negative intervals identified earlier require treatment during
preprocessing. Large positive intervals require contextual checks
rather than automatic removal.

### 7.9. Random Noise

`random_noise` is a numerical variable whose name suggests that it
contains generated noise. We inspect its observed distribution
without assuming that it is independent of the target.

The summary statistics describe its centre, spread and extremes.
A histogram and boxplot show the distribution, while a normal Q–Q
plot compares its shape with a normal distribution.

Negative values are allowed for this variable and are retained.

In [ ]:
# Select observed finite values without modifying the training data.
noise_supplied = train["random_noise"]
noise_finite_mask = noise_supplied.notna() & np.isfinite(noise_supplied)
noise_values = noise_supplied.loc[noise_finite_mask]

# Report missing and infinite values separately.
print(f"Missing values: {noise_supplied.isna().sum():,}")
print(
    "Infinite values excluded: "
    f"{(noise_supplied.notna() & ~np.isfinite(noise_supplied)).sum():,}"
)
print(f"Finite values assessed: {len(noise_values):,}")

# Summarise the observed distribution.
noise_summary = pd.Series({
    "minimum": noise_values.min(),
    "Q1": noise_values.quantile(0.25),
    "median": noise_values.median(),
    "mean": noise_values.mean(),
    "Q3": noise_values.quantile(0.75),
    "standard_deviation": noise_values.std(),
    "95th_percentile": noise_values.quantile(0.95),
    "99th_percentile": noise_values.quantile(0.99),
    "maximum": noise_values.max(),
    "negative_values": noise_values.lt(0).sum(),
    "exact_zeros": noise_values.eq(0).sum(),
    "skewness": noise_values.skew(),
    # Pandas reports excess kurtosis: a normal distribution has value zero.
    "excess_kurtosis": noise_values.kurt(),
}).to_frame("random_noise")

display(noise_summary)

#### Distribution Plots

The histogram includes a standard normal density curve as a reference.
Its vertical axis shows density so the curve and histogram are comparable.

The Q–Q plot compares observed quantiles with theoretical normal
quantiles. Approximate alignment suggests a similar distributional
shape, while departures highlight differences, particularly in the tails.

Neither plot establishes independence from other variables or
predictive usefulness.

In [ ]:
# Plot the distribution only when finite observations are available.
if noise_values.empty:
    print("No finite observations are available for plotting.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Compare the observed density with a standard normal reference.
    sns.histplot(
        noise_values,
        bins=40,
        stat="density",
        color="steelblue",
        edgecolor="white",
        ax=axes[0],
    )

    reference_x = np.linspace(
        min(noise_values.min(), -4),
        max(noise_values.max(), 4),
        400,
    )
    axes[0].plot(
        reference_x,
        stats.norm.pdf(reference_x),
        color="darkorange",
        linestyle="--",
        label="Standard normal reference",
    )
    axes[0].axvline(
        noise_values.mean(),
        color="green",
        linestyle=":",
        label=f"Observed mean: {noise_values.mean():.3f}",
    )
    axes[0].set_title("Observed distribution")
    axes[0].set_xlabel("random_noise")
    axes[0].set_ylabel("Density")
    axes[0].legend()

    # Show the central spread and values beyond the usual IQR whiskers.
    sns.boxplot(
        x=noise_values,
        color="steelblue",
        whis=1.5,
        ax=axes[1],
    )
    axes[1].set_title("Boxplot — 1.5 × IQR whiskers")
    axes[1].set_xlabel("random_noise")

    # Compare the distribution's shape with theoretical normal quantiles.
    stats.probplot(noise_values, dist="norm", plot=axes[2])
    axes[2].set_title("Normal Q–Q plot")
    axes[2].set_xlabel("Theoretical normal quantiles")
    axes[2].set_ylabel("Observed quantiles")

    fig.suptitle("Distribution of random_noise", fontsize=16)
    plt.tight_layout()
    plt.show()

#### Interpretation — Random Noise

The distribution of `random_noise` is approximately symmetric and
centred close to zero, with an observed mean of −0.022.

The histogram closely follows the standard normal reference curve.
The normal Q–Q plot also shows approximate alignment with the fitted
line, with small departures in the extreme tails. These visual
patterns are compatible with an approximately normal distribution.

The boxplot identifies values beyond the 1.5 × IQR whiskers on both
sides. Such tail observations can occur naturally in normally
distributed data and do not, by themselves, indicate recording errors.

Negative values are valid for this variable and should not be treated
as anomalies merely because of their sign.

Its distribution and name do not establish independence from the
target or other features. Its relationship with the target will be
examined later, and its inclusion in modelling will be reviewed
during feature selection.

### 7.10. Categorical Variables

We examine the frequency of each category to identify dominant labels,
rare categories and missing information.

The tables report percentages relative to all training observations
and to non-missing observations. Missing values are shown separately
in the plots.

Category frequencies describe purchase observations, so customers
with more observations contribute more often. Relationships with
the target will be explored in the next section.

In [ ]:
def plot_category_distribution(column, top_n=15):
    """Display all category frequencies and plot the most frequent labels."""

    # Count observed categories without changing their original labels.
    supplied = train[column]
    counts = supplied.value_counts(dropna=True)
    observed_count = int(supplied.notna().sum())
    missing_count = int(supplied.isna().sum())

    print(f"Variable: {column}")
    print(f"Observed categories: {len(counts):,}")
    print(f"Missing values: {missing_count:,} ({supplied.isna().mean():.2%})")

    # Keep the complete table, even when the plot shows only top categories.
    frequency_table = counts.rename("observations").to_frame()
    frequency_table["pct_of_all_rows"] = counts / len(train) * 100
    frequency_table["pct_of_observed_rows"] = (
        counts / observed_count * 100 if observed_count else np.nan
    )
    frequency_table.index.name = "category"

    display(frequency_table)

    # Create labels and counts for plotting on a separate object.
    plotted_counts = counts.iloc[:top_n].tolist()
    plotted_labels = counts.index[:top_n].astype(str).tolist()

    # Combine remaining labels only for display, retaining their total count.
    if len(counts) > top_n:
        remaining_categories = len(counts) - top_n
        plotted_counts.append(int(counts.iloc[top_n:].sum()))
        plotted_labels.append(
            f"[Other observed categories: {remaining_categories}]"
        )

    # Show missingness separately from the observed categories.
    if missing_count:
        plotted_counts.append(missing_count)
        plotted_labels.append("[Missing values]")

    if not plotted_counts:
        print("No observations are available for plotting.")
        return frequency_table

    # Use numeric positions so long category labels remain readable.
    positions = np.arange(len(plotted_counts))
    fig, ax = plt.subplots(
        figsize=(12, max(4, 0.45 * len(plotted_counts)))
    )

    bars = ax.barh(positions, plotted_counts, color="steelblue")

    # Distinguish the missing-value bar visually.
    if missing_count:
        bars[-1].set_color("grey")

    ax.set_yticks(positions)
    ax.set_yticklabels(plotted_labels)
    ax.invert_yaxis()

    # Annotate every bar with its count and percentage of all training rows.
    for bar, count in zip(bars, plotted_counts):
        ax.text(
            bar.get_width() + max(plotted_counts) * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{count:,} ({count / len(train):.2%})",
            va="center",
        )

    ax.set_xlim(0, max(plotted_counts) * 1.30)
    ax.set_title(f"Distribution of {column}")
    ax.set_xlabel("Training observations")
    ax.set_ylabel("Category")

    plt.tight_layout()
    plt.show()

    return frequency_table

#### 7.10.1. Payment Type

`payment_type` identifies the recorded payment category for the purchase.

We inspect how observations are distributed across its labels and
whether a small number of categories account for most purchases.
The labels are treated as nominal categories, without assuming
an ordering from the numbers in their names.

In [ ]:
# Display every payment category and its share of training observations.
payment_type_frequencies = plot_category_distribution("payment_type")

#### Interpretation — Payment Type

`payment_type` contains nine observed categories, with a strongly
uneven distribution.

`Maybe Tomorrow Payment 7` accounts for 4,565 observations:
69.87% of all training rows and 72.76% of non-missing values.
Together, payment categories 7, 12 and 9 represent 96.56% of
observed payment labels.

Several categories have very few observations. Payment categories
4, 11 and 3 appear only nine, eight and four times respectively.
Their relationships with the target will require cautious
interpretation because such small groups provide limited evidence.

There are 260 missing values, representing 3.98% of training rows.
These are displayed separately from the observed payment categories.

The numbers in the labels identify categories; they do not establish
a meaningful numerical order. Any decision to combine rare categories
will be considered during preprocessing and fitted within the
appropriate training split.

#### 7.10.2. Salesperson

`salesperson` identifies the recorded salesperson category associated
with a purchase.

We examine whether observations are concentrated among a small number
of salespeople and identify categories with limited representation.
The labels alone do not explain differences in customer portfolios
or sales responsibilities.

In [ ]:
# Inspect salesperson frequencies, including missing information.
salesperson_frequencies = plot_category_distribution("salesperson")

#### Interpretation — Salesperson

`salesperson` contains nine observed categories. Most observations
are associated with `VLOOKUP Master 1` and `VLOOKUP Master 4`,
which together account for 87.89% of non-missing values.

`VLOOKUP Master 1` appears in 4,054 observations, representing
62.04% of all training rows. `VLOOKUP Master 4` appears in
1,461 observations, representing 22.36%.

Several categories have limited representation. Categories 6, 5
and 8 appear only 12, seven and three times respectively.
Target rates for these small groups may be unstable.

There are 259 missing values, representing 3.96% of training rows.

These frequencies measure recorded purchase observations, not
salesperson performance or the number of distinct customers served.
Differences may reflect customer portfolios, purchasing frequency
or the period covered by the data. Further analysis is needed
before drawing conclusions about their relationship with repeat
purchases.

#### 7.10.3. Commercial Zone

`commercial_zone` identifies the recorded commercial zone associated
with a purchase.

We examine the distribution across zones to assess their representation
and whether the variable is dominated by a single category.

In [ ]:
# Inspect commercial-zone frequencies and missing information.
commercial_zone_frequencies = plot_category_distribution("commercial_zone")

#### Interpretation — Commercial Zone

`commercial_zone` contains five observed categories but is dominated
by `Decorative Dashboard Zone 4`. This category appears in 6,260
observations, representing 95.81% of all training rows and 96.77%
of non-missing values.

Zones 2 and 3 appear in 151 and 56 observations respectively.
Zones 6 and 7 each appear only once, providing insufficient evidence
for reliable comparisons of their target rates.

There are 65 missing values, representing approximately 0.99% of
training rows. The earlier missingness checks confirmed that these
are the observations with all assessed purchase fields missing.

The concentration in one category makes this variable a candidate
for review during feature selection. However, limited variation
alone does not establish that it has no predictive value.

The labels do not provide enough information to interpret the zones
as specific geographic locations or assume an ordering between them.

#### 7.10.4. Transport Method

`transport_method` identifies the recorded transport or delivery
category associated with a purchase.

We inspect category frequencies and their concentration. Because
this variable has many labels, the chart displays the 15 most
frequent categories and combines the remaining categories for
visualisation only. The table retains every observed category.

In [ ]:
# Show all transport categories in the table and the top 15 in the chart.
transport_method_frequencies = plot_category_distribution(
    "transport_method",
    top_n=15,
)

#### Interpretation — Transport Method

`transport_method` contains 80 observed categories. The most frequent,
`Maybe Tomorrow Delivery 72`, appears in 1,119 observations:
17.13% of all training rows and 17.84% of non-missing values.

The 15 most frequent categories account for 5,032 observations,
representing 80.22% of observed transport labels. The remaining
65 categories collectively account for 1,241 observations.

The “Other observed categories” bar represents this combined total,
not a single transport method. Its size should therefore not be
compared with individual categories as though it were one label.

Many categories have limited representation: 42 appear ten times
or fewer, including 26 that appear only once. These small groups
provide limited evidence for estimating category-specific target rates.

There are 261 missing values, representing 3.99% of training rows.

The large number of categories and their uneven frequencies require
attention during categorical encoding. Combining rare labels may be
considered during preprocessing, but the grouping used in this chart
is only for visualisation and does not change the data.

#### 7.10.5. Warehouse

`warehouse` identifies the recorded warehouse category associated
with a purchase.

We examine its category frequencies to assess how much variation
the variable provides and how well the less frequent category
is represented.

In [ ]:
# Inspect warehouse frequencies and missing information.
warehouse_frequencies = plot_category_distribution("warehouse")

#### Interpretation — Warehouse

`warehouse` contains two observed categories and is almost constant
among non-missing values.

`Final Excel Warehouse 1` appears in 6,253 observations, representing
95.70% of all training rows and 99.67% of observed warehouse labels.
`Final Excel Warehouse 3` appears only 21 times.

There are 260 missing values, representing 3.98% of training rows.
Missing information is therefore more frequent than the less common
warehouse category.

The limited variation makes this variable a candidate for review
during feature selection. However, its distribution alone does not
establish whether the rare category or missingness is associated
with repeat purchases.

Any target comparison involving Warehouse 3 must account for its
small number of observations and the number of distinct customers
represented.

#### 7.10.6. Document Series

`document_series` identifies the recorded document-series category
associated with a purchase.

We inspect its frequency distribution without assuming that the
numbers in the labels imply an ordering or identify document types.

In [ ]:
# Inspect document-series frequencies and missing information.
document_series_frequencies = plot_category_distribution("document_series")

#### Interpretation — Document Series

`document_series` contains three observed categories, each with
substantial representation.

`Ctrl Z Series 2` is the most frequent, with 2,738 observations,
representing 42.33% of non-missing values. Series 3 and Series 1
account for 29.42% and 28.26% respectively.

The distribution is more evenly spread than those of payment type,
salesperson, commercial zone and warehouse. None of the observed
document-series categories is rare.

There are 65 missing values, corresponding to the observations
previously identified as lacking all assessed purchase fields.

The labels are treated as nominal categories. Their frequencies
do not explain the underlying document types or establish their
relationship with repeat purchases.

### 7.11. Summary of Univariate Analysis

- Purchase amounts, historical spending and purchase counts generally
  have right-skewed distributions. Medians and quartiles complement
  means when describing typical observations.
- Shipping, global discounts and short-window activity contain many
  zeros. These observed zeros must be distinguished from missing values.
- Large values flagged by IQR rules are not automatically errors.
  Earlier consistency checks provide stronger evidence for investigating
  particular historical values.
- Historical missingness often reflects insufficient prior purchases,
  although additional missing information was also identified.
- Diagnostic plots made distributions easier to inspect without
  modifying the original data.
- Commercial zone and warehouse are dominated by one category.
  Payment type and salesperson also have uneven category frequencies.
- Transport method contains many rare labels, whereas all three
  document-series categories have substantial representation.
- `random_noise` has a distribution compatible with approximately
  normal noise. Its predictive usefulness remains to be assessed.

These findings describe individual variables. The next section
examines how their distributions and category-level repeat-purchase
rates differ between the target classes.

## 8. Relationships Between Features and the Target

We examine how purchase characteristics and customer history differ
between observations with and without a repeat purchase within 90 days.

The analysis compares numerical distributions and repeat-purchase
rates across categorical groups. Group sizes and missing information
are reported alongside the results.

Observed relationships do not establish causation or guarantee
predictive performance. Customers have repeated observations, so
the rows cannot be assumed to be independent.

### 8.1. Current Purchase Amounts

We first compare current purchase amounts between the two target
classes:

- `0`: No repeat purchase within 90 days.
- `1`: Repeat purchase within 90 days.

Means, medians and quartiles describe differences in location and
spread. Counts show how many observations contribute to each comparison.

In [ ]:
# Compare current purchase amounts using observed finite values.
current_amount_features = [
    "Total Bruto",
    "Total Liquido",
    "Total a Pagar",
    "Total Imp_",
    "Custo",
]

amount_target_results = []

for column in current_amount_features:
    for target_value in [0, 1]:
        # Select the target class without modifying the original data.
        supplied = train.loc[
            train["repeat_purchase_90d"].eq(target_value), column
        ]
        finite_mask = supplied.notna() & np.isfinite(supplied)
        values = supplied.loc[finite_mask]

        # Report coverage alongside the descriptive statistics.
        amount_target_results.append({
            "variable": column,
            "target": target_value,
            "class_rows": len(supplied),
            "finite_observations": len(values),
            "missing": supplied.isna().sum(),
            "infinite_values": (
                supplied.notna() & ~np.isfinite(supplied)
            ).sum(),
            "mean": values.mean(),
            "median": values.median(),
            "Q1": values.quantile(0.25),
            "Q3": values.quantile(0.75),
            "standard_deviation": values.std(),
        })

amount_target_summary = (
    pd.DataFrame(amount_target_results)
    .set_index(["variable", "target"])
)

display(amount_target_summary)

#### Interpretation — Current Purchase Amounts by Target

Observations followed by a repeat purchase have higher means and
medians for gross, net, payable and tax amounts.

For example, the median `Total a Pagar` is 517.46 for target 1,
compared with 443.12 for target 0. The corresponding means are
1,017.58 and 808.56.

The pattern differs for `Custo`: target 1 has a higher mean
(350.45 versus 299.83) but a lower median (147.89 versus 157.53).
Its larger standard deviation and upper quartile indicate a more
dispersed distribution, with larger values influencing the mean.

The interquartile ranges overlap for every variable. Higher average
amounts therefore do not imply a clear separation between classes,
and the plots will help assess the extent of overlap.

Each comparison uses 2,848 finite observations from target 0 and
3,621 from target 1. There are 32 and 33 missing values respectively,
with no infinite values.

These are descriptive associations. Several amount variables are
mathematically related, and customers contribute repeated observations,
so these differences should not be interpreted as independent effects
or evidence of causation.

#### Visual Comparison of Purchase Amounts

For each variable, we compare the target classes using:

- A boxplot on the original scale, retaining extreme observations.
- A boxplot of log1p values to make the central distributions easier to see.
- An empirical cumulative distribution function (ECDF) of log1p values.

The ECDF shows the proportion of observations at or below each value
within each class. Each curve is normalised separately, allowing
comparison despite different class sizes.

Log1p preserves ordering and accommodates zeros. Negative values,
if present, are excluded only from the log1p panels and reported.

In [ ]:
def plot_amount_by_target(column):
    """Compare an amount variable across the two target classes."""

    target_column = "repeat_purchase_90d"

    # Retain finite amounts and valid target labels on a separate object.
    finite_mask = train[column].notna() & np.isfinite(train[column])
    valid_target_mask = train[target_column].isin([0, 1])

    comparison = train.loc[
        finite_mask & valid_target_mask,
        [column, target_column],
    ].copy()

    print(f"Variable: {column}")
    print(f"Finite observations with valid targets: {len(comparison):,}")

    if comparison.empty:
        print("No observations are available for comparison.")
        return

    # Use descriptive class labels consistently across all plots.
    class_labels = {
        0: "0 — No repeat purchase",
        1: "1 — Repeat purchase",
    }
    class_order = list(class_labels.values())
    comparison["target_class"] = comparison[target_column].map(class_labels)

    # Preserve all finite values in the original-scale panel.
    # Use non-negative values for the separate log1p panels.
    log_comparison = comparison.loc[comparison[column].ge(0)].copy()
    log_comparison["log1p_value"] = np.log1p(log_comparison[column])

    print(
        "Negative values excluded from log1p panels: "
        f"{comparison[column].lt(0).sum():,}"
    )

    palette = dict(zip(class_order, sns.color_palette("colorblind", 2)))
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Compare original amounts, including visible extreme observations.
    sns.boxplot(
        data=comparison,
        x="target_class",
        y=column,
        order=class_order,
        hue="target_class",
        hue_order=class_order,
        palette=palette,
        dodge=False,
        legend=False,
        ax=axes[0],
    )
    axes[0].set_title("Original scale")

    if log_comparison.empty:
        for ax in axes[1:]:
            ax.text(
                0.5, 0.5,
                "No non-negative values available",
                ha="center", va="center",
                transform=ax.transAxes,
            )
    else:
        # Compress the right tail without removing large positive amounts.
        sns.boxplot(
            data=log_comparison,
            x="target_class",
            y="log1p_value",
            order=class_order,
            hue="target_class",
            hue_order=class_order,
            palette=palette,
            dodge=False,
            legend=False,
            ax=axes[1],
        )
        axes[1].set_title("Log1p scale")
        axes[1].set_ylabel(f"log(1 + {column})")

        # Compare cumulative proportions separately within each target class.
        sns.ecdfplot(
            data=log_comparison,
            x="log1p_value",
            hue="target_class",
            hue_order=class_order,
            palette=palette,
            ax=axes[2],
        )
        axes[2].set_title("ECDF — log1p scale")
        axes[2].set_xlabel(f"log(1 + {column})")
        axes[2].set_ylabel("Cumulative proportion")

    for ax in axes[:2]:
        ax.set_xlabel("Target class")
        ax.tick_params(axis="x", labelrotation=10)

    fig.suptitle(f"{column} by repeat-purchase outcome", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare payable amounts between repeat and non-repeat purchases.
plot_amount_by_target("Total a Pagar")

#### Interpretation — Payable Amount by Target

The median `Total a Pagar` is higher for observations followed by
a repeat purchase: 517.46 compared with 443.12 for target 0.

The original-scale boxplots show long upper tails in both classes.
The log1p boxplots make the central distributions easier to compare,
revealing a modest difference in their medians and substantial
overlap in their interquartile ranges.

Across much of the middle and upper distribution, the target 1
ECDF lies below the target 0 curve. At those thresholds, a smaller
proportion of repeat-purchase observations have payable amounts
at or below the threshold.

The curves cross at lower amounts, so the difference is not uniform
throughout the distribution. Payable amount alone does not clearly
separate the two outcomes.

All 6,469 finite observations are included in every panel because
there are no negative amounts. The log1p transformation changes
the display scale without removing large purchases or modifying
the original data.

#### Gross Purchase Amount by Target

We compare `Total Bruto` between the target classes to examine
whether the difference in payable amounts is also present in
purchase amounts before discounts and tax.

These variables are related, so similar patterns should not be
interpreted as separate, independent sources of evidence.

In [ ]:
# Compare gross purchase amounts across the two target classes.
plot_amount_by_target("Total Bruto")

#### Interpretation — Gross Purchase Amount by Target

The median `Total Bruto` is 493.18 for target 1 and 389.91 for
target 0. The corresponding means are 983.17 and 762.81.

Both distributions have long upper tails. The log1p boxplots show
a higher central gross amount for repeat-purchase observations,
while retaining substantial overlap between classes.

The target 1 ECDF lies below the target 0 curve across much of
the middle and upper distribution. This indicates relatively
higher gross amounts in that part of the repeat-purchase group.

The curves are close at lower amounts, and gross amount alone
does not clearly distinguish the outcomes.

The pattern resembles that of payable amounts. Because these
variables are mathematically related, their similar associations
should not be treated as independent evidence.

All 6,469 finite observations are retained in every panel.

#### Net Purchase Amount by Target

We compare `Total Liquido` between the target classes to examine
purchase amounts after the recorded line and global discounts.

This comparison describes whether the difference observed in gross
amounts remains present in net amounts. It does not isolate the
effect of discounts on repeat purchasing.

In [ ]:
# Compare net purchase amounts across the two target classes.
plot_amount_by_target("Total Liquido")

#### Interpretation — Net Purchase Amount by Target

The median `Total Liquido` is 422.05 for target 1 and 362.19 for
target 0. The corresponding means are 846.70 and 680.06.

The repeat-purchase group also has a higher upper quartile:
1,009.45 compared with 790.54. However, its lower quartile is
slightly lower, at 152.22 compared with 158.07.

The crossing ECDF curves reflect this mixed pattern. Repeat-purchase
observations include relatively more small net amounts, alongside
relatively higher amounts across much of the middle and upper
distribution.

The log1p boxplots show substantial overlap between classes.
The higher median therefore does not imply that repeat-purchase
observations consistently have larger net amounts.

Both classes retain long upper tails. All 6,469 finite observations
are included in every panel, with no negative values excluded.

This comparison does not isolate the effect of discounts:
gross amounts, customer characteristics and other purchase
conditions may also contribute to the observed differences.

#### Tax Amount by Target

We compare `Total Imp_` between target classes.

Tax amounts depend on purchase amounts and the applicable tax
treatment. Their association with the target should therefore
be interpreted alongside the related gross, net and payable amounts.

In [ ]:
# Compare recorded tax amounts across the two target classes.
plot_amount_by_target("Total Imp_")

#### Interpretation — Tax Amount by Target

The median `Total Imp_` is 88.78 for target 1 and 69.85 for target 0.
The corresponding means are 170.88 and 128.49.

The log1p boxplots show higher central tax amounts for repeat-purchase
observations, with substantial overlap between the classes.
Both distributions have long upper tails.

The target 1 ECDF generally lies below the target 0 curve,
indicating relatively higher tax amounts across much of the
observed distribution. The step at zero shows that both classes
also contain purchases with zero recorded tax.

Tax amounts are related to purchase size and tax treatment.
This association therefore does not establish an independent
effect of tax on repeat purchasing.

All 6,469 finite observations are retained in every panel.
Zero tax values remain included in the log1p plots.

#### Purchase Cost by Target

We compare `Custo` between target classes.

The earlier summary showed a higher mean but a lower median for
target 1. The plots help examine this difference by showing the
central distribution, upper tail and observations with zero cost.

In [ ]:
# Compare recorded purchase costs across the two target classes.
plot_amount_by_target("Custo")

#### Interpretation — Purchase Cost by Target

`Custo` shows a mixed relationship with the target. Repeat-purchase
observations have a higher mean cost (350.45 versus 299.83) but
a slightly lower median (147.89 versus 157.53).

Their interquartile range is wider, extending from 43.82 to 426.85,
compared with 57.40 to 365.62 for target 0. This indicates greater
variation in the central portion of the repeat-purchase group.

The ECDF shows a larger proportion of zero costs for target 1.
Across lower positive amounts, its curve also lies above that of
target 0, before crossing and lying below it at higher amounts.

Repeat-purchase observations therefore include both relatively
more low-cost purchases and a larger upper tail. The higher mean
does not describe a uniform increase across the distribution.

The classes overlap substantially. Zero costs require contextual
interpretation and should not automatically be treated as missing
values or errors.

All 6,469 finite observations are retained in every panel.

#### Findings from Current Purchase Amounts

Gross, net, payable and tax amounts have higher means and medians
among observations followed by a repeat purchase. However, their
distributions overlap substantially.

Cost has a different pattern: a higher mean accompanies a lower
median, more zero values and a wider distribution for target 1.

These comparisons suggest possible associations with repeat
purchasing, but do not establish predictive performance. Related
amount variables may also provide overlapping information.

### 8.2. Shipping and Discounts

Shipping and discount amounts contain many zeros. We therefore
separate two questions:

- Does the repeat-purchase rate differ between zero and positive amounts?
- Among positive amounts, does the distribution differ between target classes?

Missing values are kept separate from observed zeros.
Group sizes and distinct customer counts accompany the target rates.

These comparisons describe associations. They do not establish
that shipping charges or discounts cause changes in repeat purchasing.

In [ ]:
# Variables with substantial concentrations at zero.
shipping_discount_features = [
    "Total Portes",
    "Total Desc_ Global",
    "Total Desc_ Linha",
]

amount_status_order = [
    "Zero",
    "Positive",
    "Missing",
    "Negative",
    "Infinite",
]

shipping_discount_results = []

for column in shipping_discount_features:
    supplied = train[column]
    finite_mask = supplied.notna() & np.isfinite(supplied)

    # Classify values on a separate Series, preserving the original data.
    status = pd.Series("Missing", index=train.index, dtype="string")
    status.loc[finite_mask & supplied.eq(0)] = "Zero"
    status.loc[finite_mask & supplied.gt(0)] = "Positive"
    status.loc[finite_mask & supplied.lt(0)] = "Negative"
    status.loc[supplied.notna() & ~np.isfinite(supplied)] = "Infinite"

    # Restrict target-rate calculations to valid binary labels.
    valid_target_mask = train["repeat_purchase_90d"].isin([0, 1])

    for group in amount_status_order:
        group_mask = status.eq(group) & valid_target_mask
        group_data = train.loc[
            group_mask, ["customer_id", "repeat_purchase_90d"]
        ]

        # Omit empty groups; unexpected observed values remain visible.
        if group_data.empty:
            continue

        shipping_discount_results.append({
            "variable": column,
            "amount_status": group,
            "observations": len(group_data),
            "unique_customers": group_data["customer_id"].nunique(),
            "target_0": group_data["repeat_purchase_90d"].eq(0).sum(),
            "target_1": group_data["repeat_purchase_90d"].eq(1).sum(),
            "repeat_purchase_pct": (
                group_data["repeat_purchase_90d"].mean() * 100
            ),
        })

shipping_discount_target_summary = (
    pd.DataFrame(shipping_discount_results)
    .set_index(["variable", "amount_status"])
)

display(shipping_discount_target_summary)

#### Repeat-Purchase Rates by Amount Status

The plots compare the observed repeat-purchase rates for zero,
positive and missing amounts.

The dashed line marks the overall training repeat-purchase rate.
Bar labels show the number of observations contributing to each rate.

Customers may appear in multiple groups and contribute repeated
observations. These are descriptive rates rather than estimates
based on independent rows.

In [ ]:
def plot_amount_status_target(column):
    """Plot repeat-purchase rates by zero, positive or missing status."""

    # Retrieve the previously calculated summary for this variable.
    summary = shipping_discount_target_summary.loc[column].copy()

    # Keep the same group order across variables.
    present_groups = [
        group for group in amount_status_order if group in summary.index
    ]
    summary = summary.loc[present_groups]

    # Calculate the reference rate using valid training targets.
    valid_targets = train.loc[
        train["repeat_purchase_90d"].isin([0, 1]),
        "repeat_purchase_90d",
    ]
    baseline_pct = valid_targets.mean() * 100

    colours = [
        "grey" if group == "Missing" else "steelblue"
        for group in summary.index
    ]

    fig, ax = plt.subplots(figsize=(9, 5))
    positions = np.arange(len(summary))

    bars = ax.bar(
        positions,
        summary["repeat_purchase_pct"],
        color=colours,
    )

    # Use a common percentage scale for comparisons.
    ax.set_ylim(0, 100)
    ax.set_xticks(positions)
    ax.set_xticklabels(summary.index)
    ax.set_ylabel("Repeat-purchase rate (%)")
    ax.set_xlabel("Amount status")
    ax.set_title(f"{column} — repeat-purchase rate by amount status")

    ax.axhline(
        baseline_pct,
        color="darkorange",
        linestyle="--",
        label=f"Overall training rate: {baseline_pct:.2f}%",
    )

    # Report both rates and sample sizes directly on the chart.
    for bar, (_, row) in zip(bars, summary.iterrows()):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            max(bar.get_height() - 3, 2),
            f"{row['repeat_purchase_pct']:.2f}%\n"
            f"n = {int(row['observations']):,}",
            ha="center",
            va="top" if bar.get_height() >= 5 else "bottom",
        )

    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare outcomes for zero, positive and missing shipping amounts.
plot_amount_status_target("Total Portes")

#### Interpretation — Shipping Amount Status

Observations with positive shipping amounts have a repeat-purchase
rate of 58.12%, compared with 55.56% for observations with zero
shipping amounts: a difference of 2.56 percentage points.

The positive-shipping group contains 1,053 observations from
296 customers. The zero-shipping group contains 5,416 observations
from 508 customers. Customers can contribute to both groups,
so these customer counts are not additive.

The missing group has a repeat-purchase rate of 50.77%, based on
65 observations from 53 customers. These are the records previously
identified as lacking all assessed purchase fields, so this rate
cannot be attributed specifically to missing shipping information.

The zero and positive groups both have rates close to the overall
training rate of 55.92%. The observed difference does not establish
that shipping charges affect repeat purchasing or provide a stable
predictive signal.

#### Global Discount Status and Repeat Purchasing

We compare repeat-purchase rates between observations with zero
and positive global discounts.

Positive global discounts are uncommon, so the number of observations
and distinct customers is particularly relevant when interpreting
the difference.

In [ ]:
# Compare outcomes for zero, positive and missing global discounts.
plot_amount_status_target("Total Desc_ Global")

#### Interpretation — Global Discounts and Repeat Purchases

Observations with a positive global discount have a repeat-purchase rate
of 40.00%, compared with 56.25% for observations without a global discount.
The difference is approximately 16.25 percentage points.

However, positive global discounts occur in only 110 observations from
61 customers, compared with 6,359 observations without a global discount.
The smaller group and repeated observations per customer should be
considered when interpreting this difference.

This association does not establish that global discounts reduce repeat
purchases. The groups may differ in purchase characteristics, customer
history or other factors.

The 65 observations with missing global discounts are the records that
lack all purchase information. Their repeat-purchase rate is 50.77%.

In [ ]:
# Compare repeat-purchase rates for zero, positive and missing line discounts.
plot_amount_status_target("Total Desc_ Linha")

#### Interpretation — Line Discounts and Repeat Purchases

Observations with positive line discounts have a repeat-purchase rate
of 62.59%, compared with 45.06% for observations without line discounts.
The difference is approximately 17.52 percentage points.

Both groups contain substantial numbers of observations: 4,028 with
positive line discounts and 2,441 without them. They represent 469 and
411 distinct customers respectively. Customers may appear in both groups.

Positive line discounts are associated with a higher repeat-purchase
rate in the training data. However, this does not establish that discounts
cause repeat purchases. Purchase size, customer history and commercial
arrangements may help explain the difference.

The missing group contains the same 65 observations with no purchase
information and has a repeat-purchase rate of 50.77%.

#### Positive Shipping and Discount Amounts by Target

The previous graphs compared observations with zero, positive and missing
amounts. We now examine the size of positive amounts separately for each
target class.

Restricting this comparison to positive values prevents the concentration
at zero from dominating the summaries. Original data remain unchanged.

In [ ]:
# Summarise positive amounts separately for each target class.
positive_amount_rows = []

for column in shipping_discount_features:
    for target in [0, 1]:
        # Select finite positive amounts with the requested target value.
        mask = (
            train["repeat_purchase_90d"].eq(target)
            & np.isfinite(train[column])
            & train[column].gt(0)
        )

        values = train.loc[mask, column]

        positive_amount_rows.append({
            "variable": column,
            "target": target,
            "observations": len(values),
            "unique_customers": train.loc[mask, "customer_id"].nunique(),
            "minimum": values.min(),
            "Q1": values.quantile(0.25),
            "median": values.median(),
            "mean": values.mean(),
            "Q3": values.quantile(0.75),
            "maximum": values.max(),
        })

# Display all three variables together before plotting them separately.
positive_amount_target_summary = (
    pd.DataFrame(positive_amount_rows)
    .set_index(["variable", "target"])
)

display(positive_amount_target_summary)

#### Interpretation — Positive Amounts by Target

**Shipping charges**

Positive shipping charges have similar medians in both classes:
4.85 for target 0 and 4.95 for target 1. The mean is slightly higher
for target 0, at 7.47 compared with 6.79.

The quartiles show a mixed pattern, so there is no consistent shift
towards higher shipping charges in either class.

**Global discounts**

Among observations with positive global discounts, target 1 has a higher
median discount: 12.88 compared with 8.71. Its mean is also higher,
although its third quartile is slightly lower.

This comparison contains only 44 observations for target 1 and 66 for
target 0, so individual purchases can noticeably influence the summaries.

The earlier lower repeat-purchase rate for observations with a global
discount concerns whether a discount is present. This comparison concerns
its amount among observations where it is positive.

**Line discounts**

Positive line discounts tend to be larger for target 1. Its median is
74.29 compared with 44.28 for target 0, and both quartiles are higher.
The means are 195.59 and 155.59 respectively.

Absolute discounts may reflect purchase size as well as discount policy.
A larger discount does not necessarily imply a higher discount percentage.

These summaries describe purchase observations. Repeated observations
from the same customer contribute multiple times to each comparison.

In [ ]:
def plot_positive_amount_by_target(column):
    """Compare positive amount distributions between the two target classes."""

    # Keep finite positive amounts with valid binary targets.
    mask = (
        train["repeat_purchase_90d"].isin([0, 1])
        & np.isfinite(train[column])
        & train[column].gt(0)
    )

    # Create a separate plotting object without changing the original data.
    plot_data = train.loc[
        mask, [column, "repeat_purchase_90d"]
    ].copy()

    # Compress the right tail so differences in typical amounts remain visible.
    plot_data["log_amount"] = np.log1p(plot_data[column])

    # Use readable labels and consistent colours for both panels.
    class_labels = {
        0: "0 — No repeat purchase",
        1: "1 — Repeat purchase",
    }
    class_order = list(class_labels.values())
    plot_data["target_class"] = (
        plot_data["repeat_purchase_90d"].map(class_labels)
    )
    class_palette = dict(zip(class_order, sns.color_palette("colorblind", 2)))

    # Report how many positive observations contribute to each distribution.
    print(f"Variable: {column}")
    display(
        plot_data.groupby("target_class")[[column]]
        .count()
        .rename(columns={column: "positive_observations"})
        .reindex(class_order, fill_value=0)
    )

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Compare medians, quartiles and tails within the positive observations.
    sns.boxplot(
        data=plot_data,
        x="target_class",
        y="log_amount",
        hue="target_class",
        order=class_order,
        hue_order=class_order,
        palette=class_palette,
        dodge=False,
        legend=False,
        ax=axes[0],
    )
    axes[0].set_title("Positive amounts — boxplot")
    axes[0].set_xlabel("Target class")
    axes[0].set_ylabel(f"log1p({column})")

    # Normalise each class separately to compare distribution shapes.
    sns.ecdfplot(
        data=plot_data,
        x="log_amount",
        hue="target_class",
        hue_order=class_order,
        palette=class_palette,
        ax=axes[1],
    )
    axes[1].set_title("Positive amounts — cumulative distribution")
    axes[1].set_xlabel(f"log1p({column})")
    axes[1].set_ylabel("Proportion within target class")

    fig.suptitle(f"{column} — positive amounts by target", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare positive shipping charges between target classes.
plot_positive_amount_by_target("Total Portes")

#### Interpretation — Positive Shipping Charges by Target

The boxplots show similar medians and substantial overlap between the
two target classes. Target 1 has a slightly wider interquartile range,
consistent with its lower first quartile and higher third quartile.

The cumulative distribution curves remain close and cross, indicating
that neither class consistently has higher positive shipping charges.

Both classes contain unusually large charges relative to their typical
values. The boxplot markers identify distributional extremes, which are
not necessarily recording errors.

Positive shipping amounts show limited separation between target classes
in these plots. Their contribution to prediction will be assessed during
modelling.

In [ ]:
# Compare positive global discounts between target classes.
plot_positive_amount_by_target("Total Desc_ Global")

#### Interpretation — Positive Global Discounts by Target

Among observations with positive global discounts, target 1 has a higher
median amount: 12.88 compared with 8.71 for target 0. However, the
boxplots overlap substantially and their third quartiles are similar.

The cumulative distribution curves cross. Target 1 generally has larger
amounts through the lower and middle portions of the distribution,
but this pattern does not hold across the entire range.

The comparison includes only 44 observations for target 1 and 66 for
target 0. These small groups and repeated observations per customer
limit how confidently the differences can be generalised.

This describes discount amounts among positive-discount observations.
It does not contradict the earlier finding that the positive-discount
group has a lower repeat-purchase rate than the zero-discount group.

In [ ]:
# Compare positive line discounts between target classes.
plot_positive_amount_by_target("Total Desc_ Linha")

#### Interpretation — Positive Line Discounts by Target

The boxplot for target 1 has a higher median and higher quartiles.
On the original scale, median positive line discounts are 74.29
for target 1 and 44.28 for target 0.

The target 1 cumulative distribution curve lies below the target 0
curve through most of the range. This indicates that positive line
discounts generally tend to be larger for repeat-purchase observations.
The curves converge towards the upper tail.

Despite this shift, the distributions overlap substantially. Line
discount amounts alone do not clearly distinguish the two classes.

Together with the earlier zero-versus-positive comparison, these results
show associations with both the presence and size of line discounts.
Purchase size and customer characteristics may contribute to these
patterns. Discount percentages could therefore be useful to explore
during feature engineering.

### 8.3. Current Purchase Counts and the Target

We now examine whether the number of purchase details and the number
of invoices differ between target classes.

For `N_ Detalhes`, we will compare class distributions and repeat-purchase
rates across count ranges. For `invoice_count_current_purchase`, which
has only five observed values, we will examine each count separately.

Missing values will be reported separately.

In [ ]:
# Summarise current purchase counts separately for each target class.
current_count_features = [
    "N_ Detalhes",
    "invoice_count_current_purchase",
]

current_count_rows = []

for column in current_count_features:
    for target in [0, 1]:
        # Retain the full class to report missing and infinite values.
        class_data = train.loc[
            train["repeat_purchase_90d"].eq(target)
        ]
        supplied_values = class_data[column]

        # Calculate statistics using finite observed values only.
        finite_mask = np.isfinite(supplied_values)
        values = supplied_values.loc[finite_mask]

        current_count_rows.append({
            "variable": column,
            "target": target,
            "class_rows": len(class_data),
            "finite_observations": len(values),
            "missing": supplied_values.isna().sum(),
            "infinite_values": (
                supplied_values.notna() & ~finite_mask
            ).sum(),
            "minimum": values.min(),
            "Q1": values.quantile(0.25),
            "median": values.median(),
            "mean": values.mean(),
            "Q3": values.quantile(0.75),
            "95th_percentile": values.quantile(0.95),
            "maximum": values.max(),
        })

current_count_target_summary = (
    pd.DataFrame(current_count_rows)
    .set_index(["variable", "target"])
)

display(current_count_target_summary)

#### Interpretation — Current Purchase Counts by Target

**Number of purchase details**

`N_ Detalhes` has a median of 12 for target 0 and 10 for target 1.
However, target 1 has a slightly higher mean, at 22.10 compared with
20.79, and a higher 95th percentile, at 79 compared with 69.

These summaries suggest differences in distribution shape rather than
a consistent shift towards larger counts. We will inspect the plots
and repeat-purchase rates across count ranges.

**Number of invoices**

Both classes have a median and quartiles of one invoice. The mean is
slightly higher for target 1: 1.080 compared with 1.049.

The 95th percentile is two invoices for target 1 and one for target 0.
Multiple-invoice purchases therefore appear more common in target 1,
but a frequency table is needed to examine the individual counts.

Both variables have 32 missing observations in target 0 and 33 in
target 1, with no infinite values.

In [ ]:
# Compare the full and log1p distributions of purchase-detail counts.
plot_amount_by_target("N_ Detalhes")

#### Interpretation — Purchase-Detail Counts by Target

The distributions overlap substantially. Target 1 has a lower median
count but a slightly higher third quartile and a longer upper tail.

At lower counts, the target 1 cumulative distribution curve lies above
target 0, indicating a greater proportion of small purchases by number
of details. The curves cross and the ordering reverses at higher counts.

Neither class therefore consistently has more purchase details.
Comparing repeat-purchase rates across count ranges may reveal patterns
that a single mean or median would conceal.

The original-scale boxplots flag many large counts. These are
distributional extremes and are not automatically invalid observations.

#### Repeat-Purchase Rates by Purchase-Detail Range

We use fixed, readable count ranges to explore how the repeat-purchase
rate varies with the number of details. These ranges are descriptive
groups for EDA and do not define a preprocessing decision.

Group sizes and distinct customer counts are reported alongside rates.
Missing values are shown separately.

In [ ]:
# Select observations with valid binary targets.
detail_range_data = train.loc[
    train["repeat_purchase_90d"].isin([0, 1]),
    ["N_ Detalhes", "customer_id", "repeat_purchase_90d"],
].copy()

# Define fixed ranges independently of the target.
detail_labels = ["1–5", "6–10", "11–20", "21–50", "51–100", "101+"]

detail_values = detail_range_data["N_ Detalhes"]
valid_counts = (
    np.isfinite(detail_values)
    & detail_values.ge(1)
    & detail_values.eq(np.floor(detail_values))
)

# Keep missing and invalid values distinct from valid count ranges.
detail_groups = pd.Series(
    "Invalid", index=detail_range_data.index, dtype="string"
)
detail_groups.loc[detail_values.isna()] = "Missing"

detail_groups.loc[valid_counts] = pd.cut(
    detail_values.loc[valid_counts],
    bins=[0, 5, 10, 20, 50, 100, np.inf],
    labels=detail_labels,
    right=True,
).astype("string")

detail_range_data["detail_range"] = detail_groups

# Calculate observed rates without treating repeated customers as independent.
detail_range_summary = (
    detail_range_data.groupby("detail_range")
    .agg(
        observations=("repeat_purchase_90d", "size"),
        unique_customers=("customer_id", "nunique"),
        target_0=("repeat_purchase_90d", lambda values: values.eq(0).sum()),
        target_1=("repeat_purchase_90d", "sum"),
        repeat_purchase_pct=("repeat_purchase_90d", lambda values: values.mean() * 100),
    )
)

# Preserve the natural order and display only groups present in the data.
detail_order = detail_labels + ["Missing", "Invalid"]
detail_range_summary = detail_range_summary.reindex(
    [group for group in detail_order if group in detail_range_summary.index]
)

display(detail_range_summary)

# Compare group rates with the overall training rate.
fig, ax = plt.subplots(figsize=(11, 5))
positions = np.arange(len(detail_range_summary))

ax.bar(
    positions,
    detail_range_summary["repeat_purchase_pct"],
    color=[
        "gray" if group in ["Missing", "Invalid"] else "steelblue"
        for group in detail_range_summary.index
    ],
)

overall_rate = detail_range_data["repeat_purchase_90d"].mean() * 100
ax.axhline(
    overall_rate,
    color="darkorange",
    linestyle="--",
    label=f"Overall training rate: {overall_rate:.2f}%",
)

# Annotate both rates and sample sizes to support interpretation.
for position, row in enumerate(detail_range_summary.itertuples()):
    ax.text(
        position,
        row.repeat_purchase_pct + 2,
        f"{row.repeat_purchase_pct:.2f}%\nn = {row.observations:,}",
        ha="center",
        fontsize=10,
    )

ax.set_xticks(positions)
ax.set_xticklabels(detail_range_summary.index)
ax.set_ylim(0, 100)
ax.set_xlabel("Number of purchase details")
ax.set_ylabel("Repeat-purchase rate (%)")
ax.set_title("Repeat-purchase rate by purchase-detail range")
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

#### Interpretation — Repeat Purchases by Purchase-Detail Range

The repeat-purchase rate decreases from 60.34% for purchases with
1–5 details to 50.72% for those with 11–20 details. It then increases
across the larger-count ranges, reaching 65.73% for purchases with
101 or more details.

This roughly U-shaped pattern helps explain why the earlier means
and medians gave a mixed picture. The association is not a simple
increase or decrease with the number of details.

The 101+ group contains only 143 observations from 47 customers,
so its rate should be interpreted more cautiously than those of
the larger groups.

The pattern depends on the chosen descriptive ranges and may also
reflect differences in customer history or purchase characteristics.
It does not establish a causal relationship.

#### Repeat-Purchase Rates by Invoice Count

We compare each observed invoice count separately, reporting the number
of observations and distinct customers alongside the repeat-purchase rate.

Higher invoice counts are uncommon, so their group sizes are particularly
important when interpreting differences.

In [ ]:
# Keep observations with valid binary targets.
invoice_target_data = train.loc[
    train["repeat_purchase_90d"].isin([0, 1]),
    ["invoice_count_current_purchase", "customer_id", "repeat_purchase_90d"],
].copy()

invoice_values = invoice_target_data["invoice_count_current_purchase"]

# Identify valid positive integer counts.
valid_invoice_counts = (
    np.isfinite(invoice_values)
    & invoice_values.ge(1)
    & invoice_values.eq(np.floor(invoice_values))
)

# Create readable labels while keeping missing and invalid values separate.
invoice_groups = pd.Series(
    "Invalid", index=invoice_target_data.index, dtype="string"
)
invoice_groups.loc[invoice_values.isna()] = "Missing"
invoice_groups.loc[valid_invoice_counts] = (
    invoice_values.loc[valid_invoice_counts].astype("int64").astype("string")
)

invoice_target_data["invoice_count_group"] = invoice_groups

# Summarise each observed count.
invoice_target_summary = (
    invoice_target_data.groupby("invoice_count_group")
    .agg(
        observations=("repeat_purchase_90d", "size"),
        unique_customers=("customer_id", "nunique"),
        target_0=("repeat_purchase_90d", lambda values: values.eq(0).sum()),
        target_1=("repeat_purchase_90d", "sum"),
        repeat_purchase_pct=(
            "repeat_purchase_90d", lambda values: values.mean() * 100
        ),
    )
)

# Order counts numerically, followed by missing or invalid groups if present.
invoice_order = [
    str(count)
    for count in sorted(
        invoice_values.loc[valid_invoice_counts].astype("int64").unique()
    )
] + ["Missing", "Invalid"]

invoice_target_summary = invoice_target_summary.reindex(
    [group for group in invoice_order if group in invoice_target_summary.index]
)

display(invoice_target_summary)

# Plot observed rates with group sizes and the overall training baseline.
fig, ax = plt.subplots(figsize=(10, 5))
positions = np.arange(len(invoice_target_summary))

ax.bar(
    positions,
    invoice_target_summary["repeat_purchase_pct"],
    color=[
        "gray" if group in ["Missing", "Invalid"] else "steelblue"
        for group in invoice_target_summary.index
    ],
)

overall_rate = invoice_target_data["repeat_purchase_90d"].mean() * 100
ax.axhline(
    overall_rate,
    color="darkorange",
    linestyle="--",
    label=f"Overall training rate: {overall_rate:.2f}%",
)

# Place annotations inside bars so rates near 100% remain readable.
for position, row in enumerate(invoice_target_summary.itertuples()):
    ax.text(
        position,
        max(row.repeat_purchase_pct - 10, 3),
        f"{row.repeat_purchase_pct:.2f}%\nn = {row.observations:,}",
        ha="center",
        va="center",
        fontsize=10,
    )

ax.set_xticks(positions)
ax.set_xticklabels(invoice_target_summary.index)
ax.set_ylim(0, 100)
ax.set_xlabel("Number of invoices in the current purchase")
ax.set_ylabel("Repeat-purchase rate (%)")
ax.set_title("Repeat-purchase rate by invoice count")
ax.legend(loc="upper right")

plt.tight_layout()
plt.show()

#### Interpretation — Invoice Counts and Repeat Purchases

One-invoice purchases dominate the data, with 6,116 observations and
a repeat-purchase rate of 55.28%. Two-invoice purchases have a higher
rate of 67.95%, based on 312 observations from 110 customers.

The rate does not increase consistently with invoice count. Although
the three-invoice group reaches 85.71%, it contains only 14 observations.
The four- and five-invoice groups contain 21 and six observations,
representing just six and two customers respectively.

Rates for these rare counts can change substantially with a few
observations and should not be treated as reliable evidence of a trend.

Invoice count shows a descriptive association with the target,
particularly between one and two invoices. Its predictive contribution
will need to be assessed alongside purchase size and customer history.

### 8.4. Purchase History and Customer Tenure by Target

We compare the number of prior purchases and recorded customer tenure
between target classes.

Class summaries and distribution plots will show whether repeat-purchase
observations tend to have more extensive purchase histories. We will also
examine repeat-purchase rates across history and tenure ranges.

These statistics describe purchase observations, so customers with more
observations contribute more often.

In [ ]:
# Compare purchase history and tenure between target classes.
history_tenure_features = [
    "prior_purchase_count",
    "customer_tenure_days",
]

history_tenure_rows = []

for column in history_tenure_features:
    for target in [0, 1]:
        # Select the class without changing the original training data.
        class_data = train.loc[
            train["repeat_purchase_90d"].eq(target)
        ]
        supplied_values = class_data[column]

        # Use finite observations for descriptive statistics.
        finite_mask = np.isfinite(supplied_values)
        values = supplied_values.loc[finite_mask]

        history_tenure_rows.append({
            "variable": column,
            "target": target,
            "class_rows": len(class_data),
            "finite_observations": len(values),
            "missing": supplied_values.isna().sum(),
            "infinite_values": (
                supplied_values.notna() & ~finite_mask
            ).sum(),
            "exact_zeros": values.eq(0).sum(),
            "zero_pct": (
                values.eq(0).mean() * 100 if len(values) else np.nan
            ),
            "minimum": values.min(),
            "Q1": values.quantile(0.25),
            "median": values.median(),
            "mean": values.mean(),
            "Q3": values.quantile(0.75),
            "95th_percentile": values.quantile(0.95),
            "maximum": values.max(),
        })

# Display both variables in one table before plotting them separately.
history_tenure_target_summary = (
    pd.DataFrame(history_tenure_rows)
    .set_index(["variable", "target"])
)

display(history_tenure_target_summary)

#### Interpretation — Purchase History and Tenure by Target

**Prior purchase count**

Repeat-purchase observations have more recorded prior purchases.
Target 1 has a median of 17, compared with six for target 0.
Its first and third quartiles are also higher, at six and 40
compared with two and 15.

The mean is 33.62 for target 1 and 10.54 for target 0.
Both distributions have long upper tails, but the difference
extends to their central values.

No prior purchases are recorded in 14.12% of finite observations
for target 0, compared with 5.55% for target 1. These percentages
describe the composition of each class, rather than the
repeat-purchase rate among observations without prior purchases.

**Customer tenure**

Target 1 also has longer recorded tenure, with a median of
1,032 days compared with 781 days for target 0. Both quartiles
are higher for target 1.

The zero counts match those for prior purchase count because
the earlier consistency check established that zero tenure
and no prior purchases occur in the same observations.

These findings suggest associations between recorded customer
history and repeat purchases. They do not establish independent
effects, since tenure and purchase count may be related.

In [ ]:
# Compare prior-purchase counts between the two target classes.
# The existing function includes zero counts in its log1p panels.
plot_amount_by_target("prior_purchase_count")

#### Interpretation — Prior Purchase Count by Target

The boxplots show higher prior-purchase counts for target 1, including
a higher median and higher quartiles. The log1p scale makes this
difference easier to see without removing zero counts or large values.

The target 1 cumulative distribution curve lies below target 0
through almost the entire range. At a given count, a smaller proportion
of repeat-purchase observations falls at or below that value.
This indicates a broad shift towards more prior purchases.

The distributions still overlap, so prior purchase count does not
perfectly distinguish the classes.

Large counts flagged by the boxplots are not automatically errors.
The earlier checks found valid integer counts and no inconsistencies
in their progression across assessed customer observations.

Prior purchase count shows a clear descriptive association with the
target. Its predictive usefulness will be assessed under the chosen
validation strategy.

In [ ]:
# Compare recorded customer tenure between target classes.
# Zero-tenure observations remain included in all panels.
plot_amount_by_target("customer_tenure_days")

#### Interpretation — Customer Tenure by Target

Target 1 has a higher median tenure and higher quartiles. Its cumulative
distribution curve lies below target 0 through most of the range,
indicating generally longer recorded tenure among repeat-purchase
observations.

The steps at zero reflect observations without recorded prior purchases.
These account for 14.12% of finite observations in target 0 and 5.55%
in target 1.

The log1p boxplots flag some low-tenure values, including zeros, even
though the original-scale boxplots show no values beyond their fences.
The transformation changes the quartiles and IQR fences. These markers
therefore do not establish that short tenure is an error.

Tenure shows a descriptive association with repeat purchases, but there
is substantial overlap between classes. Its relationship with prior
purchase count will also need to be considered.

#### Repeat-Purchase Rates by Prior Purchase Count

We compare repeat-purchase rates across fixed purchase-history ranges.
Observations with no prior purchases are shown separately.

These descriptive ranges help reveal whether the association is
consistent across different levels of recorded purchase history.
They do not define the final modelling transformations.

In [ ]:
# Select labelled observations and keep the original training data unchanged.
prior_count_data = train.loc[
    train["repeat_purchase_90d"].isin([0, 1]),
    ["prior_purchase_count", "customer_id", "repeat_purchase_90d"],
].copy()

counts = prior_count_data["prior_purchase_count"]

# Only finite, non-negative integer counts enter the history ranges.
valid_counts = (
    np.isfinite(counts)
    & counts.ge(0)
    & counts.eq(np.floor(counts))
)

# Keep zero history separate from increasingly larger purchase counts.
prior_count_labels = ["0", "1", "2–5", "6–10", "11–20", "21–50", "51+"]

prior_count_groups = pd.Series(
    "Invalid", index=prior_count_data.index, dtype="string"
)
prior_count_groups.loc[counts.isna()] = "Missing"

prior_count_groups.loc[valid_counts] = pd.cut(
    counts.loc[valid_counts],
    bins=[-0.5, 0, 1, 5, 10, 20, 50, np.inf],
    labels=prior_count_labels,
    right=True,
).astype("string")

prior_count_data["prior_purchase_range"] = prior_count_groups

# Report group sizes alongside observed repeat-purchase rates.
prior_count_target_summary = (
    prior_count_data.groupby("prior_purchase_range")
    .agg(
        observations=("repeat_purchase_90d", "size"),
        unique_customers=("customer_id", "nunique"),
        target_0=("repeat_purchase_90d", lambda values: values.eq(0).sum()),
        target_1=("repeat_purchase_90d", "sum"),
        repeat_purchase_pct=(
            "repeat_purchase_90d", lambda values: values.mean() * 100
        ),
    )
)

# Preserve the natural order and include only groups present in the data.
prior_count_order = prior_count_labels + ["Missing", "Invalid"]
prior_count_target_summary = prior_count_target_summary.reindex(
    [group for group in prior_count_order
     if group in prior_count_target_summary.index]
)

display(prior_count_target_summary)

In [ ]:
# Plot the history-range rates calculated in the previous cell.
fig, ax = plt.subplots(figsize=(12, 5))
positions = np.arange(len(prior_count_target_summary))

ax.bar(
    positions,
    prior_count_target_summary["repeat_purchase_pct"],
    color=[
        "gray" if group in ["Missing", "Invalid"] else "steelblue"
        for group in prior_count_target_summary.index
    ],
)

# Show the overall rate as a reference.
overall_rate = prior_count_data["repeat_purchase_90d"].mean() * 100
ax.axhline(
    overall_rate,
    color="darkorange",
    linestyle="--",
    label=f"Overall training rate: {overall_rate:.2f}%",
)

# Include observation and customer counts to show group concentration.
for position, row in enumerate(prior_count_target_summary.itertuples()):
    ax.text(
        position,
        max(row.repeat_purchase_pct - 12, 3),
        (
            f"{row.repeat_purchase_pct:.2f}%\n"
            f"n = {row.observations:,}\n"
            f"customers = {row.unique_customers:,}"
        ),
        ha="center",
        va="center",
        fontsize=9,
    )

ax.set_xticks(positions)
ax.set_xticklabels(prior_count_target_summary.index)
ax.set_ylim(0, 100)
ax.set_xlabel("Number of prior purchases")
ax.set_ylabel("Repeat-purchase rate (%)")
ax.set_title("Repeat-purchase rate by recorded purchase history")

# Keep the legend above the bars to avoid obscuring the highest rate.
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.02))
plt.tight_layout()
plt.show()

#### Interpretation — Repeat Purchases by Prior Purchase Count

The repeat-purchase rate increases across the observed history ranges,
from 33.33% for observations without prior purchases to 93.22% for
those with 51 or more prior purchases.

The 11–20 group has a rate of 56.20%, close to the overall training
rate of 55.92%. Groups with fewer prior purchases fall below this
reference, while the 21–50 and 51+ groups exceed it.

However, larger-history groups contain many observations from relatively
few customers. The 51+ group contains 708 observations from only
21 customers, while the 21–50 group contains 1,367 observations from
84 customers. Their rates may therefore be strongly influenced by
a small number of frequently observed customers.

The results show a strong descriptive association between recorded
purchase history and repeat purchases. Validation must assess whether
this pattern generalises to later purchases or different customers,
depending on the intended prediction setting.

The zero-history group is the same set of observations with zero tenure.
Its repeat-purchase rate of 33.33% therefore also describes that group.

#### Repeat-Purchase Rates by Customer Tenure

We compare repeat-purchase rates across fixed tenure ranges, measured
in days. Zero tenure is shown separately because it corresponds to
observations without recorded prior purchases.

Observation and customer counts help show how much data supports
each rate. These ranges are used for exploration only.

In [ ]:
# Select labelled observations without changing the original training data.
tenure_range_data = train.loc[
    train["repeat_purchase_90d"].isin([0, 1]),
    ["customer_tenure_days", "customer_id", "repeat_purchase_90d"],
].copy()

tenure_values = tenure_range_data["customer_tenure_days"]

# Assess finite, non-negative tenure values.
valid_tenure = np.isfinite(tenure_values) & tenure_values.ge(0)

# Keep zero tenure separate and use fixed day boundaries.
tenure_labels = [
    "0",
    "1–90",
    "91–365",
    "366–730",
    "731–1,825",
    "1,826+",
]

tenure_groups = pd.Series(
    "Invalid", index=tenure_range_data.index, dtype="string"
)
tenure_groups.loc[tenure_values.isna()] = "Missing"

tenure_groups.loc[valid_tenure] = pd.cut(
    tenure_values.loc[valid_tenure],
    bins=[-0.5, 0, 90, 365, 730, 1825, np.inf],
    labels=tenure_labels,
    right=True,
).astype("string")

tenure_range_data["tenure_range"] = tenure_groups

# Summarise class counts, customer coverage and repeat-purchase rates.
tenure_target_summary = (
    tenure_range_data.groupby("tenure_range")
    .agg(
        observations=("repeat_purchase_90d", "size"),
        unique_customers=("customer_id", "nunique"),
        target_0=("repeat_purchase_90d", lambda values: values.eq(0).sum()),
        target_1=("repeat_purchase_90d", "sum"),
        repeat_purchase_pct=(
            "repeat_purchase_90d", lambda values: values.mean() * 100
        ),
    )
)

# Preserve the order of tenure ranges and show only populated groups.
tenure_order = tenure_labels + ["Missing", "Invalid"]
tenure_target_summary = tenure_target_summary.reindex(
    [group for group in tenure_order if group in tenure_target_summary.index]
)

display(tenure_target_summary)

In [ ]:
# Compare tenure-group rates with the overall training rate.
fig, ax = plt.subplots(figsize=(12, 5))
positions = np.arange(len(tenure_target_summary))

ax.bar(
    positions,
    tenure_target_summary["repeat_purchase_pct"],
    color=[
        "gray" if group in ["Missing", "Invalid"] else "steelblue"
        for group in tenure_target_summary.index
    ],
)

overall_rate = tenure_range_data["repeat_purchase_90d"].mean() * 100
ax.axhline(
    overall_rate,
    color="darkorange",
    linestyle="--",
    label=f"Overall training rate: {overall_rate:.2f}%",
)

# Display both observation and customer counts for each range.
for position, row in enumerate(tenure_target_summary.itertuples()):
    ax.text(
        position,
        max(row.repeat_purchase_pct - 12, 3),
        (
            f"{row.repeat_purchase_pct:.2f}%\n"
            f"n = {row.observations:,}\n"
            f"customers = {row.unique_customers:,}"
        ),
        ha="center",
        va="center",
        fontsize=9,
    )

ax.set_xticks(positions)
ax.set_xticklabels(tenure_target_summary.index)
ax.set_ylim(0, 100)
ax.set_xlabel("Recorded customer tenure (days)")
ax.set_ylabel("Repeat-purchase rate (%)")
ax.set_title("Repeat-purchase rate by customer tenure")
ax.legend(loc="lower left", bbox_to_anchor=(0, 1.02))

plt.tight_layout()
plt.show()

#### Interpretation — Repeat Purchases by Customer Tenure

Zero-tenure observations have a repeat-purchase rate of 33.33%.
These are the same observations with no recorded prior purchases.

Among positive-tenure groups, rates range from 53.85% to 62.19%.
They do not increase consistently: the rate is 57.88% for 1–90 days,
then falls to 53.85% for 91–365 days before rising across the later ranges.

The longest-tenure group has the highest rate among positive-tenure
groups, at 62.19%, based on 1,518 observations from 148 customers.

Across the chosen ranges, tenure shows less pronounced differences
than prior purchase count. This descriptive comparison does not
establish their relative importance in a model.

Customers can appear in several tenure ranges as their recorded
history progresses. Group rates therefore reflect repeated purchases
and differences in customer composition.

### 8.5. Recent Purchase Activity and the Target

We examine purchase counts over the previous 30, 90, 180 and 365 days
to understand how recent activity relates to repeat purchases.

We will compare distributions between target classes and examine
repeat-purchase rates across activity levels, keeping zero activity
and missing values distinct.

These windows overlap, so their associations should not be interpreted
as independent contributions.

In [ ]:
# Compare historical purchase counts across the four lookback windows.
recent_count_features = [
    "purchase_count_30d",
    "purchase_count_90d",
    "purchase_count_180d",
    "purchase_count_365d",
]

recent_count_rows = []

for column in recent_count_features:
    for target in [0, 1]:
        # Select observations belonging to the requested target class.
        supplied_values = train.loc[
            train["repeat_purchase_90d"].eq(target), column
        ]

        # Calculate statistics using finite observations only.
        finite_mask = np.isfinite(supplied_values)
        values = supplied_values.loc[finite_mask]

        recent_count_rows.append({
            "variable": column,
            "target": target,
            "finite_observations": len(values),
            "missing": supplied_values.isna().sum(),
            "infinite_values": (
                supplied_values.notna() & ~finite_mask
            ).sum(),
            "exact_zeros": values.eq(0).sum(),
            "zero_pct": (
                values.eq(0).mean() * 100 if len(values) else np.nan
            ),
            "Q1": values.quantile(0.25),
            "median": values.median(),
            "mean": values.mean(),
            "Q3": values.quantile(0.75),
            "95th_percentile": values.quantile(0.95),
            "maximum": values.max(),
        })

# Display all windows together before examining their graphs separately.
recent_count_target_summary = (
    pd.DataFrame(recent_count_rows)
    .set_index(["variable", "target"])
)

display(recent_count_target_summary)

#### Interpretation — Recent Purchase Counts by Target

Repeat-purchase observations have higher mean purchase counts and
smaller proportions of zero activity across all four lookback windows.

**30-day activity**

Both classes have a median of zero, but their distributions differ.
No purchases are recorded in 79.53% of finite observations for target 0,
compared with 57.36% for target 1. Mean counts are 0.24 and 0.88
respectively.

The shared median therefore conceals differences in the proportion
with recent activity and in the upper tail.

**90-, 180- and 365-day activity**

Median counts for target 1 are one, three and five respectively,
compared with zero, one and two for target 0.

Target 1 also has higher third quartiles and 95th percentiles in
each window. The differences therefore extend beyond the means.

**Interpretation limits**

The zero percentages describe the composition of each target class.
They are not repeat-purchase rates among observations with zero activity.

These variables summarise historical purchases. `purchase_count_90d`
describes the preceding window, while `repeat_purchase_90d` represents
the future outcome.

The windows overlap and may reflect persistent differences in customer
activity. Their separate predictive contributions cannot be established
from these summaries alone.

In [ ]:
# Compare historical 30-day purchase counts between target classes.
# Keep zero-activity observations in the original and log1p panels.
plot_amount_by_target("purchase_count_30d")

#### Interpretation — 30-Day Purchase Count by Target

For target 0, the first quartile, median and third quartile are all zero.
Its boxplot therefore collapses to a line, and every positive count
appears beyond the IQR fences. This reflects the concentration at zero
and does not imply that positive counts are errors.

Target 1 also has a median of zero, but its third quartile is one.
Recent activity is therefore more common among repeat-purchase
observations despite the identical medians.

The cumulative distribution curve for target 1 lies below target 0
until they converge at the upper end. This shows a shift towards
higher historical 30-day purchase counts for target 1.

Both classes include observations with no recent activity, so this
variable does not perfectly distinguish the outcomes.

In [ ]:
# Compare historical 90-day purchase counts between target classes.
plot_amount_by_target("purchase_count_90d")

#### Interpretation — 90-Day Purchase Count by Target

Target 1 has a higher median historical purchase count, at one compared
with zero for target 0. Its third quartile is also higher, at three
compared with one.

The cumulative distribution curve for target 1 remains below target 0
until the curves converge at the upper end. This indicates generally
higher historical 90-day activity among repeat-purchase observations.

Zero activity accounts for 62.99% of finite observations in target 0
and 30.24% in target 1. Nevertheless, both outcomes occur among
observations with no activity in this window.

The boxplots show a longer upper tail for target 1. Large counts
are not automatically errors; the earlier count checks found no
fractional, negative or nested-window inconsistencies.

In [ ]:
# Compare historical 180-day purchase counts between target classes.
plot_amount_by_target("purchase_count_180d")

#### Interpretation — 180-Day Purchase Count by Target

Target 1 has a higher median historical purchase count, at three
compared with one for target 0. Its middle 50% spans one to five
purchases, compared with zero to two for target 0.

The cumulative distribution curve for target 1 remains below target 0
until the curves converge at the upper end. The difference therefore
extends across the distribution, rather than being driven only by
a few large counts.

Zero activity occurs in 32.76% of finite observations for target 0
and 13.17% for target 1. Target 1 also has a longer upper tail,
with a 95th percentile of 22 purchases compared with four.

There is still overlap between classes. This historical activity
measure shows a descriptive association with repeat purchases,
but its additional contribution beyond the other overlapping
windows requires modelling and validation.

In [ ]:
# Compare historical 365-day purchase counts between target classes.
plot_amount_by_target("purchase_count_365d")

#### Interpretation — 365-Day Purchase Count by Target

Target 1 has a higher median annual purchase count, at five compared
with two for target 0. Its middle 50% spans three to ten purchases,
compared with one to four for target 0.

The target 1 cumulative distribution curve remains below target 0
until the curves converge at the upper end. This indicates generally
higher historical annual activity among repeat-purchase observations.

The upper tails also differ: the 95th percentile is 46 purchases
for target 1 and seven for target 0. Zero activity is less common
in target 1, at 7.46% compared with 20.12%.

All four lookback windows show higher activity for target 1.
Because the windows overlap, these patterns may capture related
aspects of customer purchasing behaviour.

#### Repeat-Purchase Rates by Recent Activity Level

We compare repeat-purchase rates across fixed count ranges for each
lookback window. Zero activity remains a separate group.

The same ranges are used across windows to aid comparison, although
a given count represents different purchasing frequency over 30 days
and 365 days.

Observation and customer counts are reported alongside rates.
These groups are used for exploration only.

In [ ]:
# Use fixed activity ranges, keeping zero and one purchase separate.
activity_labels = ["0", "1", "2–3", "4–5", "6–10", "11+"]
activity_order = activity_labels + ["Missing", "Invalid"]

# Store a separate summary for each lookback window.
recent_activity_target_tables = {}

for column in recent_count_features:
    # Work on a copy containing only observations with valid targets.
    activity_data = train.loc[
        train["repeat_purchase_90d"].isin([0, 1]),
        [column, "customer_id", "repeat_purchase_90d"],
    ].copy()

    counts = activity_data[column]

    # Only finite, non-negative integer counts enter the activity ranges.
    valid_counts = (
        np.isfinite(counts)
        & counts.ge(0)
        & counts.eq(np.floor(counts))
    )

    groups = pd.Series("Invalid", index=activity_data.index, dtype="string")
    groups.loc[counts.isna()] = "Missing"

    groups.loc[valid_counts] = pd.cut(
        counts.loc[valid_counts],
        bins=[-0.5, 0, 1, 3, 5, 10, np.inf],
        labels=activity_labels,
        right=True,
    ).astype("string")

    activity_data["activity_range"] = groups

    # Report how many observations and customers support each rate.
    summary = (
        activity_data.groupby("activity_range")
        .agg(
            observations=("repeat_purchase_90d", "size"),
            unique_customers=("customer_id", "nunique"),
            target_0=("repeat_purchase_90d", lambda values: values.eq(0).sum()),
            target_1=("repeat_purchase_90d", "sum"),
            repeat_purchase_pct=(
                "repeat_purchase_90d", lambda values: values.mean() * 100
            ),
        )
    )

    # Preserve range order and omit groups with no observations.
    summary = summary.reindex(
        [group for group in activity_order if group in summary.index]
    )

    recent_activity_target_tables[column] = summary

# Start with the 30-day table; the other summaries are stored for later cells.
display(recent_activity_target_tables["purchase_count_30d"])

In [ ]:
def plot_recent_activity_target(column):
    """Plot repeat-purchase rates and group sizes for a lookback window."""

    summary = recent_activity_target_tables[column]

    fig, ax = plt.subplots(figsize=(12, 5))
    positions = np.arange(len(summary))

    # Keep missing and invalid groups visually distinct.
    ax.bar(
        positions,
        summary["repeat_purchase_pct"],
        color=[
            "gray" if group in ["Missing", "Invalid"] else "steelblue"
            for group in summary.index
        ],
    )

    # Use all valid training targets for the reference rate.
    overall_rate = (
        train.loc[
            train["repeat_purchase_90d"].isin([0, 1]),
            "repeat_purchase_90d",
        ].mean() * 100
    )

    ax.axhline(
        overall_rate,
        color="darkorange",
        linestyle="--",
        label=f"Overall training rate: {overall_rate:.2f}%",
    )

    # Show customer coverage alongside observation counts.
    for position, row in enumerate(summary.itertuples()):
        ax.text(
            position,
            max(row.repeat_purchase_pct - 12, 3),
            (
                f"{row.repeat_purchase_pct:.2f}%\n"
                f"n = {row.observations:,}\n"
                f"customers = {row.unique_customers:,}"
            ),
            ha="center",
            va="center",
            fontsize=9,
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(summary.index)
    ax.set_ylim(0, 100)
    ax.set_xlabel("Historical purchase count")
    ax.set_ylabel("Repeat-purchase rate (%)")
    ax.set_title(f"{column} — repeat-purchase rate by activity level")
    ax.legend(loc="lower left", bbox_to_anchor=(0, 1.02))

    plt.tight_layout()
    plt.show()

In [ ]:
# Visualise rates across historical 30-day activity levels.
plot_recent_activity_target("purchase_count_30d")

#### Interpretation — Repeat Purchases by 30-Day Activity

The repeat-purchase rate increases across the observed activity ranges:
47.84% with no historical 30-day purchases, 62.37% with one purchase
and 86.40% with two or three purchases.

The higher-count groups have particularly high rates, but their customer
coverage is limited. The 4–5 group contains 130 observations from
12 customers. The 6–10 group contains 96 observations from only two
customers, while the 11+ group contains a single observation.

The observed 100% rates therefore do not establish that high recent
activity guarantees a repeat purchase or that these rates will generalise.

Zero recent activity also does not imply no future repeat purchase:
2,077 observations in this group have target 1.

Recent activity shows a clear descriptive association with the target,
but repeated observations from a few highly active customers influence
the upper activity groups.

In [ ]:
# Examine rates and customer coverage for historical 90-day activity.
display(recent_activity_target_tables["purchase_count_90d"])

#### Interpretation — 90-Day Purchase Activity

The repeat-purchase rate increases across the historical activity groups:
from 37.90% for observations with no purchases in the previous 90 days
to 56.77% with one purchase and 74.71% with two or three purchases.
It reaches 90.36% for four or five purchases and 97.71% for six to ten.

Higher historical purchase frequency is therefore associated with a
higher observed probability of another purchase within the following
90 days. Historical activity and the target refer to different periods:
previous purchases describe the past, while the target describes the
subsequent outcome.

Customer coverage becomes narrower in the higher activity groups.
The six-to-ten group contains 306 observations from 16 customers,
while all 196 observations in the eleven-or-more group come from
just two customers. Its 100% repeat-purchase rate should therefore
be interpreted cautiously.

No historical activity does not rule out a subsequent repeat purchase:
1,095 of the 2,889 observations in the zero-purchase group have target 1.

These rates describe purchase observations rather than equally weighted
customers. Repeated observations from highly active customers can
strongly influence the results.

In [ ]:
# Examine repeat-purchase rates across historical 180-day activity groups.
display(recent_activity_target_tables["purchase_count_180d"])

In [ ]:
# Plot repeat-purchase rates and customer coverage for 180-day activity.
plot_recent_activity_target("purchase_count_180d")

#### Interpretation — 180-Day Purchase Activity

Repeat-purchase rates increase steadily across historical activity groups.
The rate is 33.83% for observations with no purchases in the previous
180 days and 41.67% for those with one purchase. It rises to 58.60%
with two or three purchases and 76.97% with four or five.

The highest activity groups have rates of 89.28% for six to ten purchases
and 99.39% for eleven or more. This supports the association between
higher historical purchase frequency and subsequent repeat purchases.

However, the eleven-or-more group contains 493 observations from only
10 customers. Its high rate therefore reflects repeated observations
from a relatively small customer group and may not generalise to
other customers.

The zero-activity group still contains 477 observations followed by
a repeat purchase. An absence of purchases during the preceding
180 days therefore does not imply that the customer will remain inactive.

These are observation-level associations. Customers can appear in
different activity groups at different purchase dates, so the groups
do not represent separate sets of customers.

In [ ]:
# Examine repeat-purchase rates across historical 365-day activity groups.
display(recent_activity_target_tables["purchase_count_365d"])

In [ ]:
# Plot repeat-purchase rates and customer coverage for annual activity.
plot_recent_activity_target("purchase_count_365d")

#### Interpretation — 365-Day Purchase Activity

The repeat-purchase rate increases across annual activity groups,
from 32.03% with no historical purchases to 34.73% with one purchase
and 40.86% with two or three.

Observations with four or five purchases have a rate of 56.45%,
close to the overall training rate of 55.92%. Rates rise further
to 78.05% with six to ten purchases and 96.46% with eleven or more.

The highest activity group contains 847 observations from 26 customers.
Although it covers more customers than the highest groups in the shorter
windows, repeated observations still influence its rate.

The same purchase count represents different activity levels depending
on the window. For example, two or three purchases within 30 days
indicate more frequent activity than two or three purchases within
365 days. Rates across windows should therefore be compared with
their different durations in mind.

Across all four windows, higher historical purchase counts are associated
with higher subsequent repeat-purchase rates. However, the windows
overlap and their counts are related. These analyses do not establish
how much additional predictive information each variable contributes.

### 8.6. Historical Purchase Values and the Target

We will compare historical spending and purchase-value statistics between
observations with and without a subsequent repeat purchase.

The analysis covers:
- Total spending before the current purchase.
- Mean previous purchase value.
- Variation in previous purchase values.
- Smallest and largest previous purchase values.

Some historical values failed the earlier consistency checks.
We will first summarise the supplied values, then use separate diagnostic
views where necessary to examine how those inconsistencies affect
the comparisons.

Missing historical statistics may reflect insufficient purchase history.
Their frequency will therefore remain visible alongside the observed values.

In [ ]:
# Historical monetary variables to compare with the target.
historical_value_features = [
    "prior_total_spend",
    "prior_mean_purchase_value",
    "prior_std_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
]

# Store one summary for each variable and target class.
historical_value_rows = []

for column in historical_value_features:
    for target_value in [0, 1]:
        # Select the supplied values belonging to this target class.
        class_data = train.loc[
            train["repeat_purchase_90d"].eq(target_value),
            [column, "customer_id"],
        ]

        values = class_data[column]

        # Use finite observations for numerical statistics.
        # Negative values remain included here so anomalies stay visible.
        finite_mask = np.isfinite(values)
        finite_values = values.loc[finite_mask]

        historical_value_rows.append({
            "variable": column,
            "target": target_value,
            "class_rows": len(class_data),
            "finite_observations": len(finite_values),
            "unique_customers_with_finite_values": (
                class_data.loc[finite_mask, "customer_id"].nunique()
            ),
            "missing": int(values.isna().sum()),
            "infinite_values": int(
                (values.notna() & ~finite_mask).sum()
            ),
            "negative_values": int(finite_values.lt(0).sum()),
            "exact_zeros": int(finite_values.eq(0).sum()),
            "minimum": finite_values.min(),
            "Q1": finite_values.quantile(0.25),
            "median": finite_values.median(),
            "mean": finite_values.mean(),
            "Q3": finite_values.quantile(0.75),
            "95th_percentile": finite_values.quantile(0.95),
            "maximum": finite_values.max(),
        })

# Keep variable and target together for an organised comparison.
historical_value_target_summary = (
    pd.DataFrame(historical_value_rows)
    .set_index(["variable", "target"])
)

display(historical_value_target_summary)

#### Interpretation — Historical Purchase Values by Target

**Total historical spending**

Observations followed by a repeat purchase have higher recorded historical
spending. Their median is 12,098.99, compared with 3,246.53 for target 0.
The first and third quartiles are also higher.

Both groups are right-skewed, with means above their medians.
Total spending reflects both the number and value of previous purchases,
so this comparison does not separate those effects.

**Mean previous purchase value**

The median previous purchase value is 789.94 for target 1 and 632.41
for target 0. However, both groups contain values of 1,000,000,000,
which heavily distort their means.

There are also six negative values in target 0 and ten in target 1.
The earlier checks identified these as sign reversals of total spending
divided by purchase count. A separate diagnostic comparison is needed
before interpreting the class differences in this variable.

**Variation in previous purchase values**

The median historical standard deviation is 650.04 for target 1,
compared with 426.38 for target 0. This indicates greater absolute
variation in previous purchase values among the observed target 1 records.

Standard deviation is measured in monetary units. Higher values may
partly reflect larger purchase amounts rather than greater relative
variability.

**Smallest and largest previous purchases**

The median smallest previous purchase is lower for target 1:
45.51 versus 88.11. Conversely, its median largest previous purchase
is higher: 2,306.25 versus 1,342.62.

These patterns are consistent with a wider range of historical purchase
values. Customers with more previous purchases also have more opportunities
to record smaller minima and larger maxima, so purchase frequency
must be considered when interpreting these differences.

The supplied maximum values require particular caution because
45 observations previously failed the maximum-versus-total-spending check.

**Coverage and missing values**

Historical mean, standard deviation, minimum and maximum values are
missing more frequently within target 0 than within target 1.
Some of this difference reflects the greater proportion of observations
with little or no purchase history in target 0.

The comparisons describe available purchase observations. They do not
give each customer equal weight or establish independent effects
on repeat purchasing.

In [ ]:
# Compare supplied total historical spending between target classes.
# Zero spending is retained because it represents no recorded prior spending.
plot_amount_by_target("prior_total_spend")

#### Interpretation — Total Historical Spending

Observations followed by a repeat purchase generally have higher
historical spending. Their median is 12,098.99, compared with
3,246.53 for observations without a repeat purchase.

The original-scale boxplots show long upper tails in both classes.
The log1p view makes the central distributions easier to compare,
although substantial overlap remains.

The target 1 ECDF lies below the target 0 curve across most of the
range. This means that a smaller proportion of target 1 observations
falls below a given spending threshold.

The jumps at zero reflect observations with no recorded prior spending:
14.12% of finite target 0 observations and 5.55% of finite target 1
observations.

High historical spending may reflect more purchases, larger purchases
or both. This analysis does not establish an effect independent of
purchase frequency or customer tenure.

Points beyond the boxplot fences are statistical flags rather than
confirmed errors. The log transformation changes those fences, so
different observations can be flagged on the two scales.

In [ ]:
# Inspect the supplied historical mean purchase values by target.
# Negative values remain in the original-scale panel.
# The plotting function excludes them from the log1p panels.
plot_amount_by_target("prior_mean_purchase_value")

#### Interpretation — Supplied Mean Previous Purchase Value

Values of 1,000,000,000 occur in both target classes and dominate
the original-scale plot. Their influence also explains the very large
class means reported in the summary table.

The log1p boxplots show substantial overlap, with a slightly higher
central distribution for target 1. The ECDF curves are also close
through much of the distribution.

The original-scale panel includes 5,672 finite observations.
The log1p panels exclude the 16 negative values, leaving 5,656.
Other inconsistent positive values remain included.

A diagnostic comparison will therefore exclude supplied means that
failed the earlier total-spending-divided-by-purchase-count check.
This will help assess whether the central class differences remain
after removing those identified inconsistencies.

#### Diagnostic Comparison — Consistent Historical Means

This comparison retains finite, non-negative historical means where
a positive prior purchase count and finite total spending allow
the mean-consistency check to be assessed and passed.

Excluded values will be counted separately for each target class.
Passing this check establishes agreement with the related supplied
variables, rather than guaranteeing that the underlying history is correct.

In [ ]:
# Identify observations for which the historical mean can be checked.
mean_check_assessable = (
    np.isfinite(train["prior_purchase_count"])
    & train["prior_purchase_count"].gt(0)
    & np.isfinite(train["prior_total_spend"])
    & np.isfinite(train["prior_mean_purchase_value"])
)

# Reuse the earlier check: True indicates an inconsistency.
mean_check_failed = historical_consistency_flags[
    "Mean equals total spending / purchase count"
].reindex(train.index).fillna(False)

# Retain supplied values that passed the assessable consistency check.
mean_diagnostic_mask = (
    train["repeat_purchase_90d"].isin([0, 1])
    & mean_check_assessable
    & ~mean_check_failed
    & train["prior_mean_purchase_value"].ge(0)
)

# Create a separate diagnostic object for this variable.
historical_mean_target_diagnostic = train.loc[
    mean_diagnostic_mask,
    ["customer_id", "repeat_purchase_90d", "prior_mean_purchase_value"],
].copy()

# Document coverage and exclusions within each target class.
mean_coverage_rows = []

for target_value in [0, 1]:
    class_mask = train["repeat_purchase_90d"].eq(target_value)
    supplied_finite = (
        class_mask
        & np.isfinite(train["prior_mean_purchase_value"])
    )
    retained = class_mask & mean_diagnostic_mask

    mean_coverage_rows.append({
        "target": target_value,
        "class_rows": int(class_mask.sum()),
        "missing_supplied_means": int(
            train.loc[class_mask, "prior_mean_purchase_value"].isna().sum()
        ),
        "finite_supplied_means": int(supplied_finite.sum()),
        "finite_means_excluded": int(
            (supplied_finite & ~mean_diagnostic_mask).sum()
        ),
        "retained_means": int(retained.sum()),
        "retained_unique_customers": (
            train.loc[retained, "customer_id"].nunique()
        ),
    })

display(pd.DataFrame(mean_coverage_rows).set_index("target"))

# Compare the retained supplied means using the same descriptive statistics.
display(
    historical_mean_target_diagnostic
    .groupby("repeat_purchase_90d")["prior_mean_purchase_value"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95])
)

#### Interpretation — Diagnostic Historical Means

The diagnostic comparison retains 5,594 observations: 2,332 with
target 0 and 3,262 with target 1. It excludes 33 finite supplied
means from target 0 and 45 from target 1, matching the 78 records
flagged by the earlier mean-consistency check.

After these exclusions, the mean previous purchase value is 805.82
for target 0 and 915.91 for target 1. The corresponding medians
are 626.62 and 781.27.

The first quartile, third quartile and 95th percentile are also
higher for target 1. The difference therefore extends beyond
the class means and remains after excluding the identified inconsistencies.

Both distributions remain right-skewed, with means above their medians.
Their interquartile ranges overlap substantially, so historical mean
purchase value alone does not clearly separate the outcomes.

These results apply to observations with an assessable and consistent
historical mean. They exclude missing values, including statistics
undefined for observations without prior purchases.

In [ ]:
# Prepare plotting labels on a separate copy of the diagnostic data.
mean_plot_data = historical_mean_target_diagnostic.copy()

target_order = [
    "0 — No repeat purchase",
    "1 — Repeat purchase",
]

mean_plot_data["target_class"] = (
    mean_plot_data["repeat_purchase_90d"].map({
        0: target_order[0],
        1: target_order[1],
    })
)

# Compress the right tail while preserving the order of purchase values.
mean_plot_data["log1p_mean_purchase_value"] = np.log1p(
    mean_plot_data["prior_mean_purchase_value"]
)

# Use consistent target colours across the three panels.
target_palette = {
    target_order[0]: "tab:blue",
    target_order[1]: "tab:orange",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Show the retained monetary values on their original scale.
sns.boxplot(
    data=mean_plot_data,
    x="target_class",
    y="prior_mean_purchase_value",
    hue="target_class",
    order=target_order,
    palette=target_palette,
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Original scale")
axes[0].set_ylabel("Mean previous purchase value")

# Compare the central distributions on a log1p scale.
sns.boxplot(
    data=mean_plot_data,
    x="target_class",
    y="log1p_mean_purchase_value",
    hue="target_class",
    order=target_order,
    palette=target_palette,
    legend=False,
    ax=axes[1],
)
axes[1].set_title("Log1p scale")
axes[1].set_ylabel("log(1 + mean previous purchase value)")

# Compare cumulative proportions without choosing histogram bins.
sns.ecdfplot(
    data=mean_plot_data,
    x="log1p_mean_purchase_value",
    hue="target_class",
    hue_order=target_order,
    palette=target_palette,
    ax=axes[2],
)
axes[2].set_title("ECDF — log1p scale")
axes[2].set_xlabel("log(1 + mean previous purchase value)")
axes[2].set_ylabel("Cumulative proportion")

# Keep the class labels readable beneath both boxplots.
for ax in axes[:2]:
    ax.set_xlabel("Target class")
    ax.tick_params(axis="x", labelrotation=15)

fig.suptitle(
    "Historical mean purchase value by target — diagnostic comparison",
    fontsize=15,
)
plt.tight_layout()
plt.show()

#### Interpretation — Diagnostic Distribution Comparison

Both boxplots show a higher median and higher quartiles for target 1,
although the distributions overlap substantially.

The target 1 ECDF lies below the target 0 curve through most of the
central range, indicating generally higher historical mean purchase
values. The curves cross at some lower values and converge in the
upper tail, so this difference is not uniform across the distribution.

The original-scale boxplots retain long upper tails after the identified
inconsistencies are excluded. Values beyond the IQR fences therefore
remain present even among records that passed the mean-consistency check.
These statistical flags alone do not establish further errors.

In [ ]:
# Compare historical purchase-value standard deviations by target.
# Retain observed zeros: identical previous purchase values can give zero SD.
plot_amount_by_target("prior_std_purchase_value")

#### Interpretation — Variation in Previous Purchase Values

The median historical standard deviation is 650.04 for target 1,
compared with 426.38 for target 0. The first and third quartiles
are also higher for target 1.

The ECDF for target 1 lies below the target 0 curve through most
of the distribution. This supports the pattern of greater absolute
variation in previous purchase amounts among observations followed
by a repeat purchase.

Both classes have right-skewed distributions and substantial overlap.
The largest recorded standard deviation belongs to target 0:
6,770.22, compared with 5,158.56 for target 1. This does not contradict
the higher central values for target 1.

Standard deviation measures variation in monetary units. Larger
purchase amounts can produce larger standard deviations, so these
results do not establish greater variability relative to purchase size.

The comparison includes 5,421 observed values. Missing statistics
include observations with insufficient purchase history, as established
in the earlier missingness analysis.

In [ ]:
# Compare the smallest recorded previous purchase value by target.
# This describes a historical purchase minimum, not total historical spending.
plot_amount_by_target("prior_min_purchase_value")

#### Interpretation — Smallest Previous Purchase Value

The median smallest previous purchase is 45.51 for target 1,
compared with 88.11 for target 0. The first and third quartiles
are also lower for target 1.

The target 1 ECDF lies above the target 0 curve across most of
the range. A larger proportion of target 1 observations therefore
has a historical minimum below a given purchase-value threshold.

Both distributions have long upper tails and substantial overlap.
A high historical minimum means that even the smallest recorded
previous purchase was relatively large.

This variable depends on the number of previous purchases.
A longer purchase history provides more opportunities to record
a small purchase, which may partly explain the lower minima
among target 1 observations.

The comparison includes 5,866 observed values. The 668 missing
values correspond to the 603 observations without prior purchases
and the 65 observations missing all purchase information.

These results do not imply that making a smaller purchase causes
a subsequent repeat purchase.

In [ ]:
# Inspect supplied historical maximum purchase values by target.
# The 45 maxima previously flagged as exceeding total spending remain visible.
plot_amount_by_target("prior_max_purchase_value")

#### Interpretation — Supplied Largest Previous Purchase Value

The median value of the largest previous purchase is 2,306.25
for target 1, compared with 1,342.62 for target 0. The first
and third quartiles are also higher for target 1.

The target 1 ECDF lies below the target 0 curve through most
of the central range. This indicates that observations followed
by a repeat purchase generally have larger previous purchases,
although substantial overlap remains.

Extreme supplied values compress the original-scale boxplots.
The earlier consistency check identified 45 observations where
the largest previous purchase exceeded total historical spending.

The log1p plots improve readability but retain these inconsistencies.
A separate diagnostic comparison will assess whether the class
difference remains after excluding them.

Customers with more previous purchases have more opportunities
to record a large purchase. Purchase frequency may therefore
partly explain the differences between the target classes.

#### Diagnostic Comparison — Largest Previous Purchase Value

This comparison retains observed largest previous purchase values
that do not exceed total historical spending, allowing one cent
for monetary rounding.

We will report exclusions for each target class and compare
the retained values. Passing this check confirms agreement with
total spending but does not guarantee that every value is correct.

In [ ]:
# Identify rows where the spending comparison can be assessed.
maximum_check_assessable = (
    np.isfinite(train["prior_purchase_count"])
    & train["prior_purchase_count"].gt(0)
    & np.isfinite(train["prior_total_spend"])
    & np.isfinite(train["prior_max_purchase_value"])
)

# Flag a largest previous purchase that exceeds total historical spending.
maximum_exceeds_total = (
    train["prior_max_purchase_value"]
    > train["prior_total_spend"] + 0.01
)

# Select values for this diagnostic comparison.
maximum_diagnostic_mask = (
    train["repeat_purchase_90d"].isin([0, 1])
    & maximum_check_assessable
    & train["prior_max_purchase_value"].ge(0)
    & ~maximum_exceeds_total
)

# Store the selected observations separately.
historical_maximum_target_diagnostic = train.loc[
    maximum_diagnostic_mask,
    ["customer_id", "repeat_purchase_90d", "prior_max_purchase_value"],
].copy()

# Count retained and excluded finite values within each target class.
maximum_coverage_rows = []

for target_value in [0, 1]:
    class_mask = train["repeat_purchase_90d"].eq(target_value)
    supplied_finite = (
        class_mask
        & np.isfinite(train["prior_max_purchase_value"])
    )
    retained = class_mask & maximum_diagnostic_mask

    maximum_coverage_rows.append({
        "target": target_value,
        "finite_supplied_values": int(supplied_finite.sum()),
        "finite_values_excluded": int(
            (supplied_finite & ~maximum_diagnostic_mask).sum()
        ),
        "retained_values": int(retained.sum()),
        "retained_unique_customers": (
            train.loc[retained, "customer_id"].nunique()
        ),
    })

display(pd.DataFrame(maximum_coverage_rows).set_index("target"))

# Summarise the retained largest previous purchase values by target.
display(
    historical_maximum_target_diagnostic
    .groupby("repeat_purchase_90d")["prior_max_purchase_value"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95])
)

#### Interpretation — Diagnostic Largest Previous Purchase Values

The comparison retains 5,821 observations after excluding 21 values
from target 0 and 24 from target 1.

The median largest previous purchase is 2,306.25 for target 1,
compared with 1,335.46 for target 0. The first quartile, third quartile
and 95th percentile are also higher for target 1.

The means decrease after the exclusions, from 2,249.21 to 1,970.02
for target 0 and from 3,445.00 to 3,118.44 for target 1.
The class difference therefore remains after removing values
that exceeded total historical spending.

Both distributions remain right-skewed, with means above their medians
and overlapping interquartile ranges.

The largest retained value is 60,072.51 for target 1.
It passes this particular spending check, but that alone does not
establish its accuracy. Other inconsistencies could remain undetected.

In [ ]:
# Prepare labels and transformed values on a separate plotting object.
maximum_plot_data = historical_maximum_target_diagnostic.copy()

maximum_plot_data["target_class"] = (
    maximum_plot_data["repeat_purchase_90d"].map({
        0: "0 — No repeat purchase",
        1: "1 — Repeat purchase",
    })
)

maximum_plot_data["log1p_previous_purchase_maximum"] = np.log1p(
    maximum_plot_data["prior_max_purchase_value"]
)

# Keep target ordering and colours consistent with earlier graphs.
target_order = ["0 — No repeat purchase", "1 — Repeat purchase"]
target_palette = {
    target_order[0]: "tab:blue",
    target_order[1]: "tab:orange",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Compare retained values on the original and log1p scales.
for ax, column, title, ylabel in [
    (
        axes[0],
        "prior_max_purchase_value",
        "Original scale",
        "Largest previous purchase value",
    ),
    (
        axes[1],
        "log1p_previous_purchase_maximum",
        "Log1p scale",
        "log(1 + largest previous purchase value)",
    ),
]:
    sns.boxplot(
        data=maximum_plot_data,
        x="target_class",
        y=column,
        hue="target_class",
        order=target_order,
        palette=target_palette,
        legend=False,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Target class")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", labelrotation=15)

# Show cumulative proportions for both target classes.
sns.ecdfplot(
    data=maximum_plot_data,
    x="log1p_previous_purchase_maximum",
    hue="target_class",
    hue_order=target_order,
    palette=target_palette,
    ax=axes[2],
)
axes[2].set_title("ECDF — log1p scale")
axes[2].set_xlabel("log(1 + largest previous purchase value)")
axes[2].set_ylabel("Cumulative proportion")

fig.suptitle(
    "Largest previous purchase value by target — diagnostic comparison",
    fontsize=15,
)
plt.tight_layout()
plt.show()

#### Interpretation — Diagnostic Distribution Comparison

After excluding the 45 flagged values, target 1 still has higher
typical largest previous purchase values. Its ECDF lies below
the target 0 curve through most of the range, although the
distributions overlap.

Both classes retain right-skewed distributions. Passing the spending
check does not guarantee accuracy, and purchase frequency may
partly explain the class differences.

### 8.7. Historical Spending Windows and the Target

We will compare spending during the previous 30, 90, 180 and
365 days between target classes.

Zeros represent no recorded spending in a window. The annual
spending variable also contains previously identified inconsistencies,
which will be examined through a separate diagnostic comparison.

In [ ]:
# Compare supplied historical spending within each target class.
window_spending_features = [
    "spend_30d",
    "spend_90d",
    "spend_180d",
    "spend_365d",
]

window_spending_rows = []

for column in window_spending_features:
    for target_value in [0, 1]:
        # Select this target class without changing the original data.
        values = train.loc[
            train["repeat_purchase_90d"].eq(target_value), column
        ]

        # Keep negative finite values visible in the supplied-data summary.
        finite_values = values.loc[np.isfinite(values)]

        window_spending_rows.append({
            "variable": column,
            "target": target_value,
            "finite_observations": len(finite_values),
            "missing": int(values.isna().sum()),
            "negative_values": int(finite_values.lt(0).sum()),
            "exact_zeros": int(finite_values.eq(0).sum()),
            "zero_pct": (
                finite_values.eq(0).mean() * 100
                if len(finite_values) else np.nan
            ),
            "Q1": finite_values.quantile(0.25),
            "median": finite_values.median(),
            "mean": finite_values.mean(),
            "Q3": finite_values.quantile(0.75),
            "95th_percentile": finite_values.quantile(0.95),
            "maximum": finite_values.max(),
        })

window_spending_target_summary = (
    pd.DataFrame(window_spending_rows)
    .set_index(["variable", "target"])
)

display(window_spending_target_summary)

#### Interpretation — Historical Spending by Target

Target 1 has higher typical spending across the historical windows.
For example, median 90-day spending is 746.19 versus zero for target 0,
while median 180-day spending is 2,106.22 versus 457.87.

Both classes have median 30-day spending of zero, but zero spending
is more common in target 0: 79.53% versus 57.36%.

Annual spending also has a higher median for target 1. However,
billion-valued entries distort both class means, and 15 negative
values remain present. A diagnostic comparison is needed for this window.

Spending reflects purchase frequency and purchase amounts.
These comparisons do not separate their contributions.

In [ ]:
# Compare spending in the previous 30 days between target classes.
# Retain zeros to show observations without recorded spending in this window.
plot_amount_by_target("spend_30d")

#### Interpretation — 30-Day Spending

Both classes have median spending of zero, but zero spending is
more common in target 0: 79.53% versus 57.36%. The ECDF shows
this difference through its larger jump at zero for target 0.

The target 1 curve remains below target 0 across most positive
thresholds, indicating generally higher spending when considering
the full distributions.

For target 0, both quartiles are zero. Its boxplot therefore flags
every positive amount beyond the IQR fence; this does not mean
those purchases are errors.

The comparison combines whether any spending occurred with
how much was spent. It does not isolate spending amounts among
active observations.

In [ ]:
# Compare spending in the previous 90 days between target classes.
plot_amount_by_target("spend_90d")

#### Interpretation — 90-Day Spending

Median historical 90-day spending is 746.19 for target 1,
compared with zero for target 0. Zero spending is also much
more common in target 0: 62.99% versus 30.24%.

The target 1 ECDF remains below target 0 across most of the range,
showing generally higher spending. Both distributions have long
upper tails and overlap.

This comparison combines purchase frequency and purchase amounts.
Historical 90-day spending describes the preceding period,
while the target concerns repeat purchasing in the following 90 days.

In [ ]:
# Compare spending in the previous 180 days between target classes.
plot_amount_by_target("spend_180d")

#### Interpretation — 180-Day Spending

Median historical 180-day spending is 2,106.22 for target 1,
compared with 457.87 for target 0. Zero spending is less common
in target 1: 13.17% versus 32.76%.

The higher quartiles and lower ECDF curve for target 1 show
generally higher spending across the distribution. Both classes
remain right-skewed and overlap.

These differences may reflect purchase frequency, purchase amounts
or both; their separate contributions have not yet been assessed.

In [ ]:
# Inspect supplied spending in the previous 365 days by target.
# Negative values remain in the original-scale panel.
# The log1p panels exclude negatives but retain inconsistent positive values.
plot_amount_by_target("spend_365d")

#### Interpretation — Supplied Annual Spending

Billion-valued entries dominate the original-scale plot.
The log1p panels reveal higher central spending for target 1,
but retain inconsistent positive values.

The original panel includes 6,274 finite values. The log1p
panels exclude 15 negative values, leaving 6,259 observations.

A diagnostic comparison will exclude values flagged by the earlier
spending checks before interpreting the annual class differences.

#### Diagnostic Comparison — Annual Spending

This view excludes negative annual spending and observations flagged
by the spending-order or zero-count spending checks.

Zeros remain included where no inconsistency was identified.
The retained values satisfy these checks but are not guaranteed
to be free of other errors.

In [ ]:
# Combine the existing spending flags, keeping their original row alignment.
annual_spending_flagged = (
    any_window_spending_flag.reindex(train.index).fillna(False)
    | any_zero_count_flag.reindex(train.index).fillna(False)
)

# Retain finite, non-negative annual spending without identified inconsistencies.
annual_target_diagnostic_mask = (
    train["repeat_purchase_90d"].isin([0, 1])
    & np.isfinite(train["spend_365d"])
    & train["spend_365d"].ge(0)
    & ~annual_spending_flagged
)

# Store a separate diagnostic dataset for the target comparison.
annual_spending_target_diagnostic = train.loc[
    annual_target_diagnostic_mask,
    ["customer_id", "repeat_purchase_90d", "spend_365d"],
].copy()

# Report coverage and exclusions within each target class.
annual_coverage_rows = []

for target_value in [0, 1]:
    class_mask = train["repeat_purchase_90d"].eq(target_value)
    supplied_finite = class_mask & np.isfinite(train["spend_365d"])
    retained = class_mask & annual_target_diagnostic_mask

    annual_coverage_rows.append({
        "target": target_value,
        "finite_supplied_values": int(supplied_finite.sum()),
        "finite_values_excluded": int(
            (supplied_finite & ~annual_target_diagnostic_mask).sum()
        ),
        "retained_values": int(retained.sum()),
        "retained_unique_customers": (
            train.loc[retained, "customer_id"].nunique()
        ),
    })

display(pd.DataFrame(annual_coverage_rows).set_index("target"))

# Summarise retained annual spending, including valid zeros.
display(
    annual_spending_target_diagnostic
    .groupby("repeat_purchase_90d")["spend_365d"]
    .describe(percentiles=[0.25, 0.5, 0.75, 0.95])
)

#### Interpretation — Diagnostic Annual Spending

After excluding 31 values from target 0 and 38 from target 1,
median annual spending remains higher for target 1:
4,487.74 versus 1,126.84.

The means are now 7,343.23 and 2,508.38 respectively, with higher
quartiles and a higher 95th percentile for target 1. The class
difference therefore remains after excluding the identified inconsistencies.

Both distributions remain right-skewed. These comparisons still
combine purchase frequency and purchase amounts.

In [ ]:
# Prepare a separate plotting object with readable target labels.
annual_plot_data = annual_spending_target_diagnostic.copy()

target_order = ["0 — No repeat purchase", "1 — Repeat purchase"]
target_palette = {
    target_order[0]: "tab:blue",
    target_order[1]: "tab:orange",
}

annual_plot_data["target_class"] = (
    annual_plot_data["repeat_purchase_90d"].map({
        0: target_order[0],
        1: target_order[1],
    })
)

# Retain zeros and compress the positive spending tail.
annual_plot_data["log1p_annual_spending"] = np.log1p(
    annual_plot_data["spend_365d"]
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Compare retained annual spending on the original and log1p scales.
for ax, column, title, ylabel in [
    (axes[0], "spend_365d", "Original scale", "Annual spending"),
    (
        axes[1],
        "log1p_annual_spending",
        "Log1p scale",
        "log(1 + annual spending)",
    ),
]:
    sns.boxplot(
        data=annual_plot_data,
        x="target_class",
        y=column,
        hue="target_class",
        order=target_order,
        palette=target_palette,
        legend=False,
        ax=ax,
    )
    ax.set_title(title)
    ax.set_xlabel("Target class")
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", labelrotation=15)

# Compare cumulative proportions, including the concentration at zero.
sns.ecdfplot(
    data=annual_plot_data,
    x="log1p_annual_spending",
    hue="target_class",
    hue_order=target_order,
    palette=target_palette,
    ax=axes[2],
)
axes[2].set_title("ECDF — log1p scale")
axes[2].set_xlabel("log(1 + annual spending)")
axes[2].set_ylabel("Cumulative proportion")

fig.suptitle(
    "Annual spending by target — diagnostic comparison",
    fontsize=15,
)
plt.tight_layout()
plt.show()

#### Interpretation — Diagnostic Annual Spending Distributions

Target 1 retains higher central annual spending after the flagged
values are excluded. Its ECDF lies below target 0 across most
of the range, with a smaller concentration at zero.

Both distributions have long upper tails and overlap. The pattern
remains associated with repeat purchasing, but does not establish
an effect independent of purchase frequency.

### 8.8. Purchase Intervals and the Target

We will compare time since the previous purchase, the average
interval between previous purchases and the variation in those
intervals between target classes.

Negative intervals identified earlier will be excluded from separate
diagnostic comparisons. Missing statistics will remain documented,
as some require sufficient previous purchase history to be defined.

In [ ]:
# Historical time variables to compare between target classes.
purchase_interval_features = [
    "days_since_previous_purchase",
    "prior_mean_gap_days",
    "prior_std_gap_days",
]

interval_target_rows = []

for column in purchase_interval_features:
    for target_value in [0, 1]:
        # Select supplied values for this target class.
        values = train.loc[
            train["repeat_purchase_90d"].eq(target_value), column
        ]

        # Summarise only finite, non-negative intervals.
        # Report excluded values alongside the diagnostic statistics.
        finite_mask = np.isfinite(values)
        diagnostic_values = values.loc[finite_mask & values.ge(0)]

        interval_target_rows.append({
            "variable": column,
            "target": target_value,
            "class_rows": len(values),
            "missing": int(values.isna().sum()),
            "infinite_values": int(
                (values.notna() & ~finite_mask).sum()
            ),
            "negative_values_excluded": int(
                (finite_mask & values.lt(0)).sum()
            ),
            "retained_observations": len(diagnostic_values),
            "minimum": diagnostic_values.min(),
            "Q1": diagnostic_values.quantile(0.25),
            "median": diagnostic_values.median(),
            "mean": diagnostic_values.mean(),
            "Q3": diagnostic_values.quantile(0.75),
            "95th_percentile": diagnostic_values.quantile(0.95),
            "maximum": diagnostic_values.max(),
        })

interval_target_summary = (
    pd.DataFrame(interval_target_rows)
    .set_index(["variable", "target"])
)

display(interval_target_summary)

#### Interpretation — Purchase Intervals by Target

Observations followed by a repeat purchase have shorter recorded
purchase intervals. The median time since the previous purchase
is 37 days for target 1, compared with 105 days for target 0.
The median average historical interval is also lower:
55.76 versus 101.75 days.

The median standard deviation of historical intervals is 43.62 days
for target 1, compared with 65.06 days for target 0. This indicates
less absolute variation, although it does not necessarily mean
greater regularity relative to the average interval.

These comparisons exclude negative intervals and missing values.
Some missing statistics reflect insufficient purchase history,
so the results describe only observations with available values.

In [ ]:
# Compare valid, non-negative purchase intervals between target classes.
def plot_interval_by_target(column):
    values = train[column]

    # Retain finite intervals and valid binary targets.
    valid_mask = (
        train["repeat_purchase_90d"].isin([0, 1])
        & np.isfinite(values)
        & values.ge(0)
    )

    plot_data = train.loc[
        valid_mask, ["repeat_purchase_90d", column]
    ].copy()

    # Transform only the plotting copy to make long tails easier to inspect.
    plot_data["log1p_value"] = np.log1p(plot_data[column])

    target_labels = {
        0: "0 — No repeat purchase",
        1: "1 — Repeat purchase",
    }
    order = list(target_labels.values())
    palette = dict(zip(order, sns.color_palette("colorblind", 2)))

    plot_data["target_class"] = (
        plot_data["repeat_purchase_90d"].map(target_labels)
    )

    print(f"Variable: {column}")
    print(f"Missing values: {values.isna().sum():,}")
    print(
        "Negative values excluded: "
        f"{(np.isfinite(values) & values.lt(0)).sum():,}"
    )
    print(f"Observations plotted: {len(plot_data):,}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Compare the distributions in days and on a logarithmic scale.
    for ax, plotted_column, title, ylabel in [
        (axes[0], column, "Original scale", "Days"),
        (axes[1], "log1p_value", "Logarithmic scale", "log1p(days)"),
    ]:
        sns.boxplot(
            data=plot_data,
            x="target_class",
            y=plotted_column,
            hue="target_class",
            order=order,
            hue_order=order,
            palette=palette,
            legend=False,
            ax=ax,
        )
        ax.set_title(title)
        ax.set_xlabel("")
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", labelsize=9)

    # Compare cumulative proportions within each target class.
    sns.ecdfplot(
        data=plot_data,
        x="log1p_value",
        hue="target_class",
        hue_order=order,
        palette=palette,
        ax=axes[2],
    )
    axes[2].set_title("Cumulative distributions")
    axes[2].set_xlabel("log1p(days)")
    axes[2].set_ylabel("Cumulative proportion")

    fig.suptitle(f"{column} by repeat purchase outcome", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
# Examine how time since the previous purchase relates to the target.
plot_interval_by_target("days_since_previous_purchase")

#### Interpretation — Time Since the Previous Purchase

The median interval is 37 days for target 1, compared with
105 days for target 0. The target 1 ECDF lies above target 0
through most of the range, indicating shorter intervals.

Both classes have long upper tails and overlap. More recent
purchasing is associated with repeat purchases, but does not
fully separate the classes.

The plots exclude 861 missing values and 16 negative intervals.

In [ ]:
# Compare average historical purchase intervals between target classes.
plot_interval_by_target("prior_mean_gap_days")

#### Interpretation — Average Historical Purchase Interval

The median average interval is 55.76 days for target 1,
compared with 101.75 days for target 0. The target 1 ECDF
lies above target 0 through most of the range, indicating
shorter average intervals between previous purchases.

Both distributions have long upper tails and overlap.
More frequent historical purchasing is associated with
repeat purchases within 90 days.

The comparison excludes 1,308 missing values and 16 negative
intervals. Some missing values reflect insufficient purchase history.

In [ ]:
# Compare variation in historical purchase intervals between target classes.
plot_interval_by_target("prior_std_gap_days")

#### Interpretation — Variation in Historical Purchase Intervals

The median standard deviation is 43.62 days for target 1,
compared with 65.06 days for target 0. The ECDF also shows
generally lower interval variation for target 1, although
the distributions overlap.

This measures absolute variation in days. Since target 1 also
has shorter average intervals, lower variation alone does not
establish greater regularity relative to those intervals.

There are 1,478 missing values, partly reflecting insufficient
purchase history. No negative values were excluded.

### 8.9. Random Noise and the Target

We will compare the distribution of `random_noise` between target
classes. Its name suggests a noise variable, but we will examine
the observed relationship before drawing conclusions.

In [ ]:
# Summarise the supplied noise variable separately for each target class.
noise_target_rows = []

for target_value in [0, 1]:
    class_data = train.loc[
        train["repeat_purchase_90d"].eq(target_value),
        ["customer_id", "random_noise"],
    ]

    # Retain finite values, including negatives, which are valid for noise.
    values = class_data["random_noise"]
    finite_mask = np.isfinite(values)
    finite_values = values.loc[finite_mask]

    noise_target_rows.append({
        "target": target_value,
        "finite_observations": len(finite_values),
        "unique_customers": class_data.loc[
            finite_mask, "customer_id"
        ].nunique(),
        "missing": int(values.isna().sum()),
        "infinite_values": int((values.notna() & ~finite_mask).sum()),
        "mean": finite_values.mean(),
        "standard_deviation": finite_values.std(),
        "minimum": finite_values.min(),
        "Q1": finite_values.quantile(0.25),
        "median": finite_values.median(),
        "Q3": finite_values.quantile(0.75),
        "maximum": finite_values.max(),
    })

noise_target_summary = pd.DataFrame(
    noise_target_rows
).set_index("target")

display(noise_target_summary)

#### Interpretation — Random Noise by Target

Both classes have means close to zero, standard deviations close
to one and nearly identical medians: −0.043 for target 0 and
−0.048 for target 1.

These summaries suggest little separation between the classes.
They do not establish independence from the target; the plots
below will also compare the distribution shapes.

In [ ]:
# Retain finite noise values and valid targets without changing train.
noise_plot_mask = (
    train["repeat_purchase_90d"].isin([0, 1])
    & np.isfinite(train["random_noise"])
)

noise_plot_data = train.loc[
    noise_plot_mask, ["repeat_purchase_90d", "random_noise"]
].copy()

target_labels = {
    0: "0 — No repeat purchase",
    1: "1 — Repeat purchase",
}
target_order = list(target_labels.values())
target_palette = dict(
    zip(target_order, sns.color_palette("colorblind", 2))
)

noise_plot_data["target_class"] = (
    noise_plot_data["repeat_purchase_90d"].map(target_labels)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Compare the centre, spread and extreme values of each class.
sns.boxplot(
    data=noise_plot_data,
    x="target_class",
    y="random_noise",
    hue="target_class",
    order=target_order,
    hue_order=target_order,
    palette=target_palette,
    legend=False,
    ax=axes[0],
)
axes[0].set_title("Noise distribution by target")
axes[0].set_xlabel("")
axes[0].set_ylabel("random_noise")
axes[0].tick_params(axis="x", labelsize=9)

# Use shared bins and normalise each class separately despite unequal sizes.
sns.histplot(
    data=noise_plot_data,
    x="random_noise",
    hue="target_class",
    hue_order=target_order,
    palette=target_palette,
    bins=40,
    stat="density",
    common_norm=False,
    common_bins=True,
    element="step",
    fill=False,
    ax=axes[1],
)
axes[1].set_title("Density comparison")
axes[1].set_xlabel("random_noise")
axes[1].set_ylabel("Density")

# Compare cumulative proportions across the full observed range.
sns.ecdfplot(
    data=noise_plot_data,
    x="random_noise",
    hue="target_class",
    hue_order=target_order,
    palette=target_palette,
    ax=axes[2],
)
axes[2].set_title("Cumulative distributions")
axes[2].set_xlabel("random_noise")
axes[2].set_ylabel("Cumulative proportion")

plt.tight_layout()
plt.show()

#### Interpretation — Random Noise Distributions

The boxplots have similar centres and spreads, while the density
histograms and ECDFs almost overlap. There is little visible
separation between target classes.

This supports treating `random_noise` as a candidate for exclusion
during feature selection. Predictive usefulness will still be
assessed through validation.

### 8.10. Categorical Variables and the Target

We will compare repeat purchase rates across payment types,
salespeople, commercial zones, transport methods, warehouses
and document series.

Each comparison will include observation counts and distinct
customer counts. Categories with few observations or customers
may have unstable rates. Missing values will be shown separately.

In [ ]:
# Store category-level summaries for later tables and plots.
categorical_target_features = [
    "payment_type",
    "salesperson",
    "commercial_zone",
    "transport_method",
    "warehouse",
    "document_series",
]

categorical_target_tables = {}

for column in categorical_target_features:
    # Use labelled observations and leave the original dataset unchanged.
    category_data = train.loc[
        train["repeat_purchase_90d"].isin([0, 1]),
        [column, "customer_id", "repeat_purchase_90d"],
    ].copy()

    # Keep missing categories as a separate group.
    category_table = (
        category_data.groupby(column, dropna=False, observed=True)
        .agg(
            observations=("repeat_purchase_90d", "size"),
            unique_customers=("customer_id", "nunique"),
            target_1=("repeat_purchase_90d", "sum"),
        )
    )

    category_table["target_0"] = (
        category_table["observations"] - category_table["target_1"]
    )
    category_table["repeat_purchase_pct"] = (
        category_table["target_1"] / category_table["observations"] * 100
    )

    # Sort by frequency rather than target rate.
    category_table = category_table[
        [
            "observations",
            "unique_customers",
            "target_0",
            "target_1",
            "repeat_purchase_pct",
        ]
    ].sort_values("observations", ascending=False)

    categorical_target_tables[column] = category_table

# Start with payment type before examining the other variables.
display(categorical_target_tables["payment_type"])

#### Interpretation — Payment Type and the Target

Repeat purchase rates are 52.73% for Payment 7 and 54.29%
for Payment 12, slightly below the overall rate of 55.92%.

Payment 9 has a much higher rate of 95.02%, but its 502
observations belong to only 10 customers. Customer composition
may therefore explain part of this difference.

Payment 10 has a lower rate of 40.30%. The remaining categories
have few observations, so their rates require caution.
Missing payment types have a rate of 55.00%.

In [ ]:
# Plot category rates in frequency order, with sample sizes beside each bar.
def plot_categorical_target(column):
    table = categorical_target_tables[column].copy()

    # Replace missing index labels only for display.
    labels = [
        "Missing" if pd.isna(value) else str(value)
        for value in table.index
    ]

    positions = np.arange(len(table))
    overall_rate = (
        train.loc[
            train["repeat_purchase_90d"].isin([0, 1]),
            "repeat_purchase_90d",
        ].mean() * 100
    )

    fig, ax = plt.subplots(
        figsize=(12, max(5, len(table) * 0.55))
    )

    ax.barh(
        positions,
        table["repeat_purchase_pct"],
        color=sns.color_palette("colorblind")[0],
    )

    # Show both row counts and customer coverage to help interpret rates.
    for position, (_, row) in enumerate(table.iterrows()):
        ax.text(
            row["repeat_purchase_pct"] + 1,
            position,
            (
                f'{row["repeat_purchase_pct"]:.1f}% | '
                f'n={int(row["observations"]):,}, '
                f'customers={int(row["unique_customers"]):,}'
            ),
            va="center",
            fontsize=9,
        )

    ax.axvline(
        overall_rate,
        color="black",
        linestyle="--",
        label=f"Overall rate: {overall_rate:.2f}%",
    )

    ax.set_yticks(positions)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlim(0, 145)
    ax.set_xticks(np.arange(0, 101, 20))
    ax.set_xlabel("Repeat purchase rate (%)")
    ax.set_ylabel("")
    ax.set_title(f"{column}: repeat purchase rates and group sizes")
    ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

In [ ]:
# Compare payment categories with the overall repeat purchase rate.
plot_categorical_target("payment_type")

In [ ]:
# Examine target rates and customer coverage for each salesperson.
display(categorical_target_tables["salesperson"])

In [ ]:
# Compare salesperson groups with the overall repeat purchase rate.
plot_categorical_target("salesperson")

#### Interpretation — Salesperson and the Target

The two largest groups have different repeat purchase rates:
62.43% for VLOOKUP Master 1 and 39.43% for VLOOKUP Master 4.
They represent 330 and 199 distinct customers, respectively.

These differences may reflect customer composition and purchasing
behaviour, rather than an effect of the salesperson alone.

The 0% and 100% rates for Masters 6 and 8 are based on only
12 and 3 observations. Missing salesperson values have a rate
of 57.53%, close to the overall 55.92%.

In [ ]:
# Examine repeat purchase rates and customer coverage by commercial zone.
display(categorical_target_tables["commercial_zone"])

In [ ]:
# Compare commercial zones with the overall repeat purchase rate.
plot_categorical_target("commercial_zone")

#### Interpretation — Commercial Zone and the Target

Zone 4 dominates the dataset, with 6,260 observations and a repeat
purchase rate of 56.53%, close to the overall 55.92%.

Zones 2 and 3 have lower rates of 41.06% and 35.71%, but represent
only 17 and 21 customers. These differences may partly reflect
customer composition.

Zones 6 and 7 each contain one observation, so their 0% rates
provide little evidence about those categories.

In [ ]:
# Inspect all transport categories, including missing values.
# Keep observation and customer counts visible when comparing target rates.
with pd.option_context("display.max_rows", 100):
    display(categorical_target_tables["transport_method"])

In [ ]:
# Plot category rates and group sizes, optionally selecting a range of rows.
def plot_categorical_target(column, start=0, stop=None):
    # Select categories from the table sorted by observation count.
    table = categorical_target_tables[column].iloc[start:stop].copy()

    # Display missing categories with a readable label.
    labels = [
        "Missing" if pd.isna(value) else str(value)
        for value in table.index
    ]
    positions = np.arange(len(table))

    # Calculate the overall rate using valid target values.
    overall_rate = (
        train.loc[
            train["repeat_purchase_90d"].isin([0, 1]),
            "repeat_purchase_90d",
        ].mean() * 100
    )

    fig, ax = plt.subplots(
        figsize=(12, max(5, len(table) * 0.55))
    )

    # Draw one horizontal bar per category.
    ax.barh(
        positions,
        table["repeat_purchase_pct"],
        color=sns.color_palette("colorblind")[0],
    )

    # Label each rate with its observation and customer counts.
    for position, (_, row) in enumerate(table.iterrows()):
        ax.text(
            row["repeat_purchase_pct"] + 1,
            position,
            (
                f'{row["repeat_purchase_pct"]:.1f}% | '
                f'n={int(row["observations"]):,}, '
                f'customers={int(row["unique_customers"]):,}'
            ),
            va="center",
            fontsize=9,
        )

    # Add the overall training rate as a reference.
    ax.axvline(
        overall_rate,
        color="black",
        linestyle="--",
        label=f"Overall rate: {overall_rate:.2f}%",
    )

    ax.set_yticks(positions)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()

    # Leave room for annotations beyond the percentage scale.
    ax.set_xlim(0, 145)
    ax.set_xticks(np.arange(0, 101, 20))
    ax.set_xlabel("Repeat purchase rate (%)")
    ax.set_ylabel("")
    ax.set_title(f"{column}: repeat purchase rates and group sizes")
    ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

In [ ]:
# Show the 15 most frequent transport groups.
# Categories are selected by observation count, not by target rate.
plot_categorical_target("transport_method", start=0, stop=15)

In [ ]:
# Show the next 15 transport groups in descending observation count.
plot_categorical_target("transport_method", start=15, stop=30)

In [ ]:
# Transport groups 31–45 in descending observation count.
plot_categorical_target("transport_method", start=30, stop=45)

In [ ]:
# Transport groups 46–60.
plot_categorical_target("transport_method", start=45, stop=60)

In [ ]:
# Transport groups 61–75.
plot_categorical_target("transport_method", start=60, stop=75)

In [ ]:
# Remaining transport groups.
plot_categorical_target("transport_method", start=75)

#### Interpretation — Transport Method and the Target

The most common method, Delivery 72, has a repeat purchase rate
of 49.33%. Delivery 61 has a higher rate of 60.13%.

Delivery 11 reaches 98.02%, but its 253 observations belong to
only two customers. Repeated observations from a few customers
can therefore dominate a category's rate.

Many methods have very few observations, including categories
with a single record. Their 0% or 100% rates provide little
evidence of a stable relationship.

Missing transport methods have a repeat purchase rate of 54.02%,
close to the overall training rate of 55.92%.

In [ ]:
# Compare target counts and repeat purchase rates across warehouses.
display(categorical_target_tables["warehouse"])

In [ ]:
# Compare warehouse repeat purchase rates with the overall training rate.
plot_categorical_target("warehouse")

#### Interpretation — Warehouse and the Target

Warehouse 1 dominates the dataset with 6,253 observations. Its repeat
purchase rate of 55.65% is close to the overall rate of 55.92%.

Warehouse 3 has a higher rate of 80.95%, but only 21 observations
from 13 customers. This small group limits the strength of the comparison.

The 260 observations with missing warehouse information have a
repeat purchase rate of 60.38%.

In [ ]:
# Show observation counts, customer counts and target rates by document series.
display(categorical_target_tables["document_series"])

In [ ]:
# Compare document series rates with the overall training rate.
plot_categorical_target("document_series")

#### Interpretation — Document Series and the Target

The three document series have very similar repeat purchase rates,
ranging from 55.81% to 56.13%, close to the overall rate of 55.92%.
This comparison shows little separation between the target classes.

The 65 observations with missing document series have a lower rate
of 50.77%. These are the same observations previously identified
as missing all purchase fields, so this difference cannot be
attributed specifically to document series.

### 8.11. Missing Information and the Target

We compare repeat purchase rates across observations with different
levels of missing purchase information.

Historical statistics will also be examined separately, since some
are undefined when there are too few previous purchases.

In [ ]:
# Assess purchase fields without including identifiers or the target.
purchase_fields = [
    column for column in train.columns
    if column not in ["ID", "customer_id", "repeat_purchase_90d"]
]

# Count missing purchase fields while keeping the original data unchanged.
missing_field_count = train[purchase_fields].isna().sum(axis=1)

# Assign each observation to one mutually exclusive group.
missing_information_group = pd.Series(
    "Partially missing",
    index=train.index,
    name="purchase_information",
)

missing_information_group.loc[missing_field_count.eq(0)] = "Complete"
missing_information_group.loc[
    missing_field_count.eq(len(purchase_fields))
] = "All purchase fields missing"

# Use observations with valid binary targets.
missing_target_data = train.loc[
    train["repeat_purchase_90d"].isin([0, 1]),
    ["customer_id", "repeat_purchase_90d"],
].copy()

missing_target_data["purchase_information"] = missing_information_group

# Report group sizes, distinct customers and target counts.
missing_information_summary = (
    missing_target_data
    .groupby("purchase_information")
    .agg(
        observations=("repeat_purchase_90d", "size"),
        unique_customers=("customer_id", "nunique"),
        target_1=("repeat_purchase_90d", "sum"),
    )
    .reindex(["Complete", "Partially missing", "All purchase fields missing"])
)

missing_information_summary["target_0"] = (
    missing_information_summary["observations"]
    - missing_information_summary["target_1"]
)

missing_information_summary["repeat_purchase_pct"] = (
    missing_information_summary["target_1"]
    / missing_information_summary["observations"] * 100
)

display(missing_information_summary[
    ["observations", "unique_customers", "target_0",
     "target_1", "repeat_purchase_pct"]
])

In [ ]:
# Compare group rates with the overall rate among valid targets.
overall_rate = missing_target_data["repeat_purchase_90d"].mean() * 100

fig, ax = plt.subplots(figsize=(11, 5))

rates = missing_information_summary["repeat_purchase_pct"]
positions = np.arange(len(rates))

ax.barh(
    positions,
    rates,
    color=sns.color_palette("colorblind")[0],
)

# Include observation counts so small groups are easy to identify.
for position, (_, row) in enumerate(missing_information_summary.iterrows()):
    ax.text(
        row["repeat_purchase_pct"] + 1,
        position,
        f"{row['repeat_purchase_pct']:.2f}% | "
        f"n={int(row['observations']):,}",
        va="center",
    )

ax.axvline(
    overall_rate,
    color="black",
    linestyle="--",
    label=f"Overall rate: {overall_rate:.2f}%",
)

ax.set_yticks(positions)
ax.set_yticklabels(missing_information_summary.index)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Repeat purchase rate (%)")
ax.set_title("Purchase information completeness and the target")
ax.legend(loc="lower right")

plt.tight_layout()
plt.show()

#### Interpretation — Missing Purchase Information and the Target

Complete observations have a repeat purchase rate of 60.58%,
compared with 48.86% for partially missing observations,
a difference of 11.72 percentage points.

This association may partly reflect purchase history: customers
with fewer previous purchases have more undefined historical statistics.
We will examine these patterns separately.

The 65 observations with all purchase fields missing have a rate
of 50.77%. Their small group size limits the comparison.

Complete observations belong to 309 customers, compared with 607
for partially missing observations. These groups can share customers,
and their rates are influenced by repeated observations.

#### 8.11.1. Historical Missingness and Purchase History

The earlier checks showed that historical statistics depend on the
number of previous purchases.

We compare observed values, missing values with insufficient history
and missing values despite sufficient history.

Purchase-value summaries require at least one previous purchase.
A sample standard deviation requires at least two values.
For intervals between previous purchases, the mean requires two
previous purchases and the sample standard deviation requires three.

These thresholds follow the apparent definitions in the supplied data.
They help distinguish missingness patterns without establishing the
cause of every missing value.

In [ ]:
# Minimum previous-purchase counts implied by each historical statistic.
history_requirements = {
    "days_since_previous_purchase": 1,
    "prior_mean_purchase_value": 1,
    "prior_std_purchase_value": 2,
    "prior_min_purchase_value": 1,
    "prior_max_purchase_value": 1,
    "prior_mean_gap_days": 2,
    "prior_std_gap_days": 3,
}

# Restrict this comparison to valid targets and usable purchase counts.
prior_count = train["prior_purchase_count"]

valid_history_count = (
    prior_count.notna()
    & np.isfinite(prior_count)
    & prior_count.ge(0)
    & prior_count.eq(np.floor(prior_count))
)

history_missingness_data = train.loc[
    train["repeat_purchase_90d"].isin([0, 1]) & valid_history_count
].copy()

print(f"Rows assessed: {len(history_missingness_data):,}")
print(f"Rows excluded: {len(train) - len(history_missingness_data):,}")

# Store separate results so each variable can be plotted afterwards.
history_missingness_tables = {}

status_order = [
    "Observed",
    "Missing — insufficient history",
    "Missing — sufficient history",
]

for column, minimum_count in history_requirements.items():
    # Classify missingness using the count, independently of the target.
    missing = history_missingness_data[column].isna()
    sufficient_history = (
        history_missingness_data["prior_purchase_count"].ge(minimum_count)
    )

    status = pd.Series(
        "Observed",
        index=history_missingness_data.index,
        name="history_status",
    )

    status.loc[missing & ~sufficient_history] = status_order[1]
    status.loc[missing & sufficient_history] = status_order[2]

    comparison = history_missingness_data[
        ["customer_id", "repeat_purchase_90d"]
    ].copy()
    comparison["history_status"] = status

    # Include customer counts to show how much each group is concentrated.
    summary = (
        comparison.groupby("history_status")
        .agg(
            observations=("repeat_purchase_90d", "size"),
            unique_customers=("customer_id", "nunique"),
            target_1=("repeat_purchase_90d", "sum"),
        )
        .reindex(status_order, fill_value=0)
    )

    summary["target_0"] = (
        summary["observations"] - summary["target_1"]
    )

    # Empty groups have no rate, rather than an artificial rate of zero.
    summary["repeat_purchase_pct"] = (
        summary["target_1"]
        / summary["observations"].replace(0, np.nan) * 100
    )

    history_missingness_tables[column] = summary[
        ["observations", "unique_customers", "target_0",
         "target_1", "repeat_purchase_pct"]
    ]

# Display all seven comparisons in one organised table.
history_missingness_summary = pd.concat(
    history_missingness_tables,
    names=["variable", "history_status"],
)

display(history_missingness_summary)

In [ ]:
def plot_history_missingness(columns):
    """Compare target rates across historical missingness groups."""
    fig, axes = plt.subplots(
        len(columns), 1,
        figsize=(11, 3.5 * len(columns)),
        squeeze=False,
    )

    # Use the same assessed population as the summary table.
    overall_rate = (
        history_missingness_data["repeat_purchase_90d"].mean() * 100
    )

    for ax, column in zip(axes.ravel(), columns):
        # Empty groups have no rate and are omitted from the graph.
        table = history_missingness_tables[column]
        table = table.loc[table["observations"].gt(0)]

        positions = np.arange(len(table))

        ax.barh(
            positions,
            table["repeat_purchase_pct"],
            color=sns.color_palette("colorblind")[0],
        )

        # Show rates and group sizes beside each bar.
        for position, (_, row) in enumerate(table.iterrows()):
            ax.text(
                row["repeat_purchase_pct"] + 1,
                position,
                f"{row['repeat_purchase_pct']:.2f}% | "
                f"n={int(row['observations']):,}, "
                f"customers={int(row['unique_customers']):,}",
                va="center",
                fontsize=9,
            )

        ax.axvline(
            overall_rate,
            color="black",
            linestyle="--",
            label=f"Assessed population rate: {overall_rate:.2f}%",
        )

        ax.set_yticks(positions)
        ax.set_yticklabels(table.index)
        ax.invert_yaxis()
        ax.set_xlim(0, 110)
        ax.set_xticks(np.arange(0, 101, 20))
        ax.set_xlabel("Repeat purchase rate (%)")
        ax.set_title(column)
        ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

In [ ]:
# Previous-purchase recency and average purchase value.
plot_history_missingness([
    "days_since_previous_purchase",
    "prior_mean_purchase_value",
])

In [ ]:
# Variation, smallest value and largest value of previous purchases.
plot_history_missingness([
    "prior_std_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
])

In [ ]:
# Mean and variation of intervals between previous purchases.
plot_history_missingness([
    "prior_mean_gap_days",
    "prior_std_gap_days",
])

#### Interpretation — Historical Missingness and the Target

Groups with insufficient purchase history have repeat purchase rates
between 33.33% and 36.31%, compared with 58.30%–61.47% where the
corresponding values are observed. This suggests that historical
missingness partly reflects the lower repeat purchase rate of
observations with little previous activity.

Missing values despite sufficient history show a different pattern:
rates are 57.00% for previous-purchase recency, 58.25% for average
purchase value and 65.64% for the mean purchase interval.

Among assessed rows, missing smallest and largest purchase values
and both historical standard deviations occur only where history
is insufficient under the stated thresholds.

Missingness should therefore be interpreted according to its context.
These comparisons describe associations and remain influenced by
repeated observations from the same customers.

## 9. Relationships Between Features

This section examines associations between explanatory variables
and identifies features that may contain overlapping information.

Pearson correlation measures linear relationships. Spearman correlation
measures relationships between ranks and is useful for the skewed
distributions observed earlier.

Correlations use observations where both variables are available.
Strong correlation alone will not determine which features are retained.


### 9.1. Current Purchase Variables

We compare purchase amounts, taxes, costs, shipping, discounts,
detail counts and invoice counts.

The accounting relationships identified earlier will help interpret
correlations between monetary variables.

In [ ]:
# Group current purchase variables for a readable correlation analysis.
current_purchase_features = [
    "Total Bruto",
    "Total Liquido",
    "Total a Pagar",
    "Total Imp_",
    "Custo",
    "Total Portes",
    "Total Desc_ Global",
    "Total Desc_ Linha",
    "N_ Detalhes",
    "invoice_count_current_purchase",
]

# Create a separate diagnostic object without changing the original data.
current_purchase_data = train[current_purchase_features].copy()
current_purchase_data = current_purchase_data.replace(
    [np.inf, -np.inf], np.nan
)

# Compute linear and rank-based associations using available pairs.
current_purchase_pearson = current_purchase_data.corr(method="pearson")
current_purchase_spearman = current_purchase_data.corr(method="spearman")

# Show the number of observations supporting each correlation.
observed_values = current_purchase_data.notna().astype("int64")
current_purchase_pair_counts = observed_values.T @ observed_values

display(current_purchase_pair_counts)

In [ ]:
# Display one triangle because the correlation matrix is symmetric.
upper_triangle = np.triu(
    np.ones(current_purchase_pearson.shape, dtype=bool),
    k=1,
)

fig, ax = plt.subplots(figsize=(12, 9))

sns.heatmap(
    current_purchase_pearson,
    mask=upper_triangle,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)

ax.set_title("Current purchase variables — Pearson correlation")
plt.tight_layout()
plt.show()

In [ ]:
# Compare rank-based relationships using the same variables and scale.
fig, ax = plt.subplots(figsize=(12, 9))

sns.heatmap(
    current_purchase_spearman,
    mask=upper_triangle,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    vmin=-1,
    vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Spearman correlation"},
    ax=ax,
)

ax.set_title("Current purchase variables — Spearman correlation")
plt.tight_layout()
plt.show()

#### Interpretation — Correlations Between Current Purchase Variables

Gross, net and payable amounts are strongly correlated in both matrices.
Some correlations round to 1.00, which does not imply an exact relationship.
The earlier accounting checks also showed that these totals are linked.

Costs are strongly associated with purchase amounts. Detail counts also
increase with amounts and costs, with stronger Spearman than Pearson
correlations, suggesting a relationship that is not strictly linear.

Line discounts correlate more strongly with gross amounts under Pearson
(0.72) than Spearman (0.46), suggesting that large values influence
the linear association.

Shipping, global discounts and invoice counts generally have weaker
correlations with the other variables. Low correlation does not
establish that they are uninformative for prediction.

The overlap between monetary totals will be considered during feature
selection, together with validation results.

#### 9.1.1. Selected Scatterplots

Scatterplots examine the overlap between net and payable amounts,
the relationship between purchase size and detail count and the
association between gross amounts and line discounts.

Original-scale and log1p views are shown together to make both large
values and the central concentration visible. Original values remain
unchanged.

In [ ]:
def plot_current_purchase_relationship(x_column, y_column):
    """Show a feature relationship on original and log1p scales."""
    pair = current_purchase_data[[x_column, y_column]].dropna()

    # Only non-negative pairs can enter the log1p diagnostic panel.
    non_negative_pair = pair.loc[pair.ge(0).all(axis=1)]

    print(f"Relationship: {x_column} and {y_column}")
    print(f"Finite pairs: {len(pair):,}")
    print(f"Pairs excluded from log1p view: {len(pair) - len(non_negative_pair):,}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Transparency makes overlapping observations easier to see.
    axes[0].scatter(
        pair[x_column],
        pair[y_column],
        s=15,
        alpha=0.25,
        color=sns.color_palette("colorblind")[0],
        edgecolors="none",
    )
    axes[0].set_xlabel(x_column)
    axes[0].set_ylabel(y_column)
    axes[0].set_title("Original values")

    # Transform only the displayed coordinates.
    axes[1].scatter(
        np.log1p(non_negative_pair[x_column]),
        np.log1p(non_negative_pair[y_column]),
        s=15,
        alpha=0.25,
        color=sns.color_palette("colorblind")[0],
        edgecolors="none",
    )
    axes[1].set_xlabel(f"log1p({x_column})")
    axes[1].set_ylabel(f"log1p({y_column})")
    axes[1].set_title("Log1p diagnostic view")

    fig.suptitle(f"{x_column} and {y_column}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Inspect the close association between two accounting totals.
plot_current_purchase_relationship("Total Liquido", "Total a Pagar")

In [ ]:
# Examine how payable amounts vary with the number of purchase details.
plot_current_purchase_relationship("N_ Detalhes", "Total a Pagar")

In [ ]:
# Examine the association between gross amounts and line discounts.
plot_current_purchase_relationship("Total Bruto", "Total Desc_ Linha")

#### Interpretation — Selected Current Purchase Relationships

Net and payable amounts follow a very close relationship, consistent
with the earlier finding that payable equals net plus tax.
The visible bands are compatible with differences in tax relative
to net amounts.

Purchases with more details generally have higher payable amounts,
but substantial variation remains at each detail count.
The vertical bands reflect the discrete nature of the count.

Line discounts generally increase with gross amounts, although many
purchases have no line discount across a wide range of gross values.
The log1p view reveals variation that is compressed on the original scale.

These patterns support reviewing overlapping monetary features and
exploring relative measures, such as discount proportions, during
feature engineering.

### 9.2. Purchase Activity, Tenure and Purchase Intervals

We examine how purchase frequency relates to customer tenure and
the time between purchases.

The rolling purchase counts cover overlapping windows, so some
correlation is expected. We also compare recent activity with the
total number of previous purchases.

Negative elapsed times identified earlier are excluded from this
diagnostic comparison. Missing values remain missing and the
original training data remains unchanged.

In [ ]:
# Group purchase activity and timing variables.
activity_time_features = [
    "prior_purchase_count",
    "purchase_count_30d",
    "purchase_count_90d",
    "purchase_count_180d",
    "purchase_count_365d",
    "customer_tenure_days",
    "days_since_previous_purchase",
    "prior_mean_gap_days",
    "prior_std_gap_days",
]

# Create a separate diagnostic copy and exclude infinite values.
activity_time_data = train[activity_time_features].copy()
activity_time_data = activity_time_data.replace(
    [np.inf, -np.inf], np.nan
)

# These variables represent non-negative counts or durations.
# Exclude negative entries individually rather than removing entire rows.
negative_counts = activity_time_data.lt(0).sum()
activity_time_data = activity_time_data.mask(activity_time_data.lt(0))

# Document how many values remain available for each variable.
activity_time_availability = pd.DataFrame({
    "original_missing": train[activity_time_features].isna().sum(),
    "negative_values_excluded": negative_counts,
    "retained_observations": activity_time_data.notna().sum(),
})

display(activity_time_availability)

# Calculate correlations using available pairs without imputing values.
activity_time_pearson = activity_time_data.corr(method="pearson")
activity_time_spearman = activity_time_data.corr(method="spearman")

# Pair counts show the sample size behind each correlation.
activity_observed = activity_time_data.notna().astype("int64")
activity_time_pair_counts = activity_observed.T @ activity_observed

display(activity_time_pair_counts)

In [ ]:
def plot_feature_correlations(correlation_matrix, title, method):
    """Display a correlation matrix with a consistent scale."""
    upper_triangle = np.triu(
        np.ones(correlation_matrix.shape, dtype=bool),
        k=1,
    )

    fig, ax = plt.subplots(figsize=(12, 9))

    sns.heatmap(
        correlation_matrix,
        mask=upper_triangle,
        annot=True,
        fmt=".2f",
        cmap="vlag",
        vmin=-1,
        vmax=1,
        center=0,
        square=True,
        linewidths=0.5,
        cbar_kws={"label": f"{method} correlation"},
        ax=ax,
    )

    ax.set_title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
# Inspect linear relationships between activity and timing variables.
plot_feature_correlations(
    activity_time_pearson,
    "Purchase activity and timing — Pearson correlation",
    "Pearson",
)

In [ ]:
# Compare rank-based relationships for these skewed variables.
plot_feature_correlations(
    activity_time_spearman,
    "Purchase activity and timing — Spearman correlation",
    "Spearman",
)

#### Interpretation — Purchase Activity and Timing

Rolling purchase counts are strongly correlated, partly because their
windows overlap. The 180-day and 365-day counts have correlations
of 0.95 under Pearson and 0.89 under Spearman.

Previous-purchase recency is strongly negatively associated with
recent activity under Spearman: −0.84 for the 30-day count and
−0.86 for the 90-day count. These relationships partly reflect the
definitions: a recent previous purchase contributes to the recent windows.

Customer tenure is more strongly related to total previous purchases
under Spearman (0.79) than Pearson (0.40). Longer recorded histories
generally accompany more purchases, although the relationship is
not strictly linear.

The mean and standard deviation of previous purchase intervals are
strongly positively correlated (Pearson 0.87; Spearman 0.81).
Larger absolute variation therefore partly accompanies longer intervals
and should not automatically be interpreted as less regular behaviour.

In [ ]:
def plot_activity_relationship(x_column, y_column):
    """Compare activity and timing variables on two scales."""
    # Use the diagnostic copy, where negative values were excluded.
    pair = activity_time_data[[x_column, y_column]].dropna()
    print(f"Observations plotted: {len(pair):,}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Both variables are non-negative in this diagnostic dataset.
    for ax, use_log in zip(axes, [False, True]):
        x_values = pair[x_column]
        y_values = pair[y_column]

        if use_log:
            x_values = np.log1p(x_values)
            y_values = np.log1p(y_values)

        ax.scatter(
            x_values,
            y_values,
            s=15,
            alpha=0.25,
            color=sns.color_palette("colorblind")[0],
            edgecolors="none",
        )

        ax.set_xlabel(f"log1p({x_column})" if use_log else x_column)
        ax.set_ylabel(f"log1p({y_column})" if use_log else y_column)
        ax.set_title("Log1p diagnostic view" if use_log else "Original values")

    fig.suptitle(f"{x_column} and {y_column}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Examine how accumulated purchase counts relate to recorded tenure.
plot_activity_relationship("customer_tenure_days", "prior_purchase_count")

In [ ]:
# Examine whether absolute interval variation grows with the mean interval.
plot_activity_relationship("prior_mean_gap_days", "prior_std_gap_days")

#### Interpretation — Selected Activity and Timing Relationships

Longer customer tenure generally accompanies more previous purchases,
but customers with similar tenure can have very different purchase counts.
The visible trajectories may reflect repeated observations from individual
customers; customer-level analysis will investigate this.

The mean purchase interval and its standard deviation increase together,
although substantial variation remains. This supports considering variation
relative to the mean when exploring purchase regularity.

The tenure comparison contains 6,469 observations. The interval comparison
contains 4,870 observations with both values available and non-negative.
These subsets therefore represent different amounts of purchase history.

### 9.3. Historical Spending and Purchase Activity

We examine relationships between historical spending, previous purchase
values and the number of previous purchases.

A diagnostic copy excludes supplied average purchase values that failed
the total-spending/count check, largest previous purchase values that
exceeded total historical spending and annual spending values flagged
by the spending consistency checks.

Only the affected values are excluded. Other available values from the
same observation remain usable. Missing values are not imputed and
passing these checks does not establish that every retained value is correct.

In [ ]:
# Group historical monetary variables and total previous purchase count.
historical_spending_features = [
    "prior_purchase_count",
    "prior_total_spend",
    "prior_mean_purchase_value",
    "prior_std_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
    "spend_30d",
    "spend_90d",
    "spend_180d",
    "spend_365d",
]

# Preserve the supplied data in a separate object for comparison.
historical_spending_supplied = train[
    historical_spending_features
].copy().replace([np.inf, -np.inf], np.nan)

historical_spending_data = historical_spending_supplied.copy()

# Reuse the inconsistency flags established in the earlier checks.
spending_exclusion_flags = {
    "prior_mean_purchase_value": historical_consistency_flags[
        "Mean equals total spending / purchase count"
    ],
    "prior_max_purchase_value": maximum_exceeds_total,
    "spend_365d": any_window_spending_flag | any_zero_count_flag,
}

for column, flag in spending_exclusion_flags.items():
    # Align flags with the original row index before excluding values.
    aligned_flag = flag.reindex(
        historical_spending_data.index, fill_value=False
    ).fillna(False)

    historical_spending_data.loc[aligned_flag, column] = np.nan

# Exclude any remaining negative counts or monetary values in this view.
historical_spending_data = historical_spending_data.mask(
    historical_spending_data.lt(0)
)

# Document exclusions separately from values that were already missing.
spending_availability = pd.DataFrame({
    "original_missing": train[historical_spending_features].isna().sum(),
    "finite_supplied_values": historical_spending_supplied.notna().sum(),
    "finite_values_excluded": (
        historical_spending_supplied.notna()
        & historical_spending_data.isna()
    ).sum(),
    "retained_observations": historical_spending_data.notna().sum(),
})

display(spending_availability)

# Calculate both correlation measures using available pairs.
historical_spending_pearson = historical_spending_data.corr(
    method="pearson"
)
historical_spending_spearman = historical_spending_data.corr(
    method="spearman"
)

# Show how many observations support each pairwise comparison.
spending_observed = historical_spending_data.notna().astype("int64")
historical_spending_pair_counts = spending_observed.T @ spending_observed

display(historical_spending_pair_counts)

In [ ]:
# Examine linear relationships after excluding flagged values.
plot_feature_correlations(
    historical_spending_pearson,
    "Historical spending and purchase activity — Pearson correlation",
    "Pearson",
)

In [ ]:
# Compare rank-based relationships for the skewed spending variables.
plot_feature_correlations(
    historical_spending_spearman,
    "Historical spending and purchase activity — Spearman correlation",
    "Spearman",
)

#### Interpretation — Historical Spending Relationships

Total historical spending is strongly associated with previous purchase
count, particularly under Spearman (0.90). Accumulated spending therefore
reflects purchase frequency as well as purchase values.

Spending over overlapping windows is also strongly correlated.
The 180-day and 365-day spending variables have correlations of
0.92 under Pearson and 0.90 under Spearman.

Average previous purchase value and its standard deviation are strongly
related (Pearson 0.88; Spearman 0.90). Larger purchase values generally
accompany greater absolute variation.

The smallest previous purchase value decreases as purchase count rises
(Spearman −0.67), while the largest increases (0.56). More purchases
provide more opportunities to observe both small and large values.

These associations indicate overlapping information, but do not establish
which variables should be removed. Feature selection will consider their
predictive contribution during validation.

In [ ]:
# Confirm which supplied values were excluded from the diagnostic copy.
display(spending_availability)

#### 9.3.1. Selected Historical Spending Relationships

We examine how total historical spending varies with purchase count
and how average previous purchase value relates to its standard deviation.

The plots use the diagnostic copy created above. Original-scale and
log1p views help distinguish the central pattern from large values.

In [ ]:
def plot_historical_spending_relationship(x_column, y_column):
    """Plot historical spending relationships using diagnostic values."""
    # Retain observations where both diagnostic values are available.
    pair = historical_spending_data[[x_column, y_column]].dropna()

    print(f"Relationship: {x_column} and {y_column}")
    print(f"Observations plotted: {len(pair):,}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # All retained values are non-negative, so log1p also accommodates zeros.
    for ax, use_log in zip(axes, [False, True]):
        x_values = pair[x_column]
        y_values = pair[y_column]

        if use_log:
            x_values = np.log1p(x_values)
            y_values = np.log1p(y_values)

        ax.scatter(
            x_values,
            y_values,
            s=15,
            alpha=0.25,
            color=sns.color_palette("colorblind")[0],
            edgecolors="none",
        )

        ax.set_xlabel(f"log1p({x_column})" if use_log else x_column)
        ax.set_ylabel(f"log1p({y_column})" if use_log else y_column)
        ax.set_title("Log1p diagnostic view" if use_log else "Original values")

    fig.suptitle(f"{x_column} and {y_column}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare accumulated spending with the number of previous purchases.
plot_historical_spending_relationship(
    "prior_purchase_count",
    "prior_total_spend",
)

In [ ]:
# Compare typical previous purchase value with absolute variation.
plot_historical_spending_relationship(
    "prior_mean_purchase_value",
    "prior_std_purchase_value",
)

#### Interpretation — Selected Historical Spending Relationships

Total historical spending generally increases with previous purchase
count, but observations with similar counts can have very different
spending totals. Purchase frequency and average purchase value therefore
describe different aspects of customer activity.

The visible trajectories may reflect repeated observations from the
same customers, which will be examined in the customer-level analysis.

Higher average purchase values generally accompany larger standard
deviations. Absolute variation therefore partly reflects purchase size;
variation relative to the mean may provide additional information.

The spending/count plot includes 6,469 observations. The average/variation
plot includes 5,175 observations with both diagnostic values available.

### 9.4. Relationships Between Categorical Features

We use bias-corrected Cramér's V to examine associations between
payment type, salesperson, commercial zone, transport method,
warehouse and document series.

Values closer to zero indicate weaker association, while larger
values indicate stronger association. The measure has no direction.

Each comparison uses observations where both categories are available.
We also report customer counts and sparse expected cell counts to
help assess the results. Repeated customer observations and rare
categories require caution; no hypothesis-test conclusions are drawn.

In [ ]:
def categorical_pair_association(data, first, second):
    """Summarise a categorical pair using bias-corrected Cramér's V."""
    # Exclude missing labels for this pair without modifying the data.
    pair = data[[first, second, "customer_id"]].dropna(
        subset=[first, second]
    )

    contingency = pd.crosstab(pair[first], pair[second])
    n = int(contingency.to_numpy().sum())
    rows, columns = contingency.shape

    result = {
        "feature_1": first,
        "feature_2": second,
        "observations": n,
        "unique_customers": pair["customer_id"].nunique(),
        "categories_1": rows,
        "categories_2": columns,
        "corrected_cramers_v": np.nan,
        "expected_cells_below_5_pct": np.nan,
    }

    # Association is undefined if either variable has only one category.
    if n <= 1 or min(rows, columns) < 2:
        return result

    # Calculate the statistic without the 2×2 continuity correction.
    chi_squared, _, _, expected = stats.chi2_contingency(
        contingency,
        correction=False,
    )

    # Correct the association estimate for table dimensions and sample size.
    phi_squared = chi_squared / n
    corrected_phi_squared = max(
        0, phi_squared - ((rows - 1) * (columns - 1)) / (n - 1)
    )

    corrected_rows = rows - ((rows - 1) ** 2) / (n - 1)
    corrected_columns = columns - ((columns - 1) ** 2) / (n - 1)
    denominator = min(corrected_rows - 1, corrected_columns - 1)

    if denominator > 0:
        result["corrected_cramers_v"] = np.sqrt(
            corrected_phi_squared / denominator
        )

    # Describe sparsity; the chi-square p-value is not used.
    result["expected_cells_below_5_pct"] = (
        (expected < 5).mean() * 100
    )

    return result


# Evaluate each distinct pair once.
categorical_pair_results = []

for position, first in enumerate(categorical_target_features):
    for second in categorical_target_features[position + 1:]:
        categorical_pair_results.append(
            categorical_pair_association(train, first, second)
        )

categorical_association_summary = (
    pd.DataFrame(categorical_pair_results)
    .sort_values("corrected_cramers_v", ascending=False)
    .reset_index(drop=True)
)

display(categorical_association_summary)

In [ ]:
# Arrange pairwise associations into a symmetric matrix.
categorical_association_matrix = pd.DataFrame(
    np.nan,
    index=categorical_target_features,
    columns=categorical_target_features,
)

for _, row in categorical_association_summary.iterrows():
    first = row["feature_1"]
    second = row["feature_2"]
    value = row["corrected_cramers_v"]

    categorical_association_matrix.loc[first, second] = value
    categorical_association_matrix.loc[second, first] = value

# Hide the diagonal and upper triangle to avoid repeated information.
mask = np.triu(
    np.ones(categorical_association_matrix.shape, dtype=bool)
)

fig, ax = plt.subplots(figsize=(10, 7))

sns.heatmap(
    categorical_association_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    vmin=0,
    vmax=1,
    square=True,
    linewidths=0.5,
    cbar_kws={"label": "Bias-corrected Cramér's V"},
    ax=ax,
)

ax.set_title("Associations between categorical features")
plt.tight_layout()
plt.show()

###### Interpretation — Associations Between Categorical Features

The strongest association is between transport method and warehouse
(corrected Cramér's V = 0.83). However, Warehouse 3 has only 21
observations in the full training data, so we will inspect the
category combinations before interpreting this as broad overlap.

Payment type and commercial zone also show an association (0.55).
Transport method is associated with several other categorical features,
with values between 0.46 and 0.50.

Many comparisons contain numerous expected cell counts below five,
reflecting rare categories and sparse combinations. Together with
repeated customer observations, this limits interpretation.

Corrected values of zero do not prove independence. These results
identify relationships for inspection rather than automatic feature removal.

#### 9.4.1. Transport Method and Warehouse

We will inspect transport methods within each warehouse to understand
the strong association found in the categorical comparison.

Observation counts and distinct customer counts help assess whether
the pattern is concentrated among a few customers.

In [ ]:
# Use the same complete pairs as the association calculation.
transport_warehouse_data = train[
    ["transport_method", "warehouse", "customer_id"]
].dropna(subset=["transport_method", "warehouse"]).copy()

# Count observations for each transport–warehouse combination.
transport_warehouse_counts = pd.crosstab(
    transport_warehouse_data["transport_method"],
    transport_warehouse_data["warehouse"],
)

# Order transport methods by their total number of observations.
transport_order = (
    transport_warehouse_counts.sum(axis=1)
    .sort_values(ascending=False)
    .index
)
transport_warehouse_counts = transport_warehouse_counts.loc[transport_order]

# Show distinct customers alongside observation counts.
transport_warehouse_customers = (
    transport_warehouse_data
    .groupby(["transport_method", "warehouse"])
    ["customer_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(
        index=transport_warehouse_counts.index,
        columns=transport_warehouse_counts.columns,
        fill_value=0,
    )
)

# Focus the displayed table on methods observed with the rare warehouse.
rare_warehouse = "Final Excel Warehouse 3"
methods_with_warehouse_3 = (
    transport_warehouse_counts[rare_warehouse].gt(0)
)

print("Transport methods observed with Warehouse 3:")
display(pd.concat(
    {
        "observations": transport_warehouse_counts,
        "unique_customers": transport_warehouse_customers,
    },
    axis=1,
).loc[methods_with_warehouse_3])

# Report how many rare-warehouse records entered this complete-pair analysis.
print(
    "Warehouse 3 observations with known transport method:",
    int(transport_warehouse_counts[rare_warehouse].sum()),
)

In [ ]:
# Compare transport composition within each warehouse.
# Each warehouse column sums to 100%.
transport_warehouse_pct = (
    transport_warehouse_counts
    .div(transport_warehouse_counts.sum(axis=0), axis=1) * 100
)

# Display methods present in Warehouse 3 and the ten most common overall.
selected_methods = transport_warehouse_counts.index[
    methods_with_warehouse_3
].union(transport_order[:10], sort=False)

fig, ax = plt.subplots(
    figsize=(9, max(5, len(selected_methods) * 0.45))
)

sns.heatmap(
    transport_warehouse_pct.loc[selected_methods],
    annot=True,
    fmt=".1f",
    cmap="Blues",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    cbar_kws={"label": "% of observations within warehouse"},
    ax=ax,
)

ax.set_title("Selected transport methods — composition within warehouse")
ax.set_xlabel("Warehouse")
ax.set_ylabel("Transport method")

plt.tight_layout()
plt.show()

##### Interpretation — Transport Method and Warehouse

Of the 21 observations from Warehouse 3, 20 have a recorded
transport method. Delivery 18 accounts for 16 of these observations
(80%), while Delivery 78 and Delivery 24 each account for two (10%).

Delivery 18 appears only four times with Warehouse 1. This concentration
helps explain the strong association between transport method and
warehouse.

However, Warehouse 3 contains few observations, so the association
should be interpreted cautiously. The two features are related,
but they are not interchangeable.

The percentages describe observations with both fields recorded.
Only selected transport methods are displayed for Warehouse 1.

#### 9.4.2. Payment Type and Commercial Zone

We will compare payment types across commercial zones to understand
the association identified in the previous heatmap.

Counts and percentages are shown together because the zones contain
very different numbers of observations. Missing values are excluded
from this comparison.

In [ ]:
# Keep observations with both categorical fields recorded.
payment_zone_data = train[
    ["payment_type", "commercial_zone", "customer_id"]
].dropna(subset=["payment_type", "commercial_zone"]).copy()

# Count purchase observations for each combination.
payment_zone_counts = pd.crosstab(
    payment_zone_data["payment_type"],
    payment_zone_data["commercial_zone"],
)

# Order payment types and zones by their observation counts.
payment_order = payment_zone_counts.sum(axis=1).sort_values(
    ascending=False
).index

zone_order = payment_zone_counts.sum(axis=0).sort_values(
    ascending=False
).index

payment_zone_counts = payment_zone_counts.loc[payment_order, zone_order]

# Count distinct customers within each combination.
payment_zone_customers = (
    payment_zone_data
    .groupby(["payment_type", "commercial_zone"])["customer_id"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(index=payment_order, columns=zone_order, fill_value=0)
)

print(f"Observations assessed: {len(payment_zone_data):,}")
print(f"Observations excluded: {len(train) - len(payment_zone_data):,}")

print("\nPurchase observations:")
display(payment_zone_counts)

print("\nDistinct customers:")
display(payment_zone_customers)

In [ ]:
# Calculate the payment-type composition within each commercial zone.
payment_zone_pct = payment_zone_counts.div(
    payment_zone_counts.sum(axis=0), axis=1
) * 100

# Display percentages and counts together to make small groups visible.
payment_zone_annotations = pd.DataFrame(
    index=payment_zone_counts.index,
    columns=payment_zone_counts.columns,
    dtype="object",
)

for payment in payment_zone_counts.index:
    for zone in payment_zone_counts.columns:
        percentage = payment_zone_pct.loc[payment, zone]
        count = payment_zone_counts.loc[payment, zone]

        payment_zone_annotations.loc[payment, zone] = (
            f"{percentage:.1f}%\n(n={count:,})"
        )

fig, ax = plt.subplots(figsize=(12, 8))

sns.heatmap(
    payment_zone_pct,
    annot=payment_zone_annotations,
    fmt="",
    cmap="Blues",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    cbar_kws={"label": "% of observations within commercial zone"},
    ax=ax,
)

ax.set_title("Payment-type composition within commercial zones")
ax.set_xlabel("Commercial zone")
ax.set_ylabel("Payment type")

plt.xticks(rotation=25, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

#### Interpretation — Payment Type and Commercial Zone

Payment types differ considerably across zones. Payment 7 accounts
for 75.2% of assessed observations in Zone 4, while Payment 10
dominates Zone 2 (76.9%). Payment 12 is most common in Zone 3 (51.9%).
These differences help explain the association of 0.55.

Some combinations contain repeated observations from few customers.
For example, the 113 observations using Payment 10 in Zone 2
belong to 14 customers.

Zone 7 contains only one assessed observation, so its 100% share
provides little evidence of a stable pattern. Zone 6 is absent
because its only observation has a missing payment type.

The features contain overlapping information, but this association
alone does not justify removing either feature.

### 9.5. Relationships Across Numerical Feature Groups

We will compare current purchase values, customer activity and
historical spending to identify relationships between these groups.

The comparison uses the diagnostic copies created earlier.
Previously flagged values remain excluded, without changing
the original training data.

Pairs are ranked by absolute Spearman correlation. Pearson correlation
and the number of available observations are also reported.

In [ ]:
# Combine the diagnostic numerical data from the previous sections.
numerical_relationship_data = pd.concat(
    [
        current_purchase_data,
        activity_time_data,
        historical_spending_data,
    ],
    axis=1,
)

# Keep one copy of variables that appear in more than one group.
numerical_relationship_data = numerical_relationship_data.loc[
    :, ~numerical_relationship_data.columns.duplicated()
].copy()

# Assign each variable to a group for the cross-group comparison.
feature_groups = {}

for column in current_purchase_data.columns:
    feature_groups[column] = "Current purchase"

for column in activity_time_data.columns:
    feature_groups[column] = "Activity and time"

for column in historical_spending_data.columns:
    # Preserve the activity classification of prior_purchase_count.
    feature_groups.setdefault(column, "Historical spending")

# Calculate correlations using the available values for each pair.
overall_pearson = numerical_relationship_data.corr(method="pearson")
overall_spearman = numerical_relationship_data.corr(method="spearman")

# Count observations with both values available.
available_values = numerical_relationship_data.notna().astype("int64")
overall_pair_counts = available_values.T @ available_values

# Collect each cross-group pair once.
cross_group_records = []
numerical_columns = numerical_relationship_data.columns.tolist()

for position, first in enumerate(numerical_columns):
    for second in numerical_columns[position + 1:]:
        if feature_groups[first] == feature_groups[second]:
            continue

        cross_group_records.append({
            "feature_1": first,
            "group_1": feature_groups[first],
            "feature_2": second,
            "group_2": feature_groups[second],
            "paired_observations": overall_pair_counts.loc[first, second],
            "pearson": overall_pearson.loc[first, second],
            "spearman": overall_spearman.loc[first, second],
        })

# Rank relationships by strength, regardless of positive or negative sign.
cross_group_summary = pd.DataFrame(cross_group_records)

cross_group_summary["absolute_spearman"] = (
    cross_group_summary["spearman"].abs()
)

cross_group_summary = cross_group_summary.sort_values(
    "absolute_spearman", ascending=False
).reset_index(drop=True)

print("Strongest cross-group numerical relationships:")
display(cross_group_summary.head(15))

#### Interpretation — Cross-Group Numerical Relationships

Purchase counts and spending are strongly associated within the same
window. Spearman correlation ranges from 0.83 for the 180-day and
365-day windows to 0.96 for the 30-day window.

Pearson correlations are lower, indicating that spending does not
increase proportionally with purchase counts. Shared zero values
also contribute to the strong rank correlations.

Days since the previous purchase is negatively associated with
30-day and 90-day spending. This partly reflects the window definitions:
a previous purchase outside a window implies no purchases within it.

Customer tenure is positively associated with total historical
spending (Spearman 0.72). Longer histories provide more opportunities
to accumulate spending, although purchase frequency and value also vary.

In [ ]:
# Compare numerical features using the diagnostic values from Section 9.
def plot_cross_group_relationship(first, second):
    pair_data = numerical_relationship_data[[first, second]].dropna()

    # Restrict these panels to finite, non-negative values.
    valid_pairs = (
        np.isfinite(pair_data).all(axis=1)
        & pair_data.ge(0).all(axis=1)
    )
    plot_data = pair_data.loc[valid_pairs]

    print(f"Relationship: {first} and {second}")
    print(f"Available pairs: {len(pair_data):,}")
    print(f"Pairs plotted: {len(plot_data):,}")

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Original values show the size and spread of observations.
    axes[0].scatter(
        plot_data[first],
        plot_data[second],
        alpha=0.2,
        s=15,
    )
    axes[0].set_title("Original scale")
    axes[0].set_xlabel(first)
    axes[0].set_ylabel(second)

    # Log1p compresses large values while retaining zeros.
    axes[1].scatter(
        np.log1p(plot_data[first]),
        np.log1p(plot_data[second]),
        alpha=0.2,
        s=15,
    )
    axes[1].set_title("Log1p scale")
    axes[1].set_xlabel(f"log1p({first})")
    axes[1].set_ylabel(f"log1p({second})")

    fig.suptitle(f"{first} and {second}")
    plt.tight_layout()
    plt.show()

In [ ]:
# Examine variation in spending at each recorded purchase count.
plot_cross_group_relationship(
    "purchase_count_90d",
    "spend_90d",
)

In [ ]:
# Examine how spending within 90 days relates to purchase recency.
plot_cross_group_relationship(
    "days_since_previous_purchase",
    "spend_90d",
)

#### Interpretation — Purchase Count and 90-Day Spending

Spending generally increases with purchase frequency, but observations
with the same count have widely different spending values. Some of
the largest amounts occur with relatively few purchases.

The vertical bands reflect integer purchase counts. Shared zeros
contribute to the strong Spearman correlation (0.91), while variation
in purchase values helps explain the lower Pearson correlation (0.50).

#### Interpretation — Purchase Recency and 90-Day Spending

Positive 90-day spending is concentrated among observations with
a recent previous purchase. Longer intervals are associated with
zero spending in this window.

This pattern partly follows from the variable definitions: when the
previous purchase occurred more than 90 days earlier, no purchases
fall within the preceding 90-day window.

Among observations with recent purchases, spending varies widely.
Recency and spending therefore describe related but different
aspects of customer activity.

### 9.6. Exploratory Principal Component Analysis

PCA summarises related numerical features into components that capture
variation in the data. We will examine how much variation these components
retain and which variables contribute to them.

This analysis uses the diagnostic numerical values from Section 9.5.
Identifiers, the target, categorical variables and random_noise are
excluded so the analysis focuses on recorded purchase behaviour.


#### 9.6.1. Preparation for PCA

Observations with no available numerical purchase information are excluded.
The remaining features are transformed with log1p to reduce skewness,
missing values are filled with each transformed column's median and
the columns are standardised.

Median imputation is a practical choice for this exploratory view.
It does not distinguish insufficient purchase history from other missing
values, so it may influence the component structure.

The original datasets remain unchanged. These transformations are used
only for exploration; any PCA used for modelling will be fitted within
the appropriate training splits.

In [ ]:
# Tools for exploratory imputation, scaling and PCA.
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Copy the diagnostic numerical features assembled in Section 9.5.
pca_input = numerical_relationship_data.copy()

# Explicitly exclude identifiers, the target and the supplied noise variable.
pca_input = pca_input.drop(
    columns=["ID", "customer_id", "repeat_purchase_90d", "random_noise"],
    errors="ignore",
)

# Ensure infinite values are treated as unavailable.
pca_input = pca_input.replace([np.inf, -np.inf], np.nan)

# These purchase features should be non-negative after diagnostic checks.
# Stop if unexpected negative values remain rather than silently altering them.
if pca_input.lt(0).any().any():
    raise ValueError("Review remaining negative values before applying log1p.")

# Exclude observations with no available numerical information.
pca_rows_retained = pca_input.notna().any(axis=1)
pca_input = pca_input.loc[pca_rows_retained].copy()

# Exclude columns with no observed variation.
pca_columns_retained = pca_input.nunique(dropna=True).gt(1)
pca_excluded_columns = pca_input.columns[~pca_columns_retained].tolist()
pca_input = pca_input.loc[:, pca_columns_retained]

# Record how much imputation each feature requires.
pca_preparation_summary = pd.DataFrame({
    "observed_values": pca_input.notna().sum(),
    "values_to_impute": pca_input.isna().sum(),
    "imputed_pct": pca_input.isna().mean() * 100,
})

# Reduce skewness while preserving zero values.
pca_log_data = np.log1p(pca_input)

# Fill missing values using medians fitted on this exploratory training copy.
pca_imputer = SimpleImputer(strategy="median")
pca_imputed_values = pca_imputer.fit_transform(pca_log_data)

# Give each feature unit variance before fitting PCA.
pca_scaler = StandardScaler()
pca_scaled_values = pca_scaler.fit_transform(pca_imputed_values)

print(f"Observations retained: {len(pca_input):,}")
print(f"Observations excluded: {(~pca_rows_retained).sum():,}")
print(f"Numerical features retained: {pca_input.shape[1]}")
print(f"Columns excluded for insufficient variation: {pca_excluded_columns}")

display(pca_preparation_summary)

#### 9.6.2. Explained Variance

We will fit PCA using all retained numerical features and examine
the variation captured by each component.

The cumulative curve shows how many components are needed to retain
90% or 95% of the variation in the prepared data. These thresholds
describe data compression, not classification performance.

In [ ]:
# Fit PCA without using the target.
pca_exploratory = PCA(svd_solver="full")
pca_coordinates = pca_exploratory.fit_transform(pca_scaled_values)

# Summarise individual and cumulative explained variance.
pca_variance_summary = pd.DataFrame({
    "component": np.arange(1, len(pca_exploratory.explained_variance_ratio_) + 1),
    "explained_variance_pct": (
        pca_exploratory.explained_variance_ratio_ * 100
    ),
})

pca_variance_summary["cumulative_variance_pct"] = (
    pca_variance_summary["explained_variance_pct"].cumsum()
)

display(pca_variance_summary)

# Report the first component count reaching each threshold.
for threshold in [90, 95]:
    components_needed = int(
        np.searchsorted(
            pca_variance_summary["cumulative_variance_pct"].to_numpy(),
            threshold,
        ) + 1
    )
    print(f"Components needed for {threshold}% variance: {components_needed}")

In [ ]:
# Plot individual and cumulative explained variance.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(
    pca_variance_summary["component"],
    pca_variance_summary["explained_variance_pct"],
)
axes[0].set_title("Explained variance by component")
axes[0].set_xlabel("Principal component")
axes[0].set_ylabel("Explained variance (%)")

axes[1].plot(
    pca_variance_summary["component"],
    pca_variance_summary["cumulative_variance_pct"],
    marker="o",
)

# Show the 90% and 95% variance-retention thresholds.
axes[1].axhline(90, color="grey", linestyle="--", label="90%")
axes[1].axhline(95, color="black", linestyle=":", label="95%")
axes[1].set_title("Cumulative explained variance")
axes[1].set_xlabel("Number of principal components")
axes[1].set_ylabel("Cumulative explained variance (%)")
axes[1].set_ylim(0, 105)
axes[1].legend()

plt.tight_layout()
plt.show()

#### Interpretation — Explained Variance

The first component captures 30.74% of the variation and the second
captures 21.92%, giving a combined 52.66%. A two-dimensional plot
will therefore show just over half of the variation in the prepared data.

Eleven components retain 90.99% of the variation, while fourteen
retain 95.01%, compared with the original 28 numerical features.
This indicates substantial overlap between the features.

These percentages refer to the transformed, imputed and standardised
data. They do not measure predictive performance or determine how many
components should be used for modelling.

#### 9.6.3. Features Defining the Principal Components

We will inspect the feature weights in the first three components,
which together capture 60.97% of the variation.

Larger absolute weights indicate a greater role in defining a component.
Features with the same sign contribute in the same direction, while
opposite signs indicate a contrast within that component.

A component's overall sign is arbitrary. These weights describe
variation in the prepared features, not predictive importance.

In [ ]:
# Store the weights defining each principal component.
pca_feature_weights = pd.DataFrame(
    pca_exploratory.components_.T,
    index=pca_input.columns,
    columns=[
        f"PC{number}"
        for number in range(1, pca_exploratory.n_components_ + 1)
    ],
)

pca_feature_weights.index.name = "variable"

# Squared weights sum to one within each component.
# Express them as percentages to compare feature contributions.
pca_feature_contributions = pca_feature_weights.pow(2) * 100

# Inspect the eight largest contributions to each of the first three components.
for component in ["PC1", "PC2", "PC3"]:
    component_summary = pd.DataFrame({
        "weight": pca_feature_weights[component],
        "contribution_pct": pca_feature_contributions[component],
    }).sort_values("contribution_pct", ascending=False)

    print(f"\nLargest feature contributions: {component}")
    display(component_summary.head(8))

In [ ]:
# Compare signed weights across the first three components.
selected_components = ["PC1", "PC2", "PC3"]

# Order features by their combined squared weights in these components.
feature_order = (
    pca_feature_contributions[selected_components]
    .sum(axis=1)
    .sort_values(ascending=False)
    .index
)

weights_to_plot = pca_feature_weights.loc[
    feature_order, selected_components
]

# Use symmetrical colour limits so positive and negative weights are comparable.
colour_limit = weights_to_plot.abs().to_numpy().max()

fig, ax = plt.subplots(figsize=(8, 12))

sns.heatmap(
    weights_to_plot,
    annot=True,
    fmt=".2f",
    cmap="vlag",
    center=0,
    vmin=-colour_limit,
    vmax=colour_limit,
    linewidths=0.5,
    cbar_kws={"label": "Feature weight"},
    ax=ax,
)

ax.set_title("Feature weights in the first three principal components")
ax.set_xlabel("Principal component")
ax.set_ylabel("Variable")

plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

#### Interpretation — Principal Component Feature Weights

PC1 mainly describes purchase activity. Purchase counts and recent
spending have positive weights, while days since the previous purchase
and historical purchase intervals have negative weights.

PC2 mainly describes purchase size. Gross, net and payable amounts
have the largest contributions, followed by taxes, purchase details
and cost.

PC3 contrasts longer customer histories and wider purchase intervals
with recent activity. Customer tenure contributes 25.91%, followed
by total historical spending and purchase-interval statistics.

These descriptions summarise patterns in the prepared data.
They do not establish which features are most useful for prediction.

#### 9.6.4. Two-Dimensional PCA View

Each point represents a purchase observation projected onto the first
two components, which together capture 52.66% of the prepared variation.

The target is used only to colour the points; it was not used to fit PCA.
The plot provides an exploratory view of class overlap and separation,
rather than an assessment of classification performance.

In [ ]:
# Preserve the row index so target values align with PCA coordinates.
pca_plot_data = pd.DataFrame(
    pca_coordinates[:, :2],
    index=pca_input.index,
    columns=["PC1", "PC2"],
)

# Attach the target after fitting PCA.
pca_plot_data["target"] = train.loc[
    pca_plot_data.index, "repeat_purchase_90d"
]

# Keep observations with a valid binary target.
pca_plot_data = pca_plot_data.loc[
    pca_plot_data["target"].isin([0, 1])
].copy()

# Use readable class labels.
pca_plot_data["target_class"] = pca_plot_data["target"].map({
    0: "0 — No repeat purchase",
    1: "1 — Repeat purchase",
})

# Shuffle drawing order to reduce systematic overlap from plotting one class last.
pca_plot_data = pca_plot_data.sample(frac=1, random_state=42)

print(f"Observations plotted: {len(pca_plot_data):,}")

fig, ax = plt.subplots(figsize=(11, 7))

sns.scatterplot(
    data=pca_plot_data,
    x="PC1",
    y="PC2",
    hue="target_class",
    hue_order=[
        "0 — No repeat purchase",
        "1 — Repeat purchase",
    ],
    palette=sns.color_palette("colorblind", 2),
    alpha=0.3,
    s=20,
    linewidth=0,
    ax=ax,
)

# Include explained variance in the axis labels.
pc1_variance = pca_exploratory.explained_variance_ratio_[0] * 100
pc2_variance = pca_exploratory.explained_variance_ratio_[1] * 100

ax.set_title("Purchase observations in the first two principal components")
ax.set_xlabel(f"PC1 ({pc1_variance:.2f}% explained variance)")
ax.set_ylabel(f"PC2 ({pc2_variance:.2f}% explained variance)")
ax.legend(title="Target")

plt.tight_layout()
plt.show()

#### Interpretation — Two-Dimensional PCA View

Observations followed by a repeat purchase are more concentrated
at higher PC1 values. This agrees with PC1's emphasis on greater
purchase activity and shorter intervals between purchases.

The classes overlap substantially, particularly in the central region.
PC2, which mainly reflects purchase size, shows less visible separation.

These two components capture 52.66% of the prepared variation.
The overlap does not rule out useful information in other components
or relationships that classification models could capture.

Repeated customer observations and median imputation may also influence
the visible concentrations. The plot does not establish distinct
customer groups or predictive performance.

## 10. Temporal and Customer-Level Analysis

We will examine how purchase observations and repeat purchase rates
vary over time and across customers.

These summaries describe the supplied training observations.
Changes may reflect customer composition or dataset coverage,
as well as changes in purchasing behaviour.


### 10.1. Yearly Purchase Observations and Repeat Purchase Rates

We will compare observation counts, distinct customers and target
rates by purchase year.

The first and last years cover only part of a calendar year,
so their observation counts are not directly comparable with
those of the intervening years.

In [ ]:
# Create a separate copy for temporal exploration.
temporal_data = train[
    ["ID", "customer_id", "purchase_date", "repeat_purchase_90d"]
].copy()

# Parse dates without changing the original training column.
temporal_data["parsed_purchase_date"] = pd.to_datetime(
    temporal_data["purchase_date"],
    format="%Y-%m-%d",
    errors="coerce",
)

# Assess observations with a valid date and binary target.
valid_temporal_rows = (
    temporal_data["parsed_purchase_date"].notna()
    & temporal_data["repeat_purchase_90d"].isin([0, 1])
)

temporal_data = temporal_data.loc[valid_temporal_rows].copy()
temporal_data["year"] = temporal_data["parsed_purchase_date"].dt.year

# Summarise the target and customer coverage within each year.
yearly_target_summary = temporal_data.groupby("year").agg(
    observations=("ID", "size"),
    unique_customers=("customer_id", "nunique"),
    target_1=("repeat_purchase_90d", "sum"),
    first_observed_date=("parsed_purchase_date", "min"),
    last_observed_date=("parsed_purchase_date", "max"),
)

yearly_target_summary["target_0"] = (
    yearly_target_summary["observations"]
    - yearly_target_summary["target_1"]
)

yearly_target_summary["repeat_purchase_pct"] = (
    yearly_target_summary["target_1"]
    / yearly_target_summary["observations"] * 100
)

yearly_target_summary = yearly_target_summary[
    [
        "observations",
        "unique_customers",
        "target_0",
        "target_1",
        "repeat_purchase_pct",
        "first_observed_date",
        "last_observed_date",
    ]
]

print(f"Observations assessed: {len(temporal_data):,}")
print(f"Observations excluded: {len(train) - len(temporal_data):,}")
display(yearly_target_summary)

In [ ]:
# Use the assessed population's target rate as the reference.
temporal_overall_rate = (
    temporal_data["repeat_purchase_90d"].mean() * 100
)

year_labels = yearly_target_summary.index.astype(str)
positions = np.arange(len(year_labels))

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Counts provide context for interpreting the yearly percentages.
axes[0].bar(
    positions,
    yearly_target_summary["observations"],
)

axes[0].set_title("Training observations by purchase year")
axes[0].set_ylabel("Observations")

# Plot yearly rates without implying an independently fitted trend.
axes[1].plot(
    positions,
    yearly_target_summary["repeat_purchase_pct"],
    marker="o",
)

axes[1].axhline(
    temporal_overall_rate,
    color="grey",
    linestyle="--",
    label=f"Overall assessed rate: {temporal_overall_rate:.2f}%",
)

axes[1].set_title("Repeat purchase rate by purchase year")
axes[1].set_xlabel("Purchase year")
axes[1].set_ylabel("Repeat purchase rate (%)")
axes[1].set_ylim(0, 100)
axes[1].set_xticks(positions)
axes[1].set_xticklabels(year_labels)
axes[1].legend()

plt.tight_layout()
plt.show()

#### Interpretation — Yearly Patterns

Repeat purchase rates vary across years, from 48.09% in 2022
to 63.28% in 2017. There is no consistent upward or downward trend
across the full period, although rates increase from 2022 to 2024.

Observation counts and customer coverage also vary. The largest
yearly count occurs in 2021, with 782 observations from 228 customers.
Changes in customer composition may partly explain the rate differences.

The boundary years have partial coverage: 2015 begins in July and
2025 ends on 15 February. The 2025 rate of 62.82% is based on only
78 observations and should not be treated as a full-year result.

These summaries describe the supplied observations rather than
complete business transaction volumes.

### 10.2. Calendar-Month Patterns

We will compare observation counts and repeat purchase rates by
calendar month using 2016–2024, excluding the partial boundary years.

Pooling years provides an overview of possible seasonal patterns.
However, differences may also reflect customer composition and
variation between years, so they do not establish seasonality alone.

In [ ]:
# Use the intervening calendar years for the monthly comparison.
calendar_month_data = temporal_data.loc[
    temporal_data["year"].between(2016, 2024)
].copy()

# Extract month numbers to maintain chronological ordering.
calendar_month_data["month"] = (
    calendar_month_data["parsed_purchase_date"].dt.month
)

month_labels = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]

# Summarise purchase observations and customer coverage by calendar month.
calendar_month_summary = (
    calendar_month_data.groupby("month")
    .agg(
        observations=("ID", "size"),
        unique_customers=("customer_id", "nunique"),
        years_represented=("year", "nunique"),
        target_1=("repeat_purchase_90d", "sum"),
    )
    .reindex(range(1, 13))
)

calendar_month_summary["target_0"] = (
    calendar_month_summary["observations"]
    - calendar_month_summary["target_1"]
)

calendar_month_summary["repeat_purchase_pct"] = (
    calendar_month_summary["target_1"]
    / calendar_month_summary["observations"] * 100
)

calendar_month_summary.insert(0, "month_label", month_labels)

print(f"Observations assessed: {len(calendar_month_data):,}")
print(
    "Dated observations excluded from partial boundary years:",
    f"{len(temporal_data) - len(calendar_month_data):,}",
)

display(calendar_month_summary[
    [
        "month_label",
        "observations",
        "unique_customers",
        "years_represented",
        "target_0",
        "target_1",
        "repeat_purchase_pct",
    ]
])

In [ ]:
# Calculate the reference rate using the same observations as the monthly analysis.
calendar_month_overall_rate = (
    calendar_month_data["repeat_purchase_90d"].mean() * 100
)

positions = np.arange(12)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Show the number of supplied observations in each month.
axes[0].bar(
    positions,
    calendar_month_summary["observations"],
)
axes[0].set_title("Training observations by calendar month — 2016–2024")
axes[0].set_ylabel("Observations")

# Compare repeat purchase rates alongside the observation counts.
axes[1].plot(
    positions,
    calendar_month_summary["repeat_purchase_pct"],
    marker="o",
)

axes[1].axhline(
    calendar_month_overall_rate,
    color="grey",
    linestyle="--",
    label=f"Overall assessed rate: {calendar_month_overall_rate:.2f}%",
)

axes[1].set_title("Repeat purchase rate by calendar month — 2016–2024")
axes[1].set_xlabel("Purchase month")
axes[1].set_ylabel("Repeat purchase rate (%)")
axes[1].set_ylim(0, 100)
axes[1].set_xticks(positions)
axes[1].set_xticklabels(month_labels)
axes[1].legend()

plt.tight_layout()
plt.show()

#### Interpretation — Calendar-Month Patterns

Repeat purchase rates range from 43.84% in March to 72.45% in May.
September (71.12%) and January (69.44%) also have relatively high rates.

March, November and July contain the most observations, but their
repeat purchase rates are below the overall assessed rate of 56.03%.
More observations therefore do not necessarily mean a higher target rate.

All months are represented in nine years, although observation counts
and customer coverage differ substantially. May's highest rate is based
on 196 observations from 95 customers.

These pooled differences suggest that purchase month may be useful
to investigate further. We still need to check whether the pattern
recurs across years before describing it as consistent seasonality.

#### 10.2.1. Monthly Patterns Within Each Year

We will compare observation counts and repeat purchase rates for
each year–month combination from 2016 to 2024.

The count heatmap provides context for the rate heatmap. Percentages
based on few observations can vary substantially, and repeated
observations from the same customers may influence the patterns.

In [ ]:
# Summarise each year–month combination using the same population as Section 10.2.
year_month_summary = calendar_month_data.groupby(
    ["year", "month"]
).agg(
    observations=("ID", "size"),
    unique_customers=("customer_id", "nunique"),
    target_1=("repeat_purchase_90d", "sum"),
)

year_month_summary["repeat_purchase_pct"] = (
    year_month_summary["target_1"]
    / year_month_summary["observations"] * 100
)

# Arrange years as rows and calendar months as columns.
year_month_counts = (
    year_month_summary["observations"]
    .unstack("month")
    .reindex(index=range(2016, 2025), columns=range(1, 13))
    .fillna(0)
    .astype(int)
)

# Leave rates missing where no observations are available.
year_month_rates = (
    year_month_summary["repeat_purchase_pct"]
    .unstack("month")
    .reindex(index=range(2016, 2025), columns=range(1, 13))
)

print("Observations per year–month:")
display(year_month_counts)

# Show the combinations with the smallest observed samples.
print("\nYear–month combinations with the fewest observations:")
display(
    year_month_summary.sort_values("observations").head(10)
)

In [ ]:
# Display counts to identify uneven coverage across years and months.
fig, ax = plt.subplots(figsize=(14, 6))

sns.heatmap(
    year_month_counts,
    annot=True,
    fmt="d",
    cmap="Blues",
    linewidths=0.5,
    xticklabels=month_labels,
    yticklabels=year_month_counts.index,
    cbar_kws={"label": "Observations"},
    ax=ax,
)

ax.set_title("Training observations by year and month — 2016–2024")
ax.set_xlabel("Purchase month")
ax.set_ylabel("Purchase year")

plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Include each cell's sample size alongside its repeat purchase rate.
year_month_annotations = pd.DataFrame(
    "",
    index=year_month_rates.index,
    columns=year_month_rates.columns,
)

for year in year_month_rates.index:
    for month in year_month_rates.columns:
        rate = year_month_rates.loc[year, month]
        count = year_month_counts.loc[year, month]

        if pd.notna(rate):
            year_month_annotations.loc[year, month] = (
                f"{rate:.1f}%\n(n={count})"
            )

fig, ax = plt.subplots(figsize=(15, 7))

sns.heatmap(
    year_month_rates,
    mask=year_month_rates.isna(),
    annot=year_month_annotations,
    fmt="",
    cmap="YlGnBu",
    vmin=0,
    vmax=100,
    linewidths=0.5,
    xticklabels=month_labels,
    yticklabels=year_month_rates.index,
    cbar_kws={"label": "Repeat purchase rate (%)"},
    ax=ax,
)

ax.set_title("Repeat purchase rates by year and month — 2016–2024")
ax.set_xlabel("Purchase month")
ax.set_ylabel("Purchase year")

plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

#### Interpretation — Monthly Patterns Within Years

Some monthly differences recur across years. January and May have
relatively high repeat purchase rates throughout the period, while
March remains below 55% in every year.

March, July and November also consistently contain relatively many
observations. Their rates are often lower, although exceptions occur:
November reaches 72.2% in 2020.

Sample sizes vary substantially, from four observations in February
2019 to 135 in March 2021. High percentages in small groups require
particular caution.

The recurring differences support investigating calendar month as
a potential feature. However, customer composition and repeated
observations may contribute to these patterns, so the heatmaps
do not establish a seasonal effect on their own.

### 10.3. Customer Concentration

Customers contribute different numbers of purchase observations.
This analysis measures how strongly training is concentrated among
the most frequently observed customers.

All training observations are included, including those with missing
purchase dates. The counts describe the supplied training dataset,
rather than each customer's complete purchase history.

In [ ]:
# Count observations and target outcomes for each customer.
customer_summary = (
    train.groupby("customer_id")
    .agg(
        observations=("ID", "size"),
        target_0=("repeat_purchase_90d", lambda values: values.eq(0).sum()),
        target_1=("repeat_purchase_90d", lambda values: values.eq(1).sum()),
    )
    .sort_values("observations", ascending=False)
)

customer_summary["repeat_purchase_pct"] = (
    customer_summary["target_1"]
    / customer_summary["observations"]
    * 100
)

# Rank customers by their contribution to training.
customer_summary["customer_rank"] = np.arange(
    1, len(customer_summary) + 1
)
customer_summary["share_of_training_rows_pct"] = (
    customer_summary["observations"] / len(train) * 100
)
customer_summary["cumulative_row_share_pct"] = (
    customer_summary["share_of_training_rows_pct"].cumsum()
)
customer_summary["cumulative_customer_share_pct"] = (
    customer_summary["customer_rank"] / len(customer_summary) * 100
)

# Compare several concentration thresholds.
top_ten_pct_count = int(np.ceil(len(customer_summary) * 0.10))

concentration_records = []

for label, requested_count in [
    ("Top 1 customer", 1),
    ("Top 5 customers", 5),
    ("Top 10 customers", 10),
    ("Top 20 customers", 20),
    ("Top 10% of customers", top_ten_pct_count),
]:
    selected_count = min(requested_count, len(customer_summary))
    selected_customers = customer_summary.head(selected_count)
    selected_rows = selected_customers["observations"].sum()

    concentration_records.append({
        "group": label,
        "customers": selected_count,
        "share_of_customers_pct": (
            selected_count / len(customer_summary) * 100
        ),
        "observations": selected_rows,
        "share_of_training_rows_pct": selected_rows / len(train) * 100,
        "repeat_purchase_pct": (
            selected_customers["target_1"].sum() / selected_rows * 100
        ),
    })

customer_concentration_summary = (
    pd.DataFrame(concentration_records).set_index("group")
)

display(customer_concentration_summary.round(3))

print("\nTen customers with the most training observations:")
display(
    customer_summary[
        [
            "observations",
            "share_of_training_rows_pct",
            "target_0",
            "target_1",
            "repeat_purchase_pct",
        ]
    ].head(10).round(3)
)

In [ ]:
# Show how quickly observations accumulate when customers are
# ordered from most to least frequently observed.
customer_share = np.r_[
    0, customer_summary["cumulative_customer_share_pct"].to_numpy()
]
observation_share = np.r_[
    0, customer_summary["cumulative_row_share_pct"].to_numpy()
]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(
    customer_share,
    observation_share,
    color="steelblue",
    linewidth=2.5,
    label="Observed concentration",
)

# Equal contributions would follow the diagonal.
ax.plot(
    [0, 100],
    [0, 100],
    color="grey",
    linestyle="--",
    label="Equal observations per customer",
)

top_ten_pct_row_share = customer_summary.iloc[
    top_ten_pct_count - 1
]["cumulative_row_share_pct"]

top_ten_pct_customer_share = (
    top_ten_pct_count / len(customer_summary) * 100
)

ax.scatter(
    top_ten_pct_customer_share,
    top_ten_pct_row_share,
    color="darkorange",
    s=70,
    zorder=3,
)

ax.annotate(
    f"Top {top_ten_pct_count} customers:\n"
    f"{top_ten_pct_row_share:.1f}% of observations",
    xy=(top_ten_pct_customer_share, top_ten_pct_row_share),
    xytext=(12, -35),
    textcoords="offset points",
)

ax.set(
    title="Concentration of Training Observations Across Customers",
    xlabel="Cumulative share of customers (%) — most observed first",
    ylabel="Cumulative share of training observations (%)",
    xlim=(0, 100),
    ylim=(0, 100),
)

ax.grid(alpha=0.2)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

#### Interpretation — Customer Concentration

The 61 most frequently observed customers represent 10% of customers
but contribute 50.4% of training observations.

The curve lies well above the equal-contribution diagonal, showing
that observations are concentrated among a relatively small group
of customers.

These customers therefore have more influence on overall statistics
and target rates. We will next compare the observation-level repeat
purchase rate with a rate that gives each customer equal weight.

### 10.4. Repeat Purchase Rates with Equal Customer Weight

The observation-level rate gives every purchase observation equal weight,
so customers with more observations contribute more.

The equal-customer rate first calculates each customer's repeat purchase
rate, then averages those rates. Each customer contributes equally,
regardless of their number of observations.

Both describe the supplied training data from different perspectives.

In [ ]:
# Give each purchase observation equal weight.
observation_level_rate = (
    customer_summary["target_1"].sum()
    / customer_summary["observations"].sum()
    * 100
)

# Give each customer equal weight by averaging their individual rates.
equal_customer_rate = customer_summary["repeat_purchase_pct"].mean()

rate_comparison = pd.DataFrame({
    "approach": [
        "Equal weight per observation",
        "Equal weight per customer",
    ],
    "observations": [
        customer_summary["observations"].sum(),
        customer_summary["observations"].sum(),
    ],
    "customers": [
        len(customer_summary),
        len(customer_summary),
    ],
    "repeat_purchase_pct": [
        observation_level_rate,
        equal_customer_rate,
    ],
}).set_index("approach")

display(rate_comparison.round(3))

print(
    "Observation-level rate minus equal-customer rate: "
    f"{observation_level_rate - equal_customer_rate:.2f} percentage points"
)

In [ ]:
# Compare rates using the same percentage scale.
fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    ["Equal weight\nper observation", "Equal weight\nper customer"],
    [observation_level_rate, equal_customer_rate],
    color=["steelblue", "darkorange"],
    width=0.55,
)

ax.bar_label(bars, fmt="%.2f%%", padding=5)

ax.set(
    title="Repeat Purchase Rate Under Different Weighting Approaches",
    ylabel="Repeat purchase rate (%)",
    ylim=(0, 100),
)

ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

### 10.5. Target Outcomes Across Customers

Customers are grouped according to whether their training observations
contain only target 0, only target 1 or both outcomes.

These groups describe recorded outcomes in training. A customer with
only one observation necessarily belongs to an “only” group, so we
also separate customers with one observation from those with several.

In [ ]:
# Classify customers using their observed training outcomes.
customer_outcomes = customer_summary.copy()

customer_outcomes["outcome_group"] = np.select(
    [
        customer_outcomes["target_1"].eq(0),
        customer_outcomes["target_0"].eq(0),
    ],
    [
        "Only target 0",
        "Only target 1",
    ],
    default="Both outcomes",
)

customer_outcomes["observation_group"] = np.where(
    customer_outcomes["observations"].eq(1),
    "One observation",
    "Multiple observations",
)

outcome_order = [
    "Only target 0",
    "Only target 1",
    "Both outcomes",
]

customer_outcome_summary = (
    customer_outcomes.groupby("outcome_group")
    .agg(
        customers=("observations", "size"),
        observations=("observations", "sum"),
        median_observations_per_customer=("observations", "median"),
        maximum_observations_per_customer=("observations", "max"),
    )
    .reindex(outcome_order)
)

customer_outcome_summary["share_of_customers_pct"] = (
    customer_outcome_summary["customers"]
    / len(customer_outcomes)
    * 100
)

customer_outcome_summary["share_of_training_rows_pct"] = (
    customer_outcome_summary["observations"] / len(train) * 100
)

display(customer_outcome_summary.round(3))

# Separate single-observation customers from repeated customers.
customer_outcome_counts = pd.crosstab(
    customer_outcomes["observation_group"],
    customer_outcomes["outcome_group"],
).reindex(
    index=["One observation", "Multiple observations"],
    columns=outcome_order,
    fill_value=0,
)

print("\nCustomers by observation count and target outcomes:")
display(customer_outcome_counts)

In [ ]:
# Display customer counts while retaining the distinction between
# single-observation customers and repeated customers.
ax = customer_outcome_counts.T.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6),
    color=["steelblue", "darkorange"],
    rot=0,
)

ax.set(
    title="Target Outcomes Across Training Customers",
    xlabel="Observed target outcomes",
    ylabel="Number of customers",
)

ax.legend(title="Training observations")
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

#### Interpretation — Target Outcomes Across Customers

Most customers have both target outcomes: 349 customers account for
89.61% of training observations. Repeat purchase outcomes therefore
vary across observations for many customers.

The 258 customers with only target 0 contribute just 7.06% of rows.
This group includes all 157 customers with a single observation;
their limited recorded history does not establish that they never
make repeat purchases.

Only three customers have exclusively target 1. They contribute
218 observations, of which 203 belong to one customer.

These differences reinforce the importance of considering customer
concentration and repeated observations when interpreting results
and designing validation.

### 10.6. Target Consistency with Later Observed Purchases

For each dated observation, we identify the next dated purchase
recorded for the same customer in training.

A later purchase within 90 days should correspond to target 1,
assuming these records follow the target's purchase definition.

The absence of a later purchase within 90 days does not establish
target 0. Training may omit purchases, and the latest training date
does not necessarily mark the end of target follow-up.

This check uses future observations only to assess label consistency.
The derived fields must not be used as prediction features.

In [ ]:
# Work on a separate copy of dated observations with valid targets.
target_followup_data = temporal_data.sort_values(
    ["customer_id", "parsed_purchase_date", "ID"]
).copy()

# Identify the next purchase observed in training for each customer.
target_followup_data["next_observed_purchase_date"] = (
    target_followup_data.groupby("customer_id")[
        "parsed_purchase_date"
    ].shift(-1)
)

target_followup_data["days_to_next_observed_purchase"] = (
    target_followup_data["next_observed_purchase_date"]
    - target_followup_data["parsed_purchase_date"]
).dt.days

observed_repeat_within_90d = (
    target_followup_data["days_to_next_observed_purchase"]
    .between(1, 90)
)

# Flag target 0 only where a later observed purchase provides
# direct evidence of a repeat within the target window.
target_followup_data["potential_target_inconsistency"] = (
    observed_repeat_within_90d
    & target_followup_data["repeat_purchase_90d"].eq(0)
)

target_followup_data["observed_followup_group"] = np.select(
    [
        observed_repeat_within_90d,
        target_followup_data[
            "days_to_next_observed_purchase"
        ].gt(90),
        target_followup_data[
            "days_to_next_observed_purchase"
        ].isna(),
    ],
    [
        "Next observed purchase within 90 days",
        "Next observed purchase after 90 days",
        "No later dated purchase observed",
    ],
    default="Non-positive observed gap — investigate",
)

followup_order = [
    "Next observed purchase within 90 days",
    "Next observed purchase after 90 days",
    "No later dated purchase observed",
    "Non-positive observed gap — investigate",
]

target_followup_summary = (
    target_followup_data.groupby("observed_followup_group")
    .agg(
        observations=("ID", "size"),
        unique_customers=("customer_id", "nunique"),
        target_0=("repeat_purchase_90d", lambda values: values.eq(0).sum()),
        target_1=("repeat_purchase_90d", lambda values: values.eq(1).sum()),
    )
    .reindex(followup_order, fill_value=0)
)

display(target_followup_summary)

print(f"\nDated observations assessed: {len(target_followup_data):,}")
print(f"Observations excluded: {len(train) - len(target_followup_data):,}")

evidence_rows = int(observed_repeat_within_90d.sum())
flagged_rows = int(
    target_followup_data["potential_target_inconsistency"].sum()
)

print(f"Observed repeats within 90 days: {evidence_rows:,}")
print(f"Potential target inconsistencies: {flagged_rows:,}")

if evidence_rows > 0:
    print(
        "Potential inconsistencies among observed repeats: "
        f"{flagged_rows / evidence_rows * 100:.3f}%"
    )

# Display flagged records for investigation without changing labels.
if flagged_rows > 0:
    display(
        target_followup_data.loc[
            target_followup_data["potential_target_inconsistency"],
            [
                "ID",
                "customer_id",
                "parsed_purchase_date",
                "next_observed_purchase_date",
                "days_to_next_observed_purchase",
                "repeat_purchase_90d",
            ],
        ].head(30)
    )

#### Interpretation — Target Consistency

All 3,565 observations followed by another recorded purchase within
90 days have target 1. No inconsistencies were found in this check.

Another 56 observations have target 1 without a later dated purchase
within 90 days appearing in training. These are not necessarily errors:
relevant purchases may be absent from the supplied training records.

The check supports label consistency where a repeat purchase is
observed, but cannot verify every label. The 65 observations with
missing dates were excluded.

### 10.7. Main Findings from Temporal and Customer Analysis

- Repeat purchase rates vary across years and calendar months.
  Some monthly patterns recur, but customer composition and repeated
  observations may contribute to these differences.
- The 61 most frequently observed customers account for 50.4% of
  training observations, despite representing only 10% of customers.
- Customers with both target outcomes contribute 89.61% of training
  rows. Outcomes vary across purchases for many customers.
- All 3,565 observed repeat purchases within 90 days agree with target 1.
  Missing follow-up records prevent complete verification of labels.
- Validation must account for chronological order, the 90-day target
  window and whether predictions concern existing or unseen customers.

## 11. Customer Data Analysis and Integration

The `clientes` table contains customer attributes that may complement
the purchase information in training.

We first examine this table independently, then assess matching
coverage and create `train_clientes` through a validated join.



### 11.1. Description of Customer Variables

The descriptions below are inferred from column names and supplied
values. Anonymised labels identify categories but do not reveal their
original business meanings.

In [ ]:
# Document each customer variable and its intended analytical role.
customer_variable_definitions = [
    (
        "customer_id",
        "Customer identifier",
        "Identifier / join key",
        "Links customers to purchase observations; numeric values "
        "do not represent quantities.",
    ),
    (
        "country_or_region",
        "Customer's recorded country or territory",
        "Categorical / geographic",
        "Broad geographic grouping; original locations are anonymised.",
    ),
    (
        "commercial_region",
        "Commercial region associated with the customer",
        "Categorical / commercial",
        "May describe sales organisation rather than administrative geography.",
    ),
    (
        "island",
        "Island associated with the customer's recorded location",
        "Categorical / geographic",
        "Missing values may indicate non-applicability or unavailable information.",
    ),
    (
        "municipality",
        "Municipality associated with the customer's recorded location",
        "Categorical / geographic",
        "Potential geographic grouping; relationships with other fields "
        "require inspection.",
    ),
    (
        "city",
        "City associated with the customer's recorded location",
        "Categorical / geographic",
        "Many distinct labels and substantial missingness.",
    ),
    (
        "postal_area",
        "Postal area associated with the customer",
        "Categorical / geographic",
        "A geographic label; its relationship with postal code "
        "requires inspection.",
    ),
    (
        "postal_code",
        "Customer's recorded postal code",
        "Categorical / geographic",
        "Treat as a label rather than a numerical quantity.",
    ),
    (
        "street",
        "Street associated with the customer's recorded location",
        "Categorical / address",
        "Many distinct labels; may identify small groups of customers.",
    ),
    (
        "address",
        "Customer's recorded address label",
        "Categorical / potential identifier",
        "Almost unique across customers; review its usefulness "
        "and availability before modelling.",
    ),
    (
        "customer_type",
        "Intended customer classification",
        "Categorical / unavailable",
        "Entirely missing in the supplied table; its observed float dtype "
        "does not establish a numerical meaning.",
    ),
    (
        "company_name",
        "Customer's recorded company name label",
        "Categorical / potential identifier",
        "Almost unique across customers; the anonymised text "
        "does not establish company characteristics.",
    ),
    (
        "customer_since_year",
        "Year recorded as the start of the customer relationship",
        "Temporal / year",
        "Confirm its definition and check consistency with purchase dates; "
        "it may differ from the first recorded purchase year.",
    ),
]

customer_dictionary = pd.DataFrame(
    customer_variable_definitions,
    columns=[
        "variable",
        "description",
        "analytical_type",
        "notes",
    ],
).set_index("variable")

# Attach observed properties so definitions and data characteristics
# can be reviewed together.
customer_dictionary["observed_dtype"] = clientes.dtypes.astype(str)
customer_dictionary["missing_pct"] = clientes.isna().mean() * 100
customer_dictionary["unique_non_missing"] = clientes.nunique(dropna=True)

# Show complete variable descriptions and notes.
pd.set_option("display.max_colwidth", None)
display(customer_dictionary)

### 11.2. Customer Keys and Duplicate Records

The intended structure is one row per customer. We check missing and
repeated customer IDs, exact duplicate rows and records that match
across all customer attributes despite having different IDs.

Matching attributes alone do not establish that two customers are
duplicates.

In [ ]:
# Verify whether customer_id supports a many-to-one purchase join.
customer_key_summary = pd.DataFrame({
    "count": {
        "Total customer rows": len(clientes),
        "Missing customer IDs": clientes["customer_id"].isna().sum(),
        "Unique non-missing customer IDs": (
            clientes["customer_id"].nunique(dropna=True)
        ),
        "Repeated non-missing IDs beyond first": (
            clientes.loc[
                clientes["customer_id"].notna(), "customer_id"
            ].duplicated().sum()
        ),
        "Exact duplicate rows beyond first": clientes.duplicated().sum(),
    }
})

display(customer_key_summary)

# Inspect repeated keys without removing any records.
repeated_customer_keys = (
    clientes["customer_id"].notna()
    & clientes["customer_id"].duplicated(keep=False)
)

if repeated_customer_keys.any():
    print("\nRecords with repeated customer IDs:")
    display(
        clientes.loc[repeated_customer_keys]
        .sort_values("customer_id")
        .head(30)
    )
else:
    print("No repeated non-missing customer IDs found.")

# Check matching attributes independently of the customer identifier.
customer_attribute_columns = [
    column for column in clientes.columns
    if column != "customer_id"
]

matching_customer_attributes = clientes.duplicated(
    subset=customer_attribute_columns,
    keep=False,
)

print(
    "\nRows with matching customer attributes: "
    f"{matching_customer_attributes.sum():,}"
)

if matching_customer_attributes.any():
    display(clientes.loc[matching_customer_attributes].head(30))

#### Interpretation — Customer Keys and Duplicates

The customer table contains 1,143 unique customer IDs, with no
missing or repeated keys. No exact duplicate rows or records with
identical customer attributes were found.

This supports a many-to-one join: several purchase observations
can match one customer record without multiplying training rows.
Matching coverage still needs to be checked.

### 11.3. Missing Customer Information

We examine missingness across customer attributes and the number
of missing fields per customer.

Because `customer_type` is entirely missing, it is reported separately
when assessing the completeness of the remaining attributes.

In [ ]:
# Exclude the customer identifier when assessing attribute completeness.
customer_attribute_columns = [
    column for column in clientes.columns
    if column != "customer_id"
]

customer_missingness = pd.DataFrame({
    "missing_count": clientes[customer_attribute_columns].isna().sum(),
    "missing_pct": (
        clientes[customer_attribute_columns].isna().mean() * 100
    ),
}).sort_values("missing_pct", ascending=False)

display(customer_missingness.round(3))

# Separate entirely missing attributes from those containing information.
all_missing_customer_columns = [
    column for column in customer_attribute_columns
    if clientes[column].isna().all()
]

informative_customer_columns = [
    column for column in customer_attribute_columns
    if column not in all_missing_customer_columns
]

print("Entirely missing attributes:", all_missing_customer_columns)

# Count missing fields per customer among attributes with observed values.
missing_customer_fields = (
    clientes[informative_customer_columns].isna().sum(axis=1)
)

customer_completeness_summary = (
    missing_customer_fields.value_counts()
    .sort_index()
    .rename_axis("missing_customer_fields")
    .to_frame("customers")
)

customer_completeness_summary["customer_pct"] = (
    customer_completeness_summary["customers"] / len(clientes) * 100
)

print("\nCompleteness excluding entirely missing attributes:")
display(customer_completeness_summary.round(3))

In [ ]:
# Display all customer attributes, including the entirely missing field.
plot_data = customer_missingness.sort_values("missing_pct")

fig, ax = plt.subplots(figsize=(10, 6))

bars = ax.barh(
    plot_data.index,
    plot_data["missing_pct"],
    color="steelblue",
)

ax.bar_label(bars, fmt="%.2f%%", padding=4)

ax.set(
    title="Missing Values in Customer Attributes",
    xlabel="Customers with missing values (%)",
    ylabel="",
    xlim=(0, 110),
)

ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)

plt.tight_layout()
plt.show()

#### Interpretation — Missing Customer Information

`customer_type` is entirely missing and provides no observed information.

Missingness is also substantial in `street` (52.32%), `island` (50.57%)
and `city` (47.77%). Missing island values may reflect non-applicability,
but this cannot be established from the missingness percentages alone.

Excluding `customer_type`, only 121 customers (10.59%) have complete
attributes. Most customers have one or two missing fields.

`country_or_region`, `company_name` and `customer_since_year` are
complete, while `address` has only six missing values. Completeness
alone does not establish accuracy or usefulness for prediction.

### 11.4. Customer Categorical Label Quality

We inspect text attributes for blank labels, surrounding whitespace
and potential category variants after trimming spaces and ignoring case.

Normalisation is used only for comparison. The supplied labels remain
unchanged, and matching normalised labels require review before merging.

In [ ]:
# Select text attributes; customer_type is entirely missing
# and has no observed labels to inspect.
customer_text_columns = clientes.select_dtypes(
    include=["object", "string"]
).columns.tolist()

customer_label_records = []
customer_label_variants = {}

for column in customer_text_columns:
    # Preserve missing values when converting to pandas string dtype.
    labels = clientes[column].astype("string")
    trimmed_labels = labels.str.strip()

    blank_labels = (
        labels.notna() & trimmed_labels.eq("").fillna(False)
    )
    surrounding_whitespace = (
        labels.notna() & labels.ne(trimmed_labels).fillna(False)
    )

    # Compare distinct non-blank labels without changing the source data.
    observed_labels = labels.loc[
        labels.notna() & ~blank_labels
    ]

    label_mapping = pd.DataFrame({
        "supplied_label": observed_labels,
        "normalised_label": observed_labels.str.strip().str.casefold(),
    }).drop_duplicates()

    supplied_labels_per_normalised = (
        label_mapping.groupby("normalised_label")[
            "supplied_label"
        ].nunique()
    )

    variant_keys = supplied_labels_per_normalised.loc[
        supplied_labels_per_normalised.gt(1)
    ].index

    customer_label_records.append({
        "variable": column,
        "missing_values": int(labels.isna().sum()),
        "blank_labels": int(blank_labels.sum()),
        "rows_with_surrounding_whitespace": (
            int(surrounding_whitespace.sum())
        ),
        "distinct_non_blank_labels": observed_labels.nunique(),
        "distinct_normalised_non_blank_labels": (
            label_mapping["normalised_label"].nunique()
        ),
        "potential_variant_groups": len(variant_keys),
    })

    if len(variant_keys) > 0:
        customer_label_variants[column] = (
            label_mapping.loc[
                label_mapping["normalised_label"].isin(variant_keys)
            ]
            .sort_values(["normalised_label", "supplied_label"])
        )

customer_label_quality = (
    pd.DataFrame(customer_label_records).set_index("variable")
)

display(customer_label_quality)

if customer_label_variants:
    for column, variants in customer_label_variants.items():
        print(f"\nPotential label variants: {column}")
        display(variants)
else:
    print("No label variants found after trimming and ignoring case.")

#### Interpretation — Customer Label Quality

No blank labels, surrounding whitespace or category variants were
found after trimming and ignoring case.

No label consolidation is indicated by these checks. However, they
do not establish that geographic assignments or customer details
are accurate.

### 11.5. Customer Category Distributions

We examine category frequencies, dominant labels and categories
represented by only one customer.

Each row represents one customer. Percentages among observed values
exclude missing labels, while percentages of all customers include
the entire table in their denominator.

In [ ]:
# Compare category concentration and rarity across text attributes.
customer_category_records = []

for column in customer_text_columns:
    counts = clientes[column].value_counts(dropna=True)
    observed_count = int(counts.sum())

    customer_category_records.append({
        "variable": column,
        "observed_customers": observed_count,
        "missing_customers": int(clientes[column].isna().sum()),
        "unique_categories": len(counts),
        "most_frequent_category": (
            counts.index[0] if len(counts) else pd.NA
        ),
        "most_frequent_count": (
            int(counts.iloc[0]) if len(counts) else 0
        ),
        "most_frequent_pct_of_observed": (
            counts.iloc[0] / observed_count * 100
            if observed_count else np.nan
        ),
        "categories_with_one_customer": int(counts.eq(1).sum()),
        "customers_in_single_customer_categories_pct": (
            counts.loc[counts.eq(1)].sum() / observed_count * 100
            if observed_count else np.nan
        ),
    })

customer_category_overview = (
    pd.DataFrame(customer_category_records).set_index("variable")
)

display(customer_category_overview.round(3))

In [ ]:
def plot_customer_categories(column, top_n=15):
    """Display category frequencies and a readable customer-count graph."""
    counts = clientes[column].value_counts(dropna=True)
    observed_count = int(counts.sum())

    frequency_table = counts.rename_axis("category").to_frame("customers")
    frequency_table["pct_of_all_customers"] = (
        frequency_table["customers"] / len(clientes) * 100
    )
    frequency_table["pct_of_observed_customers"] = (
        frequency_table["customers"] / observed_count * 100
        if observed_count else np.nan
    )

    print(f"Variable: {column}")
    print(f"Observed categories: {len(counts):,}")
    print(f"Missing customers: {clientes[column].isna().sum():,}")
    display(frequency_table.head(top_n).round(3))

    # Retain the most frequent categories and account for the remainder.
    graph_counts = counts.head(top_n).copy()
    remaining_count = int(counts.iloc[top_n:].sum())

    if remaining_count > 0:
        graph_counts.loc["Other observed categories (combined)"] = (
            remaining_count
        )

    missing_count = int(clientes[column].isna().sum())
    if missing_count > 0:
        graph_counts.loc["Missing"] = missing_count

    fig, ax = plt.subplots(
        figsize=(11, max(4, len(graph_counts) * 0.4))
    )

    bars = ax.barh(
        graph_counts.index.astype(str),
        graph_counts.values,
        color="steelblue",
    )

    ax.invert_yaxis()
    ax.bar_label(bars, padding=4)
    ax.set(
        title=f"Customer Distribution — {column}",
        xlabel="Number of customers",
        ylabel="",
    )
    ax.margins(x=0.15)
    ax.grid(axis="x", alpha=0.2)
    ax.set_axisbelow(True)

    plt.tight_layout()
    plt.show()

In [ ]:
plot_customer_categories("country_or_region")

#### Interpretation — Country or Region

Country CT10 accounts for 1,079 customers (94.40%), followed by
CT5 with 49 (4.29%). The remaining nine categories contain only
15 customers combined, including six categories with one customer each.

The variable is complete but strongly concentrated in one category.
Rare categories offer limited evidence for comparisons, and the
anonymised labels do not identify the original countries.

In [ ]:
plot_customer_categories("commercial_region")

#### Interpretation — Commercial Region

Region RG17 contains 519 customers: 45.41% of all customers and
50.44% of those with an observed commercial region.

The remaining customers are spread across 36 other regions.
The next largest, RG23 and RG18, contain 63 and 59 customers,
respectively.

Commercial region is missing for 114 customers (9.97%). The graph
combines categories outside the top 15 for readability; this combined
group is not an original region.

In [ ]:
plot_customer_categories("island")

#### Interpretation — Island

Island information is missing for 578 customers (50.57%).
The remaining 565 customers are distributed across ten categories.

IS7 is the most common observed category, with 133 customers
(23.54% of observed values). IS10 contains only three customers,
so comparisons involving this category will have limited support.

Missing values may indicate non-applicability or unavailable information.
We should inspect their relationship with other geographic fields
before deciding how to handle them.

In [ ]:
plot_customer_categories("municipality")

#### Interpretation — Municipality

The 984 customers with observed municipality information are spread
across 183 categories. MU1 is the most common, with 58 customers
(5.89% of observed values), followed by MU9 with 51 and MU16 with 48.

The 168 categories outside the top 15 contain 476 customers combined.
Their large combined bar does not represent a single municipality.

Municipality is missing for 159 customers (13.91%). The many categories
may create sparse groups when analysing their relationship with the target.

In [ ]:
plot_customer_categories("city")

#### Interpretation — City

City is missing for 546 customers (47.77%). The 597 observed values
are spread across 417 categories, creating many small groups.

The most common city, CY194, contains only 14 customers.
Categories outside the top 15 account for 468 customers combined.

The combination of substantial missingness and many distinct labels
requires care when encoding this variable and comparing target rates.

In [ ]:
plot_customer_categories("postal_area")

#### Interpretation — Postal Area

Postal area is observed for 1,072 customers, with 71 missing values
(6.21%). The observed customers are spread across 271 categories.

PA259 is the most common category, with 57 customers, followed by
PA261 with 48. Categories outside the top 15 contain 598 customers
combined, showing that much of the distribution lies in smaller groups.

Postal area has lower missingness than city, but its many categories
still require attention when encoding and comparing target rates.

In [ ]:
plot_customer_categories("postal_code")

#### Interpretation — Postal Code

Postal code is observed for 973 customers across 828 categories.
Even the most frequent code, PC545, contains only six customers,
showing that individual codes have very limited representation.

Categories outside the top 15 contain 915 customers combined.
Another 170 customers (14.87%) have missing postal codes.

The field may act partly as a customer identifier. Its usefulness
for prediction requires validation, particularly for unseen customers.
The text “#REF!” appears within anonymised labels and does not,
by itself, establish a spreadsheet error.

In [ ]:
plot_customer_categories("street")

#### Interpretation — Street

Street information is missing for 598 customers (52.32%).
The 545 observed values contain 511 distinct labels; even the
most common street appears for only five customers.

This field is therefore very sparse and may act partly as a
customer identifier. Its usefulness for modelling requires
careful assessment, particularly for previously unseen customers.

In [ ]:
plot_customer_categories("address")

#### Interpretation — Address

Address is almost complete, with only six missing values (0.52%).
However, its 1,137 observed values contain 1,120 distinct labels.
The most common address appears for only three customers.

The field therefore behaves largely like a customer identifier.
High completeness alone does not establish modelling usefulness,
particularly when predicting outcomes for previously unseen customers.

In [ ]:
plot_customer_categories("company_name")

#### Interpretation — Company Name

Company name has no missing values and contains 1,141 distinct
labels across 1,143 customers. Only two labels appear twice;
all remaining labels appear once.

This field behaves largely like a customer identifier. The anonymised
names do not provide evidence about company size or business activity,
and repeated names alone do not establish duplicate customers.

### 11.6. Customer Relationship Start Year

We examine the distribution and validity of `customer_since_year`.
Consistency with purchase dates will be assessed after joining
the customer and training tables.

In [ ]:
# Assess the recorded years without modifying clientes.
customer_year = pd.to_numeric(
    clientes["customer_since_year"], errors="coerce"
)
finite_year = customer_year.notna() & np.isfinite(customer_year)
fractional_year = finite_year & customer_year.mod(1).ne(0)

customer_year_checks = pd.Series(
    {
        "Total customers": len(clientes),
        "Originally missing years": clientes["customer_since_year"].isna().sum(),
        "Non-missing values that failed conversion": (
            clientes["customer_since_year"].notna() & customer_year.isna()
        ).sum(),
        "Infinite values": (
            customer_year.notna() & ~np.isfinite(customer_year)
        ).sum(),
        "Fractional years": fractional_year.sum(),
        "Distinct finite years": customer_year.loc[finite_year].nunique(),
        "Earliest recorded year": customer_year.loc[finite_year].min(),
        "Latest recorded year": customer_year.loc[finite_year].max(),
    },
    name="value",
).to_frame()

display(customer_year_checks)

# Show every recorded finite year in chronological order.
customer_year_distribution = (
    customer_year.loc[finite_year]
    .value_counts()
    .sort_index()
    .rename_axis("customer_since_year")
    .to_frame("customers")
)
customer_year_distribution["pct_of_all_customers"] = (
    customer_year_distribution["customers"] / len(clientes) * 100
)

display(customer_year_distribution.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(
    customer_year_distribution.index.astype(str),
    customer_year_distribution["customers"],
    color="steelblue",
)
ax.bar_label(bars, padding=3)
ax.set_title("Customer Distribution by Relationship Start Year")
ax.set_xlabel("Recorded relationship start year")
ax.set_ylabel("Number of customers")
ax.margins(y=0.15)
ax.grid(axis="y", alpha=0.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

#### Interpretation — Customer Relationship Start Year

Recorded start years range from 2015 to 2026. Most customers
(664, or 58.09%) have 2015 recorded as their start year.
This concentration may reflect a recording boundary or an existing
customer base, rather than customers all joining during that year.

The table also contains customers with start years in 2025 and 2026.
These are not automatically errors: the customer table may cover
a later period than training. After joining, we will check whether
any purchase predates its customer's recorded start year.

### 11.7. Geographic Relationships and Missingness

We examine whether missing island information is concentrated
in particular countries or territories. This may help distinguish
non-applicable fields from unavailable information, although the
anonymised labels cannot establish the reason for missingness.

In [ ]:
# Compare island availability across recorded countries or territories.
island_availability = (
    clientes.assign(island_missing=clientes["island"].isna())
    .groupby("country_or_region", dropna=False)
    .agg(
        customers=("customer_id", "size"),
        missing_island=("island_missing", "sum"),
        distinct_observed_islands=("island", "nunique"),
    )
    .sort_values("customers", ascending=False)
)

island_availability["observed_island"] = (
    island_availability["customers"]
    - island_availability["missing_island"]
)
island_availability["missing_island_pct"] = (
    island_availability["missing_island"]
    / island_availability["customers"]
    * 100
)

display(island_availability.round(3))

# Show proportions alongside each country's customer count.
plot_data = island_availability[
    ["observed_island", "missing_island"]
].div(island_availability["customers"], axis=0) * 100

ax = plot_data.plot.barh(
    stacked=True,
    figsize=(11, 6),
    color=["steelblue", "lightgrey"],
)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Percentage of customers within each country or territory")
ax.set_ylabel("")
ax.set_title("Island Information Availability by Country or Territory")
ax.legend(
    ["Island observed", "Island missing"],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=2,
)

for position, count in enumerate(island_availability["customers"]):
    ax.text(101, position, f"n={count:,}", va="center")

plt.tight_layout()
plt.show()

#### Interpretation — Island Availability by Country

Island information is observed only for customers in Country CT10.
Within this country, 565 of 1,079 customers have an island recorded
(52.36%). All 64 customers in the other countries have missing values.

This pattern suggests that island information may depend on geographic
applicability. However, it does not explain the 514 missing values within
CT10 or establish that missing values represent mainland locations.

In [ ]:
# Examine geographic mappings using customers with both fields observed.
# Multiple associated labels require interpretation, not automatic correction.
geographic_pairs = [
    ("island", "country_or_region"),
    ("municipality", "island"),
    ("postal_code", "postal_area"),
    ("postal_area", "municipality"),
]

geographic_mapping_records = []
geographic_mapping_details = {}

for source, destination in geographic_pairs:
    paired = clientes[
        ["customer_id", source, destination]
    ].dropna(subset=[source, destination])

    associated_labels = (
        paired.groupby(source)[destination]
        .nunique()
        .sort_values(ascending=False)
    )
    multiple_labels = associated_labels[associated_labels > 1]

    geographic_mapping_records.append(
        {
            "source_field": source,
            "associated_field": destination,
            "assessed_customers": len(paired),
            "excluded_customers": len(clientes) - len(paired),
            "distinct_source_labels": len(associated_labels),
            "source_labels_with_multiple_associations": len(multiple_labels),
            "customers_in_these_groups": paired[source].isin(
                multiple_labels.index
            ).sum(),
        }
    )

    geographic_mapping_details[(source, destination)] = paired.loc[
        paired[source].isin(multiple_labels.index)
    ].sort_values([source, destination])

geographic_mapping_summary = pd.DataFrame(
    geographic_mapping_records
)

display(geographic_mapping_summary)

for pair, records in geographic_mapping_details.items():
    if not records.empty:
        print(f"\nMultiple associations: {pair[0]} → {pair[1]}")
        display(records.head(20))

#### Interpretation — Geographic Relationships

Among customers with both fields observed, each island maps to one
country, each municipality maps to one island and each postal code
maps to one postal area. These findings apply only to the assessed records.

Six postal areas are associated with multiple municipalities,
covering 32 customers. Postal areas may cross administrative boundaries,
so these associations do not establish errors or justify corrections.

### 11.8. Customer Matching Coverage

Before joining the tables, we check whether every training customer
has a corresponding record in `clientes`. Coverage is measured both
by purchase observations and by distinct customers.

In [ ]:
# Assess matching without changing either source table.
customer_match = (
    train["customer_id"].notna()
    & train["customer_id"].isin(clientes["customer_id"])
)

training_customer_ids = pd.Index(
    train["customer_id"].dropna().unique()
)
customer_table_ids = pd.Index(
    clientes["customer_id"].dropna().unique()
)

unmatched_customer_ids = training_customer_ids.difference(
    customer_table_ids
)
matched_customer_ids = training_customer_ids.intersection(
    customer_table_ids
)

customer_matching_summary = pd.Series(
    {
        "Training observations": len(train),
        "Observations with missing customer IDs": (
            train["customer_id"].isna().sum()
        ),
        "Matched observations": customer_match.sum(),
        "Unmatched observations": (~customer_match).sum(),
        "Matched observations (%)": customer_match.mean() * 100,
        "Distinct training customers": len(training_customer_ids),
        "Matched distinct customers": len(matched_customer_ids),
        "Unmatched distinct customers": len(unmatched_customer_ids),
        "Customer-table IDs absent from training": len(
            customer_table_ids.difference(training_customer_ids)
        ),
    },
    name="value",
).to_frame()

display(customer_matching_summary)

if not unmatched_customer_ids.empty:
    print("Unmatched training customers — first 20:")
    display(
        pd.DataFrame(
            {"customer_id": unmatched_customer_ids}
        ).head(20)
    )
else:
    print("Every non-missing training customer ID exists in clientes.")

#### Interpretation — Customer Matching Coverage

All 6,534 training observations match a customer record in `clientes`,
covering all 610 distinct training customers. No customer IDs are missing
or unmatched.

The customer table contains 533 additional customers absent from training.
These may include test customers. Complete matching ensures that the join
introduces no missing values through unmatched keys, although customer
attributes already contain missing information.

### 11.9. Joining Training and Customer Data

We create `train_clientes` using a left join on `customer_id`.
The join must preserve every training observation and its target,
while attaching at most one customer record to each observation.

In [ ]:
# Prevent overlapping attribute names from creating ambiguous columns.
overlapping_columns = (
    set(train.columns) & set(clientes.columns)
) - {"customer_id"}

assert not overlapping_columns, (
    f"Overlapping columns require review: {sorted(overlapping_columns)}"
)
assert clientes["customer_id"].notna().all()
assert clientes["customer_id"].is_unique

# Attach customer attributes while preserving training row order.
train_clientes = train.merge(
    clientes,
    on="customer_id",
    how="left",
    validate="many_to_one",
    indicator="customer_match_status",
    sort=False,
)

# Verify matching, row preservation and original training values.
assert len(train_clientes) == len(train)
assert train_clientes["customer_match_status"].eq("both").all()

pd.testing.assert_frame_equal(
    train_clientes.loc[:, train.columns].reset_index(drop=True),
    train.reset_index(drop=True),
)

# Remove the temporary join diagnostic after verification.
train_clientes = train_clientes.drop(columns="customer_match_status")

join_summary = pd.Series(
    {
        "Original training rows": len(train),
        "Joined rows": len(train_clientes),
        "Original training columns": train.shape[1],
        "Customer attributes added": clientes.shape[1] - 1,
        "Joined columns": train_clientes.shape[1],
        "Distinct joined customers": train_clientes["customer_id"].nunique(),
    },
    name="value",
).to_frame()

display(join_summary)
print("Join verified: all original training rows and values are preserved.")

### 11.10. Customer Start Year and Purchase-Date Consistency

A purchase occurring before the recorded customer start year requires
investigation. Purchases within the same year cannot be assessed more
precisely because `customer_since_year` contains no month or day.

In [ ]:
# Create a diagnostic copy without changing train_clientes.
customer_date_check = train_clientes[
    ["ID", "customer_id", "purchase_date", "customer_since_year"]
].copy()

customer_date_check["parsed_purchase_date"] = pd.to_datetime(
    customer_date_check["purchase_date"],
    format="%Y-%m-%d",
    errors="coerce",
)
customer_date_check["recorded_start_year"] = pd.to_numeric(
    customer_date_check["customer_since_year"],
    errors="coerce",
)

start_year = customer_date_check["recorded_start_year"]
valid_start_year = (
    start_year.notna()
    & np.isfinite(start_year)
    & start_year.mod(1).eq(0)
    & start_year.between(1, 9999)
)
valid_purchase_date = customer_date_check["parsed_purchase_date"].notna()
assessed = valid_purchase_date & valid_start_year

customer_date_check["purchase_year"] = (
    customer_date_check["parsed_purchase_date"].dt.year
)
purchase_before_start = (
    assessed
    & customer_date_check["purchase_year"].lt(start_year)
)

customer_start_date_summary = pd.Series(
    {
        "Total observations": len(customer_date_check),
        "Missing or unparseable purchase dates": (~valid_purchase_date).sum(),
        "Missing or invalid start years": (~valid_start_year).sum(),
        "Assessed observations": assessed.sum(),
        "Unassessable observations": (~assessed).sum(),
        "Purchases before recorded start year": purchase_before_start.sum(),
        "Distinct customers with these purchases": customer_date_check.loc[
            purchase_before_start, "customer_id"
        ].nunique(),
        "Flagged share of assessed observations (%)": (
            purchase_before_start.sum() / assessed.sum() * 100
            if assessed.any()
            else np.nan
        ),
    },
    name="value",
).to_frame()

display(customer_start_date_summary)

if purchase_before_start.any():
    print("Purchases before recorded customer start year — first 20:")
    display(
        customer_date_check.loc[
            purchase_before_start,
            [
                "ID",
                "customer_id",
                "purchase_date",
                "customer_since_year",
                "purchase_year",
            ],
        ].head(20)
    )
else:
    print("No assessed purchases predate the recorded customer start year.")

#### Interpretation — Customer Start Year Consistency

None of the 6,469 assessed purchases occurred before the recorded
customer start year. All start years were valid for this check;
65 observations could not be assessed because their purchase dates
were missing.

This confirms consistency at the year level, but does not establish
the exact relationship start date or the historical availability
of other customer attributes.

### 11.11. Customer Attribute Missingness After Joining

We compare missingness across all customers, customers represented
in training and joined purchase observations. The last comparison
gives more weight to customers with more purchase observations.

In [ ]:
# Select each training customer once for an equally weighted comparison.
training_customers = clientes.loc[
    clientes["customer_id"].isin(train["customer_id"])
].copy()

customer_attributes = [
    column for column in clientes.columns
    if column != "customer_id"
]

joined_missingness_comparison = pd.DataFrame(
    {
        "all_customers_missing_pct": (
            clientes[customer_attributes].isna().mean() * 100
        ),
        "training_customers_missing_pct": (
            training_customers[customer_attributes].isna().mean() * 100
        ),
        "training_observations_missing_pct": (
            train_clientes[customer_attributes].isna().mean() * 100
        ),
    }
).sort_values("all_customers_missing_pct", ascending=False)

print(f"All customer records: {len(clientes):,}")
print(f"Distinct training customer records: {len(training_customers):,}")
print(f"Joined purchase observations: {len(train_clientes):,}")
display(joined_missingness_comparison.round(3))

ax = joined_missingness_comparison.plot.barh(
    figsize=(11, 7),
    color=["lightgrey", "steelblue", "darkorange"],
    width=0.8,
)
ax.invert_yaxis()
ax.set_xlim(0, 105)
ax.set_xlabel("Missing values (%)")
ax.set_ylabel("")
ax.set_title("Customer Attribute Missingness Across Analysis Populations")
ax.legend(
    [
        "All customers",
        "Training customers — counted once",
        "Training purchase observations",
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, -0.12),
    ncol=1,
)
ax.grid(axis="x", alpha=0.2)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

#### Interpretation — Missingness After Joining

Training customers have less missing information than the full customer
table for several fields. Postal-code missingness falls from 14.87%
to 3.28%, while commercial-region missingness falls from 9.97% to 1.97%.

Repeated observations further change these proportions. Island missingness
is 46.89% when training customers are counted once, compared with 33.84%
across purchase observations. Street missingness increases from 54.10%
to 58.68%. These differences reflect how frequently customers appear.

`customer_type` remains entirely missing. Address, country, company name
and relationship start year are complete for training customers.

### 11.12. Customer Attributes and the Target

We compare repeat-purchase rates across customer groups, reporting both
purchase observations and distinct customers. Rates describe associations;
small groups and repeated observations can limit their interpretation.

In [ ]:
# Summarise country groups using observations with valid targets.
country_target_data = train_clientes.loc[
    train_clientes["repeat_purchase_90d"].isin([0, 1]),
    ["customer_id", "country_or_region", "repeat_purchase_90d"],
].copy()

country_target_summary = (
    country_target_data
    .groupby("country_or_region", dropna=False)
    .agg(
        observations=("repeat_purchase_90d", "size"),
        unique_customers=("customer_id", "nunique"),
        target_0=("repeat_purchase_90d", lambda values: values.eq(0).sum()),
        target_1=("repeat_purchase_90d", lambda values: values.eq(1).sum()),
    )
    .sort_values("observations", ascending=False)
)

country_target_summary["repeat_purchase_pct"] = (
    country_target_summary["target_1"]
    / country_target_summary["observations"]
    * 100
)

display(country_target_summary.round(3))

overall_rate = (
    country_target_data["repeat_purchase_90d"].mean() * 100
)

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    country_target_summary.index.astype(str),
    country_target_summary["repeat_purchase_pct"],
    color="steelblue",
)
ax.invert_yaxis()
ax.set_xlim(0, 105)
ax.set_xlabel("Repeat purchase within 90 days (%)")
ax.set_ylabel("")
ax.set_title("Repeat-Purchase Rate by Country or Territory")
ax.axvline(
    overall_rate,
    color="darkorange",
    linestyle="--",
    label=f"Overall rate: {overall_rate:.2f}%",
)

ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

#### Interpretation — Country and the Target

Country CT10 has a repeat-purchase rate of approximately 56.5%,
close to the overall rate of 55.92%. CT5 has a lower rate of 39.2%.

Other countries show rates between 0% and 50%, but their observation
and customer counts must be considered before drawing conclusions.
A 0% observed rate does not establish that future customers from
that country will never repeat a purchase.

In [ ]:
# Compare commercial regions, retaining missing values as a separate group.
region_target_data = train_clientes.loc[
    train_clientes["repeat_purchase_90d"].isin([0, 1]),
    ["customer_id", "commercial_region", "repeat_purchase_90d"],
].copy()

region_target_summary = (
    region_target_data
    .groupby("commercial_region", dropna=False)
    .agg(
        observations=("repeat_purchase_90d", "size"),
        unique_customers=("customer_id", "nunique"),
        target_0=("repeat_purchase_90d", lambda values: values.eq(0).sum()),
        target_1=("repeat_purchase_90d", lambda values: values.eq(1).sum()),
    )
    .sort_values("observations", ascending=False)
)

region_target_summary["repeat_purchase_pct"] = (
    region_target_summary["target_1"]
    / region_target_summary["observations"]
    * 100
)

display(region_target_summary.round(3))

# Plot the 15 largest groups by observation count.
# The full table above retains every group.
region_plot = region_target_summary.head(15)
region_labels = [
    "Missing" if pd.isna(label) else str(label)
    for label in region_plot.index
]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(
    region_labels,
    region_plot["repeat_purchase_pct"],
    color="steelblue",
)
ax.invert_yaxis()
ax.set_xlim(0, 105)
ax.set_xlabel("Repeat purchase within 90 days (%)")
ax.set_ylabel("")
ax.set_title("Repeat-Purchase Rate — 15 Most Frequent Commercial Regions")
ax.axvline(
    region_target_data["repeat_purchase_90d"].mean() * 100,
    color="darkorange",
    linestyle="--",
    label="Overall repeat-purchase rate",
)
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

#### Interpretation — Commercial Region and the Target

Repeat-purchase rates vary across the 15 most frequent regions.
RG17 records 51.9%, below the overall rate of 55.92%, while RG23
and RG24 record 82.4% and 94.5%. RG32 has a lower rate of 25.7%.

These differences may reflect customer composition and purchase frequency.
The observation and distinct-customer counts in the table are needed
to assess whether high rates are dominated by a few frequent customers.

In [ ]:
# Compare island groups using valid targets.
island_target_data = train_clientes.loc[
    train_clientes["repeat_purchase_90d"].isin([0, 1]),
    ["customer_id", "island", "repeat_purchase_90d"],
].copy()

island_target_summary = (
    island_target_data
    .groupby("island", dropna=False)
    .agg(
        observations=("repeat_purchase_90d", "size"),
        unique_customers=("customer_id", "nunique"),
        target_0=("repeat_purchase_90d", lambda values: values.eq(0).sum()),
        target_1=("repeat_purchase_90d", lambda values: values.eq(1).sum()),
    )
    .sort_values("observations", ascending=False)
)

island_target_summary["repeat_purchase_pct"] = (
    island_target_summary["target_1"]
    / island_target_summary["observations"]
    * 100
)

display(island_target_summary.round(3))

island_labels = [
    "Missing" if pd.isna(label) else str(label)
    for label in island_target_summary.index
]

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    island_labels,
    island_target_summary["repeat_purchase_pct"],
    color="steelblue",
)
ax.invert_yaxis()
ax.set_xlim(0, 105)
ax.set_xlabel("Repeat purchase within 90 days (%)")
ax.set_ylabel("")
ax.set_title("Repeat-Purchase Rate by Recorded Island")
ax.axvline(
    island_target_data["repeat_purchase_90d"].mean() * 100,
    color="darkorange",
    linestyle="--",
    label="Overall repeat-purchase rate",
)
ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

#### Interpretation — Island and the Target

Repeat-purchase rates vary across recorded islands, from approximately
30.0% for IS10 to 62.7% for IS3. Observations with missing island
information have a rate of 64.0%, above the overall rate of 55.92%.

The missing group combines customers from different geographic contexts
and should not be interpreted as a mainland category. Differences may
also reflect customer composition and repeated observations; the table's
customer counts are needed to assess the support for each rate.

In [ ]:
# Summarise every category and plot the most frequent groups.
def plot_joined_customer_target(column, top_n=15):
    data = train_clientes.loc[
        train_clientes["repeat_purchase_90d"].isin([0, 1]),
        ["customer_id", column, "repeat_purchase_90d"],
    ].copy()

    summary = (
        data.groupby(column, dropna=False)
        .agg(
            observations=("repeat_purchase_90d", "size"),
            unique_customers=("customer_id", "nunique"),
            target_0=(
                "repeat_purchase_90d",
                lambda values: values.eq(0).sum(),
            ),
            target_1=(
                "repeat_purchase_90d",
                lambda values: values.eq(1).sum(),
            ),
        )
        .sort_values("observations", ascending=False)
    )
    summary["repeat_purchase_pct"] = (
        summary["target_1"] / summary["observations"] * 100
    )

    print(f"Variable: {column}")
    print(f"Groups, including missing where present: {len(summary):,}")
    display(summary.head(top_n).round(3))

    plotted = summary.head(top_n)
    labels = [
        "Missing" if pd.isna(label) else str(label)
        for label in plotted.index
    ]

    fig, ax = plt.subplots(figsize=(11, 7))
    bars = ax.barh(
        labels,
        plotted["repeat_purchase_pct"],
        color="steelblue",
    )
    ax.invert_yaxis()
    ax.set_xlim(0, 105)
    ax.set_xlabel("Repeat purchase within 90 days (%)")
    ax.set_ylabel("")
    ax.set_title(
        f"Repeat-Purchase Rate — {column}\n"
        f"Top {len(plotted)} groups by observation count"
    )
    ax.axvline(
        data["repeat_purchase_90d"].mean() * 100,
        color="darkorange",
        linestyle="--",
        label="Overall repeat-purchase rate",
    )
    ax.bar_label(bars, fmt="%.1f%%", padding=3)
    ax.legend(loc="lower right")
    plt.tight_layout()
    plt.show()

    return summary

In [ ]:
municipality_target_summary = plot_joined_customer_target("municipality")

#### Interpretation — Municipality and the Target

Among the 15 most frequent groups, MU1 has a repeat-purchase rate
of 54.9%, close to the overall rate of 55.92%. MU70 and MU81
have higher rates of 89.0% and 98.6%, while missing municipality
information has a rate of 38.2%.

These are observation-level associations. High rates may be dominated
by repeated purchases from a few customers, so the distinct-customer
counts should be considered before assessing their modelling usefulness.

In [ ]:
city_target_summary = plot_joined_customer_target("city")

#### Interpretation — City and the Target

Among the 15 most frequent groups, CY265 and CY281 have repeat-purchase
rates of 99.3% and 100%, while CY151 and CY196 record 39.5% and 38.9%.
Missing city information has a rate of 55.1%, close to the overall 55.92%.

City labels are sparse, and repeated observations from a few customers
may dominate individual rates. A 100% observed rate does not guarantee
future repeat purchases; distinct-customer counts remain essential
when interpreting these differences.

In [ ]:
postal_area_target_summary = plot_joined_customer_target("postal_area")

#### Interpretation — Postal Area and the Target

PA259 has a repeat-purchase rate of 55.0%, close to the overall 55.92%.
PA84 and PA189 record higher rates of 90.6% and 98.6%, while missing
postal-area information has a rate of 41.4%.

These differences may reflect customer composition and repeated
observations. Distinct-customer counts are needed to assess how broadly
each rate is supported.

In [ ]:
# Compare recorded customer start years with the target.
start_year_target = (
    train_clientes.loc[
        train_clientes["repeat_purchase_90d"].isin([0, 1])
    ]
    .groupby("customer_since_year", dropna=False)
    .agg(
        observations=("repeat_purchase_90d", "size"),
        unique_customers=("customer_id", "nunique"),
        target_0=("repeat_purchase_90d", lambda x: x.eq(0).sum()),
        target_1=("repeat_purchase_90d", lambda x: x.eq(1).sum()),
    )
    .sort_index()
)

start_year_target["repeat_purchase_pct"] = (
    start_year_target["target_1"]
    / start_year_target["observations"]
    * 100
)

display(start_year_target.round(3))

overall_rate = (
    train_clientes.loc[
        train_clientes["repeat_purchase_90d"].isin([0, 1]),
        "repeat_purchase_90d",
    ].mean()
    * 100
)

fig, ax = plt.subplots(figsize=(11, 5))

bars = ax.bar(
    start_year_target.index.astype(str),
    start_year_target["repeat_purchase_pct"],
    color="steelblue",
)

ax.axhline(
    overall_rate,
    color="darkorange",
    linestyle="--",
    label=f"Overall rate: {overall_rate:.2f}%",
)

ax.bar_label(bars, fmt="%.1f%%", padding=3)
ax.set(
    title="Recorded Customer Start Year and Repeat Purchase",
    xlabel="Recorded customer start year",
    ylabel="Repeat purchase rate (%)",
    ylim=(0, 110),
)
ax.legend()
plt.tight_layout()
plt.show()

### Review of Fields with Many Distinct Labels

This review measures how many distinct training customers share each
observed label. Labels associated with only one customer may act as
customer identifiers, even when repeated across many purchase observations.

Missing values are assessed separately. No features are removed at this stage.

In [ ]:
fields_to_review = [
    "municipality",
    "city",
    "postal_area",
    "postal_code",
    "street",
    "address",
    "company_name",
]

label_support_records = []

# Count each customer once when assessing support for a label.
distinct_training_customers = train_clientes.drop_duplicates(
    subset="customer_id"
)

for column in fields_to_review:
    customer_values = distinct_training_customers[column]
    customers_per_label = customer_values.value_counts(dropna=True)

    single_customer_labels = customers_per_label.index[
        customers_per_label.eq(1)
    ]

    observed_rows = train_clientes[column].notna()
    single_customer_rows = train_clientes[column].isin(
        single_customer_labels
    )

    label_support_records.append({
        "variable": column,
        "missing_customers": int(customer_values.isna().sum()),
        "observed_customers": int(customer_values.notna().sum()),
        "distinct_labels": len(customers_per_label),
        "labels_with_1_customer": int(
            customers_per_label.eq(1).sum()
        ),
        "labels_with_2_to_4_customers": int(
            customers_per_label.between(2, 4).sum()
        ),
        "labels_with_5_plus_customers": int(
            customers_per_label.ge(5).sum()
        ),
        "observed_rows_with_single_customer_labels_pct": (
            single_customer_rows.sum() / observed_rows.sum() * 100
            if observed_rows.any()
            else np.nan
        ),
    })

label_support_summary = pd.DataFrame(
    label_support_records
).set_index("variable")

display(label_support_summary.round(3))

#### Interpretation — Support Across Customers

Labels belonging to only one training customer account for 97.44% of
observed address rows and 99.68% of company-name rows. These fields
may encourage customer memorisation.

Street and postal code also have limited support across customers.
Municipality and postal area are more widely shared, with only 3.56%
and 8.07% of observed rows carrying labels unique to one customer.

Feature decisions will consider unseen labels in test and the
validation strategy.

### Customer Data and Integration — Summary

All 6,534 training observations matched customer records. The join
added 12 attributes while preserving the original training rows and values.

No assessed purchase predates the recorded customer start year.
Customer attributes contain substantial missingness, including an
entirely missing customer_type field.

Some geographic categories show different repeat-purchase rates,
but sparse groups and repeated observations limit interpretation.
Address and company name behave almost like customer identifiers.

These findings will inform preprocessing and feature selection.

## 12. Train–Test Comparison

This section compares structure, missingness, dates, customer overlap
and feature distributions across the supplied training and test datasets.
The comparisons inform validation and preprocessing decisions.

In [ ]:
target_column = "repeat_purchase_90d"

train_feature_columns = train.columns.drop(target_column)
test_feature_columns = test.columns

schema_summary = pd.Series({
    "Training observations": len(train),
    "Test observations": len(test),
    "Training features excluding target": len(train_feature_columns),
    "Test features": len(test_feature_columns),
    "Target present in training": target_column in train.columns,
    "Target present in test": target_column in test.columns,
    "Matching feature names": (
        set(train_feature_columns) == set(test_feature_columns)
    ),
    "Matching feature order": (
        list(train_feature_columns) == list(test_feature_columns)
    ),
}).to_frame("value")

display(schema_summary)

print(
    "Features absent from test:",
    sorted(set(train_feature_columns) - set(test_feature_columns)),
)
print(
    "Features absent from training:",
    sorted(set(test_feature_columns) - set(train_feature_columns)),
)

common_columns = [
    column for column in train_feature_columns
    if column in test.columns
]

dtype_comparison = pd.DataFrame({
    "train_dtype": train[common_columns].dtypes.astype(str),
    "test_dtype": test[common_columns].dtypes.astype(str),
})

dtype_differences = dtype_comparison.loc[
    dtype_comparison["train_dtype"]
    != dtype_comparison["test_dtype"]
]

if dtype_differences.empty:
    print("All shared features have matching pandas dtypes.")
else:
    display(dtype_differences)

#### Interpretation — Train–Test Structure

Training and test contain the same 38 input columns in the same order.
The target appears only in training, as expected.

Eight numerical columns are stored as floats in training and integers
in test. This is consistent with missing values in training and does
not, by itself, indicate different feature meanings.

### 12.1. Missingness Comparison

Missing-value percentages are compared because training and test
contain different numbers of observations. Differences are expressed
in percentage points.

In [ ]:
missingness_comparison = pd.DataFrame({
    "train_missing_count": train[common_columns].isna().sum(),
    "test_missing_count": test[common_columns].isna().sum(),
    "train_missing_pct": train[common_columns].isna().mean() * 100,
    "test_missing_pct": test[common_columns].isna().mean() * 100,
})

missingness_comparison["test_minus_train_pp"] = (
    missingness_comparison["test_missing_pct"]
    - missingness_comparison["train_missing_pct"]
)

# Show fields with missing values in either dataset.
missingness_to_show = missingness_comparison.loc[
    missingness_comparison[
        ["train_missing_count", "test_missing_count"]
    ].gt(0).any(axis=1)
].sort_values(
    "train_missing_pct",
    ascending=False,
)

display(missingness_to_show.round(3))

In [ ]:
if not missingness_to_show.empty:
    plot_data = missingness_to_show.sort_values(
        "train_missing_pct"
    )

    positions = np.arange(len(plot_data))
    fig, ax = plt.subplots(
        figsize=(11, max(5, len(plot_data) * 0.35))
    )

    ax.barh(
        positions - 0.2,
        plot_data["train_missing_pct"],
        height=0.4,
        color="steelblue",
        label="Training",
    )
    ax.barh(
        positions + 0.2,
        plot_data["test_missing_pct"],
        height=0.4,
        color="darkorange",
        label="Test",
    )

    ax.set_yticks(positions)
    ax.set_yticklabels(plot_data.index)
    ax.set(
        title="Missing Values — Training and Test",
        xlabel="Missing observations (%)",
    )
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Neither dataset contains missing values.")

#### Interpretation — Train–Test Missingness

Historical statistics have substantially less missing information in test.
For example, prior_std_gap_days is missing in 7.00% of test observations,
compared with 22.62% in training.

Test has no missing purchase dates, current-purchase totals or purchase
counts. It therefore contains no observations with all purchase fields
missing, unlike the 65 such training observations.

These differences may reflect customer history or sampling differences.
Lower missingness does not establish that the observed values are correct.

### 12.2. Purchase-Date Coverage

Compare the observed periods to assess whether test represents later
purchases, an overlapping period or a mixture of both.

In [ ]:
comparison_dates = {}
date_coverage_records = []

for name, dataset in {"Training": train, "Test": test}.items():
    raw_dates = dataset["purchase_date"]

    parsed_dates = pd.to_datetime(
        raw_dates,
        format="%Y-%m-%d",
        errors="coerce",
    )

    comparison_dates[name] = parsed_dates

    date_coverage_records.append({
        "dataset": name,
        "observations": len(dataset),
        "originally_missing_dates": int(raw_dates.isna().sum()),
        "non_missing_dates_failing_parsing": int(
            (raw_dates.notna() & parsed_dates.isna()).sum()
        ),
        "valid_dates": int(parsed_dates.notna().sum()),
        "earliest_date": parsed_dates.min(),
        "latest_date": parsed_dates.max(),
    })

date_coverage_summary = pd.DataFrame(
    date_coverage_records
).set_index("dataset")

display(date_coverage_summary)

latest_training_date = comparison_dates["Training"].max()
valid_test_dates = comparison_dates["Test"].dropna()

if pd.notna(latest_training_date) and not valid_test_dates.empty:
    after_training = valid_test_dates.gt(latest_training_date)

    print(f"Latest training date: {latest_training_date:%Y-%m-%d}")
    print(
        "Test observations after the latest training date: "
        f"{after_training.sum():,} of {len(valid_test_dates):,} "
        f"({after_training.mean() * 100:.2f}%)"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

for name, dates in comparison_dates.items():
    monthly_counts = (
        dates.dropna()
        .dt.to_period("M")
        .value_counts()
        .sort_index()
    )

    if monthly_counts.empty:
        continue

    # Include intervening months with no observations.
    monthly_counts = monthly_counts.reindex(
        pd.period_range(
            monthly_counts.index.min(),
            monthly_counts.index.max(),
            freq="M",
        ),
        fill_value=0,
    )

    ax.plot(
        monthly_counts.index.to_timestamp(),
        monthly_counts.values,
        label=name,
    )

ax.set(
    title="Monthly Purchase Observations — Training and Test",
    xlabel="Purchase month",
    ylabel="Observations",
)
ax.legend()
plt.tight_layout()
plt.show()

#### Interpretation — Purchase-Date Coverage

All 986 test observations occur after the latest training purchase.
Training ends on 15 February 2025, while test covers 24 May 2025
to 6 October 2026.

The task is binary classification with a temporal prediction horizon.
Validation should reflect prediction on later purchases and account
for the 90 days required to observe the target outcome.

### 12.3. Customer Overlap

Identify test customers who already appear in training and those absent
from training. “Unseen” refers to the supplied training dataset, not
necessarily to a newly acquired customer.

In [ ]:
training_customer_ids = set(train["customer_id"].dropna())

test_customer_status = pd.Series(
    "Unseen in training",
    index=test.index,
    name="customer_status",
)

test_customer_status.loc[
    test["customer_id"].isin(training_customer_ids)
] = "Seen in training"

test_customer_status.loc[
    test["customer_id"].isna()
] = "Missing customer ID"

customer_overlap_data = test[["customer_id"]].copy()
customer_overlap_data["customer_status"] = test_customer_status

customer_overlap_summary = (
    customer_overlap_data
    .groupby("customer_status")
    .agg(
        observations=("customer_status", "size"),
        unique_customers=("customer_id", "nunique"),
    )
    .reindex(
        [
            "Seen in training",
            "Unseen in training",
            "Missing customer ID",
        ],
        fill_value=0,
    )
)

customer_overlap_summary["test_observations_pct"] = (
    customer_overlap_summary["observations"] / len(test) * 100
)

display(customer_overlap_summary.round(3))

test_customer_ids = set(test["customer_id"].dropna())

print(f"Distinct training customers: {len(training_customer_ids):,}")
print(f"Distinct test customers: {len(test_customer_ids):,}")
print(
    "Customers shared by training and test: "
    f"{len(training_customer_ids & test_customer_ids):,}"
)
print(
    "Distinct test customers absent from clientes: "
    f"{len(test_customer_ids - set(clientes['customer_id'].dropna())):,}"
)

#### Interpretation — Customer Overlap

Of the 986 test observations, 932 (94.52%) belong to 184 customers
already present in training. The remaining 54 observations belong
to 22 customers unseen in training.

All test customer IDs exist in clientes, so customer attributes
can be joined for every test observation.

Validation should reflect later purchases and report performance
separately for seen and unseen customers where sample sizes permit.

### 12.4. Categories Unseen in Training

Compare observed labels across training and test. Missing values are
reported separately and are not treated as unseen categories.

In [ ]:
# Join customer attributes while preserving the original test data.
assert clientes["customer_id"].notna().all()
assert clientes["customer_id"].is_unique
assert not (
    (set(test.columns) & set(clientes.columns)) - {"customer_id"}
)

test_clientes = test.merge(
    clientes,
    on="customer_id",
    how="left",
    validate="many_to_one",
    sort=False,
    indicator="_customer_match",
)

assert len(test_clientes) == len(test)
assert test_clientes["_customer_match"].eq("both").all()

pd.testing.assert_frame_equal(
    test_clientes[test.columns].reset_index(drop=True),
    test.reset_index(drop=True),
)

test_clientes = test_clientes.drop(columns="_customer_match")

categorical_comparison_columns = [
    "payment_type",
    "salesperson",
    "commercial_zone",
    "transport_method",
    "warehouse",
    "document_series",
    "country_or_region",
    "commercial_region",
    "island",
    "municipality",
    "city",
    "postal_area",
    "postal_code",
    "street",
    "address",
    "company_name",
]

unseen_category_records = []
unseen_category_labels = {}

for column in categorical_comparison_columns:
    train_labels = set(train_clientes[column].dropna())
    test_labels = set(test_clientes[column].dropna())
    unseen_labels = test_labels - train_labels

    observed_test = test_clientes[column].notna()
    unseen_rows = test_clientes[column].isin(unseen_labels)

    unseen_category_labels[column] = sorted(unseen_labels)

    unseen_category_records.append({
        "variable": column,
        "training_categories": len(train_labels),
        "test_categories": len(test_labels),
        "unseen_test_categories": len(unseen_labels),
        "test_missing": int((~observed_test).sum()),
        "test_rows_with_unseen_labels": int(unseen_rows.sum()),
        "unseen_pct_of_observed_test_rows": (
            unseen_rows.sum() / observed_test.sum() * 100
            if observed_test.any()
            else np.nan
        ),
    })

unseen_category_summary = (
    pd.DataFrame(unseen_category_records)
    .set_index("variable")
    .sort_values(
        "unseen_pct_of_observed_test_rows",
        ascending=False,
    )
)

display(unseen_category_summary.round(3))

#### Interpretation — Categories Unseen in Training

Two commercial-zone labels unseen in training appear in 209 test
observations (21.20%). Unseen transport-method labels affect 6.59%
of non-missing test observations.

Company name and address also contain unfamiliar labels, reinforcing
their limited usefulness for customers unseen during training.

All categorical preprocessing must handle unseen labels. Matching
category counts alone does not establish that the labels are the same.

### 12.5. Numerical Distribution Comparison

Compare numerical features using finite observations. Supplied values
are retained in this initial comparison, including previously identified
inconsistencies. Medians and quartiles complement means that may be
distorted by extreme values.

In [ ]:
# Exclude identifiers; retain random_noise as a diagnostic reference.
comparison_numeric_columns = [
    column for column in common_columns
    if pd.api.types.is_numeric_dtype(train[column])
    and pd.api.types.is_numeric_dtype(test[column])
    and column not in ["ID", "customer_id"]
]

numeric_comparison_records = []

for column in comparison_numeric_columns:
    for name, dataset in {"Training": train, "Test": test}.items():
        values = dataset[column]
        finite_values = values.loc[
            values.notna() & np.isfinite(values)
        ]

        numeric_comparison_records.append({
            "variable": column,
            "dataset": name,
            "finite_observations": len(finite_values),
            "missing": int(values.isna().sum()),
            "infinite_values": int(np.isinf(values).sum()),
            "negative_values": int(finite_values.lt(0).sum()),
            "Q1": finite_values.quantile(0.25),
            "median": finite_values.median(),
            "mean": finite_values.mean(),
            "Q3": finite_values.quantile(0.75),
            "95th_percentile": finite_values.quantile(0.95),
            "maximum": finite_values.max(),
        })

numeric_train_test_summary = (
    pd.DataFrame(numeric_comparison_records)
    .set_index(["variable", "dataset"])
)

display(numeric_train_test_summary.round(3))

In [ ]:
features_to_plot = [
    "Total a Pagar",
    "prior_purchase_count",
    "customer_tenure_days",
]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, column in zip(axes, features_to_plot):
    for name, dataset, color in [
        ("Training", train, "steelblue"),
        ("Test", test, "darkorange"),
    ]:
        values = dataset[column]
        finite_values = values.loc[
            values.notna() & np.isfinite(values)
        ]

        # Plot non-negative values on a log1p scale.
        plotted_values = finite_values.loc[finite_values.ge(0)]

        print(
            f"{column} — {name}: "
            f"{len(plotted_values):,} plotted; "
            f"{finite_values.lt(0).sum():,} negatives excluded"
        )

        sns.ecdfplot(
            x=np.log1p(plotted_values),
            ax=ax,
            label=name,
            color=color,
        )

    ax.set(
        title=column,
        xlabel="log1p(value)",
        ylabel="Cumulative proportion",
    )
    ax.legend()

fig.suptitle("Numerical Distributions — Training and Test")
plt.tight_layout()
plt.show()

#### Interpretation — Numerical Distribution Differences

Test has higher median purchase values (663.89 versus 476.74),
more previous purchases (28 versus 11) and longer customer tenure
(2,662 versus 918 days).

These differences are consistent with later purchases from a customer
population largely represented in training, although customer composition
also differs.

Negative intervals and extreme supplied historical values also occur
in test. The same cleaning rules will therefore need to apply consistently
across datasets.

In [ ]:
features_to_plot = [
    "purchase_count_90d",
    "spend_90d",
    "days_since_previous_purchase",
]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

for ax, column in zip(axes, features_to_plot):
    for name, dataset, color in [
        ("Training", train, "steelblue"),
        ("Test", test, "darkorange"),
    ]:
        values = dataset[column]
        finite_values = values.loc[
            values.notna() & np.isfinite(values)
        ]
        plotted_values = finite_values.loc[finite_values.ge(0)]

        print(
            f"{column} — {name}: "
            f"{len(plotted_values):,} plotted; "
            f"{finite_values.lt(0).sum():,} negatives excluded"
        )

        sns.ecdfplot(
            x=np.log1p(plotted_values),
            ax=ax,
            label=name,
            color=color,
        )

    ax.set(
        title=column,
        xlabel="log1p(value)",
        ylabel="Cumulative proportion",
    )
    ax.legend()

fig.suptitle("Recent Activity — Training and Test")
plt.tight_layout()
plt.show()

#### Interpretation — Recent Activity Differences

The 90-day purchase-count distributions are broadly similar, although
training has a heavier upper tail. Both datasets have a median of
one purchase.

Median 90-day spending is higher in test: 472.72 versus 198.50.
Previous-purchase intervals are similar, with supplied medians of
59 and 57 days respectively.

The interval plots exclude 16 negative training values and two negative
test values. Longer customer histories in test therefore do not imply
uniformly greater recent purchase frequency.

### 12.6. Categorical Distribution Comparison

Compare category percentages across training and test, including missing
values as a separate group. Report the three largest absolute changes
per feature, measured in percentage points.

In [ ]:
category_shift_tables = {}
largest_category_shifts = []

for column in categorical_comparison_columns:
    # Keep missingness separate from actual labels.
    train_shares = (
        train_clientes[column]
        .value_counts(dropna=True)
        .div(len(train_clientes))
        .mul(100)
    )
    test_shares = (
        test_clientes[column]
        .value_counts(dropna=True)
        .div(len(test_clientes))
        .mul(100)
    )

    comparison = pd.concat(
        [
            train_shares.rename("train_pct"),
            test_shares.rename("test_pct"),
        ],
        axis=1,
    ).fillna(0)

    comparison.index.name = "category"
    comparison = comparison.reset_index()
    comparison["is_missing"] = False

    missing_row = pd.DataFrame([{
        "category": None,
        "train_pct": train_clientes[column].isna().mean() * 100,
        "test_pct": test_clientes[column].isna().mean() * 100,
        "is_missing": True,
    }])

    comparison = pd.concat(
        [comparison, missing_row],
        ignore_index=True,
    )

    comparison["test_minus_train_pp"] = (
        comparison["test_pct"] - comparison["train_pct"]
    )
    comparison["absolute_change_pp"] = (
        comparison["test_minus_train_pp"].abs()
    )

    comparison = comparison.sort_values(
        "absolute_change_pp",
        ascending=False,
    )
    category_shift_tables[column] = comparison

    top_changes = comparison.head(3).copy()
    top_changes.insert(0, "variable", column)
    largest_category_shifts.append(top_changes)

category_shift_summary = pd.concat(
    largest_category_shifts,
    ignore_index=True,
)

# Display missing groups clearly without changing the source data.
category_shift_display = category_shift_summary.copy()
category_shift_display.loc[
    category_shift_display["is_missing"], "category"
] = "(Missing)"

display(
    category_shift_display.drop(columns="is_missing").round(3)
)

#### Interpretation — Categorical Distribution Differences

All test observations use document series 2, compared with 41.90%
of training observations. Commercial zone 4 falls from 95.81% to
70.08%, while zone 1 appears only in test and accounts for 19.17%.

Salesperson and transport-method frequencies also change substantially.
These differences may reflect changes over time and customer composition.
They do not establish changes in repeat-purchase rates because test
targets are unavailable.

### Train–Test Comparison — Summary

Training and test share the same input columns, but cover separate
periods. All test purchases occur after the latest training purchase,
and 94.52% belong to customers already represented in training.

Test has less missing purchase information, longer customer histories
and generally higher spending. Several category frequencies also change,
and some test labels are unseen in training.

These findings support chronological validation, consistent cleaning
rules and preprocessing that handles unfamiliar categories.

## 13. Leakage and Validation Strategy

### 13.1. Information Available at Prediction Time

The working prediction point is immediately after the current purchase
is completed. The model predicts whether another purchase will occur
within the following 90 days.

Current-purchase fields can be used only if available at that point.
Historical statistics must use earlier purchases and rolling windows
ending at the prediction point, without including future transactions.

Customer attributes require confirmation that their supplied values
were available at the purchase date. A successful join does not establish
historical availability.

The target and diagnostic fields derived from later purchases must never
be included as predictors. ID is retained for submission and customer_id
for joining and validation analysis; neither is a numerical quantity.

Imputation, encoding, scaling, feature selection and any modelling PCA
must be fitted within each training split. The PCA fitted during EDA
is exploratory and will not be reused for validation.

### 13.2. Candidate Chronological Validation Split

Validation begins on 1 January 2024. Training observations are eligible
only when their full 90-day target window ends before that date.

Observations between the eligible training period and validation are
excluded from this split. Rows without usable purchase dates cannot
be assigned chronologically and are reported separately.

This cell assesses the proposed split without modifying the source data
or fitting preprocessing.

In [ ]:
validation_start = pd.Timestamp("2024-01-01")
target_horizon = pd.Timedelta(days=90)

split_dates = pd.to_datetime(
    train["purchase_date"],
    format="%Y-%m-%d",
    errors="coerce",
)

target_window_end = split_dates + target_horizon

training_mask = (
    split_dates.notna()
    & target_window_end.lt(validation_start)
)

validation_mask = (
    split_dates.notna()
    & split_dates.ge(validation_start)
)

gap_mask = (
    split_dates.notna()
    & ~training_mask
    & ~validation_mask
)

undated_mask = split_dates.isna()

split_masks = {
    "Training subset": training_mask,
    "Excluded — outcome-window gap": gap_mask,
    "Validation subset": validation_mask,
    "Excluded — missing or invalid date": undated_mask,
}

split_summary_records = []

for name, mask in split_masks.items():
    subset = train.loc[mask]
    dates = split_dates.loc[mask]
    targets = subset["repeat_purchase_90d"]

    split_summary_records.append({
        "subset": name,
        "observations": len(subset),
        "unique_customers": subset["customer_id"].nunique(),
        "earliest_purchase": dates.min(),
        "latest_purchase": dates.max(),
        "target_0": int(targets.eq(0).sum()),
        "target_1": int(targets.eq(1).sum()),
        "repeat_purchase_pct": (
            targets.mean() * 100 if len(subset) else np.nan
        ),
    })

chronological_split_summary = (
    pd.DataFrame(split_summary_records)
    .set_index("subset")
)

display(chronological_split_summary)

assert sum(mask.sum() for mask in split_masks.values()) == len(train)
assert not (training_mask & validation_mask).any()

if training_mask.any():
    assert target_window_end.loc[training_mask].max() < validation_start

print(f"Validation begins: {validation_start:%Y-%m-%d}")
print(
    "Latest eligible training target-window end:",
    target_window_end.loc[training_mask].max(),
)

In [ ]:
subset_training_customers = set(
    train.loc[training_mask, "customer_id"].dropna()
)

validation_customer_data = train.loc[
    validation_mask,
    ["customer_id", "repeat_purchase_90d"],
].copy()

validation_customer_data["customer_status"] = np.where(
    validation_customer_data["customer_id"].isin(
        subset_training_customers
    ),
    "Seen in training subset",
    "Unseen in training subset",
)

validation_customer_summary = (
    validation_customer_data
    .groupby("customer_status")
    .agg(
        observations=("repeat_purchase_90d", "size"),
        unique_customers=("customer_id", "nunique"),
        repeat_purchase_pct=(
            "repeat_purchase_90d",
            lambda values: values.mean() * 100,
        ),
    )
)

validation_customer_summary["validation_observations_pct"] = (
    validation_customer_summary["observations"]
    / len(validation_customer_data)
    * 100
)

display(validation_customer_summary.round(3))